## RPA diagnóstico contratos SIPSE vs SECOP

In [67]:
import pandas as pd
import numpy as np
import os
import re
import unicodedata

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
KEYWORDS = {
    'PLAZO_SECOP': ['DURA', 'CONTRATO'],
    'FECHA_INI_SECOP': ['FECHA', 'INICIO'],
    'KEY_SECOP': ['REFERENCIA', 'CONTRATO']
}

COLS_FIJAS = {
    'KEY_SIPSE': 'NUMERO_CONTRATO', 
    'CEDULA_SIPSE': 'DOC_CONTRATISTA',  
    'CEDULA_SECOP': 'Documento Proveedor',   
    'FECHA_INI_SIPSE': 'FECHA_INICIO',        
    'PLAZO_SIPSE': 'PLAZO',
    'SOLICITUD_SIPSE': 'SOLICITUD' # <--- NUEVA COLUMNA A BUSCAR
}

# ==========================================
# 2. FUNCIONES DE LIMPIEZA
# ==========================================

def eliminar_tildes(texto):
    if not isinstance(texto, str): return str(texto)
    try:
        texto = texto.encode('cp1252').decode('utf-8')
    except:
        pass
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

def encontrar_columna_por_keywords(df, keywords_lista):
    """Busca columna flexible (ej. DURA + CONTRATO)."""
    for col_real in df.columns:
        col_norm = str(col_real).upper()
        col_norm = eliminar_tildes(col_norm)
        coincide = True
        for palabra in keywords_lista:
            if palabra not in col_norm:
                coincide = False
                break
        if coincide:
            print(f"   ✅ Columna encontrada: '{col_real}' (keywords: {keywords_lista})")
            return col_real
    return None

def limpiar_cedula(serie):
    s = serie.astype(str).str.strip().str.upper()
    s = s.str.replace(r'\.0$', '', regex=True)
    s = s.str.replace(r'[.,\-\s]|NIT|CC', '', regex=True)
    return s.replace('NAN', 'SIN_DATO')

def limpiar_numero_contrato(serie):
    # Extrae el primer número (ej. 440) para cruzar
    s = serie.astype(str).str.extract(r'(\d+)', expand=False).fillna('0')
    def safe_int(x):
        try: return str(int(x))
        except: return str(x)
    return s.apply(safe_int).replace('0', 'SIN_DATO')

def extraer_solicitud_secop(serie):
    """
    Busca el número entre paréntesis en SECOP.
    Ej: "440... (153540)" -> "153540"
    """
    s = serie.astype(str)
    # Regex: Busca paréntesis abierto \(, captura digitos \d+, paréntesis cerrado \)
    extracted = s.str.extract(r'\((\d+)\)', expand=False)
    
    # Limpieza estándar (quitar ceros izq si aplica, espacios)
    extracted = extracted.fillna('SIN_SOLICITUD').astype(str).str.strip()
    return extracted

def procesar_fecha_segura(serie):
    fechas = pd.to_datetime(serie, errors='coerce')
    return fechas.dt.strftime('%Y-%m-%d').fillna('SIN_FECHA')

def limpiar_texto_plazo(serie):
    s = serie.astype(str).str.strip().str.upper()
    s = s.apply(eliminar_tildes)
    s = s.str.replace(r'[().,-]', '', regex=True)
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()
    return s.replace('NAN', 'SIN_DATO')

def obtener_nombre_real_columna(df, nombre_original, sufijo_preferido):
    if nombre_original in df.columns: return nombre_original
    nombre = f"{nombre_original}{sufijo_preferido}"
    if nombre in df.columns: return nombre
    return None

# ==========================================
# 3. PROCESO PRINCIPAL
# ==========================================
def generar_reporte_auditoria(ruta_archivo):
    # 1. GENERACIÓN DE NOMBRE DINÁMICO
    nombre_archivo_base = os.path.basename(ruta_archivo) # "Usaquen.xlsx"
    nombre_sin_ext = os.path.splitext(nombre_archivo_base)[0] # "Usaquen"
    ruta_salida = os.path.join(os.path.dirname(ruta_archivo), f"Diagnóstico\\Resultado_{nombre_sin_ext}.xlsx")

    print(f"\n--- Procesando: {nombre_sin_ext} ---")

    try:
        xls = pd.ExcelFile(ruta_archivo)
        df_sipse = pd.read_excel(xls, 'SIPSE')
        df_secop = pd.read_excel(xls, 'SECOP II')
    except Exception as e:
        return f"Error leyendo Excel: {e}"

    print("🔎 Auto-detectando columnas...")
    COLS_FINAL = COLS_FIJAS.copy()
    col_ref = encontrar_columna_por_keywords(df_secop, KEYWORDS['KEY_SECOP'])
    col_fec = encontrar_columna_por_keywords(df_secop, KEYWORDS['FECHA_INI_SECOP'])
    col_plazo = encontrar_columna_por_keywords(df_secop, KEYWORDS['PLAZO_SECOP'])
    
    if not col_ref or not col_fec or not col_plazo:
        return "❌ Error: No se detectaron columnas en SECOP."

    COLS_FINAL['KEY_SECOP'] = col_ref
    COLS_FINAL['FECHA_INI_SECOP'] = col_fec
    COLS_FINAL['PLAZO_SECOP'] = col_plazo

    print("🧹 Depurando Datos (Incluyendo Solicitud)...")
    
    # Llaves Contrato
    df_sipse['KEY_JOIN'] = limpiar_numero_contrato(df_sipse[COLS_FINAL['KEY_SIPSE']])
    df_secop['KEY_JOIN'] = limpiar_numero_contrato(df_secop[COLS_FINAL['KEY_SECOP']])
    
    # Cédulas
    df_sipse['CEDULA_CLEAN'] = limpiar_cedula(df_sipse[COLS_FINAL['CEDULA_SIPSE']])
    df_secop['CEDULA_CLEAN'] = limpiar_cedula(df_secop[COLS_FINAL['CEDULA_SECOP']])

    # --- NUEVO: PREPARAR DATOS PARA METRICA 1 ESTRICTA ---
    # 1. Solicitud (Sipse: Columna Solicitud, Secop: Paréntesis en Referencia)
    # Validamos si existe columna solicitud en Sipse
    if COLS_FINAL['SOLICITUD_SIPSE'] in df_sipse.columns:
        df_sipse['SOL_CLEAN'] = df_sipse[COLS_FINAL['SOLICITUD_SIPSE']].astype(str).str.strip()
    else:
        df_sipse['SOL_CLEAN'] = "NO_COL_SOLICITUD"
        print("⚠️ Advertencia: No existe columna 'SOLICITUD' en SIPSE.")

    df_secop['SOL_CLEAN'] = extraer_solicitud_secop(df_secop[COLS_FINAL['KEY_SECOP']])

    # 2. Fechas (Limpieza previa para comparar mas rapido)
    df_sipse['FECHA_CLEAN'] = procesar_fecha_segura(df_sipse[COLS_FINAL['FECHA_INI_SIPSE']])
    df_secop['FECHA_CLEAN'] = procesar_fecha_segura(df_secop[COLS_FINAL['FECHA_INI_SECOP']])

    # 3. Plazos (Limpieza previa)
    df_sipse['PLAZO_CLEAN'] = limpiar_texto_plazo(df_sipse[COLS_FINAL['PLAZO_SIPSE']])
    df_secop['PLAZO_CLEAN'] = limpiar_texto_plazo(df_secop[COLS_FINAL['PLAZO_SECOP']])

    # Duplicados
    df_sipse = df_sipse.drop_duplicates(subset=['KEY_JOIN'])
    df_secop = df_secop.drop_duplicates(subset=['KEY_JOIN'])

    print("🔄 Cruzando Bases de Datos...")
    df_master = pd.merge(
        df_sipse, df_secop, on='KEY_JOIN', how='outer', indicator=True, suffixes=('_SIPSE', '_SECOP')
    )

    # Visualización
    col_sipse = obtener_nombre_real_columna(df_master, COLS_FINAL['KEY_SIPSE'], '_SIPSE')
    col_secop = obtener_nombre_real_columna(df_master, COLS_FINAL['KEY_SECOP'], '_SECOP')
    df_master['CONTRATO_VISIBLE'] = df_master[col_sipse].fillna(df_master[col_secop])
    
    cols = ['CONTRATO_VISIBLE', 'KEY_JOIN', 'CEDULA_CLEAN_SIPSE', 'CEDULA_CLEAN_SECOP']
    rest_cols = [c for c in df_master.columns if c not in cols]
    df_master = df_master[cols + rest_cols]

    resultados = {}

    # Base
    base_ambos = df_master[df_master['_merge'] == 'both'].copy()
    resultados['2. Solo en SECOP'] = df_master[df_master['_merge'] == 'right_only'].copy()
    resultados['3. Solo en SIPSE'] = df_master[df_master['_merge'] == 'left_only'].copy()

    # --- EVALUACIÓN DE DISCREPANCIAS (Para definir M1 estricta) ---
    print("⚖️ Evaluando Coincidencia Estricta (Contrato+Cedula+Fecha+Plazo+Solicitud)...")

    # 1. Comparación Cédula
    mask_cedula_ok = base_ambos['CEDULA_CLEAN_SIPSE'] == base_ambos['CEDULA_CLEAN_SECOP']
    
    # 2. Comparación Fecha
    mask_fecha_ok = base_ambos['FECHA_CLEAN_SIPSE'] == base_ambos['FECHA_CLEAN_SECOP']
    
    # 3. Comparación Plazo
    mask_plazo_ok = base_ambos['PLAZO_CLEAN_SIPSE'] == base_ambos['PLAZO_CLEAN_SECOP']

    # 4. Comparación Solicitud (Nuevo)
    # Ojo: Comparamos strings. "153540" == "153540"
    mask_solicitud_ok = base_ambos['SOL_CLEAN_SIPSE'] == base_ambos['SOL_CLEAN_SECOP']

    # --- DEFINICIÓN DE MÉTRICAS ---

    # M5: Cédula Diferente
    resultados['5. Cedula Diferente'] = base_ambos[~mask_cedula_ok].copy()
    
    # M6: Fecha Diferente
    resultados['6. Fecha Ini Diferente'] = base_ambos[~mask_fecha_ok].copy()

    # M7: Plazo Diferente
    resultados['7. Plazo Diferente'] = base_ambos[~mask_plazo_ok].copy()
    
    # Extra: Solicitud Diferente (Para que sepas cuáles fallaron por esto)
    # Filtramos: Que las cedulas coincidan (si no es otro problema) pero la solicitud falle
    mask_sol_fail = (~mask_solicitud_ok) & (mask_cedula_ok)
    df_sol_fail = base_ambos[mask_sol_fail].copy()
    df_sol_fail['DEBUG_SOL_SIPSE'] = base_ambos.loc[mask_sol_fail, 'SOL_CLEAN_SIPSE']
    df_sol_fail['DEBUG_SOL_SECOP_EXTRAIDA'] = base_ambos.loc[mask_sol_fail, 'SOL_CLEAN_SECOP']
    resultados['8. Solicitud Diferente'] = df_sol_fail

    # --- M1: COINCIDENCIA TOTAL (ESTRICTA) ---
    # Deben cumplirse TODAS las condiciones a la vez
    mask_total_ok = mask_cedula_ok & mask_fecha_ok & mask_plazo_ok & mask_solicitud_ok
    resultados['1. Coincidencia TOTAL'] = base_ambos[mask_total_ok].copy()

    # M4: Contrato diferente (por Cédula)
    print("🔍 Analizando Métrica 4...")
    sub_sipse = df_sipse[['CEDULA_CLEAN', 'KEY_JOIN', COLS_FINAL['KEY_SIPSE'], COLS_FINAL['CEDULA_SIPSE']]].rename(columns={'KEY_JOIN': 'K_S'})
    sub_secop = df_secop[['CEDULA_CLEAN', 'KEY_JOIN', COLS_FINAL['KEY_SECOP'], COLS_FINAL['CEDULA_SECOP']]].rename(columns={'KEY_JOIN': 'K_2'})
    df_ced = pd.merge(sub_sipse, sub_secop, on='CEDULA_CLEAN', how='inner')
    resultados['4. Num Contrato Diferente'] = df_ced[df_ced['K_S'] != df_ced['K_2']].copy()

    # EXPORTAR
    try:
        with pd.ExcelWriter(ruta_salida, engine='openpyxl') as writer:
            resumen = []
            for k in sorted(resultados.keys()):
                df_res = resultados[k]
                cant = len(df_res)
                resumen.append({'Indicador': k, 'Cantidad': cant})
                name = k[:30].replace(':', '').replace('.', '')
                df_res.to_excel(writer, sheet_name=name, index=False, startrow=2)
                writer.sheets[name].cell(row=1, column=1, value=f"{k}: {cant}")
            pd.DataFrame(resumen).to_excel(writer, sheet_name='RESUMEN', index=False)
            
        print(f"\n✅ ¡Éxito! Archivo generado: {ruta_salida}")

    except Exception as e:
        print(f"❌ Error guardando: {e}")

# --- EJECUCIÓN ---
from pathlib import Path
# Encuentra todos los archivos .txt en el directorio actual
archivos_excel = [f.name for f in Path('C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO').glob("*.xlsx")]
for archivo in archivos_excel:
    ruta = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\' + archivo
    print("Analizando ruta: ", ruta)
    generar_reporte_auditoria(ruta)
#ruta = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Antonio Nariño.xlsx'
#generar_reporte_auditoria(ruta)

Analizando ruta:  C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\Antonio Nariño.xlsx

--- Procesando: Antonio Nariño ---
🔎 Auto-detectando columnas...
   ✅ Columna encontrada: 'Referencia del Contrato' (keywords: ['REFERENCIA', 'CONTRATO'])
   ✅ Columna encontrada: 'Fecha de Inicio del Contrato' (keywords: ['FECHA', 'INICIO'])
   ✅ Columna encontrada: 'DuraciÃ³n del contrato' (keywords: ['DURA', 'CONTRATO'])
🧹 Depurando Datos (Incluyendo Solicitud)...
🔄 Cruzando Bases de Datos...
⚖️ Evaluando Coincidencia Estricta (Contrato+Cedula+Fecha+Plazo+Solicitud)...
🔍 Analizando Métrica 4...


C:\Users\Pc\AppData\Local\Temp\ipykernel_10800\3521084550.py:80: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas = pd.to_datetime(serie, errors='coerce')



✅ ¡Éxito! Archivo generado: C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\Diagnóstico\Resultado_Antonio Nariño.xlsx
Analizando ruta:  C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\Barrios Unidos.xlsx

--- Procesando: Barrios Unidos ---
🔎 Auto-detectando columnas...
   ✅ Columna encontrada: 'Referencia del Contrato' (keywords: ['REFERENCIA', 'CONTRATO'])
   ✅ Columna encontrada: 'Fecha de Inicio del Contrato' (keywords: ['FECHA', 'INICIO'])
   ✅ Columna encontrada: 'DuraciÃ³n del contrato' (keywords: ['DURA', 'CONTRATO'])
🧹 Depurando Datos (Incluyendo Solicitud)...
🔄 Cruzando Bases de Datos...
⚖️ Evaluando Coincidencia Estricta (Contrato+Cedula+Fecha+Plazo+Solicitud)...
🔍 Analizando Métrica 4...


C:\Users\Pc\AppData\Local\Temp\ipykernel_10800\3521084550.py:80: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas = pd.to_datetime(serie, errors='coerce')
C:\Users\Pc\AppData\Local\Temp\ipykernel_10800\3521084550.py:80: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas = pd.to_datetime(serie, errors='coerce')



✅ ¡Éxito! Archivo generado: C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\Diagnóstico\Resultado_Barrios Unidos.xlsx
Analizando ruta:  C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\Bosa.xlsx

--- Procesando: Bosa ---
🔎 Auto-detectando columnas...
   ✅ Columna encontrada: 'Referencia del Contrato' (keywords: ['REFERENCIA', 'CONTRATO'])
   ✅ Columna encontrada: 'Fecha de Inicio del Contrato' (keywords: ['FECHA', 'INICIO'])
   ✅ Columna encontrada: 'DuraciÃ³n del contrato' (keywords: ['DURA', 'CONTRATO'])
🧹 Depurando Datos (Incluyendo Solicitud)...
🔄 Cruzando Bases de Datos...
⚖️ Evaluando Coincidencia Estricta (Contrato+Cedula+Fecha+Plazo+Solicitud)...
🔍 Analizando Métrica 4...


C:\Users\Pc\AppData\Local\Temp\ipykernel_10800\3521084550.py:80: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas = pd.to_datetime(serie, errors='coerce')
C:\Users\Pc\AppData\Local\Temp\ipykernel_10800\3521084550.py:80: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas = pd.to_datetime(serie, errors='coerce')


... (21 outputs left out)

C:\Users\Pc\AppData\Local\Temp\ipykernel_10800\3521084550.py:80: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas = pd.to_datetime(serie, errors='coerce')



✅ ¡Éxito! Archivo generado: C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\Diagnóstico\Resultado_Usaquen.xlsx
Analizando ruta:  C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\Usme.xlsx

--- Procesando: Usme ---
🔎 Auto-detectando columnas...
   ✅ Columna encontrada: 'Referencia del Contrato' (keywords: ['REFERENCIA', 'CONTRATO'])
   ✅ Columna encontrada: 'Fecha de Inicio del Contrato' (keywords: ['FECHA', 'INICIO'])
   ✅ Columna encontrada: 'DuraciÃ³n del contrato' (keywords: ['DURA', 'CONTRATO'])
🧹 Depurando Datos (Incluyendo Solicitud)...
🔄 Cruzando Bases de Datos...
⚖️ Evaluando Coincidencia Estricta (Contrato+Cedula+Fecha+Plazo+Solicitud)...
🔍 Analizando Métrica 4...


C:\Users\Pc\AppData\Local\Temp\ipykernel_10800\3521084550.py:80: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas = pd.to_datetime(serie, errors='coerce')



✅ ¡Éxito! Archivo generado: C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\Diagnóstico\Resultado_Usme.xlsx


### Pruebas

In [1]:
import pandas as pd
import numpy as np
import os
import re
import unicodedata

# ==========================================
# 1. CONFIGURACIÓN (MAYÚSCULAS Y SIN TILDES)
# ==========================================
# Revisa que estos nombres coincidan con los de tu archivo (ignorando mayúsculas/tildes)
COLS = {
    'KEY_SIPSE': 'NUMERO_CONTRATO', 
    'KEY_SECOP': 'REFERENCIA DEL CONTRATO', # ¿Seguro se llama así en esta nueva entidad?

    'CEDULA_SIPSE': 'DOC_CONTRATISTA',  
    'CEDULA_SECOP': 'DOCUMENTO PROVEEDOR',   

    'FECHA_INI_SIPSE': 'FECHA_INICIO',        
    'FECHA_INI_SECOP': 'FECHA DE INICIO DEL CONTRATO',     

    'PLAZO_SIPSE': 'PLAZO',                  
    'PLAZO_SECOP': 'DURACION DEL CONTRATO'
}

# ==========================================
# 2. FUNCIONES
# ==========================================
def eliminar_tildes(texto):
    if not isinstance(texto, str): return str(texto)
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

def normalizar_encabezados(df):
    """Convierte encabezados a MAYÚSCULAS y SIN TILDES automáticamente."""
    nuevas = []
    for col in df.columns:
        col_limpia = str(col).strip().upper()
        col_limpia = eliminar_tildes(col_limpia)
        nuevas.append(col_limpia)
    df.columns = nuevas
    return df

def limpiar_cedula(serie):
    s = serie.astype(str).str.strip().str.upper()
    s = s.str.replace(r'[.,\-\s]|NIT|CC', '', regex=True)
    s = s.str.replace(r'\.0$', '', regex=True)
    return s.replace('NAN', 'SIN_DATO')

def limpiar_numero_contrato(serie):
    """
    Logica robusta:
    1. Busca digitos.
    2. Convierte a entero (quita ceros izq).
    3. Maneja errores.
    """
    # Regex: busca primer grupo de digitos
    s = serie.astype(str).str.extract(r'(\d+)', expand=False).fillna('0')
    
    def safe_int(x):
        try: return str(int(x))
        except: return str(x)
        
    return s.apply(safe_int).replace('0', 'SIN_DATO')

def procesar_fecha_segura(serie):
    fechas = pd.to_datetime(serie, errors='coerce')
    return fechas.dt.strftime('%Y-%m-%d').fillna('SIN_FECHA')

def limpiar_texto_plazo(serie):
    s = serie.astype(str).str.strip().str.upper()
    s = s.apply(eliminar_tildes)
    s = s.str.replace(r'[().,-]', '', regex=True)
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()
    return s.replace('NAN', 'SIN_DATO')

def obtener_nombre_real_columna(df, nombre_original, sufijo_preferido):
    if nombre_original in df.columns: return nombre_original
    nombre = f"{nombre_original}{sufijo_preferido}"
    if nombre in df.columns: return nombre
    return None

# ==========================================
# 3. PROCESO
# ==========================================
def generar_reporte_auditoria(ruta_archivo):
    print(f"\n--- Procesando: {ruta_archivo} ---")

    try:
        xls = pd.ExcelFile(ruta_archivo)
        df_sipse = pd.read_excel(xls, 'SIPSE')
        df_secop = pd.read_excel(xls, 'SECOP II')
        
        # 1. Normalizar Encabezados
        df_sipse = normalizar_encabezados(df_sipse)
        df_secop = normalizar_encabezados(df_secop)
        
    except Exception as e:
        return f"Error leyendo Excel: {e}"

    # 2. VALIDACIÓN VISUAL (DEBUG) - ¡MIRA LA CONSOLA!
    print("\n🔍 --- MODO DETECTIVE (Verificación de Columnas) ---")
    if COLS['KEY_SECOP'] not in df_secop.columns:
        print(f"❌ ERROR CRÍTICO: No encuentro la columna '{COLS['KEY_SECOP']}' en SECOP.")
        print(f"   Columnas disponibles en SECOP: {list(df_secop.columns)}")
        return "Detenido por falta de columna."
    else:
        print(f"✅ Columna '{COLS['KEY_SECOP']}' encontrada en SECOP.")

    print("\n🧹 Depurando llaves de contrato...")
    df_sipse['KEY_JOIN'] = limpiar_numero_contrato(df_sipse[COLS['KEY_SIPSE']])
    df_secop['KEY_JOIN'] = limpiar_numero_contrato(df_secop[COLS['KEY_SECOP']])
    
    # 3. IMPRIMIR EJEMPLOS PARA QUE VEAS QUE PASA
    print("\n👀 --- MUESTRA DE DATOS LIMPIOS (Primeros 5 registros) ---")
    print(f"SIPSE Original: {df_sipse[COLS['KEY_SIPSE']].head(5).tolist()}")
    print(f"SIPSE Limpio:   {df_sipse['KEY_JOIN'].head(5).tolist()}")
    print("-" * 40)
    print(f"SECOP Original: {df_secop[COLS['KEY_SECOP']].head(5).tolist()}")
    print(f"SECOP Limpio:   {df_secop['KEY_JOIN'].head(5).tolist()}")
    print("----------------------------------------------------------\n")

    # Limpieza de Cédulas
    df_sipse['CEDULA_CLEAN'] = limpiar_cedula(df_sipse[COLS['CEDULA_SIPSE']])
    df_secop['CEDULA_CLEAN'] = limpiar_cedula(df_secop[COLS['CEDULA_SECOP']])

    # Eliminar duplicados de llave
    df_sipse = df_sipse.drop_duplicates(subset=['KEY_JOIN'])
    df_secop = df_secop.drop_duplicates(subset=['KEY_JOIN'])

    # CRUCE
    print("🔄 Cruzando Bases de Datos...")
    df_master = pd.merge(
        df_sipse, df_secop, on='KEY_JOIN', how='outer', indicator=True, suffixes=('_SIPSE', '_SECOP')
    )

    # Visualización
    col_sipse = obtener_nombre_real_columna(df_master, COLS['KEY_SIPSE'], '_SIPSE')
    col_secop = obtener_nombre_real_columna(df_master, COLS['KEY_SECOP'], '_SECOP')
    df_master['CONTRATO_VISIBLE'] = df_master[col_sipse].fillna(df_master[col_secop])
    
    cols = ['CONTRATO_VISIBLE', 'KEY_JOIN', 'CEDULA_CLEAN_SIPSE', 'CEDULA_CLEAN_SECOP']
    rest_cols = [c for c in df_master.columns if c not in cols]
    df_master = df_master[cols + rest_cols]

    resultados = {}

    # Clasificación
    base_ambos = df_master[df_master['_merge'] == 'both'].copy()
    resultados['2. Solo en SECOP'] = df_master[df_master['_merge'] == 'right_only'].copy()
    resultados['3. Solo en SIPSE'] = df_master[df_master['_merge'] == 'left_only'].copy()

    # M1 vs M5
    mask_cedula_dif = base_ambos['CEDULA_CLEAN_SIPSE'] != base_ambos['CEDULA_CLEAN_SECOP']
    resultados['5. Cedula Diferente'] = base_ambos[mask_cedula_dif].copy()
    resultados['1. Coincidencia (Contrato y Cedula)'] = base_ambos[~mask_cedula_dif].copy()

    # M4
    sub_sipse = df_sipse[['CEDULA_CLEAN', 'KEY_JOIN', COLS['KEY_SIPSE'], COLS['CEDULA_SIPSE']]].rename(columns={'KEY_JOIN': 'K_S'})
    sub_secop = df_secop[['CEDULA_CLEAN', 'KEY_JOIN', COLS['KEY_SECOP'], COLS['CEDULA_SECOP']]].rename(columns={'KEY_JOIN': 'K_2'})
    df_ced = pd.merge(sub_sipse, sub_secop, on='CEDULA_CLEAN', how='inner')
    resultados['4. Num Contrato Diferente'] = df_ced[df_ced['K_S'] != df_ced['K_2']].copy()

    # M6 (Fechas)
    col_f_sip = obtener_nombre_real_columna(base_ambos, COLS['FECHA_INI_SIPSE'], '_SIPSE')
    col_f_sec = obtener_nombre_real_columna(base_ambos, COLS['FECHA_INI_SECOP'], '_SECOP')
    
    if col_f_sip and col_f_sec:
        fecha_a = procesar_fecha_segura(base_ambos[col_f_sip])
        fecha_b = procesar_fecha_segura(base_ambos[col_f_sec])
        mask_f = fecha_a != fecha_b
        df_f = base_ambos[mask_f].copy()
        df_f['DEBUG_FECHA_SIPSE'] = fecha_a[mask_f]
        df_f['DEBUG_FECHA_SECOP'] = fecha_b[mask_f]
        resultados['6. Fecha Ini Diferente'] = df_f
    else:
        resultados['6. Fecha Ini Diferente'] = pd.DataFrame()

    # M7 (Plazo)
    col_p_sip = obtener_nombre_real_columna(base_ambos, COLS['PLAZO_SIPSE'], '_SIPSE')
    col_p_sec = obtener_nombre_real_columna(base_ambos, COLS['PLAZO_SECOP'], '_SECOP')
    
    if col_p_sip and col_p_sec:
        val_a = limpiar_texto_plazo(base_ambos[col_p_sip])
        val_b = limpiar_texto_plazo(base_ambos[col_p_sec])
        mask_p = val_a != val_b
        df_p = base_ambos[mask_p].copy()
        df_p['DEBUG_PLAZO_SIPSE'] = val_a[mask_p]
        df_p['DEBUG_PLAZO_SECOP'] = val_b[mask_p]
        resultados['7. Plazo Diferente'] = df_p
    else:
        resultados['7. Plazo Diferente'] = pd.DataFrame()

    # EXPORTAR
    nombre_salida = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Resultado_Auditoria_V5_Debug.xlsx'
    try:
        with pd.ExcelWriter(nombre_salida, engine='openpyxl') as writer:
            resumen = []
            for k in sorted(resultados.keys()):
                df_res = resultados[k]
                cant = len(df_res)
                resumen.append({'Indicador': k, 'Cantidad': cant})
                name = k[:30].replace(':', '').replace('.', '')
                df_res.to_excel(writer, sheet_name=name, index=False, startrow=2)
                writer.sheets[name].cell(row=1, column=1, value=f"{k}: {cant}")
            pd.DataFrame(resumen).to_excel(writer, sheet_name='RESUMEN', index=False)
            
        print(f"\n✅ ¡Éxito! Archivo generado: {nombre_salida}")

    except Exception as e:
        print(f"❌ Error guardando: {e}")

# --- EJECUCIÓN ---
ruta = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Engativa.xlsx'
generar_reporte_auditoria(ruta)


--- Procesando: C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\Engativa.xlsx ---

🔍 --- MODO DETECTIVE (Verificación de Columnas) ---
✅ Columna 'REFERENCIA DEL CONTRATO' encontrada en SECOP.

🧹 Depurando llaves de contrato...

👀 --- MUESTRA DE DATOS LIMPIOS (Primeros 5 registros) ---
SIPSE Original: [653, 652, 651, 650, 649]
SIPSE Limpio:   ['653', '652', '651', '650', '649']
----------------------------------------
SECOP Original: ['119 DE 2026', '045 DE 2026', '393 DE 2026', '577 DE 2026', '617 DE 2026']
SECOP Limpio:   ['119', '45', '393', '577', '617']
----------------------------------------------------------

🔄 Cruzando Bases de Datos...


C:\Users\Pc\AppData\Local\Temp\ipykernel_18408\1155374753.py:65: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas = pd.to_datetime(serie, errors='coerce')
C:\Users\Pc\AppData\Local\Temp\ipykernel_18408\1155374753.py:65: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas = pd.to_datetime(serie, errors='coerce')



✅ ¡Éxito! Archivo generado: C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\Resultado_Auditoria_V5_Debug.xlsx


In [49]:
import re

def prueba_manual(texto):
    # Busca el primer grupo de números
    match = re.search(r'(\d+)', str(texto)) 
    if match:
        numero = int(match.group(1)) # Convierte a entero (quita ceros izq)
        return str(numero)           # Devuelve como texto
    return "SIN_DATO"

print(prueba_manual("440-2026-2026-CPS-P (153540)")) 
# Resultado debería ser: '440'

440


In [60]:
import glob

# Encuentra todos los archivos .txt en el directorio actual
archivos_txt = glob.glob('C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\*.xlsx')
print(archivos_txt)

['C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Antonio Nariño.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Barrios Unidos.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Bosa.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Candelaria.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Chapinero.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Ciudad Bolivar.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Engativa.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Fontibon.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Kennedy.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Martires.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Puente Aranda.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Rafael Uribe Uribe.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\San Cristobal.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\Santa Fe.xlsx', 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 0

In [62]:
from pathlib import Path

# Buscamos archivos .txt en la carpeta actual
archivos = [f.name for f in Path('C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO').glob("*.xlsx")]

print(archivos)

['Antonio Nariño.xlsx', 'Barrios Unidos.xlsx', 'Bosa.xlsx', 'Candelaria.xlsx', 'Chapinero.xlsx', 'Ciudad Bolivar.xlsx', 'Engativa.xlsx', 'Fontibon.xlsx', 'Kennedy.xlsx', 'Martires.xlsx', 'Puente Aranda.xlsx', 'Rafael Uribe Uribe.xlsx', 'San Cristobal.xlsx', 'Santa Fe.xlsx', 'Suba.xlsx', 'Sumapaz.xlsx', 'Teusaquillo.xlsx', 'Tunjuelito.xlsx', 'Usaquen.xlsx', 'Usme.xlsx', '~$Engativa.xlsx', '~$Fontibon.xlsx', '~$Usaquen.xlsx']


In [42]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

### Entregas

#### Entrega 18/02/2026

In [ ]:
import os
import shutil
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.formatting.rule import CellIsRule
from collections import defaultdict
import re

# --- CONFIGURACIÓN DE CARPETAS ---
CARPETA_ORIGEN = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO'
CARPETA_DESTINO = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\DiagnóstivoV2_1'

# --- RANGOS A OCULTAR (CORREGIDOS PARA USAR HIDDEN=TRUE) ---
# Nota: He ajustado 'A-AH' a 'Z-AH' en SECOP para NO ocultar las columnas L, M, T, X, Y que necesitas ver.
RANGOS_OCULTAR_SECOP = ['A-K', 'N-S', 'U-W', 'Z-AH', 'AJ-BR', 'BT-CI']
RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-H', 'K', 'N', 'O', 'Q', 'S-BK', 'BM-BW', 'BY-CE', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']

# --- DEFINICIÓN DE FUNCIONES ---

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def ocultar_columnas_por_lista(ws, lista_rangos):
    """
    Recibe una lista de cadenas como ['A-K', 'M'] y establece la propiedad HIDDEN = True.
    """
    for item in lista_rangos:
        item = item.upper().strip()
        
        # Caso 1: Es un rango (ej: "A-K")
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio)
            idx_fin = column_index_from_string(fin)
            
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                # Esta es la forma correcta de ocultar en Excel
                ws.column_dimensions[letra].hidden = True 
                
        # Caso 2: Es una columna única (ej: "A")
        else:
            ws.column_dimensions[item].hidden = True

def aplicar_estilo_ferrocarril(ws):
    """
    Encabezados Azul Oscuro + Letra Blanca.
    Filas con patrón intercalado.
    """
    # 1. Encabezados
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    
    for cell in ws[1]:
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas (Patrón Ferrocarril)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    max_col = ws.max_column
    
    for i_fila, row in enumerate(ws.iter_rows(min_row=2, max_col=max_col), start=0):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF":
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for cell in row:
                cell.fill = relleno

def dejar_columna_blanca(ws, indice_columna):
    relleno_blanco = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    
    # Recorremos todas las celdas de la columna indicada
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=indice_columna, max_col=indice_columna):
        for cell in row:
            cell.fill = relleno_blanco

def aplicar_formato_condicional_alertas(ws, col_inicio_eval):
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    max_row = ws.max_row
    
    reglas = [
        (0, 'equal', '"Numero contrato NO coincide"'),
        (2, 'equal', '"Cedula NO coincide"'),
        (3, 'greaterThan', '1'),
        (4, 'greaterThan', '1'),
        (5, 'equal', '"No coinciden fechas"'),
        (6, 'equal', '"No coinciden fechas"'),
        (7, 'equal', '"Duracion NO coincide"'),
        (8, 'equal', '"Valor NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        rango = f"{letra_col}2:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

def normalizar_numero_contrato(valor):
    if valor is None: return 0
    texto = str(valor).strip()
    # Busca el primer grupo de números en la cadena
    match = re.search(r'\d+', texto)
    if match:
        return int(match.group())
    return 0

# --- LÓGICA PRINCIPAL ---
def procesar_archivos():
    if not os.path.exists(CARPETA_DESTINO):
        os.makedirs(CARPETA_DESTINO)

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith('.xlsx')]
    print(f"Se encontraron {len(archivos)} archivos de Excel.")

    for archivo in archivos:
        ruta_orig = os.path.join(CARPETA_ORIGEN, archivo)
        ruta_dest = os.path.join(CARPETA_DESTINO, archivo)
        
        print(f"Procesando: {archivo}...")
        shutil.copy2(ruta_orig, ruta_dest)

        try:
            wb = openpyxl.load_workbook(ruta_dest)
        except Exception as e:
            print(f"  Error al abrir el archivo: {e}")
            continue

        nombres_hojas = [h.lower() for h in wb.sheetnames]
        if 'sipse' not in nombres_hojas or 'secop ii' not in nombres_hojas:
            print(f"  -> FALTA INFORMACIÓN: Saltando archivo...")
            wb.close()
            continue 

        # ==========================================
        # PARTE 1: Procesar SECOP II (Crear aux1)
        # ==========================================
        nombre_real_secop = next(n for n in wb.sheetnames if n.lower() == 'secop ii')
        ws_origen = wb[nombre_real_secop]
        
        if "SECOP II aux1" in wb.sheetnames:
            del wb["SECOP II aux1"]
            
        ws_secop_aux = wb.copy_worksheet(ws_origen)
        ws_secop_aux.title = "SECOP II aux1"

        filas = []
        for row in ws_secop_aux.iter_rows(min_row=2, values_only=True):
            filas.append(list(row))
        
        filas_ordenadas = sorted(filas, key=lambda x: normalizar_numero_contrato(x[11] if len(x) > 11 else 0))

        for row in ws_secop_aux.iter_rows(min_row=2):
            for cell in row:
                cell.value = None

        for i_fila, datos_fila in enumerate(filas_ordenadas, start=2):
            for i_col, valor in enumerate(datos_fila, start=1):
                ws_secop_aux.cell(row=i_fila, column=i_col, value=valor)

        max_col_secop = ws_secop_aux.max_column
        ws_secop_aux.cell(row=1, column=max_col_secop + 1, value="Columna1")
        col_L = get_column_letter(12)
        
        for i in range(2, ws_secop_aux.max_row + 1):
            #formula = f'''=MID({col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), FIND("-", {col_L}{i}) - MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0))'''
            formula = (
                f'=IFERROR(MID({col_L}{i}, '
                f'MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), '
                f'MIN(IFERROR(FIND(" ", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999), '
                f'IFERROR(FIND("-", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999)) '
                f'- MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), "")'
            )
            ws_secop_aux.cell(row=i, column=max_col_secop + 1, value=formula)

        # ESTILOS SECOP
        print("  -> Aplicando estilos y ocultando columnas en SECOP II aux1...")
        aplicar_estilo_ferrocarril(ws_secop_aux)
        ocultar_columnas_por_lista(ws_secop_aux, RANGOS_OCULTAR_SECOP)

        # ==========================================
        # PARTE 2 y 3: ALINEACIÓN EN 'ANALISIS'
        # ==========================================
        print("  -> Generando hoja 'Analisis'...")

        nombre_real_sipse = next(n for n in wb.sheetnames if n.lower() == 'sipse')
        ws_sipse = wb[nombre_real_sipse]
        
        if 'Analisis' in wb.sheetnames:
            del wb['Analisis']
        ws_analisis = wb.create_sheet("Analisis")
        
        dict_sipse = defaultdict(list)
        dict_secop = defaultdict(list)
        
        filas_sipse = list(ws_sipse.values)
        header_sipse = filas_sipse[0] if filas_sipse else []
        for fila in filas_sipse[1:]:
            contrato = normalizar_numero_contrato(fila[3] if len(fila) > 3 else 0)
            dict_sipse[contrato].append(fila)

        filas_secop = list(ws_secop_aux.values)
        header_secop = filas_secop[0] if filas_secop else []
        for fila in filas_secop[1:]:
            contrato = normalizar_numero_contrato(fila[11] if len(fila) > 11 else 0)
            dict_secop[contrato].append(fila)

        ws_analisis.append(header_sipse) 
        col_inicio_secop = column_index_from_string("BM")
        for i, val in enumerate(header_secop):
            ws_analisis.cell(row=1, column=col_inicio_secop + i, value=val)

        todos_contratos = sorted(list(set(dict_sipse.keys()) | set(dict_secop.keys())))
        current_row = 2 
        
        for contrato in todos_contratos:
            filas_de_sipse = dict_sipse.get(contrato, [])
            filas_de_secop = dict_secop.get(contrato, [])
            max_filas = max(len(filas_de_sipse), len(filas_de_secop))
            
            for i in range(max_filas):
                if i < len(filas_de_sipse):
                    datos = filas_de_sipse[i]
                    for c_idx, valor in enumerate(datos, start=1):
                        ws_analisis.cell(row=current_row, column=c_idx, value=valor)
                
                if i < len(filas_de_secop):
                    datos = filas_de_secop[i]
                    for c_idx, valor in enumerate(datos):
                        col_excel = col_inicio_secop + c_idx
                        if c_idx == len(datos) - 1 and str(valor).startswith('='):
                             #nueva_formula = f'''=MID(BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), FIND("-", BX{current_row}) - MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0))'''
                             nueva_formula = (
                                f'=IFERROR(MID(BX{current_row}, '
                                f'MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), '
                                f'MIN(IFERROR(FIND(" ", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999), '
                                f'IFERROR(FIND("-", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999)) '
                                f'- MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), "")'
                            )
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula)
                        else:
                            ws_analisis.cell(row=current_row, column=col_excel, value=valor)
                current_row += 1

        # ==========================================
        # PARTE 4: Columnas de Evaluación
        # ==========================================
        print("  -> Agregando columnas de evaluación y estilos...")
        
        ultima_col_actual = ws_analisis.max_column
        col_inicio_eval = ultima_col_actual + 1
        ultima_fila_analisis = ws_analisis.max_row
        
        headers_nuevos = [
            "Ev. Nro contrato", "Aux Nro contrato", "Ev. Cédula", "Repeticiones contrato SIPSE", 
            "Repeticiones contrato SECOP", "Fecha inicio", "Fecha fin", "Duración contratos", "Valor contratos"
        ]

        for idx, titulo in enumerate(headers_nuevos):
            ws_analisis.cell(row=1, column=col_inicio_eval + idx, value=titulo)

        for i in range(2, ultima_fila_analisis + 1):

            # 🔹 AUX NRO CONTRATO (EX) → llave única
            f_aux = f'=IF(D{i}<>"",D{i},EV{i})'
            ws_analisis.cell(row=i, column=col_inicio_eval + 1, value=f_aux)

            # 🔹 ESTADO CONTRATO
            f1 = f'''=IF(COUNTIF($EX$2:EX{i},EX{i})>1,"",IF(AND(COUNTIF($D$2:$D${ultima_fila_analisis},EX{i})>0,COUNTIF($EV$2:$EV${ultima_fila_analisis},EX{i})>0),"Numero contrato coincide",IF(COUNTIF($D$2:$D${ultima_fila_analisis},EX{i})>0,"Contrato no existe en SECOP",IF(COUNTIF($EV$2:$EV${ultima_fila_analisis},EX{i})>0,"Contrato no existe en SIPSE","No existe"))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval, value=f1)

            # 🔹 CÉDULA
            f2 = f'=IF(AND(BY{i}="",M{i}=""),"",IF(OR(BY{i}="",M{i}=""),"",IF(BY{i}="-","Sin cedula en SECOP",IF(M{i}="-","Sin cedula en SIPSE",IF(VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(CJ{i}),"CC",""),"NIT",""),"TI",""))=VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(M{i}),"CC",""),"NIT",""),"TI","")),"Cedula coincide","Cedula NO coincide")))))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 2, value=f2)

            # 🔹 REPETICIONES SIPSE
            f3 = f'=IF(EW{i}="","",COUNTIF(D$2:D${ultima_fila_analisis},D{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 3, value=f3)

            # 🔹 REPETICIONES SECOP
            f4 = f'=IF(EW{i}="","",COUNTIF(EV$2:EV${ultima_fila_analisis},EV{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 4, value=f4)

            # 🔹 FECHA ACTA DE INICIO
            cf_clean = f'SUBSTITUTE(SUBSTITUTE(CF{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CF{i}), DAY(CF{i}), MONTH(CF{i}))'
            parte_es_texto = f'DATE(RIGHT({cf_clean}, 4), LEFT({cf_clean}, FIND("/", {cf_clean}) - 1), MID({cf_clean}, FIND("/", {cf_clean}) + 1, FIND("/", {cf_clean}, FIND("/", {cf_clean}) + 1) - FIND("/", {cf_clean}) - 1))'

            f5 = f'''=IF(AND(I{i}="",CF{i}=""),"",IF(AND(I{i}="-",CF{i}="-"),"Ambos sin fechas",IF(I{i}="-","sin fecha SIPSE",IF(CF{i}="-","sin fecha SECOP",IF(OR(I{i}="",CF{i}=""),"",IF(INT(VALUE(SUBSTITUTE(I{i},"-","/"))) =INT(IF(ISNUMBER(CF{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 5, value=f5)

            # 🔹 FECHA FIN
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CG{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CG{i}), DAY(CG{i}), MONTH(CG{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f6 = f'''=IF(AND(J{i}="",CG{i}=""),"",IF(AND(J{i}="-",CG{i}="-"),"Ambos sin fechas",IF(J{i}="-","sin fecha SIPSE",IF(CG{i}="-","sin fecha SECOP",IF(OR(J{i}="",CG{i}=""),"",IF(INT(VALUE(SUBSTITUTE(J{i},"-","/"))) =INT(IF(ISNUMBER(CG{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 6, value=f6)

            # 🔹 DURACIÓN
            f7 = f'''=IF(AND(R{i}="",EE{i}=""),"",IF(AND(R{i}="-",EE{i}="-"),"Ambos sin duracion",IF(R{i}="-","sin duracion en SECOP",IF(EE{i}="-","sin duracion en SIPSE",IF(OR(R{i}="",EE{i}=""),"",IF(TRIM(UPPER(R{i}))=TRIM(UPPER(EE{i})),"Duracion coincide","Duracion NO coincide"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 7, value=f7)

            # 🔹 VALOR CONTRATO (Comparación Robusta)
            f8 = f'''=IF(AND(P{i}="",CU{i}=""),"",
                IF(AND(P{i}="-",CU{i}="-"),"Ambos sin valor",
                IF(P{i}="-","Sin valor en SECOP",
                IF(CU{i}="-","Sin valor en SIPSE",
                IF(OR(P{i}="",CU{i}=""),"",
                IF(P{i}=CU{i},"Valor coincide","Valor NO coincide"))))))'''

            ws_analisis.cell(row=i, column=col_inicio_eval + 8, value=f8)

        # --- ESTILOS ANALISIS ---
        print("  -> Aplicando estilos y ocultando columnas en Analisis...")
        aplicar_estilo_ferrocarril(ws_analisis)
        aplicar_formato_condicional_alertas(ws_analisis, col_inicio_eval)
        
        # Ocultar columnas hoja analisis
        ocultar_columnas_por_lista(ws_analisis, RANGOS_OCULTAR_ANALISIS)

        # ==========================================
        # PARTE 5: Crear HOJA RESULTADOS
        # ==========================================
        print("  -> Creando hoja 'Resultados'...")
        if 'Resultados' in wb.sheetnames:
            del wb['Resultados']
        ws_res = wb.create_sheet("Resultados")
        ws_res['B1'] = f"Métricas alcaldía local {archivo[:-5]}"
        
        letra_ev_nro = get_column_letter(col_inicio_eval)       
        letra_ev_ced = get_column_letter(col_inicio_eval + 2)   
        letra_rep_sipse = get_column_letter(col_inicio_eval + 3)
        letra_rep_secop = get_column_letter(col_inicio_eval + 4)
        letra_fecha_inicio = get_column_letter(col_inicio_eval + 5)
        letra_fecha_fin = get_column_letter(col_inicio_eval + 6)
        letra_duracion = get_column_letter(col_inicio_eval + 7)
        letra_valor = get_column_letter(col_inicio_eval + 8)

        metricas = [
            ("Nro de contratos registrados en SIPSE",'=SUMPRODUCT((Analisis!A2:A10000<>"")/COUNTIF(Analisis!A2:A10000,Analisis!A2:A10000&""))'),
            ("Nro de contratos registrados SECOP",'=SUMPRODUCT((Analisis!EV2:EV10000<>"")/COUNTIF(Analisis!EV2:EV10000,Analisis!EV2:EV10000&""))'),
            ("Nro de contratos registrados en SECOP y SIPSE con numero de contrato igual", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")"),
            ("Nro de contratos solo en SECOP y no en SIPSE", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos solo en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos diferentes en SIPSE-SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato NO coincide\")"),
            ("Nro de contratos Cédula diferente", f"=COUNTIF('Analisis'!{letra_ev_ced}:{letra_ev_ced}, \"Cedula NO coincide\")"),
            ("Nro de contratos dobles en SIPSE", f"=COUNTIF('Analisis'!{letra_rep_sipse}:{letra_rep_sipse}, 2)"),
            ("Nro de contratos dobles en SECOP", f"=COUNTIF('Analisis'!{letra_rep_secop}:{letra_rep_secop}, 2)"),
            ("Fecha de acta de inicio Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Coincide fecha\")"),
            ("Fecha de acta de inicio no coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"No coinciden fechas\")"),
            ("Fecha de Fin conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"Coincide fecha\")"),
            ("Fecha de Fin no conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"No coinciden fechas\")"),
            ("Contratos que coinciden en Duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion Coincide\")"),
            ("Contratos que tienen diferente duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion NO coincide\")"),
            ("Contratos que tienen Fecha de inicio en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Sin fecha SECOP\")"),
            ("Datos en la matriz de seguimiento", f"=C2+C3"),
            ("Valor contrato coinciden", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor coincide\")"),
            ("Valor contrato no coincidenn", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor NO coincide\")")
        ]


        for idx, (texto, formula) in enumerate(metricas, start=2):
            ws_res.cell(row=idx, column=1, value=idx-1)
            ws_res.cell(row=idx, column=2, value=texto)
            ws_res.cell(row=idx, column=3, value=formula)
        
        # Pintar B2 de amarillo
        fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
        ws_res['B1'].fill = fill_amarillo
        ws_res.column_dimensions['B'].width = 100
        
        ws_analisis.column_dimensions['EW'].width = 25
        ws_analisis.column_dimensions['EX'].width = 20
        ws_analisis.column_dimensions['EY'].width = 20
        ws_analisis.column_dimensions['EZ'].width = 15
        ws_analisis.column_dimensions['FA'].width = 15
        ws_analisis.column_dimensions['FB'].width = 20
        ws_analisis.column_dimensions['FC'].width = 20
        ws_analisis.column_dimensions['FD'].width = 18
        ws_analisis.column_dimensions['FE'].width = 14

        dejar_columna_blanca(ws_analisis, 64)# columna 64 es la columna "BL"

        #Ajustar el texto en la fila 1 de las hojas aux y Analisis
        ajustar_texto_fila(ws_secop_aux, 1)
        ajustar_texto_fila(ws_analisis, 1)

        wb.save(ruta_dest)
        print("  -> Procesado exitosamente.")

if __name__ == "__main__":
    procesar_archivos()

Se encontraron 22 archivos de Excel.
Procesando: ANTONIO NARIÑO.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Procesado exitosamente.
Procesando: BARRIOS UNIDOS.xls...
  Error al abrir el archivo: openpyxl does not support the old .xls file format, please use xlrd to read this file, or convert it to the more recent .xlsx file format.
Procesando: BARRIOS UNIDOS.xlsx.xls...
  Error al abrir el archivo: openpyxl does not support the old .xls file format, please use xlrd to read this file, or convert it to the more recent .xlsx file format.
Procesando: BOSA.xls...
  Error al abrir el archivo: openpyxl does not support the old .xls file format, please use xlrd to read this file, or convert it to the more recent .xlsx file format.
Procesando: BOSA.xlsx.xls...
  Error al abrir el archivo

Creación de archivo con el consolidado de las 20 alcaldías

In [ ]:
import os
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
import re
import unicodedata

# ================= CONFIGURACIÓN =================
CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1' 
ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1\Consolidado\Consolidado_Localidades.xlsx"

nombres_hojas = [
    "1-Usaquén", "2-Chapinero", "3-Santa Fe", "4-San Cristóbal",
    "5-Usme", "6-Tunjuelito", "7-Bosa", "8-Kennedy", "9-Fontibón",
    "10-Engativá", "11-Suba", "12-Barrios Unidos", "13-Teusaquillo",
    "14-Los Mártires", "15-Antonio Nariño", "16-Puente Aranda",
    "17-Rafael Uribe Uribe", "18-La Candelaria", "19-Ciudad Bolívar", "20-Sumapaz"
]

RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-H', 'K', 'N', 'O', 'Q', 'S-BK', 'BM-BW', 'BY-CE', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']

# Aquí puedes añadir todas las columnas a las que quieras fijarles un ancho específico
ANCHOS_COLUMNAS = {
    'EZ': 25,
    'FA': 20,
    'FB': 20,
    'FC': 15,
    'FD': 15,
    'FE': 20,
    'FF': 20,
    'FG': 18,
    'FH': 14
}

# ================= FUNCIONES AUXILIARES =================
def normalizar_texto(texto):
    texto = texto.lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^\d+[-_ ]?', '', texto)
    texto = re.sub(r'\b(la|el|los|las)\b', '', texto).strip()
    return texto

def encontrar_hoja_destino(nombre_archivo, hojas_destino):
    archivo_norm = normalizar_texto(nombre_archivo)
    for hoja in hojas_destino:
        if normalizar_texto(hoja) in archivo_norm:
            return hoja
    return None

def ocultar_columnas_por_lista(ws, lista_rangos, offset=3):
    for item in lista_rangos:
        item = item.upper().strip()
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio) + offset
            idx_fin = column_index_from_string(fin) + offset
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                ws.column_dimensions[letra].hidden = True 
        else:
            idx = column_index_from_string(item) + offset
            letra = get_column_letter(idx)
            ws.column_dimensions[letra].hidden = True

def adaptar_formulas_resultados(formula):
    pattern = re.compile(r"'?Analisis'?!([A-Za-z]+)(\d*)(?::([A-Za-z]+)(\d*))?")
    def replacer(match):
        col1, row1, col2, row2 = match.groups()
        nueva_col1 = get_column_letter(column_index_from_string(col1) + 3)
        nueva_row1 = str(int(row1) + 21) if row1 else ""
        resultado = f"{nueva_col1}{nueva_row1}"
        if col2:
            nueva_col2 = get_column_letter(column_index_from_string(col2) + 3)
            nueva_row2 = str(int(row2) + 21) if row2 else ""
            resultado += f":{nueva_col2}{nueva_row2}"
        return resultado
    return pattern.sub(replacer, formula)

def desplazar_formulas_analisis(formula):
    formula = re.sub(r"'?Analisis'?!", "", formula, flags=re.IGNORECASE)
    pattern = re.compile(r'(\$?)([A-Za-z]{1,3})(\$?)(\d+)(?!\s*\()')
    def replacer(match):
        lock_col, col_str, lock_row, row_str = match.groups()
        nueva_col = get_column_letter(column_index_from_string(col_str) + 3)
        nueva_fila = str(int(row_str) + 21)
        return f"{lock_col}{nueva_col}{lock_row}{nueva_fila}"
    return pattern.sub(replacer, formula)

# ================= FUNCIONES DE DISEÑO =================
def aplicar_estilo_ferrocarril(ws, start_row, max_row):
    """Aplica formato estilo ferrocarril solo a columnas E:U y CA:FH desde start_row."""
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    
    # Rango 1: E (5) a U (21) | Rango 2: CA (79) a FH (164)
    def es_columna_objetivo(c):
        return (5 <= c <= 21) or (79 <= c <= 164)

    # 1. Encabezados (Fila 22)
    for col_idx in range(1, ws.max_column + 1):
        if es_columna_objetivo(col_idx):
            cell = ws.cell(row=start_row, column=col_idx)
            cell.fill = fill_header
            cell.font = font_header
            cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas de Datos (Patrón Ferrocarril)
    for i_fila, r_idx in enumerate(range(start_row + 1, max_row + 1)):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF": # Evitar procesar blanco para ahorrar memoria
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for col_idx in range(1, ws.max_column + 1):
                if es_columna_objetivo(col_idx):
                    ws.cell(row=r_idx, column=col_idx).fill = relleno

def aplicar_formato_condicional_alertas(ws, start_row, max_row):
    """Aplica reglas condicionales desde la columna EZ."""
    col_inicio_eval = column_index_from_string('EZ')
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    
    reglas = [
        (0, 'equal', '"Numero contrato NO coincide"'),
        (2, 'equal', '"Cedula NO coincide"'),
        (3, 'greaterThan', '1'),
        (4, 'greaterThan', '1'),
        (5, 'equal', '"No coinciden fechas"'),
        (6, 'equal', '"No coinciden fechas"'),
        (7, 'equal', '"Duracion NO coincide"'),
        (8, 'equal', '"Valor NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        # Se aplica desde la fila siguiente al encabezado (23 en adelante)
        rango = f"{letra_col}{start_row + 1}:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

# ================= PROGRAMA PRINCIPAL =================
def consolidar_excels():
    wb_nuevo = openpyxl.Workbook()
    
    # 1. Configurar Hoja General
    ws_general = wb_nuevo.active
    ws_general.title = "General"
    ws_general.cell(row=1, column=1, value="Nro").font = Font(bold=True)
    ws_general.cell(row=1, column=2, value="Alcaldía local").font = Font(bold=True)
    
    # Pre-llenar las localidades en General con Hipervínculos
    for i, hoja in enumerate(nombres_hojas):
        num, nombre = hoja.split("-", 1)
        row_idx = i + 2
        ws_general.cell(row=row_idx, column=1, value=int(num))
        
        celda_loc = ws_general.cell(row=row_idx, column=2, value=nombre.strip())
        celda_loc.hyperlink = f"#'{hoja}'!A1"
        celda_loc.font = Font(color="0563C1", underline="single") 

    for nombre in nombres_hojas:
        wb_nuevo.create_sheet(title=nombre)

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith('.xlsx') and not f.startswith('~')]
    metricas_extraidas = False 

    for archivo in archivos:
        hoja_destino_nombre = encontrar_hoja_destino(archivo, nombres_hojas)
        
        if not hoja_destino_nombre:
            continue
            
        print(f"-> Procesando '{archivo}' -> Destino: '{hoja_destino_nombre}'")
        
        ruta_completa = os.path.join(CARPETA_ORIGEN, archivo)
        wb_origen = openpyxl.load_workbook(ruta_completa)
        ws_destino = wb_nuevo[hoja_destino_nombre]

        # --- TAREA A: Copiar de 'Resultados' ---
        if "Resultados" in wb_origen.sheetnames:
            ws_resultados = wb_origen["Resultados"]
            row_gen = nombres_hojas.index(hoja_destino_nombre) + 2
            col_gen = 3 
            
            for r_idx in range(2, 21): 
                titulo_metrica = ws_resultados.cell(row=r_idx, column=2).value 
                
                if not metricas_extraidas:
                    ws_general.cell(row=1, column=col_gen, value=titulo_metrica).font = Font(bold=True)
                
                ws_general.cell(row=row_gen, column=col_gen, value=f"='{hoja_destino_nombre}'!C{r_idx}")
                col_gen += 1
                
            metricas_extraidas = True 
            
            for row in ws_resultados['A1':'C20']:
                for cell in row:
                    dest_coord = cell.coordinate
                    valor = cell.value
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = adaptar_formulas_resultados(valor)
                    else:
                        ws_destino[dest_coord].value = valor
        else:
            print(f"  - Faltó hoja 'Resultados' en {archivo}")

        # --- TAREA B: Copiar de 'Analisis' a la fila 22 ---
        if "Analisis" in wb_origen.sheetnames:
            ws_analisis = wb_origen["Analisis"]
            ultima_fila = ws_analisis.max_row
            max_col_origen = min(ws_analisis.max_column, column_index_from_string('FH'))
            
            ws_destino.merge_cells("D21:BN21")
            celda_sipse = ws_destino["D21"]
            celda_sipse.value = "SIPSE"
            celda_sipse.fill = PatternFill(start_color="0070C0", end_color="0070C0", fill_type="solid") 
            celda_sipse.font = Font(color="FFFFFF", bold=True)
            celda_sipse.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells("BP21:EH21")
            celda_secop = ws_destino["BP21"]
            celda_secop.value = "SECOP"
            celda_secop.fill = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid") 
            celda_secop.font = Font(color="FFFFFF", bold=True)
            celda_secop.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells("EI21:FH21")
            celda_comparacion = ws_destino["EI21"]
            celda_comparacion.value = "COMPARACIÓN"
            celda_comparacion.fill = PatternFill(start_color="7e350e", end_color="7e350e", fill_type="solid") 
            celda_comparacion.font = Font(color="FFFFFF", bold=True)
            celda_comparacion.alignment = Alignment(horizontal="center", vertical="center")

            # Pegar los datos desde la fila 22
            for r_idx, row_origen in enumerate(ws_analisis.iter_rows(min_row=1, max_row=ultima_fila, min_col=1, max_col=max_col_origen), start=22):
                for c_idx, cell in enumerate(row_origen, start=4):
                    dest_coord = f"{get_column_letter(c_idx)}{r_idx}"
                    valor = cell.value
                    
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = desplazar_formulas_analisis(valor)
                    else:
                        ws_destino[dest_coord].value = valor
            
            # 1. Ocultar columnas
            ocultar_columnas_por_lista(ws_destino, RANGOS_OCULTAR_ANALISIS, offset=3)
            # 2. Aplicar anchos de columnas
            for letra_col, ancho in ANCHOS_COLUMNAS.items():
                ws_destino.column_dimensions[letra_col].width = ancho
            # 2.1 Ajustar el texto en la fila 22 de las hojas de cada localidad
            ajustar_texto_fila(ws_destino, 22)
            # 3. Aplicar estilo Ferrocarril (La tabla empieza en la 22)
            aplicar_estilo_ferrocarril(ws_destino, start_row=22, max_row=ultima_fila + 21)
            # 4. Aplicar alertas condicionales
            aplicar_formato_condicional_alertas(ws_destino, start_row=22, max_row=ultima_fila + 21)
            # 5 Pintar y ajustar tabla de cada alcaldía
            fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
            ws_destino['B1'].fill = fill_amarillo
            ws_destino.column_dimensions['B'].width = 100
            
            # --- NUEVO: BOTÓN DE REGRESO A GENERAL ---
            celda_volver = ws_destino['A1']
            celda_volver.value = "General"
            celda_volver.hyperlink = "#'General'!A1"
            celda_volver.font = Font(color="0563C1", underline="single", bold=True)
            celda_volver.fill = PatternFill(start_color="E7E6E6", end_color="E7E6E6", fill_type="solid") # Gris claro
            celda_volver.alignment = Alignment(horizontal="center", vertical="center")

        else:
            print(f"  - Faltó hoja 'Analisis' en {archivo}")
            
        wb_origen.close()

    # --- DISEÑO FINAL HOJA GENERAL ---
    # 1. Fondo Rojo para la fila de títulos A1:U1
    relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
    fuente_blanca = Font(color="FFFFFF", bold=True)
    for col in range(1, 22): # De la A (1) a la U (21)
        celda = ws_general.cell(row=1, column=col)
        celda.fill = relleno_rojo
        celda.font = fuente_blanca
        celda.alignment = Alignment(horizontal="center", vertical="center")

    # 2. Centrar los valores de C2:U21
    alineacion_centro = Alignment(horizontal="center", vertical="center")
    for r in range(2, 22): # Filas 2 a 21
        for c in range(3, 22): # Columnas C (3) a U (21)
            ws_general.cell(row=r, column=c).alignment = alineacion_centro
            
    # Ajustar ligeramente el ancho de General para que se vea bien
    ws_general.column_dimensions['A'].width = 5
    ws_general.column_dimensions['B'].width = 25
    for c in range(3, 22):
        ws_general.column_dimensions[get_column_letter(c)].width = 15
    
    # Ajustar el texto en la fila 1 de las hoja General
    ajustar_texto_fila(ws_general, 1)

    # --- NUEVO: APLICAR GRILLA DELGADA (A1:U21) ---
    borde_delgado = Border(
        left=Side(style='thin', color='000000'),
        right=Side(style='thin', color='000000'),
        top=Side(style='thin', color='000000'),
        bottom=Side(style='thin', color='000000')
    )
    
    for r in range(1, 22):      # Filas de la 1 a la 21
        for c in range(1, 22):  # Columnas de la A(1) a la U(21)
            ws_general.cell(row=r, column=c).border = borde_delgado

    wb_nuevo.save(ARCHIVO_DESTINO)
    print(f"\n🚀 ¡Proceso completado! Archivo guardado en: {ARCHIVO_DESTINO}")

if __name__ == "__main__":
    consolidar_excels()


🚀 ¡Proceso completado! Archivo guardado en: C:\Users\Pc\Desktop\META 7 - FDL 02 FEBRERO\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx


#### Entrega 2/03/2026

Para abrir archivos XLS (1997) y XLSX

In [ ]:
import os
import shutil
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.formatting.rule import CellIsRule
from collections import defaultdict
import re
import pandas as pd

# --- CONFIGURACIÓN DE CARPETAS ---
#CARPETA_ORIGEN = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO'
#CARPETA_DESTINO = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\DiagnóstivoV2_1'
#CARPETA_ORIGEN = r'C:\\Users\\Pc\\Desktop\\META 7 - FDL 02 FEBRERO' #Dell
#CARPETA_DESTINO = r"C:\\Users\\Pc\\Desktop\\META 7 - FDL 02 FEBRERO\\Diagnóstico" #Dell
CARPETA_ORIGEN = r'C:\\Users\\Pc\\Desktop\\META 7 - FDL 02 FEBRERO' #Predator
CARPETA_DESTINO = r"C:\\Users\\Pc\\Desktop\\META 7 - FDL 02 FEBRERO\\Diagnóstico" #PRedator

# --- RANGOS A OCULTAR (CORREGIDOS PARA USAR HIDDEN=TRUE) ---
# Nota: He ajustado 'A-AH' a 'Z-AH' en SECOP para NO ocultar las columnas L, M, T, X, Y que necesitas ver.
RANGOS_OCULTAR_SECOP = ['A-K', 'N-S', 'U-W', 'Z-AH', 'AJ-BR', 'BT-CI']
RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']

# --- DEFINICIÓN DE FUNCIONES ---

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def ocultar_columnas_por_lista(ws, lista_rangos):
    """
    Recibe una lista de cadenas como ['A-K', 'M'] y establece la propiedad HIDDEN = True.
    """
    for item in lista_rangos:
        item = item.upper().strip()
        
        # Caso 1: Es un rango (ej: "A-K")
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio)
            idx_fin = column_index_from_string(fin)
            
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                # Esta es la forma correcta de ocultar en Excel
                ws.column_dimensions[letra].hidden = True 
                
        # Caso 2: Es una columna única (ej: "A")
        else:
            ws.column_dimensions[item].hidden = True

def aplicar_estilo_ferrocarril(ws):
    """
    Encabezados Azul Oscuro + Letra Blanca.
    Filas con patrón intercalado.
    """
    # 1. Encabezados
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    
    for cell in ws[1]:
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas (Patrón Ferrocarril)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    max_col = ws.max_column
    
    for i_fila, row in enumerate(ws.iter_rows(min_row=2, max_col=max_col), start=0):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF":
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for cell in row:
                cell.fill = relleno

def dejar_columna_blanca(ws, indice_columna):
    relleno_blanco = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    
    # Recorremos todas las celdas de la columna indicada
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=indice_columna, max_col=indice_columna):
        for cell in row:
            cell.fill = relleno_blanco

def aplicar_formato_condicional_alertas(ws, col_inicio_eval):
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    max_row = ws.max_row
    
    reglas = [
        (0, 'equal', '"Numero contrato NO coincide"'),
        (2, 'equal', '"Cedula NO coincide"'),
        (3, 'greaterThan', '1'),
        (4, 'greaterThan', '1'),
        (5, 'equal', '"No coinciden fechas"'),
        (6, 'equal', '"No coinciden fechas"'),
        (7, 'equal', '"No coinciden fechas"'),
        (8, 'equal', '"Duracion NO coincide"'),
        (9, 'equal', '"Valor NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        rango = f"{letra_col}2:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

def normalizar_numero_contrato(valor):
    if valor is None: return 0
    texto = str(valor).strip()
    # Busca el primer grupo de números en la cadena
    match = re.search(r'\d+', texto)
    if match:
        return int(match.group())
    return 0

# --- LÓGICA PRINCIPAL ---
def procesar_archivos():
    if not os.path.exists(CARPETA_DESTINO):
        os.makedirs(CARPETA_DESTINO)

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('CANDELARIA.xlsx', 'CANDELARIA.xls'))]
    print(f"Se encontraron {len(archivos)} archivos de Excel.")

    for archivo in archivos:
        ruta_orig = os.path.join(CARPETA_ORIGEN, archivo)
        
        # Definimos la ruta de destino siempre con extensión .xlsx
        nombre_base = os.path.splitext(archivo)[0]
        ruta_dest = os.path.join(CARPETA_DESTINO, nombre_base + ".xlsx")
        
        print(f"Procesando: {archivo}...")

        # --- PASO DE CONVERSIÓN SI ES .XLS ---
        if archivo.lower().endswith('.xls'):
            try:
                # Leemos el .xls y lo guardamos inmediatamente como .xlsx
                # sheet_name=None lee todas las pestañas
                conversion = pd.read_excel(ruta_orig, sheet_name=None, engine='xlrd')
                with pd.ExcelWriter(ruta_dest, engine='openpyxl') as writer:
                    for nombre_hoja, datos in conversion.items():
                        datos.to_excel(writer, sheet_name=nombre_hoja, index=False)
                print(f"  -> Convertido de .xls a .xlsx con éxito.")
            except Exception as e:
                print(f"  Error convirtiendo {archivo}: {e}")
                continue
        else:
            # Si ya es .xlsx, solo lo copiamos como hacías antes
            shutil.copy2(ruta_orig, ruta_dest)
        # -------------------------------------

        try:
            # Ahora openpyxl siempre abrirá un archivo .xlsx (el copiado o el convertido)
            wb = openpyxl.load_workbook(ruta_dest)
        except Exception as e:
            print(f"   Error al abrir el archivo con openpyxl: {e}")
            continue

        # ==========================================
        # PARTE 1: Procesar SECOP II (Crear aux1)
        # ==========================================
        nombre_real_secop = next(n for n in wb.sheetnames if n.lower() == 'secop ii')
        ws_origen = wb[nombre_real_secop]
        
        if "SECOP II aux1" in wb.sheetnames:
            del wb["SECOP II aux1"]
            
        ws_secop_aux = wb.copy_worksheet(ws_origen)
        ws_secop_aux.title = "SECOP II aux1"

        filas = []
        for row in ws_secop_aux.iter_rows(min_row=2, values_only=True):
            filas.append(list(row))
        
        filas_ordenadas = sorted(filas, key=lambda x: normalizar_numero_contrato(x[11] if len(x) > 11 else 0))

        for row in ws_secop_aux.iter_rows(min_row=2):
            for cell in row:
                cell.value = None

        for i_fila, datos_fila in enumerate(filas_ordenadas, start=2):
            for i_col, valor in enumerate(datos_fila, start=1):
                ws_secop_aux.cell(row=i_fila, column=i_col, value=valor)

        max_col_secop = ws_secop_aux.max_column
        ws_secop_aux.cell(row=1, column=max_col_secop + 1, value="Columna1")
        col_L = get_column_letter(12)
        
        for i in range(2, ws_secop_aux.max_row + 1):
            #formula = f'''=MID({col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), FIND("-", {col_L}{i}) - MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0))'''
            formula = (
                f'=IFERROR(MID({col_L}{i}, '
                f'MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), '
                f'MIN(IFERROR(FIND(" ", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999), '
                f'IFERROR(FIND("-", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999)) '
                f'- MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), "")'
            )
            ws_secop_aux.cell(row=i, column=max_col_secop + 1, value=formula)

        # ESTILOS SECOP
        print("  -> Aplicando estilos y ocultando columnas en SECOP II aux1...")
        aplicar_estilo_ferrocarril(ws_secop_aux)
        ocultar_columnas_por_lista(ws_secop_aux, RANGOS_OCULTAR_SECOP)

        # ==========================================
        # PARTE 2 y 3: ALINEACIÓN EN 'ANALISIS'
        # ==========================================
        print("  -> Generando hoja 'Analisis'...")

        nombre_real_sipse = next(n for n in wb.sheetnames if n.lower() == 'sipse')
        ws_sipse = wb[nombre_real_sipse]
        
        if 'Analisis' in wb.sheetnames:
            del wb['Analisis']
        ws_analisis = wb.create_sheet("Analisis")
        
        dict_sipse = defaultdict(list)
        dict_secop = defaultdict(list)
        
        filas_sipse = list(ws_sipse.values)
        header_sipse = filas_sipse[0] if filas_sipse else []
        for fila in filas_sipse[1:]:
            contrato = normalizar_numero_contrato(fila[3] if len(fila) > 3 else 0)
            dict_sipse[contrato].append(fila)

        filas_secop = list(ws_secop_aux.values)
        header_secop = filas_secop[0] if filas_secop else []
        for fila in filas_secop[1:]:
            contrato = normalizar_numero_contrato(fila[11] if len(fila) > 11 else 0)
            dict_secop[contrato].append(fila)

        ws_analisis.append(header_sipse) 
        col_inicio_secop = column_index_from_string("BM")
        for i, val in enumerate(header_secop):
            ws_analisis.cell(row=1, column=col_inicio_secop + i, value=val)

        todos_contratos = sorted(list(set(dict_sipse.keys()) | set(dict_secop.keys())))
        current_row = 2 
        
        for contrato in todos_contratos:
            filas_de_sipse = dict_sipse.get(contrato, [])
            filas_de_secop = dict_secop.get(contrato, [])
            max_filas = max(len(filas_de_sipse), len(filas_de_secop))
            
            for i in range(max_filas):
                if i < len(filas_de_sipse):
                    datos = filas_de_sipse[i]
                    for c_idx, valor in enumerate(datos, start=1):
                        ws_analisis.cell(row=current_row, column=c_idx, value=valor)
                
                if i < len(filas_de_secop):
                    datos = filas_de_secop[i]
                    for c_idx, valor in enumerate(datos):
                        col_excel = col_inicio_secop + c_idx
                        if c_idx == len(datos) - 1 and str(valor).startswith('='):
                             #nueva_formula = f'''=MID(BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), FIND("-", BX{current_row}) - MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0))'''
                             nueva_formula = (
                                f'=IFERROR(MID(BX{current_row}, '
                                f'MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), '
                                f'MIN(IFERROR(FIND(" ", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999), '
                                f'IFERROR(FIND("-", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999)) '
                                f'- MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), "")'
                            )
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula)
                        else:
                            ws_analisis.cell(row=current_row, column=col_excel, value=valor)
                current_row += 1

        # ==========================================
        # PARTE 4: Columnas de Evaluación
        # ==========================================
        print("  -> Agregando columnas de evaluación y estilos...")
        
        ultima_col_actual = ws_analisis.max_column
        col_inicio_eval = ultima_col_actual + 1
        ultima_fila_analisis = ws_analisis.max_row
        
        headers_nuevos = [
            "Ev. Nro contrato", "Aux Nro contrato", "Ev. Cédula", "Repeticiones contrato SIPSE", 
            "Repeticiones contrato SECOP", "Fecha suscripción", "Fecha inicio", "Fecha fin", "Duración contratos", "Valor contratos"
        ]

        for idx, titulo in enumerate(headers_nuevos):
            ws_analisis.cell(row=1, column=col_inicio_eval + idx, value=titulo)

        for i in range(2, ultima_fila_analisis + 1):

            # 🔹 AUX NRO CONTRATO (EX) → llave única
            f_aux = f'=IF(D{i}<>"",D{i},EV{i})'
            ws_analisis.cell(row=i, column=col_inicio_eval + 1, value=f_aux)

            # 🔹 ESTADO CONTRATO
            f1 = f'''=IF(COUNTIF($EX$2:EX{i},EX{i})>1,"",IF(AND(COUNTIF($D$2:$D${ultima_fila_analisis},EX{i})>0,COUNTIF($EV$2:$EV${ultima_fila_analisis},EX{i})>0),"Numero contrato coincide",IF(COUNTIF($D$2:$D${ultima_fila_analisis},EX{i})>0,"Contrato no existe en SECOP",IF(COUNTIF($EV$2:$EV${ultima_fila_analisis},EX{i})>0,"Contrato no existe en SIPSE","No existe"))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval, value=f1)

            # 🔹 CÉDULA
            f2 = f'=IF(AND(BY{i}="",M{i}=""),"",IF(OR(BY{i}="",M{i}=""),"",IF(BY{i}="-","Sin cedula en SECOP",IF(M{i}="-","Sin cedula en SIPSE",IF(VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(CJ{i}),"CC",""),"NIT",""),"TI",""))=VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(M{i}),"CC",""),"NIT",""),"TI","")),"Cedula coincide","Cedula NO coincide")))))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 2, value=f2)

            # 🔹 REPETICIONES SIPSE
            f3 = f'=IF(EW{i}="","",COUNTIF(D$2:D${ultima_fila_analisis},D{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 3, value=f3)

            # 🔹 REPETICIONES SECOP
            f4 = f'=IF(EW{i}="","",COUNTIF(EV$2:EV${ultima_fila_analisis},EV{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 4, value=f4)

            # 🔹 FECHA SUSCRIPCION
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CE{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CE{i}), DAY(CE{i}), MONTH(CE{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f5 = f'''=IF(AND(H{i}="",CE{i}=""),"",IF(AND(H{i}="-",CE{i}="-"),"Ambos sin fechas",IF(H{i}="-","sin fecha SIPSE",IF(CE{i}="-","sin fecha SECOP",IF(OR(H{i}="",CE{i}=""),"",IF(INT(VALUE(SUBSTITUTE(H{i},"-","/"))) =INT(IF(ISNUMBER(CE{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 5, value=f5)

            # 🔹 FECHA ACTA DE INICIO
            cf_clean = f'SUBSTITUTE(SUBSTITUTE(CF{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CF{i}), DAY(CF{i}), MONTH(CF{i}))'
            parte_es_texto = f'DATE(RIGHT({cf_clean}, 4), LEFT({cf_clean}, FIND("/", {cf_clean}) - 1), MID({cf_clean}, FIND("/", {cf_clean}) + 1, FIND("/", {cf_clean}, FIND("/", {cf_clean}) + 1) - FIND("/", {cf_clean}) - 1))'

            f6 = f'''=IF(AND(I{i}="",CF{i}=""),"",IF(AND(I{i}="-",CF{i}="-"),"Ambos sin fechas",IF(I{i}="-","sin fecha SIPSE",IF(CF{i}="-","sin fecha SECOP",IF(OR(I{i}="",CF{i}=""),"",IF(INT(VALUE(SUBSTITUTE(I{i},"-","/"))) =INT(IF(ISNUMBER(CF{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 6, value=f6)

            # 🔹 FECHA FIN
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CG{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CG{i}), DAY(CG{i}), MONTH(CG{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f7 = f'''=IF(AND(J{i}="",CG{i}=""),"",IF(AND(J{i}="-",CG{i}="-"),"Ambos sin fechas",IF(J{i}="-","sin fecha SIPSE",IF(CG{i}="-","sin fecha SECOP",IF(OR(J{i}="",CG{i}=""),"",IF(INT(VALUE(SUBSTITUTE(J{i},"-","/"))) =INT(IF(ISNUMBER(CG{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 7, value=f7)

            # 🔹 DURACIÓN
            f8 = f'''=IF(AND(R{i}="",EE{i}=""),"",IF(AND(R{i}="-",EE{i}="-"),"Ambos sin duracion",IF(R{i}="-","sin duracion en SECOP",IF(EE{i}="-","sin duracion en SIPSE",IF(OR(R{i}="",EE{i}=""),"",IF(TRIM(UPPER(R{i}))=TRIM(UPPER(EE{i})),"Duracion coincide","Duracion NO coincide"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 8, value=f8)

            # 🔹 VALOR CONTRATO (Comparación Robusta)
            f9 = f'''=IF(AND(P{i}="",CU{i}=""),"",
                IF(AND(P{i}="-",CU{i}="-"),"Ambos sin valor",
                IF(P{i}="-","Sin valor en SECOP",
                IF(CU{i}="-","Sin valor en SIPSE",
                IF(OR(P{i}="",CU{i}=""),"",
                IF(P{i}=CU{i},"Valor coincide","Valor NO coincide"))))))'''

            ws_analisis.cell(row=i, column=col_inicio_eval + 9, value=f9)

        # --- ESTILOS ANALISIS ---
        print("  -> Aplicando estilos y ocultando columnas en Analisis...")
        aplicar_estilo_ferrocarril(ws_analisis)
        aplicar_formato_condicional_alertas(ws_analisis, col_inicio_eval)
        
        # Ocultar columnas hoja analisis
        ocultar_columnas_por_lista(ws_analisis, RANGOS_OCULTAR_ANALISIS)

        # ==========================================
        # PARTE 5: Crear HOJA RESULTADOS
        # ==========================================
        print("  -> Creando hoja 'Resultados'...")
        if 'Resultados' in wb.sheetnames:
            del wb['Resultados']
        ws_res = wb.create_sheet("Resultados")
        nombre_limpio = os.path.splitext(archivo)[0]
        ws_res['B1'] = f"Métricas alcaldía local {nombre_limpio}"
        
        letra_ev_nro = get_column_letter(col_inicio_eval)       
        letra_ev_ced = get_column_letter(col_inicio_eval + 2)   
        letra_rep_sipse = get_column_letter(col_inicio_eval + 3)
        letra_rep_secop = get_column_letter(col_inicio_eval + 4)
        letra_fecha_suscripcion = get_column_letter(col_inicio_eval + 5)
        letra_fecha_inicio = get_column_letter(col_inicio_eval + 6)
        letra_fecha_fin = get_column_letter(col_inicio_eval + 7)
        letra_duracion = get_column_letter(col_inicio_eval + 8)
        letra_valor = get_column_letter(col_inicio_eval + 9)

        metricas = [
            ("Nro de contratos registrados en SIPSE",'=SUMPRODUCT((Analisis!A2:A10000<>"")/COUNTIF(Analisis!A2:A10000,Analisis!A2:A10000&""))'),
            ("Nro de contratos registrados SECOP",'=SUMPRODUCT((Analisis!EV2:EV10000<>"")/COUNTIF(Analisis!EV2:EV10000,Analisis!EV2:EV10000&""))'),
            ("Nro de contratos registrados en SECOP y SIPSE con numero de contrato igual", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")"),
            ("Nro de contratos solo en SECOP y no en SIPSE", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos solo en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos diferentes en SIPSE-SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato NO coincide\")"),
            ("Nro de contratos Cédula diferente", f"=COUNTIF('Analisis'!{letra_ev_ced}:{letra_ev_ced}, \"Cedula NO coincide\")"),
            ("Nro de contratos dobles en SIPSE", f"=COUNTIF('Analisis'!{letra_rep_sipse}:{letra_rep_sipse}, 2)"),
            ("Nro de contratos dobles en SECOP", f"=COUNTIF('Analisis'!{letra_rep_secop}:{letra_rep_secop}, 2)"),
            ("Fecha de acta de suscriccpión Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"Coincide fecha\")"),
            ("Fecha de acta de suscriccpión no Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"No coinciden fechas\")"),
            ("Fecha de acta de inicio Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Coincide fecha\")"),
            ("Fecha de acta de inicio no coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"No coinciden fechas\")"),
            ("Fecha de Fin conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"Coincide fecha\")"),
            ("Fecha de Fin no conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"No coinciden fechas\")"),
            ("Contratos que coinciden en Duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion Coincide\")"),
            ("Contratos que tienen diferente duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion NO coincide\")"),
            ("Contratos que tienen Fecha de inicio en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Sin fecha SECOP\")"),
            ("Datos en la matriz de seguimiento", f"=C2+C3"),
            ("Valor contrato coinciden", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor coincide\")"),
            ("Valor contrato no coincidenn", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor NO coincide\")")
        ]


        for idx, (texto, formula) in enumerate(metricas, start=2):
            ws_res.cell(row=idx, column=1, value=idx-1)
            ws_res.cell(row=idx, column=2, value=texto)
            ws_res.cell(row=idx, column=3, value=formula)
        
        # Pintar B2 de amarillo
        fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
        ws_res['B1'].fill = fill_amarillo
        ws_res.column_dimensions['B'].width = 100
        
        ws_analisis.column_dimensions['EW'].width = 25
        ws_analisis.column_dimensions['EX'].width = 20
        ws_analisis.column_dimensions['EY'].width = 20
        ws_analisis.column_dimensions['EZ'].width = 15
        ws_analisis.column_dimensions['FA'].width = 15
        ws_analisis.column_dimensions['FB'].width = 15
        ws_analisis.column_dimensions['FC'].width = 20
        ws_analisis.column_dimensions['FD'].width = 20
        ws_analisis.column_dimensions['FE'].width = 18
        ws_analisis.column_dimensions['FF'].width = 14

        dejar_columna_blanca(ws_analisis, 64)# columna 64 es la columna "BL"

        #Ajustar el texto en la fila 1 de las hojas aux y Analisis
        ajustar_texto_fila(ws_secop_aux, 1)
        ajustar_texto_fila(ws_analisis, 1)

        ##--- RELLENAR CELDAS VACÍAS CON "?" EN SIPSE Y SECOP ---
        #for row in ws_analisis.iter_rows(min_row=2, max_row=ultima_fila_analisis, min_col=1, max_col=col_inicio_eval - 1):
        #    for cell in row:
        #        # Comprueba si la celda está realmente vacía (None) o si es un texto vacío ("")
        #        if cell.value is None or str(cell.value).strip() == "":
        #            cell.value = "?"

        wb.save(ruta_dest)
        print("  -> Procesado exitosamente.")

if __name__ == "__main__":
    procesar_archivos()

PermissionError: [WinError 5] Acceso denegado: 'C:\\\\Users\\\\Pc'

In [3]:
import os
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
import re
import unicodedata

# ================= CONFIGURACIÓN =================
#CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1' 
#ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1\Consolidado\Consolidado_Localidades.xlsx"
CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 02 FEBRERO\Diagnóstico'
ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 02 FEBRERO\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx"

nombres_hojas = [
    "1-Usaquén", "2-Chapinero", "3-Santa Fe", "4-San Cristóbal",
    "5-Usme", "6-Tunjuelito", "7-Bosa", "8-Kennedy", "9-Fontibón",
    "10-Engativá", "11-Suba", "12-Barrios Unidos", "13-Teusaquillo",
    "14-Los Mártires", "15-Antonio Nariño", "16-Puente Aranda",
    "17-Rafael Uribe Uribe", "18-La Candelaria", "19-Ciudad Bolívar", "20-Sumapaz"
]

RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']
# Aquí puedes añadir todas las columnas a las que quieras fijarles un ancho específico
ANCHOS_COLUMNAS = {
    'EZ': 25,
    'FA': 20,
    'FB': 20,
    'FC': 15,
    'FD': 15,
    'FE': 15,
    'FF': 20,
    'FG': 20,
    'FH': 18,
    'FI': 14
}

# ================= FUNCIONES AUXILIARES =================
def normalizar_texto(texto):
    texto = texto.lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^\d+[-_ ]?', '', texto)
    texto = re.sub(r'\b(la|el|los|las)\b', '', texto).strip()
    return texto

def encontrar_hoja_destino(nombre_archivo, hojas_destino):
    archivo_norm = normalizar_texto(nombre_archivo)
    for hoja in hojas_destino:
        if normalizar_texto(hoja) in archivo_norm:
            return hoja
    return None

def ocultar_columnas_por_lista(ws, lista_rangos, offset=3):
    for item in lista_rangos:
        item = item.upper().strip()
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio) + offset
            idx_fin = column_index_from_string(fin) + offset
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                ws.column_dimensions[letra].hidden = True 
        else:
            idx = column_index_from_string(item) + offset
            letra = get_column_letter(idx)
            ws.column_dimensions[letra].hidden = True

def adaptar_formulas_resultados(formula):
    pattern = re.compile(r"'?Analisis'?!([A-Za-z]+)(\d*)(?::([A-Za-z]+)(\d*))?")
    def replacer(match):
        col1, row1, col2, row2 = match.groups()
        nueva_col1 = get_column_letter(column_index_from_string(col1) + 3)
        nueva_row1 = str(int(row1) + 21) if row1 else ""
        resultado = f"{nueva_col1}{nueva_row1}"
        if col2:
            nueva_col2 = get_column_letter(column_index_from_string(col2) + 3)
            nueva_row2 = str(int(row2) + 21) if row2 else ""
            resultado += f":{nueva_col2}{nueva_row2}"
        return resultado
    return pattern.sub(replacer, formula)

def desplazar_formulas_analisis(formula):
    formula = re.sub(r"'?Analisis'?!", "", formula, flags=re.IGNORECASE)
    pattern = re.compile(r'(\$?)([A-Za-z]{1,3})(\$?)(\d+)(?!\s*\()')
    def replacer(match):
        lock_col, col_str, lock_row, row_str = match.groups()
        nueva_col = get_column_letter(column_index_from_string(col_str) + 3)
        nueva_fila = str(int(row_str) + 21)
        return f"{lock_col}{nueva_col}{lock_row}{nueva_fila}"
    return pattern.sub(replacer, formula)

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# ================= FUNCIONES DE DISEÑO =================
def aplicar_estilo_ferrocarril(ws, start_row, max_row):
    """Aplica formato estilo ferrocarril solo a columnas E:U y CA:FH desde start_row."""
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    
    # Rango 1: E (5) a U (21) | Rango 2: CA (79) a FI (165)
    def es_columna_objetivo(c):
        return (5 <= c <= 21) or (79 <= c <= 165)

    # 1. Encabezados (Fila 22)
    for col_idx in range(1, ws.max_column + 1):
        if es_columna_objetivo(col_idx):
            cell = ws.cell(row=start_row, column=col_idx)
            cell.fill = fill_header
            cell.font = font_header
            cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas de Datos (Patrón Ferrocarril)
    for i_fila, r_idx in enumerate(range(start_row + 1, max_row + 1)):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF": # Evitar procesar blanco para ahorrar memoria
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for col_idx in range(1, ws.max_column + 1):
                if es_columna_objetivo(col_idx):
                    ws.cell(row=r_idx, column=col_idx).fill = relleno

def aplicar_formato_condicional_alertas(ws, start_row, max_row):
    """Aplica reglas condicionales desde la columna EZ."""
    col_inicio_eval = column_index_from_string('EZ')
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    
    reglas = [
        (0, 'equal', '"Numero contrato NO coincide"'),
        (2, 'equal', '"Cedula NO coincide"'),
        (3, 'greaterThan', '1'),
        (4, 'greaterThan', '1'),
        (5, 'equal', '"No coinciden fechas"'),
        (6, 'equal', '"No coinciden fechas"'),
        (7, 'equal', '"No coinciden fechas"'),
        (8, 'equal', '"Duracion NO coincide"'),
        (9, 'equal', '"Valor NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        # Se aplica desde la fila siguiente al encabezado (23 en adelante)
        rango = f"{letra_col}{start_row + 1}:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

# ================= PROGRAMA PRINCIPAL =================
def consolidar_excels():
    wb_nuevo = openpyxl.Workbook()
    
    # 1. Configurar Hoja General
    ws_general = wb_nuevo.active
    ws_general.title = "General"
    ws_general.cell(row=1, column=1, value="Nro").font = Font(bold=True)
    ws_general.cell(row=1, column=2, value="Alcaldía local").font = Font(bold=True)
    
    # Pre-llenar las localidades en General con Hipervínculos
    for i, hoja in enumerate(nombres_hojas):
        num, nombre = hoja.split("-", 1)
        row_idx = i + 2
        ws_general.cell(row=row_idx, column=1, value=int(num))
        
        celda_loc = ws_general.cell(row=row_idx, column=2, value=nombre.strip())
        celda_loc.hyperlink = f"#'{hoja}'!A1"
        celda_loc.font = Font(color="0563C1", underline="single") 

    for nombre in nombres_hojas:
        wb_nuevo.create_sheet(title=nombre)

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith('.xlsx') and not f.startswith('~')]
    metricas_extraidas = False 

    for archivo in archivos:
        hoja_destino_nombre = encontrar_hoja_destino(archivo, nombres_hojas)
        
        if not hoja_destino_nombre:
            continue
            
        print(f"-> Procesando '{archivo}' -> Destino: '{hoja_destino_nombre}'")
        
        ruta_completa = os.path.join(CARPETA_ORIGEN, archivo)
        wb_origen = openpyxl.load_workbook(ruta_completa)
        ws_destino = wb_nuevo[hoja_destino_nombre]

        # --- TAREA A: Copiar de 'Resultados' ---
        if "Resultados" in wb_origen.sheetnames:
            ws_resultados = wb_origen["Resultados"]
            row_gen = nombres_hojas.index(hoja_destino_nombre) + 2
            col_gen = 3 
            
            for r_idx in range(2, 21): 
                titulo_metrica = ws_resultados.cell(row=r_idx, column=2).value 
                
                if not metricas_extraidas:
                    ws_general.cell(row=1, column=col_gen, value=titulo_metrica).font = Font(bold=True)
                
                ws_general.cell(row=row_gen, column=col_gen, value=f"='{hoja_destino_nombre}'!C{r_idx}")
                col_gen += 1
                
            metricas_extraidas = True 
            
            for row in ws_resultados['A1':'C20']:
                for cell in row:
                    dest_coord = cell.coordinate
                    valor = cell.value
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = adaptar_formulas_resultados(valor)
                    else:
                        ws_destino[dest_coord].value = valor
        else:
            print(f"  - Faltó hoja 'Resultados' en {archivo}")

        # --- TAREA B: Copiar de 'Analisis' a la fila 22 ---
        if "Analisis" in wb_origen.sheetnames:
            ws_analisis = wb_origen["Analisis"]
            ultima_fila = ws_analisis.max_row
            max_col_origen = min(ws_analisis.max_column, column_index_from_string('FH'))
            
            ws_destino.merge_cells("D21:BN21")
            celda_sipse = ws_destino["D21"]
            celda_sipse.value = "SIPSE"
            celda_sipse.fill = PatternFill(start_color="0070C0", end_color="0070C0", fill_type="solid") 
            celda_sipse.font = Font(color="FFFFFF", bold=True)
            celda_sipse.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells("BP21:EH21")
            celda_secop = ws_destino["BP21"]
            celda_secop.value = "SECOP"
            celda_secop.fill = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid") 
            celda_secop.font = Font(color="FFFFFF", bold=True)
            celda_secop.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells("EI21:FI21")
            celda_comparacion = ws_destino["EI21"]
            celda_comparacion.value = "COMPARACIÓN"
            celda_comparacion.fill = PatternFill(start_color="7e350e", end_color="7e350e", fill_type="solid") 
            celda_comparacion.font = Font(color="FFFFFF", bold=True)
            celda_comparacion.alignment = Alignment(horizontal="center", vertical="center")

            # Pegar los datos desde la fila 22
            for r_idx, row_origen in enumerate(ws_analisis.iter_rows(min_row=1, max_row=ultima_fila, min_col=1, max_col=max_col_origen), start=22):
                for c_idx, cell in enumerate(row_origen, start=4):
                    dest_coord = f"{get_column_letter(c_idx)}{r_idx}"
                    valor = cell.value
                    
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = desplazar_formulas_analisis(valor)
                    else:
                        ws_destino[dest_coord].value = valor
            
            # 1. Ocultar columnas
            ocultar_columnas_por_lista(ws_destino, RANGOS_OCULTAR_ANALISIS, offset=3)
            # 2. Aplicar anchos de columnas
            for letra_col, ancho in ANCHOS_COLUMNAS.items():
                ws_destino.column_dimensions[letra_col].width = ancho
            # 2.1 Ajustar el texto en la fila 22 de las hojas de cada localidad
            ajustar_texto_fila(ws_destino, 22)
            # 3. Aplicar estilo Ferrocarril (La tabla empieza en la 22)
            aplicar_estilo_ferrocarril(ws_destino, start_row=22, max_row=ultima_fila + 21)
            # 4. Aplicar alertas condicionales
            aplicar_formato_condicional_alertas(ws_destino, start_row=22, max_row=ultima_fila + 21)
            # 5 Pintar y ajustar tabla de cada alcaldía
            fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
            ws_destino['B1'].fill = fill_amarillo
            ws_destino.column_dimensions['B'].width = 100
            
            # --- NUEVO: BOTÓN DE REGRESO A GENERAL ---
            celda_volver = ws_destino['A1']
            celda_volver.value = "General"
            celda_volver.hyperlink = "#'General'!A1"
            celda_volver.font = Font(color="0563C1", underline="single", bold=True)
            celda_volver.fill = PatternFill(start_color="E7E6E6", end_color="E7E6E6", fill_type="solid") # Gris claro
            celda_volver.alignment = Alignment(horizontal="center", vertical="center")

        else:
            print(f"  - Faltó hoja 'Analisis' en {archivo}")
            
        wb_origen.close()

    # --- DISEÑO FINAL HOJA GENERAL ---
    # 1. Fondo Rojo para la fila de títulos A1:U1
    relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
    fuente_blanca = Font(color="FFFFFF", bold=True)
    for col in range(1, 22): # De la A (1) a la U (21)
        celda = ws_general.cell(row=1, column=col)
        celda.fill = relleno_rojo
        celda.font = fuente_blanca
        celda.alignment = Alignment(horizontal="center", vertical="center")

    # 2. Centrar los valores de C2:U21
    alineacion_centro = Alignment(horizontal="center", vertical="center")
    for r in range(2, 22): # Filas 2 a 21
        for c in range(3, 22): # Columnas C (3) a U (21)
            ws_general.cell(row=r, column=c).alignment = alineacion_centro
            
    # Ajustar ligeramente el ancho de General para que se vea bien
    ws_general.column_dimensions['A'].width = 5
    ws_general.column_dimensions['B'].width = 25
    for c in range(3, 22):
        ws_general.column_dimensions[get_column_letter(c)].width = 15
    
    # Ajustar el texto en la fila 1 de las hoja General
    ajustar_texto_fila(ws_general, 1)

    # --- NUEVO: APLICAR GRILLA DELGADA (A1:U21) ---
    borde_delgado = Border(
        left=Side(style='thin', color='000000'),
        right=Side(style='thin', color='000000'),
        top=Side(style='thin', color='000000'),
        bottom=Side(style='thin', color='000000')
    )
    
    for r in range(1, 22):      # Filas de la 1 a la 21
        for c in range(1, 22):  # Columnas de la A(1) a la U(21)
            ws_general.cell(row=r, column=c).border = borde_delgado

    wb_nuevo.save(ARCHIVO_DESTINO)
    print(f"\n🚀 ¡Proceso completado! Archivo guardado en: {ARCHIVO_DESTINO}")

if __name__ == "__main__":
    consolidar_excels()

-> Procesando 'ANTONIO NARIÑO.xlsx' -> Destino: '15-Antonio Nariño'
-> Procesando 'BARRIOS UNIDOS.xlsx' -> Destino: '12-Barrios Unidos'
-> Procesando 'BOSA.xlsx' -> Destino: '7-Bosa'
-> Procesando 'CANDELARIA.xlsx' -> Destino: '18-La Candelaria'
-> Procesando 'CHAPINERO.xlsx' -> Destino: '2-Chapinero'
-> Procesando 'CIUDAD BOLIVAR.xlsx' -> Destino: '19-Ciudad Bolívar'
-> Procesando 'ENGATIVA.xlsx' -> Destino: '10-Engativá'
-> Procesando 'FONTIBON.xlsx' -> Destino: '9-Fontibón'
-> Procesando 'KENNEDY.xlsx' -> Destino: '8-Kennedy'
-> Procesando 'MARTIRES.xlsx' -> Destino: '14-Los Mártires'
-> Procesando 'PUENTE ARANDA.xlsx' -> Destino: '16-Puente Aranda'
-> Procesando 'RAFAEL URIBE URIBE.xlsx' -> Destino: '17-Rafael Uribe Uribe'
-> Procesando 'SAN CRISTOBAL.xlsx' -> Destino: '4-San Cristóbal'
-> Procesando 'SANTA FE.xlsx' -> Destino: '3-Santa Fe'
-> Procesando 'SUBA.xlsx' -> Destino: '11-Suba'
-> Procesando 'SUMAPAZ.xlsx' -> Destino: '20-Sumapaz'
-> Procesando 'TEUSAQUILLO.xlsx' -> Desti

#### Entrega 6/03/2026

Para procesar lso datos re crudos de Colombia compra eficiente. Si Gustavo envía el archivo excel solo en visualización en la web, se da en "File/Info/Open in Desktop" Ahi se abre en el pc y se guarda como Excel ".XSLS" no cmo por defecto que es "Excel 1997". Y de ahí se corre el primer script. <br>

Tener en cuenta que al correr el script se abre una ventana para seleccionar donde esta el excel y donde quiere que se guarden los excels de cada localidad

In [ ]:
import pandas as pd
import unicodedata
import re
import difflib
import os
import tkinter as tk
from tkinter import filedialog, messagebox

# ==========================================
# 1. FUNCIONES DE LIMPIEZA Y COMPARACIÓN
# ==========================================

def normalizar_texto(texto):
    """Limpia el texto: quita nulos, espacios, pasa a mayúsculas, quita tildes y artículos"""
    if pd.isna(texto):
        return ""
    # Convertir a texto, mayúsculas y quitar espacios a los lados
    texto = str(texto).upper().strip()
    # Quitar tildes/acentos
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    # Quitar artículos al inicio para igualar "LOS MARTIRES" con "MARTIRES"
    texto = re.sub(r'^(EL|LA|LOS|LAS)\s+', '', texto)
    return texto

def unificar_localidades(lista_localidades, similitud_minima=0.85):
    """Agrupa nombres similares bajo un mismo nombre 'canónico'"""
    nombres_canonicos = {}
    for loc in lista_localidades:
        if not loc: continue # Saltar vacíos
        
        # Buscar si ya existe un nombre muy parecido en nuestro diccionario
        coincidencias = difflib.get_close_matches(loc, nombres_canonicos.keys(), n=1, cutoff=similitud_minima)
        
        if coincidencias:
            nombres_canonicos[loc] = coincidencias[0]
        else:
            nombres_canonicos[loc] = loc
            
    return nombres_canonicos

# ==========================================
# 2. SELECCIÓN DE ARCHIVOS Y CARPETAS
# ==========================================

# Ocultar la ventana principal de Tkinter
root = tk.Tk()
root.withdraw()

print("Por favor, selecciona el archivo Excel de origen en la ventana emergente...")
archivo_origen = filedialog.askopenfilename(
    title="Selecciona el archivo Excel (.xlsx o .xls)",
    filetypes=[("Archivos de Excel", "*.xlsx *.xls *")]
)

if not archivo_origen:
    print("No se seleccionó ningún archivo. Cancelando proceso.")
    exit()

print("Por favor, selecciona la carpeta donde deseas guardar los nuevos archivos...")
carpeta_destino = filedialog.askdirectory(title="Selecciona la carpeta de destino")

if not carpeta_destino:
    print("No se seleccionó ninguna carpeta. Cancelando proceso.")
    exit()

# ==========================================
# 3. LECTURA Y PROCESAMIENTO DEL EXCEL
# ==========================================

try:
    print(f"\nLeyendo el archivo: {archivo_origen}")
    xls = pd.ExcelFile(archivo_origen)
    nombres_hojas = xls.sheet_names
    
    # Buscar hojas independientemente de mayúsculas/minúsculas
    hoja_sipse = next((h for h in nombres_hojas if h.upper() == "SIPSE"), None)
    hoja_secop = next((h for h in nombres_hojas if h.upper() == "SECOP II"), None)
    
    if not hoja_sipse or not hoja_secop:
        raise ValueError("El archivo no contiene las hojas 'SIPSE' y/o 'SECOP II'. Verifica el archivo.")

    # Leer las hojas
    df_sipse = pd.read_excel(xls, sheet_name=hoja_sipse)
    df_secop = pd.read_excel(xls, sheet_name=hoja_secop)
    
    # Identificar columnas (SIPSE: BJ es la columna 62, índice 61 | SECOP II: A es índice 0)
    # Validamos que SIPSE tenga suficientes columnas
    if len(df_sipse.columns) <= 61:
        raise ValueError("La hoja SIPSE no tiene suficientes columnas para llegar a la columna BJ.")
        
    col_sipse_nombre = df_sipse.columns[61] 
    col_secop_nombre = df_secop.columns[0]
    
    print("Normalizando nombres de localidades...")
    # Crear columnas auxiliares limpias
    df_sipse['Loc_Limpia'] = df_sipse[col_sipse_nombre].apply(normalizar_texto)
    df_secop['Loc_Limpia'] = df_secop[col_secop_nombre].apply(normalizar_texto)
    
    # Obtener todas las localidades únicas de ambas hojas
    todas_localidades = list(set(df_sipse['Loc_Limpia'].unique()) | set(df_secop['Loc_Limpia'].unique()))
    
    # Unificar nombres similares (Ej: MARTIRES y LOS MARTIRES)
    mapa_canonico = unificar_localidades(todas_localidades)
    
    # Aplicar el nombre unificado a los DataFrames
    df_sipse['Loc_Final'] = df_sipse['Loc_Limpia'].map(mapa_canonico)
    df_secop['Loc_Final'] = df_secop['Loc_Limpia'].map(mapa_canonico)
    
    # Obtener la lista final de localidades a exportar
    localidades_finales = set(mapa_canonico.values())
    
    print(f"Se detectaron {len(localidades_finales)} localidades únicas. Generando archivos...")
    
    # ==========================================
    # 4. CREACIÓN DE LOS ARCHIVOS .XLSX
    # ==========================================
    
    archivos_creados = 0
    for loc in localidades_finales:
        if not loc: continue # Evitar crear un archivo para localidades vacías
        
        # Filtrar datos (y eliminar las columnas auxiliares de limpieza que creamos)
        filtro_sipse = df_sipse[df_sipse['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        filtro_secop = df_secop[df_secop['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        
        # Solo crear el archivo si hay datos en alguna de las dos hojas
        if not filtro_sipse.empty or not filtro_secop.empty:
            # Reemplazar caracteres no válidos para nombres de archivos
            nombre_archivo_seguro = re.sub(r'[\\/*?:"<>|]', "", loc)
            ruta_salida = os.path.join(carpeta_destino, f"{nombre_archivo_seguro}.xlsx")
            
            with pd.ExcelWriter(ruta_salida, engine='openpyxl') as writer:
                filtro_sipse.to_excel(writer, sheet_name="SIPSE", index=False)
                filtro_secop.to_excel(writer, sheet_name="SECOP II", index=False)
                
            archivos_creados += 1
            print(f" -> Creado: {nombre_archivo_seguro}.xlsx")

    print(f"\n¡Proceso Finalizado Exitosamente! Se crearon {archivos_creados} archivos en: {carpeta_destino}")
    messagebox.showinfo("Éxito", f"Se generaron {archivos_creados} archivos correctamente.")

except Exception as e:
    print(f"\nOcurrió un error: {e}")
    messagebox.showerror("Error", f"Ocurrió un error procesando el archivo:\n{e}")

Por favor, selecciona el archivo Excel de origen en la ventana emergente...
Por favor, selecciona la carpeta donde deseas guardar los nuevos archivos...

Leyendo el archivo: C:/Users/Pc/Desktop/Meta 7 - FDL 01 MARZO/Datos Virgenes/SECOP_II_-_Contratos_Electrónicos_20260309.xls
Normalizando nombres de localidades...
Se detectaron 22 localidades únicas. Generando archivos...
 -> Creado: SANTA FE.xlsx
 -> Creado: MARTIRES.xlsx
 -> Creado: TUNJUELITO.xlsx
 -> Creado: SUMAPAZ.xlsx
 -> Creado: CANDELARIA.xlsx
 -> Creado: KENNEDY.xlsx
 -> Creado: SAN CRISTOBAL.xlsx
 -> Creado: RAFAEL URIBE URIBE.xlsx
 -> Creado: SDG - NIVEL CENTRAL.xlsx
 -> Creado: PUENTE ARANDA.xlsx
 -> Creado: ANTONIO NARINO.xlsx
 -> Creado: TEUSAQUILLO.xlsx
 -> Creado: ENGATIVA.xlsx
 -> Creado: BARRIOS UNIDOS.xlsx
 -> Creado: USAQUEN.xlsx
 -> Creado: CIUDAD BOLIVAR.xlsx
 -> Creado: SUBA.xlsx
 -> Creado: RAFAEL URIBE.xlsx
 -> Creado: USME.xlsx
 -> Creado: FONTIBON.xlsx
 -> Creado: BOSA.xlsx
 -> Creado: CHAPINERO.xlsx

¡Proc

In [1]:
import os
import shutil
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.formatting.rule import CellIsRule
from collections import defaultdict
import re
import pandas as pd

# --- CONFIGURACIÓN DE CARPETAS ---
#CARPETA_ORIGEN = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO'
#CARPETA_DESTINO = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\DiagnóstivoV2_1'
CARPETA_ORIGEN = r'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 MARZO' #Dell
CARPETA_DESTINO = r"C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 MARZO\\Diagnóstico" #Dell
#CARPETA_ORIGEN = r'D:\\Trabajo\\2026\\SecretariaDeGobierno\\BasesDeDatosSipseSecop\\2026_03_06\\DatosPartidos' #Predator
#CARPETA_DESTINO = r"D:\\Trabajo\\2026\\SecretariaDeGobierno\\BasesDeDatosSipseSecop\\2026_03_06\\DatosPartidos\\Diagnóstico" #PRedator

# --- RANGOS A OCULTAR (CORREGIDOS PARA USAR HIDDEN=TRUE) ---
# Nota: He ajustado 'A-AH' a 'Z-AH' en SECOP para NO ocultar las columnas L, M, T, X, Y que necesitas ver.
RANGOS_OCULTAR_SECOP = ['A-K', 'N-S', 'U-W', 'Z-AH', 'AJ-BR', 'BT-CI']
RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']

# --- DEFINICIÓN DE FUNCIONES ---

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def ocultar_columnas_por_lista(ws, lista_rangos):
    """
    Recibe una lista de cadenas como ['A-K', 'M'] y establece la propiedad HIDDEN = True.
    """
    for item in lista_rangos:
        item = item.upper().strip()
        
        # Caso 1: Es un rango (ej: "A-K")
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio)
            idx_fin = column_index_from_string(fin)
            
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                # Esta es la forma correcta de ocultar en Excel
                ws.column_dimensions[letra].hidden = True 
                
        # Caso 2: Es una columna única (ej: "A")
        else:
            ws.column_dimensions[item].hidden = True

def aplicar_estilo_ferrocarril(ws):
    """
    Encabezados Azul Oscuro + Letra Blanca.
    Filas con patrón intercalado.
    """
    # 1. Encabezados
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    
    for cell in ws[1]:
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas (Patrón Ferrocarril)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    max_col = ws.max_column
    
    for i_fila, row in enumerate(ws.iter_rows(min_row=2, max_col=max_col), start=0):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF":
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for cell in row:
                cell.fill = relleno

def dejar_columna_blanca(ws, indice_columna):
    relleno_blanco = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    
    # Recorremos todas las celdas de la columna indicada
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=indice_columna, max_col=indice_columna):
        for cell in row:
            cell.fill = relleno_blanco

def aplicar_formato_condicional_alertas(ws, col_inicio_eval):
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    max_row = ws.max_row
    
    reglas = [
        (0, 'equal', '"Numero contrato NO coincide"'),
        (2, 'equal', '"Cedula NO coincide"'),
        (3, 'greaterThan', '1'),
        (4, 'greaterThan', '1'),
        (5, 'equal', '"No coinciden fechas"'),
        (6, 'equal', '"No coinciden fechas"'),
        (7, 'equal', '"No coinciden fechas"'),
        (8, 'equal', '"Duracion NO coincide"'),
        (9, 'equal', '"Valor NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        rango = f"{letra_col}2:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

def normalizar_numero_contrato(valor):
    if valor is None: return 0
    texto = str(valor).strip()
    # Busca el primer grupo de números en la cadena
    match = re.search(r'\d+', texto)
    if match:
        return int(match.group())
    return 0

# --- LÓGICA PRINCIPAL ---
def procesar_archivos():
    if not os.path.exists(CARPETA_DESTINO):
        os.makedirs(CARPETA_DESTINO)

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('.xlsx', '.xls'))]
    print(f"Se encontraron {len(archivos)} archivos de Excel.")

    for archivo in archivos:
        ruta_orig = os.path.join(CARPETA_ORIGEN, archivo)
        
        # Definimos la ruta de destino siempre con extensión .xlsx
        nombre_base = os.path.splitext(archivo)[0]
        ruta_dest = os.path.join(CARPETA_DESTINO, nombre_base + ".xlsx")
        
        print(f"Procesando: {archivo}...")

        # --- PASO DE CONVERSIÓN SI ES .XLS ---
        if archivo.lower().endswith('.xls'):
            try:
                # Leemos el .xls y lo guardamos inmediatamente como .xlsx
                # sheet_name=None lee todas las pestañas
                conversion = pd.read_excel(ruta_orig, sheet_name=None, engine='xlrd')
                with pd.ExcelWriter(ruta_dest, engine='openpyxl') as writer:
                    for nombre_hoja, datos in conversion.items():
                        datos.to_excel(writer, sheet_name=nombre_hoja, index=False)
                print(f"  -> Convertido de .xls a .xlsx con éxito.")
            except Exception as e:
                print(f"  Error convirtiendo {archivo}: {e}")
                continue
        else:
            # Si ya es .xlsx, solo lo copiamos como hacías antes
            shutil.copy2(ruta_orig, ruta_dest)
        # -------------------------------------

        try:
            # Ahora openpyxl siempre abrirá un archivo .xlsx (el copiado o el convertido)
            wb = openpyxl.load_workbook(ruta_dest)
        except Exception as e:
            print(f"   Error al abrir el archivo con openpyxl: {e}")
            continue

        # ==========================================
        # PARTE 1: Procesar SECOP II (Crear aux1)
        # ==========================================
        nombre_real_secop = next(n for n in wb.sheetnames if n.lower() == 'secop ii')
        ws_origen = wb[nombre_real_secop]
        
        if "SECOP II aux1" in wb.sheetnames:
            del wb["SECOP II aux1"]
            
        ws_secop_aux = wb.copy_worksheet(ws_origen)
        ws_secop_aux.title = "SECOP II aux1"

        filas = []
        for row in ws_secop_aux.iter_rows(min_row=2, values_only=True):
            filas.append(list(row))
        
        filas_ordenadas = sorted(filas, key=lambda x: normalizar_numero_contrato(x[11] if len(x) > 11 else 0))

        for row in ws_secop_aux.iter_rows(min_row=2):
            for cell in row:
                cell.value = None

        for i_fila, datos_fila in enumerate(filas_ordenadas, start=2):
            for i_col, valor in enumerate(datos_fila, start=1):
                ws_secop_aux.cell(row=i_fila, column=i_col, value=valor)

        max_col_secop = ws_secop_aux.max_column
        ws_secop_aux.cell(row=1, column=max_col_secop + 1, value="Columna1")
        col_L = get_column_letter(12)
        
        for i in range(2, ws_secop_aux.max_row + 1):
            #formula = f'''=MID({col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), FIND("-", {col_L}{i}) - MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0))'''
            formula = (
                f'=IFERROR(MID({col_L}{i}, '
                f'MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), '
                f'MIN(IFERROR(FIND(" ", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999), '
                f'IFERROR(FIND("-", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999)) '
                f'- MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), "")'
            )
            ws_secop_aux.cell(row=i, column=max_col_secop + 1, value=formula)

        # ESTILOS SECOP
        print("  -> Aplicando estilos y ocultando columnas en SECOP II aux1...")
        aplicar_estilo_ferrocarril(ws_secop_aux)
        ocultar_columnas_por_lista(ws_secop_aux, RANGOS_OCULTAR_SECOP)

        # ==========================================
        # PARTE 2 y 3: ALINEACIÓN EN 'ANALISIS'
        # ==========================================
        print("  -> Generando hoja 'Analisis'...")

        nombre_real_sipse = next(n for n in wb.sheetnames if n.lower() == 'sipse')
        ws_sipse = wb[nombre_real_sipse]
        
        if 'Analisis' in wb.sheetnames:
            del wb['Analisis']
        ws_analisis = wb.create_sheet("Analisis")
        
        dict_sipse = defaultdict(list)
        dict_secop = defaultdict(list)
        
        filas_sipse = list(ws_sipse.values)
        header_sipse = filas_sipse[0] if filas_sipse else []
        for fila in filas_sipse[1:]:
            contrato = normalizar_numero_contrato(fila[3] if len(fila) > 3 else 0)
            dict_sipse[contrato].append(fila)

        filas_secop = list(ws_secop_aux.values)
        header_secop = filas_secop[0] if filas_secop else []
        for fila in filas_secop[1:]:
            contrato = normalizar_numero_contrato(fila[11] if len(fila) > 11 else 0)
            dict_secop[contrato].append(fila)

        ws_analisis.append(header_sipse) 
        col_inicio_secop = column_index_from_string("BM")
        for i, val in enumerate(header_secop):
            ws_analisis.cell(row=1, column=col_inicio_secop + i, value=val)

        todos_contratos = sorted(list(set(dict_sipse.keys()) | set(dict_secop.keys())))
        current_row = 2 
        
        for contrato in todos_contratos:
            filas_de_sipse = dict_sipse.get(contrato, [])
            filas_de_secop = dict_secop.get(contrato, [])
            max_filas = max(len(filas_de_sipse), len(filas_de_secop))
            
            for i in range(max_filas):
                if i < len(filas_de_sipse):
                    datos = filas_de_sipse[i]
                    for c_idx, valor in enumerate(datos, start=1):
                        ws_analisis.cell(row=current_row, column=c_idx, value=valor)
                
                if i < len(filas_de_secop):
                    datos = filas_de_secop[i]
                    for c_idx, valor in enumerate(datos):
                        col_excel = col_inicio_secop + c_idx
                        if c_idx == len(datos) - 1 and str(valor).startswith('='):
                             #nueva_formula = f'''=MID(BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), FIND("-", BX{current_row}) - MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0))'''
                             nueva_formula = (
                                f'=IFERROR(MID(BX{current_row}, '
                                f'MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), '
                                f'MIN(IFERROR(FIND(" ", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999), '
                                f'IFERROR(FIND("-", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999)) '
                                f'- MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), "")'
                            )
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula)
                        else:
                            ws_analisis.cell(row=current_row, column=col_excel, value=valor)
                current_row += 1

        # ==========================================
        # PARTE 4: Columnas de Evaluación
        # ==========================================
        print("  -> Agregando columnas de evaluación y estilos...")
        
        ultima_col_actual = ws_analisis.max_column
        col_inicio_eval = ultima_col_actual + 1
        ultima_fila_analisis = ws_analisis.max_row
        
        headers_nuevos = [
            "Ev. Nro contrato", "Aux Nro contrato", "Ev. Cédula", "Repeticiones contrato SIPSE", 
            "Repeticiones contrato SECOP", "Fecha suscripción", "Fecha inicio", "Fecha fin", "Duración contratos", "Valor contratos"
        ]

        for idx, titulo in enumerate(headers_nuevos):
            ws_analisis.cell(row=1, column=col_inicio_eval + idx, value=titulo)

        for i in range(2, ultima_fila_analisis + 1):

            # 🔹 AUX NRO CONTRATO (EX) → llave única
            f_aux = f'=IF(D{i}<>"",D{i},EV{i})'
            ws_analisis.cell(row=i, column=col_inicio_eval + 1, value=f_aux)

            # 🔹 ESTADO CONTRATO
            f1 = f'''=IF(COUNTIF($EX$2:EX{i},EX{i})>1,"",IF(AND(COUNTIF($D$2:$D${ultima_fila_analisis},EX{i})>0,COUNTIF($EV$2:$EV${ultima_fila_analisis},EX{i})>0),"Numero contrato coincide",IF(COUNTIF($D$2:$D${ultima_fila_analisis},EX{i})>0,"Contrato no existe en SECOP",IF(COUNTIF($EV$2:$EV${ultima_fila_analisis},EX{i})>0,"Contrato no existe en SIPSE","No existe"))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval, value=f1)

            # 🔹 CÉDULA
            f2 = f'=IF(AND(BY{i}="",M{i}=""),"",IF(OR(BY{i}="",M{i}=""),"",IF(BY{i}="-","Sin cedula en SECOP",IF(M{i}="-","Sin cedula en SIPSE",IF(VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(CJ{i}),"CC",""),"NIT",""),"TI",""))=VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(M{i}),"CC",""),"NIT",""),"TI","")),"Cedula coincide","Cedula NO coincide")))))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 2, value=f2)

            # 🔹 REPETICIONES SIPSE
            f3 = f'=IF(EW{i}="","",COUNTIF(D$2:D${ultima_fila_analisis},D{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 3, value=f3)

            # 🔹 REPETICIONES SECOP
            f4 = f'=IF(EW{i}="","",COUNTIF(EV$2:EV${ultima_fila_analisis},EV{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 4, value=f4)

            # 🔹 FECHA SUSCRIPCION
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CE{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CE{i}), DAY(CE{i}), MONTH(CE{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f5 = f'''=IF(AND(H{i}="",CE{i}=""),"",IF(AND(H{i}="-",CE{i}="-"),"Ambos sin fechas",IF(H{i}="-","sin fecha SIPSE",IF(CE{i}="-","sin fecha SECOP",IF(OR(H{i}="",CE{i}=""),"",IF(INT(VALUE(SUBSTITUTE(H{i},"-","/"))) =INT(IF(ISNUMBER(CE{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 5, value=f5)

            # 🔹 FECHA ACTA DE INICIO
            cf_clean = f'SUBSTITUTE(SUBSTITUTE(CF{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CF{i}), DAY(CF{i}), MONTH(CF{i}))'
            parte_es_texto = f'DATE(RIGHT({cf_clean}, 4), LEFT({cf_clean}, FIND("/", {cf_clean}) - 1), MID({cf_clean}, FIND("/", {cf_clean}) + 1, FIND("/", {cf_clean}, FIND("/", {cf_clean}) + 1) - FIND("/", {cf_clean}) - 1))'

            f6 = f'''=IF(AND(I{i}="",CF{i}=""),"",IF(AND(I{i}="-",CF{i}="-"),"Ambos sin fechas",IF(I{i}="-","sin fecha SIPSE",IF(CF{i}="-","sin fecha SECOP",IF(OR(I{i}="",CF{i}=""),"",IF(INT(VALUE(SUBSTITUTE(I{i},"-","/"))) =INT(IF(ISNUMBER(CF{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 6, value=f6)

            # 🔹 FECHA FIN
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CG{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CG{i}), DAY(CG{i}), MONTH(CG{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f7 = f'''=IF(AND(J{i}="",CG{i}=""),"",IF(AND(J{i}="-",CG{i}="-"),"Ambos sin fechas",IF(J{i}="-","sin fecha SIPSE",IF(CG{i}="-","sin fecha SECOP",IF(OR(J{i}="",CG{i}=""),"",IF(INT(VALUE(SUBSTITUTE(J{i},"-","/"))) =INT(IF(ISNUMBER(CG{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 7, value=f7)

            # 🔹 DURACIÓN
            f8 = f'''=IF(AND(R{i}="",EE{i}=""),"",IF(AND(R{i}="-",EE{i}="-"),"Ambos sin duracion",IF(R{i}="-","sin duracion en SECOP",IF(EE{i}="-","sin duracion en SIPSE",IF(OR(R{i}="",EE{i}=""),"",IF(TRIM(UPPER(R{i}))=TRIM(UPPER(EE{i})),"Duracion coincide","Duracion NO coincide"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 8, value=f8)

            # 🔹 VALOR CONTRATO (Comparación Robusta)
            f9 = f'''=IF(AND(P{i}="",CU{i}=""),"",
                IF(AND(P{i}="-",CU{i}="-"),"Ambos sin valor",
                IF(P{i}="-","Sin valor en SECOP",
                IF(CU{i}="-","Sin valor en SIPSE",
                IF(OR(P{i}="",CU{i}=""),"",
                IF(P{i}=CU{i},"Valor coincide","Valor NO coincide"))))))'''

            ws_analisis.cell(row=i, column=col_inicio_eval + 9, value=f9)

        # --- ESTILOS ANALISIS ---
        print("  -> Aplicando estilos y ocultando columnas en Analisis...")
        aplicar_estilo_ferrocarril(ws_analisis)
        aplicar_formato_condicional_alertas(ws_analisis, col_inicio_eval)
        
        # Ocultar columnas hoja analisis
        ocultar_columnas_por_lista(ws_analisis, RANGOS_OCULTAR_ANALISIS)

        # ==========================================
        # PARTE 5: Crear HOJA RESULTADOS
        # ==========================================
        print("  -> Creando hoja 'Resultados'...")
        if 'Resultados' in wb.sheetnames:
            del wb['Resultados']
        ws_res = wb.create_sheet("Resultados")
        nombre_limpio = os.path.splitext(archivo)[0]
        ws_res['B1'] = f"Métricas alcaldía local {nombre_limpio}"
        
        letra_ev_nro = get_column_letter(col_inicio_eval)       
        letra_ev_ced = get_column_letter(col_inicio_eval + 2)   
        letra_rep_sipse = get_column_letter(col_inicio_eval + 3)
        letra_rep_secop = get_column_letter(col_inicio_eval + 4)
        letra_fecha_suscripcion = get_column_letter(col_inicio_eval + 5)
        letra_fecha_inicio = get_column_letter(col_inicio_eval + 6)
        letra_fecha_fin = get_column_letter(col_inicio_eval + 7)
        letra_duracion = get_column_letter(col_inicio_eval + 8)
        letra_valor = get_column_letter(col_inicio_eval + 9)

        metricas = [
            ("Nro de contratos registrados en SIPSE",'=SUMPRODUCT((Analisis!A2:A10000<>"")/COUNTIF(Analisis!A2:A10000,Analisis!A2:A10000&""))'),
            ("Nro de contratos registrados SECOP",'=SUMPRODUCT((Analisis!EV2:EV10000<>"")/COUNTIF(Analisis!EV2:EV10000,Analisis!EV2:EV10000&""))'),
            ("Nro de contratos registrados en SECOP y SIPSE con numero de contrato igual", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")"),
            ("Nro de contratos solo en SECOP y no en SIPSE", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos solo en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos diferentes en SIPSE-SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato NO coincide\")"),
            ("Nro de contratos Cédula diferente", f"=COUNTIF('Analisis'!{letra_ev_ced}:{letra_ev_ced}, \"Cedula NO coincide\")"),
            ("Nro de contratos dobles en SIPSE", f"=COUNTIF('Analisis'!{letra_rep_sipse}:{letra_rep_sipse}, 2)"),
            ("Nro de contratos dobles en SECOP", f"=COUNTIF('Analisis'!{letra_rep_secop}:{letra_rep_secop}, 2)"),
            ("Fecha de acta de suscriccpión Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"Coincide fecha\")"),
            ("Fecha de acta de suscriccpión no Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"No coinciden fechas\")"),
            ("Fecha de acta de inicio Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Coincide fecha\")"),
            ("Fecha de acta de inicio no coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"No coinciden fechas\")"),
            ("Fecha de Fin conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"Coincide fecha\")"),
            ("Fecha de Fin no conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"No coinciden fechas\")"),
            ("Contratos que coinciden en Duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion Coincide\")"),
            ("Contratos que tienen diferente duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion NO coincide\")"),
            ("Contratos que tienen Fecha de inicio en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Sin fecha SECOP\")"),
            ("Datos en la matriz de seguimiento", f"=C2+C3"),
            ("Valor contrato coinciden", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor coincide\")"),
            ("Valor contrato no coincidenn", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor NO coincide\")")
        ]


        for idx, (texto, formula) in enumerate(metricas, start=2):
            ws_res.cell(row=idx, column=1, value=idx-1)
            ws_res.cell(row=idx, column=2, value=texto)
            ws_res.cell(row=idx, column=3, value=formula)
        
        # Pintar B2 de amarillo
        fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
        ws_res['B1'].fill = fill_amarillo
        ws_res.column_dimensions['B'].width = 100
        
        ws_analisis.column_dimensions['EW'].width = 25
        ws_analisis.column_dimensions['EX'].width = 20
        ws_analisis.column_dimensions['EY'].width = 20
        ws_analisis.column_dimensions['EZ'].width = 15
        ws_analisis.column_dimensions['FA'].width = 15
        ws_analisis.column_dimensions['FB'].width = 15
        ws_analisis.column_dimensions['FC'].width = 20
        ws_analisis.column_dimensions['FD'].width = 20
        ws_analisis.column_dimensions['FE'].width = 18
        ws_analisis.column_dimensions['FF'].width = 14

        dejar_columna_blanca(ws_analisis, 64)# columna 64 es la columna "BL"

        #Ajustar el texto en la fila 1 de las hojas aux y Analisis
        ajustar_texto_fila(ws_secop_aux, 1)
        ajustar_texto_fila(ws_analisis, 1)

        ##--- RELLENAR CELDAS VACÍAS CON "?" EN SIPSE Y SECOP ---
        #for row in ws_analisis.iter_rows(min_row=2, max_row=ultima_fila_analisis, min_col=1, max_col=col_inicio_eval - 1):
        #    for cell in row:
        #        # Comprueba si la celda está realmente vacía (None) o si es un texto vacío ("")
        #        if cell.value is None or str(cell.value).strip() == "":
        #            cell.value = "?"

        wb.save(ruta_dest)
        print("  -> Procesado exitosamente.")

if __name__ == "__main__":
    procesar_archivos()

Se encontraron 20 archivos de Excel.
Procesando: ANTONIO NARINO.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Procesado exitosamente.
Procesando: BARRIOS UNIDOS.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Procesado exitosamente.
Procesando: BOSA.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Procesado exitosamente.
Procesando: CANDELARIA.xlsx...
  -> Aplicando estilos y

In [2]:
import os
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
import re
import unicodedata

# ================= CONFIGURACIÓN =================
#CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1' 
#ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1\Consolidado\Consolidado_Localidades.xlsx"
CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 01 MARZO\Diagnóstico' # Dell
ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 01 MARZO\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx" #Dell
#CARPETA_ORIGEN = r'D:\Trabajo\2026\SecretariaDeGobierno\BasesDeDatosSipseSecop\2026_03_06\DatosPartidos\Diagnóstico' # PRedator
#ARCHIVO_DESTINO = r"D:\Trabajo\2026\SecretariaDeGobierno\BasesDeDatosSipseSecop\2026_03_06\DatosPartidos\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx" #Predator

nombres_hojas = [
    "1-Usaquén", "2-Chapinero", "3-Santa Fe", "4-San Cristóbal",
    "5-Usme", "6-Tunjuelito", "7-Bosa", "8-Kennedy", "9-Fontibón",
    "10-Engativá", "11-Suba", "12-Barrios Unidos", "13-Teusaquillo",
    "14-Los Mártires", "15-Antonio Nariño", "16-Puente Aranda",
    "17-Rafael Uribe Uribe", "18-La Candelaria", "19-Ciudad Bolívar", "20-Sumapaz"
]

RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']
# Aquí puedes añadir todas las columnas a las que quieras fijarles un ancho específico
ANCHOS_COLUMNAS = {
    'EZ': 25,
    'FA': 20,
    'FB': 20,
    'FC': 15,
    'FD': 15,
    'FE': 15,
    'FF': 20,
    'FG': 20,
    'FH': 18,
    'FI': 14
}

# ================= FUNCIONES AUXILIARES =================
def normalizar_texto(texto):
    texto = texto.lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^\d+[-_ ]?', '', texto)
    texto = re.sub(r'\b(la|el|los|las)\b', '', texto).strip()
    return texto

def encontrar_hoja_destino(nombre_archivo, hojas_destino):
    archivo_norm = normalizar_texto(nombre_archivo)
    for hoja in hojas_destino:
        if normalizar_texto(hoja) in archivo_norm:
            return hoja
    return None

def ocultar_columnas_por_lista(ws, lista_rangos, offset=3):
    for item in lista_rangos:
        item = item.upper().strip()
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio) + offset
            idx_fin = column_index_from_string(fin) + offset
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                ws.column_dimensions[letra].hidden = True 
        else:
            idx = column_index_from_string(item) + offset
            letra = get_column_letter(idx)
            ws.column_dimensions[letra].hidden = True

def adaptar_formulas_resultados(formula):
    pattern = re.compile(r"'?Analisis'?!([A-Za-z]+)(\d*)(?::([A-Za-z]+)(\d*))?")
    def replacer(match):
        col1, row1, col2, row2 = match.groups()
        nueva_col1 = get_column_letter(column_index_from_string(col1) + 3)
        nueva_row1 = str(int(row1) + 21) if row1 else ""
        resultado = f"{nueva_col1}{nueva_row1}"
        if col2:
            nueva_col2 = get_column_letter(column_index_from_string(col2) + 3)
            nueva_row2 = str(int(row2) + 21) if row2 else ""
            resultado += f":{nueva_col2}{nueva_row2}"
        return resultado
    return pattern.sub(replacer, formula)

def desplazar_formulas_analisis(formula):
    formula = re.sub(r"'?Analisis'?!", "", formula, flags=re.IGNORECASE)
    pattern = re.compile(r'(\$?)([A-Za-z]{1,3})(\$?)(\d+)(?!\s*\()')
    def replacer(match):
        lock_col, col_str, lock_row, row_str = match.groups()
        nueva_col = get_column_letter(column_index_from_string(col_str) + 3)
        nueva_fila = str(int(row_str) + 21)
        return f"{lock_col}{nueva_col}{lock_row}{nueva_fila}"
    return pattern.sub(replacer, formula)

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# ================= FUNCIONES DE DISEÑO =================
def aplicar_estilo_ferrocarril(ws, start_row, max_row):
    """Aplica formato estilo ferrocarril solo a columnas E:U y CA:FH desde start_row."""
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    
    # Rango 1: E (5) a U (21) | Rango 2: CA (79) a FI (165)
    def es_columna_objetivo(c):
        return (5 <= c <= 21) or (79 <= c <= 165)

    # 1. Encabezados (Fila 22)
    for col_idx in range(1, ws.max_column + 1):
        if es_columna_objetivo(col_idx):
            cell = ws.cell(row=start_row, column=col_idx)
            cell.fill = fill_header
            cell.font = font_header
            cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas de Datos (Patrón Ferrocarril)
    for i_fila, r_idx in enumerate(range(start_row + 1, max_row + 1)):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF": # Evitar procesar blanco para ahorrar memoria
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for col_idx in range(1, ws.max_column + 1):
                if es_columna_objetivo(col_idx):
                    ws.cell(row=r_idx, column=col_idx).fill = relleno

def aplicar_formato_condicional_alertas(ws, start_row, max_row):
    """Aplica reglas condicionales desde la columna EZ."""
    col_inicio_eval = column_index_from_string('EZ')
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    
    reglas = [
        (0, 'equal', '"Numero contrato NO coincide"'),
        (2, 'equal', '"Cedula NO coincide"'),
        (3, 'greaterThan', '1'),
        (4, 'greaterThan', '1'),
        (5, 'equal', '"No coinciden fechas"'),
        (6, 'equal', '"No coinciden fechas"'),
        (7, 'equal', '"No coinciden fechas"'),
        (8, 'equal', '"Duracion NO coincide"'),
        (9, 'equal', '"Valor NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        # Se aplica desde la fila siguiente al encabezado (23 en adelante)
        rango = f"{letra_col}{start_row + 1}:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

# ================= PROGRAMA PRINCIPAL =================
def consolidar_excels():
    wb_nuevo = openpyxl.Workbook()
    
    # 1. Configurar Hoja General
    ws_general = wb_nuevo.active
    ws_general.title = "General"
    ws_general.cell(row=1, column=1, value="Nro").font = Font(bold=True)
    ws_general.cell(row=1, column=2, value="Alcaldía local").font = Font(bold=True)
    
    # Pre-llenar las localidades en General con Hipervínculos
    for i, hoja in enumerate(nombres_hojas):
        num, nombre = hoja.split("-", 1)
        row_idx = i + 2
        ws_general.cell(row=row_idx, column=1, value=int(num))
        
        celda_loc = ws_general.cell(row=row_idx, column=2, value=nombre.strip())
        celda_loc.hyperlink = f"#'{hoja}'!A1"
        celda_loc.font = Font(color="0563C1", underline="single") 

    for nombre in nombres_hojas:
        wb_nuevo.create_sheet(title=nombre)

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith('.xlsx') and not f.startswith('~')]
    metricas_extraidas = False 

    for archivo in archivos:
        hoja_destino_nombre = encontrar_hoja_destino(archivo, nombres_hojas)
        
        if not hoja_destino_nombre:
            continue
            
        print(f"-> Procesando '{archivo}' -> Destino: '{hoja_destino_nombre}'")
        
        ruta_completa = os.path.join(CARPETA_ORIGEN, archivo)
        wb_origen = openpyxl.load_workbook(ruta_completa)
        ws_destino = wb_nuevo[hoja_destino_nombre]

        # --- TAREA A: Copiar de 'Resultados' ---
        if "Resultados" in wb_origen.sheetnames:
            ws_resultados = wb_origen["Resultados"]
            row_gen = nombres_hojas.index(hoja_destino_nombre) + 2
            col_gen = 3 
            
            for r_idx in range(2, 21): 
                titulo_metrica = ws_resultados.cell(row=r_idx, column=2).value 
                
                if not metricas_extraidas:
                    ws_general.cell(row=1, column=col_gen, value=titulo_metrica).font = Font(bold=True)
                
                ws_general.cell(row=row_gen, column=col_gen, value=f"='{hoja_destino_nombre}'!C{r_idx}")
                col_gen += 1
                
            metricas_extraidas = True 
            
            for row in ws_resultados['A1':'C20']:
                for cell in row:
                    dest_coord = cell.coordinate
                    valor = cell.value
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = adaptar_formulas_resultados(valor)
                    else:
                        ws_destino[dest_coord].value = valor
        else:
            print(f"  - Faltó hoja 'Resultados' en {archivo}")

        # --- TAREA B: Copiar de 'Analisis' a la fila 22 ---
        if "Analisis" in wb_origen.sheetnames:
            ws_analisis = wb_origen["Analisis"]
            ultima_fila = ws_analisis.max_row
            max_col_origen = min(ws_analisis.max_column, column_index_from_string('FH'))
            
            ws_destino.merge_cells("D21:BN21")
            celda_sipse = ws_destino["D21"]
            celda_sipse.value = "SIPSE"
            celda_sipse.fill = PatternFill(start_color="0070C0", end_color="0070C0", fill_type="solid") 
            celda_sipse.font = Font(color="FFFFFF", bold=True)
            celda_sipse.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells("BP21:EH21")
            celda_secop = ws_destino["BP21"]
            celda_secop.value = "SECOP"
            celda_secop.fill = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid") 
            celda_secop.font = Font(color="FFFFFF", bold=True)
            celda_secop.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells("EI21:FI21")
            celda_comparacion = ws_destino["EI21"]
            celda_comparacion.value = "COMPARACIÓN"
            celda_comparacion.fill = PatternFill(start_color="7e350e", end_color="7e350e", fill_type="solid") 
            celda_comparacion.font = Font(color="FFFFFF", bold=True)
            celda_comparacion.alignment = Alignment(horizontal="center", vertical="center")

            # Pegar los datos desde la fila 22
            for r_idx, row_origen in enumerate(ws_analisis.iter_rows(min_row=1, max_row=ultima_fila, min_col=1, max_col=max_col_origen), start=22):
                for c_idx, cell in enumerate(row_origen, start=4):
                    dest_coord = f"{get_column_letter(c_idx)}{r_idx}"
                    valor = cell.value
                    
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = desplazar_formulas_analisis(valor)
                    else:
                        ws_destino[dest_coord].value = valor
            
            # 1. Ocultar columnas
            ocultar_columnas_por_lista(ws_destino, RANGOS_OCULTAR_ANALISIS, offset=3)
            # 2. Aplicar anchos de columnas
            for letra_col, ancho in ANCHOS_COLUMNAS.items():
                ws_destino.column_dimensions[letra_col].width = ancho
            # 2.1 Ajustar el texto en la fila 22 de las hojas de cada localidad
            ajustar_texto_fila(ws_destino, 22)
            # 3. Aplicar estilo Ferrocarril (La tabla empieza en la 22)
            aplicar_estilo_ferrocarril(ws_destino, start_row=22, max_row=ultima_fila + 21)
            # 4. Aplicar alertas condicionales
            aplicar_formato_condicional_alertas(ws_destino, start_row=22, max_row=ultima_fila + 21)
            # 5 Pintar y ajustar tabla de cada alcaldía
            fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
            ws_destino['B1'].fill = fill_amarillo
            ws_destino.column_dimensions['B'].width = 100
            
            # --- NUEVO: BOTÓN DE REGRESO A GENERAL ---
            celda_volver = ws_destino['A1']
            celda_volver.value = "General"
            celda_volver.hyperlink = "#'General'!A1"
            celda_volver.font = Font(color="0563C1", underline="single", bold=True)
            celda_volver.fill = PatternFill(start_color="E7E6E6", end_color="E7E6E6", fill_type="solid") # Gris claro
            celda_volver.alignment = Alignment(horizontal="center", vertical="center")

        else:
            print(f"  - Faltó hoja 'Analisis' en {archivo}")
            
        wb_origen.close()

    # --- DISEÑO FINAL HOJA GENERAL ---
    # 1. Fondo Rojo para la fila de títulos A1:U1
    relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
    fuente_blanca = Font(color="FFFFFF", bold=True)
    for col in range(1, 22): # De la A (1) a la U (21)
        celda = ws_general.cell(row=1, column=col)
        celda.fill = relleno_rojo
        celda.font = fuente_blanca
        celda.alignment = Alignment(horizontal="center", vertical="center")

    # 2. Centrar los valores de C2:U21
    alineacion_centro = Alignment(horizontal="center", vertical="center")
    for r in range(2, 22): # Filas 2 a 21
        for c in range(3, 22): # Columnas C (3) a U (21)
            ws_general.cell(row=r, column=c).alignment = alineacion_centro
            
    # Ajustar ligeramente el ancho de General para que se vea bien
    ws_general.column_dimensions['A'].width = 5
    ws_general.column_dimensions['B'].width = 25
    for c in range(3, 22):
        ws_general.column_dimensions[get_column_letter(c)].width = 15
    
    # Ajustar el texto en la fila 1 de las hoja General
    ajustar_texto_fila(ws_general, 1)

    # --- NUEVO: APLICAR GRILLA DELGADA (A1:U21) ---
    borde_delgado = Border(
        left=Side(style='thin', color='000000'),
        right=Side(style='thin', color='000000'),
        top=Side(style='thin', color='000000'),
        bottom=Side(style='thin', color='000000')
    )
    
    for r in range(1, 22):      # Filas de la 1 a la 21
        for c in range(1, 22):  # Columnas de la A(1) a la U(21)
            ws_general.cell(row=r, column=c).border = borde_delgado

    wb_nuevo.save(ARCHIVO_DESTINO)
    print(f"\n🚀 ¡Proceso completado! Archivo guardado en: {ARCHIVO_DESTINO}")

if __name__ == "__main__":
    consolidar_excels()

-> Procesando 'ANTONIO NARINO.xlsx' -> Destino: '15-Antonio Nariño'
-> Procesando 'BARRIOS UNIDOS.xlsx' -> Destino: '12-Barrios Unidos'
-> Procesando 'BOSA.xlsx' -> Destino: '7-Bosa'
-> Procesando 'CANDELARIA.xlsx' -> Destino: '18-La Candelaria'
-> Procesando 'CHAPINERO.xlsx' -> Destino: '2-Chapinero'
-> Procesando 'CIUDAD BOLIVAR.xlsx' -> Destino: '19-Ciudad Bolívar'
-> Procesando 'ENGATIVA.xlsx' -> Destino: '10-Engativá'
-> Procesando 'FONTIBON.xlsx' -> Destino: '9-Fontibón'
-> Procesando 'KENNEDY.xlsx' -> Destino: '8-Kennedy'
-> Procesando 'MARTIRES.xlsx' -> Destino: '14-Los Mártires'
-> Procesando 'PUENTE ARANDA.xlsx' -> Destino: '16-Puente Aranda'
-> Procesando 'SAN CRISTOBAL.xlsx' -> Destino: '4-San Cristóbal'
-> Procesando 'SANTA FE.xlsx' -> Destino: '3-Santa Fe'
-> Procesando 'SUBA.xlsx' -> Destino: '11-Suba'
-> Procesando 'SUMAPAZ.xlsx' -> Destino: '20-Sumapaz'
-> Procesando 'TEUSAQUILLO.xlsx' -> Destino: '13-Teusaquillo'
-> Procesando 'TUNJUELITO.xlsx' -> Destino: '6-Tunjueli

#### Entrega 09 03 2026

In [ ]:
import pandas as pd
import unicodedata
import re
import difflib
import os
import tkinter as tk
from tkinter import filedialog, messagebox

# ==========================================
# 1. FUNCIONES DE LIMPIEZA Y COMPARACIÓN
# ==========================================

def normalizar_texto(texto):
    """Limpia el texto: quita nulos, espacios, pasa a mayúsculas, quita tildes y artículos"""
    if pd.isna(texto):
        return ""
    # Convertir a texto, mayúsculas y quitar espacios a los lados
    texto = str(texto).upper().strip()
    # Quitar tildes/acentos
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    # Quitar artículos al inicio para igualar "LOS MARTIRES" con "MARTIRES"
    texto = re.sub(r'^(EL|LA|LOS|LAS)\s+', '', texto)
    return texto

def unificar_localidades(lista_localidades, similitud_minima=0.85):
    """Agrupa nombres similares bajo un mismo nombre 'canónico'"""
    nombres_canonicos = {}
    for loc in lista_localidades:
        if not loc: continue # Saltar vacíos
        
        # Buscar si ya existe un nombre muy parecido en nuestro diccionario
        coincidencias = difflib.get_close_matches(loc, nombres_canonicos.keys(), n=1, cutoff=similitud_minima)
        
        if coincidencias:
            nombres_canonicos[loc] = coincidencias[0]
        else:
            nombres_canonicos[loc] = loc
            
    return nombres_canonicos

# ==========================================
# 2. SELECCIÓN DE ARCHIVOS Y CARPETAS
# ==========================================

# Ocultar la ventana principal de Tkinter
root = tk.Tk()
root.withdraw()

print("Por favor, selecciona el archivo Excel de origen en la ventana emergente...")
archivo_origen = filedialog.askopenfilename(
    title="Selecciona el archivo Excel (.xlsx o .xls)",
    filetypes=[("Archivos de Excel", "*.xlsx *.xls *")]
)

if not archivo_origen:
    print("No se seleccionó ningún archivo. Cancelando proceso.")
    exit()

print("Por favor, selecciona la carpeta donde deseas guardar los nuevos archivos...")
carpeta_destino = filedialog.askdirectory(title="Selecciona la carpeta de destino")

if not carpeta_destino:
    print("No se seleccionó ninguna carpeta. Cancelando proceso.")
    exit()

# ==========================================
# 3. LECTURA Y PROCESAMIENTO DEL EXCEL
# ==========================================

try:
    print(f"\nLeyendo el archivo: {archivo_origen}")
    xls = pd.ExcelFile(archivo_origen)
    nombres_hojas = xls.sheet_names
    
    # Buscar hojas independientemente de mayúsculas/minúsculas
    hoja_sipse = next((h for h in nombres_hojas if h.upper() == "SIPSE"), None)
    hoja_secop = next((h for h in nombres_hojas if h.upper() == "SECOP II"), None)
    
    if not hoja_sipse or not hoja_secop:
        raise ValueError("El archivo no contiene las hojas 'SIPSE' y/o 'SECOP II'. Verifica el archivo.")

    # Leer las hojas
    df_sipse = pd.read_excel(xls, sheet_name=hoja_sipse)
    df_secop = pd.read_excel(xls, sheet_name=hoja_secop)
    
    # Identificar columnas (SIPSE: BJ es la columna 62, índice 61 | SECOP II: A es índice 0)
    # Validamos que SIPSE tenga suficientes columnas
    if len(df_sipse.columns) <= 61:
        raise ValueError("La hoja SIPSE no tiene suficientes columnas para llegar a la columna BJ.")
        
    col_sipse_nombre = df_sipse.columns[61] 
    col_secop_nombre = df_secop.columns[0]
    
    print("Normalizando nombres de localidades...")
    # Crear columnas auxiliares limpias
    df_sipse['Loc_Limpia'] = df_sipse[col_sipse_nombre].apply(normalizar_texto)
    df_secop['Loc_Limpia'] = df_secop[col_secop_nombre].apply(normalizar_texto)
    
    # Obtener todas las localidades únicas de ambas hojas
    todas_localidades = list(set(df_sipse['Loc_Limpia'].unique()) | set(df_secop['Loc_Limpia'].unique()))
    
    # Unificar nombres similares (Ej: MARTIRES y LOS MARTIRES)
    mapa_canonico = unificar_localidades(todas_localidades)
    
    # Aplicar el nombre unificado a los DataFrames
    df_sipse['Loc_Final'] = df_sipse['Loc_Limpia'].map(mapa_canonico)
    df_secop['Loc_Final'] = df_secop['Loc_Limpia'].map(mapa_canonico)
    
    # Obtener la lista final de localidades a exportar
    localidades_finales = set(mapa_canonico.values())
    
    print(f"Se detectaron {len(localidades_finales)} localidades únicas. Generando archivos...")
    
    # ==========================================
    # 4. CREACIÓN DE LOS ARCHIVOS .XLSX
    # ==========================================
    
    archivos_creados = 0
    for loc in localidades_finales:
        if not loc: continue # Evitar crear un archivo para localidades vacías
        
        # Filtrar datos (y eliminar las columnas auxiliares de limpieza que creamos)
        filtro_sipse = df_sipse[df_sipse['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        filtro_secop = df_secop[df_secop['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        
        # Solo crear el archivo si hay datos en alguna de las dos hojas
        if not filtro_sipse.empty or not filtro_secop.empty:
            # Reemplazar caracteres no válidos para nombres de archivos
            nombre_archivo_seguro = re.sub(r'[\\/*?:"<>|]', "", loc)
            ruta_salida = os.path.join(carpeta_destino, f"{nombre_archivo_seguro}.xlsx")
            
            with pd.ExcelWriter(ruta_salida, engine='openpyxl') as writer:
                filtro_sipse.to_excel(writer, sheet_name="SIPSE", index=False)
                filtro_secop.to_excel(writer, sheet_name="SECOP II", index=False)
                
            archivos_creados += 1
            print(f" -> Creado: {nombre_archivo_seguro}.xlsx")

    print(f"\n¡Proceso Finalizado Exitosamente! Se crearon {archivos_creados} archivos en: {carpeta_destino}")
    messagebox.showinfo("Éxito", f"Se generaron {archivos_creados} archivos correctamente.")

except Exception as e:
    print(f"\nOcurrió un error: {e}")
    messagebox.showerror("Error", f"Ocurrió un error procesando el archivo:\n{e}")

In [87]:
import os
import shutil
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
from collections import defaultdict
import re
import pandas as pd
import difflib # para busqueda aproximada
import datetime

# --- CONFIGURACIÓN DE CARPETAS ---
#CARPETA_ORIGEN = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO'
#CARPETA_DESTINO = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\DiagnóstivoV2_1'
CARPETA_ORIGEN = r'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 MARZO' #Dell
CARPETA_DESTINO = r"C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 MARZO\\Diagnóstico" #Dell
#CARPETA_ORIGEN = r'D:\\Trabajo\\2026\\SecretariaDeGobierno\\BasesDeDatosSipseSecop\\2026_03_06\\DatosPartidos' #Predator
#CARPETA_DESTINO = r"D:\\Trabajo\\2026\\SecretariaDeGobierno\\BasesDeDatosSipseSecop\\2026_03_06\\DatosPartidos\\Diagnóstico" #PRedator

# --- RANGOS A OCULTAR (CORREGIDOS PARA USAR HIDDEN=TRUE) ---
# Nota: He ajustado 'A-AH' a 'Z-AH' en SECOP para NO ocultar las columnas L, M, T, X, Y que necesitas ver.
RANGOS_OCULTAR_SECOP = ['A-K', 'N-S', 'U-W', 'Z-AH', 'AJ-BR', 'BT-CI']
RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']

# --- DEFINICIÓN DE FUNCIONES ---

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def ocultar_columnas_por_lista(ws, lista_rangos):
    """
    Recibe una lista de cadenas como ['A-K', 'M'] y establece la propiedad HIDDEN = True.
    """
    for item in lista_rangos:
        item = item.upper().strip()
        
        # Caso 1: Es un rango (ej: "A-K")
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio)
            idx_fin = column_index_from_string(fin)
            
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                # Esta es la forma correcta de ocultar en Excel
                ws.column_dimensions[letra].hidden = True 
                
        # Caso 2: Es una columna única (ej: "A")
        else:
            ws.column_dimensions[item].hidden = True
    
def obtener_letra_columna_cercana(nombre_ideal, lista_encabezados, default_letter="A", umbral=0.5):
    """
    Busca 'nombre_ideal' dentro de 'lista_encabezados' usando coincidencia aproximada (fuzzy matching).
    lista_encabezados debe ser una lista de tuplas: [(indice_columna, "Texto del Encabezado"), ...]
    Devuelve la letra de la columna en Excel.
    """
    # Extraer solo los textos que no sean None
    nombres_disponibles = [str(col[1]).strip() for col in lista_encabezados if col[1]]
    
    # Buscar el más parecido
    coincidencias = difflib.get_close_matches(nombre_ideal, nombres_disponibles, n=1, cutoff=umbral)
    
    if coincidencias:
        mejor_match = coincidencias[0]
        # Buscar qué índice de columna corresponde a ese match
        for idx, nombre in lista_encabezados:
            if str(nombre).strip() == mejor_match:
                return get_column_letter(idx)
                
    # Si no encuentra nada parecido, devuelve un valor por defecto para no romper las fórmulas
    print(f"  [ADVERTENCIA] No se encontró columna similar a '{nombre_ideal}'. Usando por defecto '{default_letter}'.")
    return default_letter

def aplicar_estilo_ferrocarril(ws):
    """
    Encabezados Azul Oscuro + Letra Blanca.
    Filas con patrón intercalado.
    """
    # 1. Encabezados
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    
    for cell in ws[1]:
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas (Patrón Ferrocarril)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    max_col = ws.max_column
    
    for i_fila, row in enumerate(ws.iter_rows(min_row=2, max_col=max_col), start=0):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF":
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for cell in row:
                cell.fill = relleno

def dejar_columna_blanca(ws, indice_columna):
    relleno_blanco = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    
    # Recorremos todas las celdas de la columna indicada
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=indice_columna, max_col=indice_columna):
        for cell in row:
            cell.fill = relleno_blanco

def aplicar_formato_condicional_alertas(ws, col_inicio_eval):
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    max_row = ws.max_row
    
    reglas = [
        (0, 'equal', '"Numero contrato NO coincide"'),
        (2, 'equal', '"Cedula NO coincide"'),
        (3, 'greaterThan', '1'),
        (4, 'greaterThan', '1'),
        (5, 'equal', '"No coinciden fechas"'),
        (6, 'equal', '"No coinciden fechas"'),
        (7, 'equal', '"No coinciden fechas"'),
        (8, 'equal', '"Duracion NO coincide"'),
        (9, 'equal', '"Valor NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        rango = f"{letra_col}2:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

def normalizar_numero_contrato(valor):
    if valor is None: return 0
    texto = str(valor).strip()
    # Busca el primer grupo de números en la cadena
    match = re.search(r'\d+', texto)
    if match:
        return int(match.group())
    return 0

#Funciones Hoja "Para MEMO"
def es_error_fecha(val_sipse, val_secop):
    """Replica la fórmula de Excel de fechas para detectar discrepancias imitando tu cruce de día/mes."""
    s1 = str(val_sipse).strip() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip() if pd.notna(val_secop) and val_secop is not None else ""
    
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    
    # Parsear SIPSE 
    dt_sipse = None
    if isinstance(val_sipse, (datetime.datetime, datetime.date)):
        dt_sipse = datetime.date(val_sipse.year, val_sipse.month, val_sipse.day)
    else:
        try:
            dt_sipse = pd.to_datetime(s1.replace("-", "/"), dayfirst=True).date()
        except: pass

    # Parsear SECOP imitando la fórmula FECHA(AÑO, DIA, MES)
    dt_secop = None
    if isinstance(val_secop, (datetime.datetime, datetime.date)):
        try:
            dt_secop = datetime.date(val_secop.year, val_secop.day, val_secop.month)
        except: pass
    else:
        partes = s2.replace("-", "/").replace(" ", "/").split("/")
        if len(partes) >= 3:
            try:
                # Tu fórmula usa IZQUIERDA para el mes y EXTRAE para el día
                y = int(partes[2][-4:])
                m = int(partes[0])
                d = int(partes[1])
                dt_secop = datetime.date(y, m, d)
            except: pass
                
    if dt_sipse is not None and dt_secop is not None:
        return dt_sipse != dt_secop
        
    return s1 != s2 # Fallback a comparación de texto

def es_error_duracion(val_sipse, val_secop):
    """Replica la lógica de Duración."""
    s1 = str(val_sipse).strip().upper() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip().upper() if pd.notna(val_secop) and val_secop is not None else ""
    
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    return s1 != s2

def color_memo(err_tuple):
    """Devuelve los colores agrupados tipo pastel idénticos a los de tu imagen."""
    if err_tuple[0]: return "E4DFEC" # Beige/Grisáceo claro (Suscripción)
    if err_tuple[1]: return "B4C6E7" # Azul claro (Inicio)
    if err_tuple[2]: return "DDEBF7" # Gris claro (Fin)
    if err_tuple[3]: return "F4CCCC" # Rosa claro (Plazo)
    return "FFFFFF"

# --- LÓGICA PRINCIPAL ---
def procesar_archivos():
    if not os.path.exists(CARPETA_DESTINO):
        os.makedirs(CARPETA_DESTINO)

    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('ANTONIO NARINO.xlsx', 'ANTONIO NARINO.xls'))]
    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('.xlsx', '.xls'))]
    print(f"Se encontraron {len(archivos)} archivos de Excel.")

    for archivo in archivos:
        ruta_orig = os.path.join(CARPETA_ORIGEN, archivo)
        
        # Definimos la ruta de destino siempre con extensión .xlsx
        nombre_base = os.path.splitext(archivo)[0]
        ruta_dest = os.path.join(CARPETA_DESTINO, nombre_base + ".xlsx")
        
        print(f"Procesando: {archivo}...")

        # --- PASO DE CONVERSIÓN SI ES .XLS ---
        if archivo.lower().endswith('.xls'):
            try:
                # Leemos el .xls y lo guardamos inmediatamente como .xlsx
                # sheet_name=None lee todas las pestañas
                conversion = pd.read_excel(ruta_orig, sheet_name=None, engine='xlrd')
                with pd.ExcelWriter(ruta_dest, engine='openpyxl') as writer:
                    for nombre_hoja, datos in conversion.items():
                        datos.to_excel(writer, sheet_name=nombre_hoja, index=False)
                print(f"  -> Convertido de .xls a .xlsx con éxito.")
            except Exception as e:
                print(f"  Error convirtiendo {archivo}: {e}")
                continue
        else:
            # Si ya es .xlsx, solo lo copiamos como hacías antes
            shutil.copy2(ruta_orig, ruta_dest)
        # -------------------------------------

        try:
            # Ahora openpyxl siempre abrirá un archivo .xlsx (el copiado o el convertido)
            wb = openpyxl.load_workbook(ruta_dest)
        except Exception as e:
            print(f"   Error al abrir el archivo con openpyxl: {e}")
            continue

        # ==========================================
        # PARTE 1: Procesar SECOP II (Crear aux1)
        # ==========================================
        nombre_real_secop = next(n for n in wb.sheetnames if n.lower() == 'secop ii')
        ws_origen = wb[nombre_real_secop]
        
        if "SECOP II aux1" in wb.sheetnames:
            del wb["SECOP II aux1"]
            
        ws_secop_aux = wb.copy_worksheet(ws_origen)
        ws_secop_aux.title = "SECOP II aux1"

        filas = []
        for row in ws_secop_aux.iter_rows(min_row=2, values_only=True):
            filas.append(list(row))
        
        filas_ordenadas = sorted(filas, key=lambda x: normalizar_numero_contrato(x[11] if len(x) > 11 else 0))

        for row in ws_secop_aux.iter_rows(min_row=2):
            for cell in row:
                cell.value = None

        for i_fila, datos_fila in enumerate(filas_ordenadas, start=2):
            for i_col, valor in enumerate(datos_fila, start=1):
                ws_secop_aux.cell(row=i_fila, column=i_col, value=valor)

        # Para crear columnas "Columna1"  y "Columna2" en la hoja "SECOP II aux1"
        max_col_secop = ws_secop_aux.max_column
        ws_secop_aux.cell(row=1, column=max_col_secop + 1, value="Columna1")
        ws_secop_aux.cell(row=1, column=max_col_secop + 2, value="Columna2") # NUEVO
        col_L = get_column_letter(12)
        
        for i in range(2, ws_secop_aux.max_row + 1):
            # Fórmula Columna 1
            formula1 = (
                f'=IFERROR(MID({col_L}{i}, '
                f'MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), '
                f'MIN(IFERROR(FIND(" ", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999), '
                f'IFERROR(FIND("-", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999)) '
                f'- MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), "")'
            )
            ws_secop_aux.cell(row=i, column=max_col_secop + 1, value=formula1)

            # Fórmula Columna 2 (Extrae número entre paréntesis)
            formula2 = f'=IFERROR(MID({col_L}{i}, FIND("(", {col_L}{i}) + 1, FIND(")", {col_L}{i}) - FIND("(", {col_L}{i}) - 1), "")'
            ws_secop_aux.cell(row=i, column=max_col_secop + 2, value=formula2)

        # ESTILOS SECOP
        print("  -> Aplicando estilos y ocultando columnas en SECOP II aux1...")
        aplicar_estilo_ferrocarril(ws_secop_aux)
        ocultar_columnas_por_lista(ws_secop_aux, RANGOS_OCULTAR_SECOP)

        # ==========================================
        # PARTE 2 y 3: ALINEACIÓN EN 'ANALISIS'
        # ==========================================
        print("  -> Generando hoja 'Analisis'...")

        nombre_real_sipse = next(n for n in wb.sheetnames if n.lower() == 'sipse')
        ws_sipse = wb[nombre_real_sipse]
        
        if 'Analisis' in wb.sheetnames:
            del wb['Analisis']
        ws_analisis = wb.create_sheet("Analisis")
        
        dict_sipse = defaultdict(list)
        dict_secop = defaultdict(list)
        
        filas_sipse = list(ws_sipse.values)
        header_sipse = filas_sipse[0] if filas_sipse else []
        for fila in filas_sipse[1:]:
            contrato = normalizar_numero_contrato(fila[3] if len(fila) > 3 else 0)
            dict_sipse[contrato].append(fila)

        filas_secop = list(ws_secop_aux.values)
        header_secop = filas_secop[0] if filas_secop else []
        for fila in filas_secop[1:]:
            contrato = normalizar_numero_contrato(fila[11] if len(fila) > 11 else 0)
            dict_secop[contrato].append(fila)

        ws_analisis.append(header_sipse) 
        col_inicio_secop = column_index_from_string("BM")
        for i, val in enumerate(header_secop):
            ws_analisis.cell(row=1, column=col_inicio_secop + i, value=val)

        todos_contratos = sorted(list(set(dict_sipse.keys()) | set(dict_secop.keys())))
        current_row = 2 
        
        for contrato in todos_contratos:
            filas_de_sipse = dict_sipse.get(contrato, [])
            filas_de_secop = dict_secop.get(contrato, [])
            max_filas = max(len(filas_de_sipse), len(filas_de_secop))
            
            for i in range(max_filas):
                if i < len(filas_de_sipse):
                    datos = filas_de_sipse[i]
                    for c_idx, valor in enumerate(datos, start=1):
                        ws_analisis.cell(row=current_row, column=c_idx, value=valor)
                
                if i < len(filas_de_secop):
                    datos = filas_de_secop[i]
                    for c_idx, valor in enumerate(datos):
                        col_excel = col_inicio_secop + c_idx
                        if c_idx == len(datos) - 2 and str(valor).startswith('='):
                             # Pega la fórmula de Columna1 (Ahora es la penúltima)
                             nueva_formula1 = (
                                f'=IFERROR(MID(BX{current_row}, '
                                f'MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), '
                                f'MIN(IFERROR(FIND(" ", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999), '
                                f'IFERROR(FIND("-", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999)) '
                                f'- MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), "")'
                            )
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula1)
                        elif c_idx == len(datos) - 1 and str(valor).startswith('='):
                             # Pega la fórmula de Columna2 (Extrae entre paréntesis)
                             nueva_formula2 = f'=IFERROR(MID(BX{current_row}, FIND("(", BX{current_row}) + 1, FIND(")", BX{current_row}) - FIND("(", BX{current_row}) - 1), "")'
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula2)
                        else:
                            ws_analisis.cell(row=current_row, column=col_excel, value=valor)
                current_row += 1

        # ==========================================
        # PARTE 4: Columnas de Evaluación
        # ==========================================
        print("  -> Agregando columnas de evaluación y estilos...")
        
        ultima_col_actual = ws_analisis.max_column
        col_inicio_eval = ultima_col_actual + 1
        ultima_fila_analisis = ws_analisis.max_row

        # Extraer los encabezados reales de la hoja Analisis para la búsqueda
        headers_sipse_excel = [(c, ws_analisis.cell(row=1, column=c).value) for c in range(1, col_inicio_secop)]
        headers_secop_excel = [(c, ws_analisis.cell(row=1, column=c).value) for c in range(col_inicio_secop, col_inicio_eval)]

        # --- Variables SIPSE ---
        col_1 = obtener_letra_columna_cercana("NUMERO_CONTRATO", headers_sipse_excel, "D")
        col_2 = obtener_letra_columna_cercana("DOC_CONTRATISTA", headers_sipse_excel, "M")
        col_3 = obtener_letra_columna_cercana("FECHA_SUSCRIPCION", headers_sipse_excel, "H")
        col_4 = obtener_letra_columna_cercana("FECHA_INICIO", headers_sipse_excel, "I")
        col_5 = obtener_letra_columna_cercana("FECHA_TERMINACION", headers_sipse_excel, "J")
        col_6 = obtener_letra_columna_cercana("PLAZO", headers_sipse_excel, "R")
        col_7 = obtener_letra_columna_cercana("VALOR_TOTAL", headers_sipse_excel, "P")

        # --- Variables SECOP ---
        # La fórmula normalizada de SECOP siempre la creaste al final de las columnas de SECOP
        col_idx_secop_formula = col_inicio_secop + len(header_secop) - 1
        col_8 = get_column_letter(col_idx_secop_formula) # Equivalente a antiguo EV
        
        col_9  = obtener_letra_columna_cercana("Documento Proveedor", headers_secop_excel, "CJ")
        col_10 = obtener_letra_columna_cercana("Aux Nro contrato", headers_secop_excel, "CJ")
        col_11 = obtener_letra_columna_cercana("Fecha de Firma", headers_secop_excel, "CE")
        col_12 = obtener_letra_columna_cercana("Fecha de Inicio del Contrato", headers_secop_excel, "CF")
        col_13 = obtener_letra_columna_cercana("Fecha de Fin del Contrato", headers_secop_excel, "CG")
        col_14 = obtener_letra_columna_cercana("Valor del Contrato", headers_secop_excel, "CU")
        col_15 = obtener_letra_columna_cercana("n del contrato", headers_secop_excel, "EE")
        
        headers_nuevos = [
            "Ev. Nro contrato", "Aux Nro contrato", "Ev. Cédula", "Repeticiones contrato SIPSE", 
            "Repeticiones contrato SECOP", "Fecha suscripción", "Fecha inicio", "Fecha fin", "Duración contratos", "Valor contratos"
        ]

        for idx, titulo in enumerate(headers_nuevos):
            ws_analisis.cell(row=1, column=col_inicio_eval + idx, value=titulo)

        for i in range(2, ultima_fila_analisis + 1):

            # 🔹 AUX NRO CONTRATO (EX) → llave única
            f_aux = f'=IF(D{i}<>"",D{i},EV{i})'
            ws_analisis.cell(row=i, column=col_inicio_eval + 1, value=f_aux)

            COL_SOLICITUD = "B"

            # 🔹 ESTADO CONTRATO
            f1 = f'''=IF(COUNTIF($EY$2:EY{i},EY{i})>1,"",IF(AND(D{i}<>"",EV{i}<>""),IF(EW{i}="","Contrato coincide sin SIPSE",IF(TRIM({COL_SOLICITUD}{i})=TRIM(EW{i}),"Numero contrato coincide","Numero contrato NO coincide")),IF(D{i}<>"","Contrato no existe en SECOP",IF(EV{i}<>"","Contrato no existe en SIPSE","No existe"))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval, value=f1)

            # 🔹 CÉDULA
            f2 = f'=IF(AND(BY{i}="",M{i}=""),"",IF(OR(BY{i}="",M{i}=""),"",IF(BY{i}="-","Sin cedula en SECOP",IF(M{i}="-","Sin cedula en SIPSE",IF(VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(CJ{i}),"CC",""),"NIT",""),"TI",""))=VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(M{i}),"CC",""),"NIT",""),"TI","")),"Cedula coincide","Cedula NO coincide")))))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 2, value=f2)

            # 🔹 REPETICIONES SIPSE
            f3 = f'=IF(EX{i}="","",COUNTIF(D$2:D${ultima_fila_analisis},D{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 3, value=f3)

            # 🔹 REPETICIONES SECOP
            f4 = f'=IF(EX{i}="","",COUNTIF(EV$2:EV${ultima_fila_analisis},EV{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 4, value=f4)

            # 🔹 FECHA SUSCRIPCION
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CE{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CE{i}), DAY(CE{i}), MONTH(CE{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f5 = f'''=IF(AND(H{i}="",CE{i}=""),"",IF(AND(H{i}="-",CE{i}="-"),"Ambos sin fechas",IF(H{i}="-","sin fecha SIPSE",IF(CE{i}="-","sin fecha SECOP",IF(OR(H{i}="",CE{i}=""),"",IF(INT(VALUE(SUBSTITUTE(H{i},"-","/"))) =INT(IF(ISNUMBER(CE{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 5, value=f5)

            # 🔹 FECHA ACTA DE INICIO
            cf_clean = f'SUBSTITUTE(SUBSTITUTE(CF{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CF{i}), DAY(CF{i}), MONTH(CF{i}))'
            parte_es_texto = f'DATE(RIGHT({cf_clean}, 4), LEFT({cf_clean}, FIND("/", {cf_clean}) - 1), MID({cf_clean}, FIND("/", {cf_clean}) + 1, FIND("/", {cf_clean}, FIND("/", {cf_clean}) + 1) - FIND("/", {cf_clean}) - 1))'

            f6 = f'''=IF(AND(I{i}="",CF{i}=""),"",IF(AND(I{i}="-",CF{i}="-"),"Ambos sin fechas",IF(I{i}="-","sin fecha SIPSE",IF(CF{i}="-","sin fecha SECOP",IF(OR(I{i}="",CF{i}=""),"",IF(INT(VALUE(SUBSTITUTE(I{i},"-","/"))) =INT(IF(ISNUMBER(CF{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 6, value=f6)

            # 🔹 FECHA FIN
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CG{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CG{i}), DAY(CG{i}), MONTH(CG{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f7 = f'''=IF(AND(J{i}="",CG{i}=""),"",IF(AND(J{i}="-",CG{i}="-"),"Ambos sin fechas",IF(J{i}="-","sin fecha SIPSE",IF(CG{i}="-","sin fecha SECOP",IF(OR(J{i}="",CG{i}=""),"",IF(INT(VALUE(SUBSTITUTE(J{i},"-","/"))) =INT(IF(ISNUMBER(CG{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 7, value=f7)

            # 🔹 DURACIÓN
            f8 = f'''=IF(AND(R{i}="",EE{i}=""),"",IF(AND(R{i}="-",EE{i}="-"),"Ambos sin duracion",IF(R{i}="-","sin duracion en SECOP",IF(EE{i}="-","sin duracion en SIPSE",IF(OR(R{i}="",EE{i}=""),"",IF(TRIM(UPPER(R{i}))=TRIM(UPPER(EE{i})),"Duracion coincide","Duracion NO coincide"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 8, value=f8)

            # 🔹 VALOR CONTRATO (Comparación Robusta)
            f9 = f'''=IF(AND(P{i}="",CU{i}=""),"",
                IF(AND(P{i}="-",CU{i}="-"),"Ambos sin valor",
                IF(P{i}="-","Sin valor en SECOP",
                IF(CU{i}="-","Sin valor en SIPSE",
                IF(OR(P{i}="",CU{i}=""),"",
                IF(P{i}=CU{i},"Valor coincide","Valor NO coincide"))))))'''

            ws_analisis.cell(row=i, column=col_inicio_eval + 9, value=f9)

        # --- ESTILOS ANALISIS ---
        print("  -> Aplicando estilos y ocultando columnas en Analisis...")
        aplicar_estilo_ferrocarril(ws_analisis)
        aplicar_formato_condicional_alertas(ws_analisis, col_inicio_eval)
        
        # Ocultar columnas hoja analisis
        ocultar_columnas_por_lista(ws_analisis, RANGOS_OCULTAR_ANALISIS)

        # ==========================================
        # PARTE 5: Crear HOJA RESULTADOS
        # ==========================================
        print("  -> Creando hoja 'Resultados'...")
        if 'Resultados' in wb.sheetnames:
            del wb['Resultados']
        ws_res = wb.create_sheet("Resultados")
        nombre_limpio = os.path.splitext(archivo)[0]
        ws_res['B1'] = f"Métricas alcaldía local {nombre_limpio}"
        
        letra_ev_nro = get_column_letter(col_inicio_eval)       
        letra_ev_ced = get_column_letter(col_inicio_eval + 2)   
        letra_rep_sipse = get_column_letter(col_inicio_eval + 3)
        letra_rep_secop = get_column_letter(col_inicio_eval + 4)
        letra_fecha_suscripcion = get_column_letter(col_inicio_eval + 5)
        letra_fecha_inicio = get_column_letter(col_inicio_eval + 6)
        letra_fecha_fin = get_column_letter(col_inicio_eval + 7)
        letra_duracion = get_column_letter(col_inicio_eval + 8)
        letra_valor = get_column_letter(col_inicio_eval + 9)

        metricas = [
            #("Nro de contratos registrados en SIPSE",'=SUMPRODUCT((Analisis!A2:A10000<>"")/COUNTIF(Analisis!A2:A10000,Analisis!A2:A10000&""))'),
            #("Nro de contratos registrados SECOP",'=SUMPRODUCT((Analisis!EV2:EV10000<>"")/COUNTIF(Analisis!EV2:EV10000,Analisis!EV2:EV10000&""))'),
            ("Nro de contratos registrados SIPSE",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos registrados SECOP",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos registrados en SECOP y SIPSE con numero de contrato igual", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")"),
            ("Nro de contratos solo en SECOP y no en SIPSE", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos solo en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos diferentes en SIPSE-SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato NO coincide\")"),
            ("Nro de contratos Cédula diferente", f"=COUNTIF('Analisis'!{letra_ev_ced}:{letra_ev_ced}, \"Cedula NO coincide\")"),
            ("Nro de contratos dobles en SIPSE", f"=COUNTIF('Analisis'!{letra_rep_sipse}:{letra_rep_sipse}, 2)"),
            ("Nro de contratos dobles en SECOP", f"=COUNTIF('Analisis'!{letra_rep_secop}:{letra_rep_secop}, 2)"),
            ("Fecha de acta de suscriccpión Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"Coincide fecha\")"),
            ("Fecha de acta de suscriccpión no Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"No coinciden fechas\")"),
            ("Fecha de acta de inicio Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Coincide fecha\")"),
            ("Fecha de acta de inicio no coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"No coinciden fechas\")"),
            ("Fecha de Fin conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"Coincide fecha\")"),
            ("Fecha de Fin no conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"No coinciden fechas\")"),
            ("Contratos que coinciden en Duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion Coincide\")"),
            ("Contratos que tienen diferente duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion NO coincide\")"),
            ("Contratos que tienen Fecha de inicio en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Sin fecha SECOP\")"),
            ("Datos en la matriz de seguimiento", f"=C2+C3"),
            ("Valor contrato coinciden", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor coincide\")"),
            ("Valor contrato no coincidenn", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor NO coincide\")")
        ]


        for idx, (texto, formula) in enumerate(metricas, start=2):
            ws_res.cell(row=idx, column=1, value=idx-1)
            ws_res.cell(row=idx, column=2, value=texto)
            ws_res.cell(row=idx, column=3, value=formula)
        
        # Pintar B2 de amarillo
        fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
        ws_res['B1'].fill = fill_amarillo
        ws_res.column_dimensions['B'].width = 100
        
        ws_analisis.column_dimensions['EX'].width = 29
        ws_analisis.column_dimensions['EY'].width = 10
        ws_analisis.column_dimensions['EZ'].width = 20
        ws_analisis.column_dimensions['FA'].width = 15
        ws_analisis.column_dimensions['FB'].width = 15
        ws_analisis.column_dimensions['FC'].width = 15
        ws_analisis.column_dimensions['FD'].width = 20
        ws_analisis.column_dimensions['FE'].width = 20
        ws_analisis.column_dimensions['FF'].width = 18
        ws_analisis.column_dimensions['FG'].width = 14

        dejar_columna_blanca(ws_analisis, 64)# columna 64 es la columna "BL"

        #Ajustar el texto en la fila 1 de las hojas aux y Analisis
        ajustar_texto_fila(ws_secop_aux, 1)
        ajustar_texto_fila(ws_analisis, 1)

        ##--- RELLENAR CELDAS VACÍAS CON "?" EN SIPSE Y SECOP ---
        #for row in ws_analisis.iter_rows(min_row=2, max_row=ultima_fila_analisis, min_col=1, max_col=col_inicio_eval - 1):
        #    for cell in row:
        #        # Comprueba si la celda está realmente vacía (None) o si es un texto vacío ("")
        #        if cell.value is None or str(cell.value).strip() == "":
        #            cell.value = "?"

        # ==========================================
        # PARTE 6: Crear HOJA 'Para MEMO'
        # ==========================================
        print("  -> Creando hoja 'Para MEMO'...")
        if 'Para MEMO' in wb.sheetnames:
            del wb['Para MEMO']
        ws_memo = wb.create_sheet("Para MEMO")
        
        datos_memo = []
        
        # 1. Escanear fila por fila extrayendo los datos directos
        for i in range(2, ultima_fila_analisis + 1):
            val_b = ws_analisis.cell(row=i, column=2).value # B: SIPSE (Solicitud)
            val_d = ws_analisis.cell(row=i, column=4).value # D: No. Contrato
            
            # Extraer celdas origen según tu matriz (SIPSE y SECOP)
            susc_sipse = ws_analisis.cell(row=i, column=8).value  # H
            susc_secop = ws_analisis.cell(row=i, column=83).value # CE
            ini_sipse  = ws_analisis.cell(row=i, column=9).value  # I
            ini_secop  = ws_analisis.cell(row=i, column=84).value # CF
            fin_sipse  = ws_analisis.cell(row=i, column=10).value # J
            fin_secop  = ws_analisis.cell(row=i, column=85).value # CG
            dur_sipse  = ws_analisis.cell(row=i, column=18).value # R
            dur_secop  = ws_analisis.cell(row=i, column=135).value# EE
            
            # Evaluarlas en Python
            err_susc = es_error_fecha(susc_sipse, susc_secop)
            err_ini  = es_error_fecha(ini_sipse, ini_secop)
            err_fin  = es_error_fecha(fin_sipse, fin_secop)
            err_dur  = es_error_duracion(dur_sipse, dur_secop)
            
            # Si tiene al menos 1 error, lo enlistamos
            if err_susc or err_ini or err_fin or err_dur:
                llave_orden = (err_susc, err_ini, err_fin, err_dur)
                datos_memo.append([
                    val_b, val_d,
                    "X" if err_susc else "",
                    "X" if err_ini else "",
                    "X" if err_fin else "",
                    "X" if err_dur else "",
                    llave_orden # Llave secreta para ordenar
                ])
                
        # 2. Ordenar datos para que queden agrupados por color como en tu imagen
        datos_memo.sort(key=lambda x: x[6], reverse=True)
        
        # 3. Diseño de la Cabecera
        relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
        fuente_blanca = Font(color="FFFFFF", bold=True)
        ws_memo.merge_cells("A1:A2")
        ws_memo['A1'] = "SIPSE"
        ws_memo.merge_cells("B1:B2")
        ws_memo['B1'] = "No. Contrato"
        ws_memo.merge_cells("C1:F1")
        ws_memo['C1'] = "TIPOLOGÍA DE LA INCONSISTENCIA"
        
        ws_memo['C2'] = "Fecha suscripción"
        ws_memo['D2'] = "Fecha Inicio"
        ws_memo['E2'] = "Fecha Fin"
        ws_memo['F2'] = "Plazo"
        ws_memo['A1'].fill = relleno_rojo
        ws_memo['B1'].fill = relleno_rojo
        ws_memo['C1'].fill = relleno_rojo
        ws_memo['C2'].fill = relleno_rojo
        ws_memo['D2'].fill = relleno_rojo
        ws_memo['E2'].fill = relleno_rojo
        ws_memo['F2'].fill = relleno_rojo
        
        align_header = Alignment(horizontal="center", vertical="center")
        borde_delgado = Border(left=Side(style='thin', color='000000'), right=Side(style='thin', color='000000'), top=Side(style='thin', color='000000'), bottom=Side(style='thin', color='000000'))
        
        for fila_h in ws_memo['A1:F2']:
            for celda in fila_h:
                celda.font = fuente_blanca
                celda.alignment = align_header
                celda.border = borde_delgado
                
        # 4. Volcar datos extraídos
        fila_actual = 3
        for fila_datos in datos_memo:
            for col_idx in range(1, 7):
                celda = ws_memo.cell(row=fila_actual, column=col_idx, value=fila_datos[col_idx-1])
                celda.border = borde_delgado
                celda.alignment = align_header # Centrado para todo
                
            fila_actual += 1
            
        # 5. Ajustar anchos
        ws_memo.column_dimensions['A'].width = 15
        ws_memo.column_dimensions['B'].width = 15
        ws_memo.column_dimensions['C'].width = 20
        ws_memo.column_dimensions['D'].width = 15
        ws_memo.column_dimensions['E'].width = 15
        ws_memo.column_dimensions['F'].width = 15

        wb.save(ruta_dest)
        print("  -> Procesado exitosamente.")

if __name__ == "__main__":
    procesar_archivos()

Se encontraron 20 archivos de Excel.
Procesando: ANTONIO NARINO.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 'Para MEMO'...
  -> Procesado exitosamente.
Procesando: BARRIOS UNIDOS.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 'Para MEMO'...
  -> Procesado exitosamente.
Procesando: BOSA.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 

In [33]:
import os
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
import re
import unicodedata

# ================= CONFIGURACIÓN =================
#CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1' 
#ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1\Consolidado\Consolidado_Localidades.xlsx"
CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 01 MARZO\Diagnóstico' # Dell
ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 01 MARZO\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx" #Dell
#CARPETA_ORIGEN = r'D:\Trabajo\2026\SecretariaDeGobierno\BasesDeDatosSipseSecop\2026_03_06\DatosPartidos\Diagnóstico' # PRedator
#ARCHIVO_DESTINO = r"D:\Trabajo\2026\SecretariaDeGobierno\BasesDeDatosSipseSecop\2026_03_06\DatosPartidos\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx" #Predator

nombres_hojas = [
    "1-Usaquén", "2-Chapinero", "3-Santa Fe", "4-San Cristóbal",
    "5-Usme", "6-Tunjuelito", "7-Bosa", "8-Kennedy", "9-Fontibón",
    "10-Engativá", "11-Suba", "12-Barrios Unidos", "13-Teusaquillo",
    "14-Los Mártires", "15-Antonio Nariño", "16-Puente Aranda",
    "17-Rafael Uribe", "18-La Candelaria", "19-Ciudad Bolívar", "20-Sumapaz"
]

RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']
# Aquí puedes añadir todas las columnas a las que quieras fijarles un ancho específico
ANCHOS_COLUMNAS = {
    'FA': 25,
    'FB': 20,
    'FC': 20,
    'FD': 15,
    'FE': 15,
    'FF': 15,
    'FG': 20,
    'FH': 20,
    'FI': 18,
    'FJ': 14
}

# ================= FUNCIONES AUXILIARES =================
def normalizar_texto(texto):
    texto = texto.lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^\d+[-_ ]?', '', texto)
    texto = re.sub(r'\b(la|el|los|las)\b', '', texto).strip()
    return texto

def encontrar_hoja_destino(nombre_archivo, hojas_destino):
    archivo_norm = normalizar_texto(nombre_archivo)
    for hoja in hojas_destino:
        if normalizar_texto(hoja) in archivo_norm:
            return hoja
    return None

def ocultar_columnas_por_lista(ws, lista_rangos, offset=3):
    for item in lista_rangos:
        item = item.upper().strip()
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio) + offset
            idx_fin = column_index_from_string(fin) + offset
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                ws.column_dimensions[letra].hidden = True 
        else:
            idx = column_index_from_string(item) + offset
            letra = get_column_letter(idx)
            ws.column_dimensions[letra].hidden = True

def adaptar_formulas_resultados(formula):
    pattern = re.compile(r"'?Analisis'?!([A-Za-z]+)(\d*)(?::([A-Za-z]+)(\d*))?")
    def replacer(match):
        col1, row1, col2, row2 = match.groups()
        nueva_col1 = get_column_letter(column_index_from_string(col1) + 3)
        nueva_row1 = str(int(row1) + 21) if row1 else ""
        resultado = f"{nueva_col1}{nueva_row1}"
        if col2:
            nueva_col2 = get_column_letter(column_index_from_string(col2) + 3)
            nueva_row2 = str(int(row2) + 21) if row2 else ""
            resultado += f":{nueva_col2}{nueva_row2}"
        return resultado
    return pattern.sub(replacer, formula)

def desplazar_formulas_analisis(formula):
    formula = re.sub(r"'?Analisis'?!", "", formula, flags=re.IGNORECASE)
    pattern = re.compile(r'(\$?)([A-Za-z]{1,3})(\$?)(\d+)(?!\s*\()')
    def replacer(match):
        lock_col, col_str, lock_row, row_str = match.groups()
        nueva_col = get_column_letter(column_index_from_string(col_str) + 3)
        nueva_fila = str(int(row_str) + 21)
        return f"{lock_col}{nueva_col}{lock_row}{nueva_fila}"
    return pattern.sub(replacer, formula)

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# ================= FUNCIONES DE DISEÑO =================
def aplicar_estilo_ferrocarril(ws, start_row, max_row):
    """Aplica formato estilo ferrocarril solo a columnas E:U y CA:FH desde start_row."""
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    
    # Rango 1: E (5) a U (21) | Rango 2: CA (79) a Fj (166)
    def es_columna_objetivo(c):
        return (5 <= c <= 21) or (79 <= c <= 166)

    # 1. Encabezados (Fila 22)
    for col_idx in range(1, ws.max_column + 1):
        if es_columna_objetivo(col_idx):
            cell = ws.cell(row=start_row, column=col_idx)
            cell.fill = fill_header
            cell.font = font_header
            cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas de Datos (Patrón Ferrocarril)
    for i_fila, r_idx in enumerate(range(start_row + 1, max_row + 1)):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF": # Evitar procesar blanco para ahorrar memoria
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for col_idx in range(1, ws.max_column + 1):
                if es_columna_objetivo(col_idx):
                    ws.cell(row=r_idx, column=col_idx).fill = relleno

def aplicar_formato_condicional_alertas(ws, start_row, max_row):
    """Aplica reglas condicionales desde la columna EZ."""
    col_inicio_eval = column_index_from_string('FA')
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    
    reglas = [
        (0, 'equal', '"Numero contrato NO coincide"'),
        (2, 'equal', '"Cedula NO coincide"'),
        (3, 'greaterThan', '1'),
        (4, 'greaterThan', '1'),
        (5, 'equal', '"No coinciden fechas"'),
        (6, 'equal', '"No coinciden fechas"'),
        (7, 'equal', '"No coinciden fechas"'),
        (8, 'equal', '"Duracion NO coincide"'),
        (9, 'equal', '"Valor NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        # Se aplica desde la fila siguiente al encabezado (23 en adelante)
        rango = f"{letra_col}{start_row + 1}:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

# ================= PROGRAMA PRINCIPAL =================
def consolidar_excels():
    wb_nuevo = openpyxl.Workbook()
    
    # 1. Configurar Hoja General
    ws_general = wb_nuevo.active
    ws_general.title = "General"
    ws_general.cell(row=1, column=1, value="Nro").font = Font(bold=True)
    ws_general.cell(row=1, column=2, value="Alcaldía local").font = Font(bold=True)
    
    # Pre-llenar las localidades en General con Hipervínculos
    for i, hoja in enumerate(nombres_hojas):
        num, nombre = hoja.split("-", 1)
        row_idx = i + 2
        ws_general.cell(row=row_idx, column=1, value=int(num))
        
        celda_loc = ws_general.cell(row=row_idx, column=2, value=nombre.strip())
        celda_loc.hyperlink = f"#'{hoja}'!A1"
        celda_loc.font = Font(color="0563C1", underline="single") 

    for nombre in nombres_hojas:
        wb_nuevo.create_sheet(title=nombre)

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith('.xlsx') and not f.startswith('~')]
    metricas_extraidas = False 

    for archivo in archivos:
        hoja_destino_nombre = encontrar_hoja_destino(archivo, nombres_hojas)
        
        if not hoja_destino_nombre:
            continue
            
        print(f"-> Procesando '{archivo}' -> Destino: '{hoja_destino_nombre}'")
        
        ruta_completa = os.path.join(CARPETA_ORIGEN, archivo)
        wb_origen = openpyxl.load_workbook(ruta_completa)
        ws_destino = wb_nuevo[hoja_destino_nombre]

        # --- TAREA A: Copiar de 'Resultados' ---
        if "Resultados" in wb_origen.sheetnames:
            ws_resultados = wb_origen["Resultados"]
            row_gen = nombres_hojas.index(hoja_destino_nombre) + 2
            col_gen = 3 
            
            for r_idx in range(2, 21): 
                titulo_metrica = ws_resultados.cell(row=r_idx, column=2).value 
                
                if not metricas_extraidas:
                    ws_general.cell(row=1, column=col_gen, value=titulo_metrica).font = Font(bold=True)
                
                ws_general.cell(row=row_gen, column=col_gen, value=f"='{hoja_destino_nombre}'!C{r_idx}")
                col_gen += 1
                
            metricas_extraidas = True 
            
            for row in ws_resultados['A1':'C20']:
                for cell in row:
                    dest_coord = cell.coordinate
                    valor = cell.value
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = adaptar_formulas_resultados(valor)
                    else:
                        ws_destino[dest_coord].value = valor
        else:
            print(f"  - Faltó hoja 'Resultados' en {archivo}")

        # --- TAREA B: Copiar de 'Analisis' a la fila 22 ---
        if "Analisis" in wb_origen.sheetnames:
            ws_analisis = wb_origen["Analisis"]
            ultima_fila = ws_analisis.max_row
            max_col_origen = min(ws_analisis.max_column, column_index_from_string('FH'))
            
            ws_destino.merge_cells("D21:BN21")
            celda_sipse = ws_destino["D21"]
            celda_sipse.value = "SIPSE"
            celda_sipse.fill = PatternFill(start_color="0070C0", end_color="0070C0", fill_type="solid") 
            celda_sipse.font = Font(color="FFFFFF", bold=True)
            celda_sipse.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells("BP21:EH21")
            celda_secop = ws_destino["BP21"]
            celda_secop.value = "SECOP"
            celda_secop.fill = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid") 
            celda_secop.font = Font(color="FFFFFF", bold=True)
            celda_secop.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells("EI21:FJ21")
            celda_comparacion = ws_destino["EI21"]
            celda_comparacion.value = "COMPARACIÓN"
            celda_comparacion.fill = PatternFill(start_color="7e350e", end_color="7e350e", fill_type="solid") 
            celda_comparacion.font = Font(color="FFFFFF", bold=True)
            celda_comparacion.alignment = Alignment(horizontal="center", vertical="center")

            # Pegar los datos desde la fila 22
            for r_idx, row_origen in enumerate(ws_analisis.iter_rows(min_row=1, max_row=ultima_fila, min_col=1, max_col=max_col_origen), start=22):
                for c_idx, cell in enumerate(row_origen, start=4):
                    dest_coord = f"{get_column_letter(c_idx)}{r_idx}"
                    valor = cell.value
                    
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = desplazar_formulas_analisis(valor)
                    else:
                        ws_destino[dest_coord].value = valor
            
            # 1. Ocultar columnas
            ocultar_columnas_por_lista(ws_destino, RANGOS_OCULTAR_ANALISIS, offset=3)
            # 2. Aplicar anchos de columnas
            for letra_col, ancho in ANCHOS_COLUMNAS.items():
                ws_destino.column_dimensions[letra_col].width = ancho
            # 2.1 Ajustar el texto en la fila 22 de las hojas de cada localidad
            ajustar_texto_fila(ws_destino, 22)
            # 3. Aplicar estilo Ferrocarril (La tabla empieza en la 22)
            aplicar_estilo_ferrocarril(ws_destino, start_row=22, max_row=ultima_fila + 21)
            # 4. Aplicar alertas condicionales
            aplicar_formato_condicional_alertas(ws_destino, start_row=22, max_row=ultima_fila + 21)
            # 5 Pintar y ajustar tabla de cada alcaldía
            fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
            ws_destino['B1'].fill = fill_amarillo
            ws_destino.column_dimensions['B'].width = 100
            
            # --- NUEVO: BOTÓN DE REGRESO A GENERAL ---
            celda_volver = ws_destino['A1']
            celda_volver.value = "General"
            celda_volver.hyperlink = "#'General'!A1"
            celda_volver.font = Font(color="0563C1", underline="single", bold=True)
            celda_volver.fill = PatternFill(start_color="E7E6E6", end_color="E7E6E6", fill_type="solid") # Gris claro
            celda_volver.alignment = Alignment(horizontal="center", vertical="center")

        else:
            print(f"  - Faltó hoja 'Analisis' en {archivo}")
            
        wb_origen.close()

    # --- DISEÑO FINAL HOJA GENERAL ---
    # 1. Fondo Rojo para la fila de títulos A1:U1
    relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
    fuente_blanca = Font(color="FFFFFF", bold=True)
    for col in range(1, 22): # De la A (1) a la U (21)
        celda = ws_general.cell(row=1, column=col)
        celda.fill = relleno_rojo
        celda.font = fuente_blanca
        celda.alignment = Alignment(horizontal="center", vertical="center")

    # 2. Centrar los valores de C2:U21
    alineacion_centro = Alignment(horizontal="center", vertical="center")
    for r in range(2, 22): # Filas 2 a 21
        for c in range(3, 22): # Columnas C (3) a U (21)
            ws_general.cell(row=r, column=c).alignment = alineacion_centro
            
    # Ajustar ligeramente el ancho de General para que se vea bien
    ws_general.column_dimensions['A'].width = 5
    ws_general.column_dimensions['B'].width = 25
    for c in range(3, 22):
        ws_general.column_dimensions[get_column_letter(c)].width = 15
    
    # Ajustar el texto en la fila 1 de las hoja General
    ajustar_texto_fila(ws_general, 1)

    # --- NUEVO: APLICAR GRILLA DELGADA (A1:U21) ---
    borde_delgado = Border(
        left=Side(style='thin', color='000000'),
        right=Side(style='thin', color='000000'),
        top=Side(style='thin', color='000000'),
        bottom=Side(style='thin', color='000000')
    )
    
    for r in range(1, 22):      # Filas de la 1 a la 21
        for c in range(1, 22):  # Columnas de la A(1) a la U(21)
            ws_general.cell(row=r, column=c).border = borde_delgado

    wb_nuevo.save(ARCHIVO_DESTINO)
    print(f"\n🚀 ¡Proceso completado! Archivo guardado en: {ARCHIVO_DESTINO}")

if __name__ == "__main__":
    consolidar_excels()

-> Procesando 'ANTONIO NARINO.xlsx' -> Destino: '15-Antonio Nariño'
-> Procesando 'BARRIOS UNIDOS.xlsx' -> Destino: '12-Barrios Unidos'
-> Procesando 'BOSA.xlsx' -> Destino: '7-Bosa'
-> Procesando 'CANDELARIA.xlsx' -> Destino: '18-La Candelaria'
-> Procesando 'CHAPINERO.xlsx' -> Destino: '2-Chapinero'
-> Procesando 'CIUDAD BOLIVAR.xlsx' -> Destino: '19-Ciudad Bolívar'
-> Procesando 'ENGATIVA.xlsx' -> Destino: '10-Engativá'
-> Procesando 'FONTIBON.xlsx' -> Destino: '9-Fontibón'
-> Procesando 'KENNEDY.xlsx' -> Destino: '8-Kennedy'
-> Procesando 'MARTIRES.xlsx' -> Destino: '14-Los Mártires'
-> Procesando 'PUENTE ARANDA.xlsx' -> Destino: '16-Puente Aranda'
-> Procesando 'RAFAEL URIBE.xlsx' -> Destino: '17-Rafael Uribe'
-> Procesando 'SAN CRISTOBAL.xlsx' -> Destino: '4-San Cristóbal'
-> Procesando 'SANTA FE.xlsx' -> Destino: '3-Santa Fe'
-> Procesando 'SUBA.xlsx' -> Destino: '11-Suba'
-> Procesando 'SUMAPAZ.xlsx' -> Destino: '20-Sumapaz'
-> Procesando 'TEUSAQUILLO.xlsx' -> Destino: '13-Teus

#### Entrega 18 03 2026

SIPSE debe llegar hasta "bk" </br>
SECOPII debe llegar hasta "ci" </br>
</br>
</br>
Si SECOP II llega hasta "cf", entonces poner 3 columnas vacías en "aq", "ar", "as" </br>
===================================================</br>
Alcaldía Rafael Uribe tiene solo SIPSE</br>
Alcaldía Rafael Uribe Uribe tiene solo SECOP</br>
</br>
Eliminar Nivel central</br>
Eliminar Rafael Uribe Uribe</br>

In [ ]:
import pandas as pd
import unicodedata
import re
import difflib
import os
import tkinter as tk
from tkinter import filedialog, messagebox

# ==========================================
# 1. FUNCIONES DE LIMPIEZA Y COMPARACIÓN
# ==========================================

def normalizar_texto(texto):
    """Limpia el texto: quita nulos, espacios, pasa a mayúsculas, quita tildes y artículos"""
    if pd.isna(texto):
        return ""
    # Convertir a texto, mayúsculas y quitar espacios a los lados
    texto = str(texto).upper().strip()
    # Quitar tildes/acentos
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    # Quitar artículos al inicio para igualar "LOS MARTIRES" con "MARTIRES"
    texto = re.sub(r'^(EL|LA|LOS|LAS)\s+', '', texto)
    return texto

def unificar_localidades(lista_localidades, similitud_minima=0.85):
    """Agrupa nombres similares bajo un mismo nombre 'canónico'"""
    nombres_canonicos = {}
    for loc in lista_localidades:
        if not loc: continue # Saltar vacíos
        
        # Buscar si ya existe un nombre muy parecido en nuestro diccionario
        coincidencias = difflib.get_close_matches(loc, nombres_canonicos.keys(), n=1, cutoff=similitud_minima)
        
        if coincidencias:
            nombres_canonicos[loc] = coincidencias[0]
        else:
            nombres_canonicos[loc] = loc
            
    return nombres_canonicos

# ==========================================
# 2. SELECCIÓN DE ARCHIVOS Y CARPETAS
# ==========================================

# Ocultar la ventana principal de Tkinter
root = tk.Tk()
root.withdraw()

print("Por favor, selecciona el archivo Excel de origen en la ventana emergente...")
archivo_origen = filedialog.askopenfilename(
    title="Selecciona el archivo Excel (.xlsx o .xls)",
    filetypes=[("Archivos de Excel", "*.xlsx *.xls *")]
)

if not archivo_origen:
    print("No se seleccionó ningún archivo. Cancelando proceso.")
    exit()

print("Por favor, selecciona la carpeta donde deseas guardar los nuevos archivos...")
carpeta_destino = filedialog.askdirectory(title="Selecciona la carpeta de destino")

if not carpeta_destino:
    print("No se seleccionó ninguna carpeta. Cancelando proceso.")
    exit()

# ==========================================
# 3. LECTURA Y PROCESAMIENTO DEL EXCEL
# ==========================================

try:
    print(f"\nLeyendo el archivo: {archivo_origen}")
    xls = pd.ExcelFile(archivo_origen)
    nombres_hojas = xls.sheet_names
    
    # Buscar hojas independientemente de mayúsculas/minúsculas
    hoja_sipse = next((h for h in nombres_hojas if h.upper() == "SIPSE"), None)
    hoja_secop = next((h for h in nombres_hojas if h.upper() == "SECOP II"), None)
    
    if not hoja_sipse or not hoja_secop:
        raise ValueError("El archivo no contiene las hojas 'SIPSE' y/o 'SECOP II'. Verifica el archivo.")

    # Leer las hojas
    df_sipse = pd.read_excel(xls, sheet_name=hoja_sipse)
    df_secop = pd.read_excel(xls, sheet_name=hoja_secop)
    
    # Identificar columnas (SIPSE: BJ es la columna 62, índice 61 | SECOP II: A es índice 0)
    # Validamos que SIPSE tenga suficientes columnas
    if len(df_sipse.columns) <= 61:
        raise ValueError("La hoja SIPSE no tiene suficientes columnas para llegar a la columna BJ.")
        
    col_sipse_nombre = df_sipse.columns[61] 
    col_secop_nombre = df_secop.columns[0]
    
    print("Normalizando nombres de localidades...")
    # Crear columnas auxiliares limpias
    df_sipse['Loc_Limpia'] = df_sipse[col_sipse_nombre].apply(normalizar_texto)
    df_secop['Loc_Limpia'] = df_secop[col_secop_nombre].apply(normalizar_texto)
    
    # Obtener todas las localidades únicas de ambas hojas
    todas_localidades = list(set(df_sipse['Loc_Limpia'].unique()) | set(df_secop['Loc_Limpia'].unique()))
    
    # Unificar nombres similares (Ej: MARTIRES y LOS MARTIRES)
    mapa_canonico = unificar_localidades(todas_localidades)
    
    # Aplicar el nombre unificado a los DataFrames
    df_sipse['Loc_Final'] = df_sipse['Loc_Limpia'].map(mapa_canonico)
    df_secop['Loc_Final'] = df_secop['Loc_Limpia'].map(mapa_canonico)
    
    # Obtener la lista final de localidades a exportar
    localidades_finales = set(mapa_canonico.values())
    
    print(f"Se detectaron {len(localidades_finales)} localidades únicas. Generando archivos...")
    
    # ==========================================
    # 4. CREACIÓN DE LOS ARCHIVOS .XLSX
    # ==========================================
    
    archivos_creados = 0
    for loc in localidades_finales:
        if not loc: continue # Evitar crear un archivo para localidades vacías
        
        # Filtrar datos (y eliminar las columnas auxiliares de limpieza que creamos)
        filtro_sipse = df_sipse[df_sipse['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        filtro_secop = df_secop[df_secop['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        
        # Solo crear el archivo si hay datos en alguna de las dos hojas
        if not filtro_sipse.empty or not filtro_secop.empty:
            # Reemplazar caracteres no válidos para nombres de archivos
            nombre_archivo_seguro = re.sub(r'[\\/*?:"<>|]', "", loc)
            ruta_salida = os.path.join(carpeta_destino, f"{nombre_archivo_seguro}.xlsx")
            
            with pd.ExcelWriter(ruta_salida, engine='openpyxl') as writer:
                filtro_sipse.to_excel(writer, sheet_name="SIPSE", index=False)
                filtro_secop.to_excel(writer, sheet_name="SECOP II", index=False)
                
            archivos_creados += 1
            print(f" -> Creado: {nombre_archivo_seguro}.xlsx")

    print(f"\n¡Proceso Finalizado Exitosamente! Se crearon {archivos_creados} archivos en: {carpeta_destino}")
    messagebox.showinfo("Éxito", f"Se generaron {archivos_creados} archivos correctamente.")

except Exception as e:
    print(f"\nOcurrió un error: {e}")
    messagebox.showerror("Error", f"Ocurrió un error procesando el archivo:\n{e}")

finally:
    # ==========================================
    # 5. CERRAR TKINTER (¡Esto evita que se quede colgado!)
    # ==========================================
    root.destroy()

Por favor, selecciona el archivo Excel de origen en la ventana emergente...
Por favor, selecciona la carpeta donde deseas guardar los nuevos archivos...

Leyendo el archivo: C:/Users/Pc/Desktop/Meta 7 - FDL 02 ABRIL/Datos Virgenes/Corte 20 04 - sipse vs secop ii.xls
Normalizando nombres de localidades...
Se detectaron 22 localidades únicas. Generando archivos...
 -> Creado: USAQUEN.xlsx
 -> Creado: BOSA.xlsx
 -> Creado: CANDELARIA.xlsx
 -> Creado: KENNEDY.xlsx
 -> Creado: CIUDAD BOLIVAR.xlsx
 -> Creado: SAN CRISTOBAL.xlsx
 -> Creado: BARRIOS UNIDOS.xlsx
 -> Creado: FONTIBON.xlsx
 -> Creado: TUNJUELITO.xlsx
 -> Creado: ANTONIO NARINO.xlsx
 -> Creado: SANTA FE.xlsx
 -> Creado: TEUSAQUILLO.xlsx
 -> Creado: CHAPINERO.xlsx
 -> Creado: PUENTE ARANDA.xlsx
 -> Creado: RAFAEL URIBE.xlsx
 -> Creado: SUMAPAZ.xlsx
 -> Creado: USME.xlsx
 -> Creado: SDG - NIVEL CENTRAL.xlsx
 -> Creado: SUBA.xlsx
 -> Creado: RAFAEL URIBE URIBE.xlsx
 -> Creado: MARTIRES.xlsx
 -> Creado: ENGATIVA.xlsx

¡Proceso Finaliz

Metodología para atacar la similitud entre SIPSE vs BOGDATA
1. Se captura el valor de la cédula de la persona en SIPSE y se busca en el archivo de RP de Bogdata
2. Se pueden encontrar más de un registro con el mismo número de cédula. Hay que tener en cuenta que varios de ellos son por adiciones y otros por obligaciones por pagar de vigencias anteriores. <br>
2.1. Se filtran y se toma solo el rubro que no tiene que ver con lo mencionado anteriormente. <br>
2.2. Se aplica un segundo filtro en la cloumna "Z" de "Texto Id Proyecto" donde se puede encontrar el nombre de la localidad eg. "PMR LOS MARTIRES". Con ello se debe comparar el nombre de la localidad con el nombre del archivo que es el de la localidad.
3. De ese registro, se captura el número, valor y fecha del RP y Código interno CDP
4. En el archivo de CDP de Bogdata se busca por el código interno CDP y número CDP. De ese registro se capturan el número, valor y fecha del CDP
5. Con los 6 valores capturados, se comparan con los 6 registrados en SIPSE.

In [10]:
dict_cdp

{('3039', '869313'): ('3039', '139400', '11/03/2026'),
 ('2181', '866374'): ('2181', '20691962', '02/03/2026'),
 ('2021', '868949'): ('2021', '1755250', '10/03/2026'),
 ('669', '869888'): ('669', '830000000', '16/03/2026'),
 ('668', '869887'): ('668', '100000000', '16/03/2026'),
 ('1215', '869889'): ('1215', '201300', '16/03/2026'),
 ('743', '850726'): ('743', '1090800', '20/01/2026'),
 ('2187', '869878'): ('2187', '299725696', '16/03/2026'),
 ('265', '850242'): ('265', '34720000', '21/01/2026'),
 ('1473', '857623'): ('1473', '67537260', '30/01/2026'),
 ('264', '850241'): ('264', '21266667', '21/01/2026'),
 ('310', '850289'): ('310', '33000000', '21/01/2026'),
 ('693', '850676'): ('693', '1013334', '20/01/2026'),
 ('170', '846517'): ('170', '987', '18/01/2026'),
 ('162', '826232'): ('162', '742912000', '09/01/2026'),
 ('712', '850695'): ('712', '1800000', '20/01/2026'),
 ('87', '811704'): ('87', '432600000', '06/01/2026'),
 ('764', '850747'): ('764', '2270000', '20/01/2026'),
 ('728', 

In [11]:
dict_rp

defaultdict(list,
            {'52060589': [('1',
               '77000000',
               '06/01/2026',
               '1',
               '804211',
               'PMR LOS MARTIRES')],
             '899999061': [('1084',
               '144',
               '06/01/2026',
               '1360',
               '811773',
               'PMR ENGATIVÁ'),
              ('2358',
               '6075000000',
               '04/02/2026',
               '113',
               '824965',
               'PMR CIUDAD BOLIVAR'),
              ('1053',
               '11495901',
               '04/02/2026',
               '861',
               '858577',
               'PMR TUNJUELITO'),
              ('879',
               '945000000',
               '12/02/2026',
               '683',
               '843690',
               'PMR BARRIOS UNIDOS'),
              ('1502',
               '274500000',
               '13/02/2026',
               '1873',
               '858401',
               'PMR SUMAPAZ

In [13]:
dict_cdp

{('738', '870274'): ('738', '2600000', '18/03/2026'),
 ('737', '870273'): ('737', '3000000', '18/03/2026'),
 ('736', '870271'): ('736', '183333', '18/03/2026'),
 ('735', '870270'): ('735', '200000000', '18/03/2026'),
 ('734', '870268'): ('734', '666667', '18/03/2026'),
 ('733', '870267'): ('733', '1880000', '18/03/2026'),
 ('732', '870266'): ('732', '2820000', '18/03/2026'),
 ('731', '870265'): ('731', '5640000', '18/03/2026'),
 ('730', '870264'): ('730', '1880000', '18/03/2026'),
 ('729', '870263'): ('729', '20000000', '18/03/2026'),
 ('727', '870261'): ('727', '2706667', '18/03/2026'),
 ('728', '870262'): ('728', '1345400', '18/03/2026'),
 ('726', '870259'): ('726', '2883000', '18/03/2026'),
 ('725', '870258'): ('725', '2820000', '18/03/2026'),
 ('724', '870257'): ('724', '3171300', '18/03/2026'),
 ('723', '870255'): ('723', '376000', '18/03/2026'),
 ('722', '870254'): ('722', '438667', '18/03/2026'),
 ('720', '870148'): ('720', '1065334', '18/03/2026'),
 ('719', '870147'): ('719', '

In [14]:
dict_rp

defaultdict(list,
            {'5': [('1210', '0', '16/02/2026', '1357', '1', '17'),
              ('1210', '0', '16/02/2026', '1357', '2', '17'),
              ('1', '0', '01/01/2026', '1', '1', '17'),
              ('3', '0', '01/01/2026', '3', '1', '17'),
              ('2190', '0', '27/03/2026', '1733', '1', '17'),
              ('2190', '0', '27/03/2026', '1733', '2', '17'),
              ('2190', '0', '27/03/2026', '1733', '3', '17'),
              ('2071', '0', '26/03/2026', '2188', '1', '17'),
              ('1507', '0', '03/03/2026', '1875', '1', '17'),
              ('1507', '0', '03/03/2026', '1875', '2', '17')],
             '2025': [('2050', '0', '05/03/2026', '2183', '1', '3'),
              ('2050', '0', '05/03/2026', '2183', '2', '3')],
             '2026': [('881', '0', '20/03/2026', '670', '1', '8')],
             '045': [('1', '0', '01/01/2026', '1', '1', '1'),
              ('1', '0', '01/01/2026', '1', '2', '1'),
              ('1', '0', '01/01/2026', '1', '3', '1'

In [ ]:
import os
import shutil
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
from collections import defaultdict
import re
import pandas as pd
import difflib # para busqueda aproximada
import datetime

#Para abrir explorador de archivos
import tkinter as tk
from tkinter import filedialog
import math
import unicodedata

# --- CONFIGURACIÓN DE CARPETAS ---
#CARPETA_ORIGEN = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO'
#CARPETA_DESTINO = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\DiagnóstivoV2_1'
CARPETA_ORIGEN = r'C:\\Users\\Pc\\Desktop\\META 7 - FDL 02 ABRIL' #Dell
CARPETA_DESTINO = r"C:\\Users\\Pc\\Desktop\\META 7 - FDL 02 ABRIL\\Diagnóstico" #Dell
#CARPETA_ORIGEN = r'D:\\Trabajo\\2026\\SecretariaDeGobierno\\BasesDeDatosSipseSecop\\2026_03_06\\DatosPartidos' #Predator
#CARPETA_DESTINO = r"D:\\Trabajo\\2026\\SecretariaDeGobierno\\BasesDeDatosSipseSecop\\2026_03_06\\DatosPartidos\\Diagnóstico" #PRedator

# --- RANGOS A OCULTAR (CORREGIDOS PARA USAR HIDDEN=TRUE) ---
# Nota: He ajustado 'A-AH' a 'Z-AH' en SECOP para NO ocultar las columnas L, M, T, X, Y que necesitas ver.
RANGOS_OCULTAR_SECOP = ['A-K', 'N-S', 'U-W', 'Z-AH', 'AJ-BR', 'BT-CI']
RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-AG','AJ-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']

# --- DEFINICIÓN DE FUNCIONES ---

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def ocultar_columnas_por_lista(ws, lista_rangos):
    """
    Recibe una lista de cadenas como ['A-K', 'M'] y establece la propiedad HIDDEN = True.
    """
    for item in lista_rangos:
        item = item.upper().strip()
        
        # Caso 1: Es un rango (ej: "A-K")
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio)
            idx_fin = column_index_from_string(fin)
            
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                # Esta es la forma correcta de ocultar en Excel
                ws.column_dimensions[letra].hidden = True 
                
        # Caso 2: Es una columna única (ej: "A")
        else:
            ws.column_dimensions[item].hidden = True

def aplicar_estilo_ferrocarril(ws):
    """
    Encabezados Azul Oscuro + Letra Blanca.
    Filas con patrón intercalado.
    """
    # 1. Encabezados
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    
    for cell in ws[1]:
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas (Patrón Ferrocarril)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    max_col = ws.max_column
    
    for i_fila, row in enumerate(ws.iter_rows(min_row=2, max_col=max_col), start=0):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF":
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for cell in row:
                cell.fill = relleno

def dejar_columna_blanca(ws, indice_columna):
    relleno_blanco = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    
    # Recorremos todas las celdas de la columna indicada
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=indice_columna, max_col=indice_columna):
        for cell in row:
            cell.fill = relleno_blanco

def aplicar_formato_condicional_alertas(ws, col_inicio_eval):
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    max_row = ws.max_row
    
    reglas = [
        (12, 'equal', '"Numero contrato NO coincide"'),
        (14, 'equal', '"Cedula NO coincide"'),
        (15, 'greaterThan', '1'),
        (16, 'greaterThan', '1'),
        (17, 'equal', '"No coinciden fechas"'),
        (18, 'equal', '"No coinciden fechas"'),
        (19, 'equal', '"No coinciden fechas"'),
        (20, 'equal', '"Duracion NO coincide"'),
        (21, 'equal', '"Valor NO coincide"'),
        (22, 'equal', '"El numero RP NO coincide"'),
        (23, 'equal', '"El valor RP NO coincide"'),
        (24, 'equal', '"La fecha RP NO coincide"'),
        (25, 'equal', '"El numero CDP NO coincide"'),
        (26, 'equal', '"El valor CDP NO coincide"'),
        (27, 'equal', '"La fecha CDP NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        rango = f"{letra_col}2:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

def normalizar_numero_contrato(valor):
    if valor is None: return 0
    texto = str(valor).strip()
    # Busca el primer grupo de números en la cadena
    match = re.search(r'\d+', texto)
    if match:
        return int(match.group())
    return 0

#Funciones Hoja "Para MEMO"
def es_error_fecha(val_sipse, val_secop):
    """Replica la fórmula de Excel de fechas para detectar discrepancias imitando tu cruce de día/mes."""
    s1 = str(val_sipse).strip() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip() if pd.notna(val_secop) and val_secop is not None else ""
    
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    
    # Parsear SIPSE 
    dt_sipse = None
    if isinstance(val_sipse, (datetime.datetime, datetime.date)):
        dt_sipse = datetime.date(val_sipse.year, val_sipse.month, val_sipse.day)
    else:
        try:
            dt_sipse = pd.to_datetime(s1.replace("-", "/"), dayfirst=True).date()
        except: pass

    # Parsear SECOP imitando la fórmula FECHA(AÑO, DIA, MES)
    dt_secop = None
    if isinstance(val_secop, (datetime.datetime, datetime.date)):
        try:
            dt_secop = datetime.date(val_secop.year, val_secop.day, val_secop.month)
        except: pass
    else:
        partes = s2.replace("-", "/").replace(" ", "/").split("/")
        if len(partes) >= 3:
            try:
                # Tu fórmula usa IZQUIERDA para el mes y EXTRAE para el día
                y = int(partes[2][-4:])
                m = int(partes[0])
                d = int(partes[1])
                dt_secop = datetime.date(y, m, d)
            except: pass
                
    if dt_sipse is not None and dt_secop is not None:
        return dt_sipse != dt_secop
        
    return s1 != s2 # Fallback a comparación de texto

def es_error_duracion(val_sipse, val_secop):
    """Replica la lógica de Duración."""
    s1 = str(val_sipse).strip().upper() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip().upper() if pd.notna(val_secop) and val_secop is not None else ""
    
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    return s1 != s2

def color_memo(err_tuple):
    """Devuelve los colores agrupados tipo pastel idénticos a los de tu imagen."""
    if err_tuple[0]: return "E4DFEC" # Beige/Grisáceo claro (Suscripción)
    if err_tuple[1]: return "B4C6E7" # Azul claro (Inicio)
    if err_tuple[2]: return "DDEBF7" # Gris claro (Fin)
    if err_tuple[3]: return "F4CCCC" # Rosa claro (Plazo)
    return "FFFFFF"

def normalizar_alfanumerico(valor):
    """Limpia códigos (como los de CDP) para evitar que pandas los lea como '1234.0'"""
    if pd.isna(valor) or valor is None:
        return ""
    texto = str(valor).strip().upper()
    if texto.endswith(".0"):
        texto = texto[:-2]
    return texto

def limpiar_dato(val):
    """Limpia el dato y elimina el '.0' que Pandas añade por defecto a los números."""
    if pd.isna(val) or val is None:
        return ""
    if isinstance(val, (datetime.datetime, datetime.date)):
        return val.strftime('%d/%m/%Y')
    
    val_str = str(val).strip()
    if val_str.endswith(".0"):
        val_str = val_str[:-2]
    return val_str

def normalizar_cedula(valor):
    """Limpia la cédula quitando letras y asegurando que sea un número exacto."""
    val_str = limpiar_dato(valor)
    return re.sub(r'\D', '', val_str)

def obtener_valor_seguro(row, indice):
    """Evita errores si el archivo de Excel viene con columnas faltantes."""
    try:
        return limpiar_dato(row.iloc[indice])
    except IndexError:
        return ""

def cargar_datos_bogdata():
    """Abre el explorador y procesa RP (por cédula) y CDP (por Nro + Cod Interno)."""
    root = tk.Tk()
    root.withdraw() 
    root.attributes('-topmost', True) 
    
    print("\n--- SELECCIÓN BOGDATA ---")
    print("1. Por favor selecciona el archivo de CDP (Excel)...")
    ruta_cdp = filedialog.askopenfilename(title="Seleccionar archivo CDP", filetypes=[("Excel files", "*.xlsx *.xls")])
    
    print("2. Por favor selecciona el archivo de RP (Excel)...")
    ruta_rp = filedialog.askopenfilename(title="Seleccionar archivo RP", filetypes=[("Excel files", "*.xlsx *.xls")])
         
    dict_cdp = {}
    dict_rp = defaultdict(list) # <--- Ahora es un diccionario que guarda listas

    # --- PROCESAR RP ---
    if ruta_rp:
        print(f"  -> Procesando archivo RP...")
        df_rp = pd.read_excel(ruta_rp, sheet_name="Data")
        # Índices: F=5(Fec), O=14(NumCDP), P=15(NumRP), R=17(Filtro1), Z=25(Localidad), AE=30(Ced), AL=37(Filtro2), AN=39(ValRP), AS=44(CodIntCDP)
        for _, row in df_rp.iterrows():
            val_r = obtener_valor_seguro(row, 17)
            digitos_r = re.sub(r'\D', '', val_r)
            
            # 1. Filtro 1: Columna R >= 7 dígitos
            if len(digitos_r) >= 7:
                
                # 2. Filtro 2: Columna AL NO debe ser mayor a 0
                val_al = obtener_valor_seguro(row, 37) # Columna AL
                try:
                    # Convertimos a número (quitando comas si las hay)
                    num_al = float(str(val_al).replace(',', ''))
                except ValueError:
                    # Si la celda está vacía o tiene letras, asumimos 0 para que no falle
                    num_al = 0.0 
                
                # Solo continuamos si AL es menor o igual a 0
                if num_al == 0:
                    cedula_raw = obtener_valor_seguro(row, 30)
                    llave_cedula = normalizar_cedula(cedula_raw)
                    
                    # Guardamos TODOS los RP válidos que encuentre de esa persona
                    if llave_cedula:
                        rp_num = obtener_valor_seguro(row, 15) # P
                        rp_val = obtener_valor_seguro(row, 39) # AN
                        rp_fec = obtener_valor_seguro(row, 5)  # F
                        cdp_num = obtener_valor_seguro(row, 14) # O
                        cdp_cod = obtener_valor_seguro(row, 44) # AS
                        
                        # Capturamos la localidad (Columna Z)
                        localidad_col_z = obtener_valor_seguro(row, 25) # Z
                        
                        dict_rp[llave_cedula].append((rp_num, rp_val, rp_fec, cdp_num, cdp_cod, localidad_col_z))

    # --- PROCESAR CDP ---
    if ruta_cdp:
        print(f"  -> Procesando archivo CDP...")
        df_cdp = pd.read_excel(ruta_cdp, sheet_name="Data")
        # Índices: H=7(NumCDP), Q=16(ValCDP), U=20(CodIntCDP), AC=28(FecCDP)
        for _, row in df_cdp.iterrows():
            cdp_num = obtener_valor_seguro(row, 7)  # H
            cdp_cod = obtener_valor_seguro(row, 20) # U
            
            # La llave maestra del CDP es la combinación de Número (H) + Código interno (U)
            llave_cdp = (cdp_num, cdp_cod)
            
            if llave_cdp != ("", "") and llave_cdp not in dict_cdp:
                cdp_val = obtener_valor_seguro(row, 16) # Q
                cdp_fec = obtener_valor_seguro(row, 28) # AC
                
                dict_cdp[llave_cdp] = (cdp_num, cdp_val, cdp_fec)

    return dict_cdp, dict_rp

def normalizar_texto_localidad(texto):
    """Elimina espacios, tildes y pasa a mayúsculas para un match perfecto."""
    if pd.isna(texto) or texto is None:
        return ""
    t = str(texto).upper().replace(" ", "")
    # Quitar tildes (ej: NARIÑO -> NARINO, BOGOTÁ -> BOGOTA)
    t = ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')
    return t

# --- LÓGICA PRINCIPAL ---
global dict_cdp
global dict_rp
global metricas
def procesar_archivos():
    global metricas
    if not os.path.exists(CARPETA_DESTINO):
        os.makedirs(CARPETA_DESTINO)

    #Cargar Bogdata en memoria antes del ciclo principal
    global dict_cdp
    global dict_rp
    dict_cdp, dict_rp = cargar_datos_bogdata()

    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('ANTONIO NARINO.xlsx', 'ANTONIO NARINO.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('SANTA FE.xlsx', 'SANTA FE.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('CANDELARIA.xlsx', 'CANDELARIA.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('RAFAEL URIBE.xlsx', 'RAFAEL URIBE.xls'))]
    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('.xlsx', '.xls'))]
    print(f"Se encontraron {len(archivos)} archivos de Excel.")

    for archivo in archivos:
        ruta_orig = os.path.join(CARPETA_ORIGEN, archivo)
        
        # Definimos la ruta de destino siempre con extensión .xlsx
        nombre_base = os.path.splitext(archivo)[0]
        ruta_dest = os.path.join(CARPETA_DESTINO, nombre_base + ".xlsx")
        
        print(f"Procesando: {archivo}...")

        # --- PASO DE CONVERSIÓN SI ES .XLS ---
        if archivo.lower().endswith('.xls'):
            try:
                # Leemos el .xls y lo guardamos inmediatamente como .xlsx
                # sheet_name=None lee todas las pestañas
                conversion = pd.read_excel(ruta_orig, sheet_name=None, engine='xlrd')
                with pd.ExcelWriter(ruta_dest, engine='openpyxl') as writer:
                    for nombre_hoja, datos in conversion.items():
                        datos.to_excel(writer, sheet_name=nombre_hoja, index=False)
                print(f"  -> Convertido de .xls a .xlsx con éxito.")
            except Exception as e:
                print(f"  Error convirtiendo {archivo}: {e}")
                continue
        else:
            # Si ya es .xlsx, solo lo copiamos como hacías antes
            shutil.copy2(ruta_orig, ruta_dest)
        # -------------------------------------

        try:
            # Ahora openpyxl siempre abrirá un archivo .xlsx (el copiado o el convertido)
            wb = openpyxl.load_workbook(ruta_dest)
        except Exception as e:
            print(f"   Error al abrir el archivo con openpyxl: {e}")
            continue

        # ==========================================
        # PARTE 1: Procesar SECOP II (Crear aux1)
        # ==========================================
        nombre_real_secop = next(n for n in wb.sheetnames if n.lower() == 'secop ii')
        ws_origen = wb[nombre_real_secop]
        
        if "SECOP II aux1" in wb.sheetnames:
            del wb["SECOP II aux1"]
            
        ws_secop_aux = wb.copy_worksheet(ws_origen)
        ws_secop_aux.title = "SECOP II aux1"

        filas = []
        for row in ws_secop_aux.iter_rows(min_row=2, values_only=True):
            filas.append(list(row))
        
        filas_ordenadas = sorted(filas, key=lambda x: normalizar_numero_contrato(x[11] if len(x) > 11 else 0))

        for row in ws_secop_aux.iter_rows(min_row=2):
            for cell in row:
                cell.value = None

        for i_fila, datos_fila in enumerate(filas_ordenadas, start=2):
            for i_col, valor in enumerate(datos_fila, start=1):
                ws_secop_aux.cell(row=i_fila, column=i_col, value=valor)

        # Para crear columnas "Columna1" a "Columna8" en la hoja "SECOP II aux1"
        max_col_secop = ws_secop_aux.max_column
        ws_secop_aux.cell(row=1, column=max_col_secop + 1, value="Columna1")
        ws_secop_aux.cell(row=1, column=max_col_secop + 2, value="Columna2")
        col_L = get_column_letter(12)
        
        for i in range(2, ws_secop_aux.max_row + 1):
            # Fórmula Columna 1
            f1 = (
                f'=IFERROR(MID({col_L}{i}, '
                f'MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), '
                f'MIN(IFERROR(FIND(" ", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999), '
                f'IFERROR(FIND("-", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999)) '
                f'- MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), "")'
            )
            ws_secop_aux.cell(row=i, column=max_col_secop + 1, value=f1)

            # Fórmula Columna 2 (Extrae número entre paréntesis)
            f2 = f'=IFERROR(MID({col_L}{i}, FIND("(", {col_L}{i}) + 1, FIND(")", {col_L}{i}) - FIND("(", {col_L}{i}) - 1), "")'
            ws_secop_aux.cell(row=i, column=max_col_secop + 2, value=f2)

        # ESTILOS SECOP
        print("  -> Aplicando estilos y ocultando columnas en SECOP II aux1...")
        aplicar_estilo_ferrocarril(ws_secop_aux)
        ocultar_columnas_por_lista(ws_secop_aux, RANGOS_OCULTAR_SECOP)

        # ==========================================
        # PARTE 2 y 3: ALINEACIÓN EN 'ANALISIS'
        # ==========================================
        print("  -> Generando hoja 'Analisis'...")

        nombre_real_sipse = next(n for n in wb.sheetnames if n.lower() == 'sipse')
        ws_sipse = wb[nombre_real_sipse]
        
        if 'Analisis' in wb.sheetnames:
            del wb['Analisis']
        ws_analisis = wb.create_sheet("Analisis")
        
        dict_sipse = defaultdict(list)
        dict_secop = defaultdict(list)
        
        filas_sipse = list(ws_sipse.values)
        header_sipse = filas_sipse[0] if filas_sipse else []
        for fila in filas_sipse[1:]:
            contrato = normalizar_numero_contrato(fila[3] if len(fila) > 3 else 0)
            dict_sipse[contrato].append(fila)

        filas_secop = list(ws_secop_aux.values)
        header_secop = filas_secop[0] if filas_secop else []
        for fila in filas_secop[1:]:
            contrato = normalizar_numero_contrato(fila[11] if len(fila) > 11 else 0)
            dict_secop[contrato].append(fila)

        ws_analisis.append(header_sipse) 
        col_inicio_secop = column_index_from_string("BM")
        for i, val in enumerate(header_secop):
            ws_analisis.cell(row=1, column=col_inicio_secop + i, value=val)

        todos_contratos = sorted(list(set(dict_sipse.keys()) | set(dict_secop.keys())))
        current_row = 2 
        
        for contrato in todos_contratos:
            filas_de_sipse = dict_sipse.get(contrato, [])
            filas_de_secop = dict_secop.get(contrato, [])
            max_filas = max(len(filas_de_sipse), len(filas_de_secop))
            
            for i in range(max_filas):
                if i < len(filas_de_sipse):
                    datos = filas_de_sipse[i]
                    for c_idx, valor in enumerate(datos, start=1):
                        ws_analisis.cell(row=current_row, column=c_idx, value=valor)
                
                if i < len(filas_de_secop):
                    datos = filas_de_secop[i]
                    for c_idx, valor in enumerate(datos):
                        col_excel = col_inicio_secop + c_idx
                        if c_idx == len(datos) - 2 and str(valor).startswith('='):
                             # Pega la fórmula de Columna1 (Ahora es la penúltima)
                             nueva_formula1 = (
                                f'=IFERROR(MID(BX{current_row}, '
                                f'MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), '
                                f'MIN(IFERROR(FIND(" ", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999), '
                                f'IFERROR(FIND("-", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999)) '
                                f'- MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), "")'
                            )
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula1)
                        elif c_idx == len(datos) - 1 and str(valor).startswith('='):
                             # Pega la fórmula de Columna2 (Extrae entre paréntesis)
                             nueva_formula2 = f'=IFERROR(MID(BX{current_row}, FIND("(", BX{current_row}) + 1, FIND(")", BX{current_row}) - FIND("(", BX{current_row}) - 1), "")'
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula2)
                        else:
                            ws_analisis.cell(row=current_row, column=col_excel, value=valor)
                current_row += 1

        # ==========================================
        # PARTE 4: Columnas de Evaluación
        # ==========================================
        print("  -> Agregando columnas de evaluación y estilos...")
        
        ultima_col_actual = ws_analisis.max_column
        col_inicio_eval = ultima_col_actual + 1
        ultima_fila_analisis = ws_analisis.max_row
        
        headers_nuevos = [
            "Columna3\n# RP\n(SIPSE)", "Columna4\nValor RP\n(SIPSE)", "Columna5\nFecha RP\n(SIPSE)", "Columna6\n# CDP\n(SIPSE)", "Columna7\nValor CDP\n(SIPSE)", "Columna8\nFecha CDP\n(SIPSE)", "Col.9\n# RP\n(BOGD)", "Col.10\nValor RP\n(BOGD)", "Col.11\nFecha RP\n(BOGD)", "Col.12\n# CDP\n(BOGD)", "Col.13\nValor CDP\n(BOGD)", "Col.14\nFecha CDP\n(BOGD)",
            "Ev. Nro contrato", "Aux Nro contrato", "Ev. Cédula", "Repeticiones contrato SIPSE", "Repeticiones contrato SECOP", "Fecha suscripción", "Fecha inicio", "Fecha fin", "Duración contratos", "Valor contratos", "Ev. # RP", "Ev. Valor RP", "Ev. Fecha RP", "Ev. # CDP", "Ev. Valor CDP", "Ev. Fecha CDP"
        ]

        #ESCRIBIR LOS TITULOS EN LOS HEADERS
        for idx, titulo in enumerate(headers_nuevos):
            ws_analisis.cell(row=1, column=col_inicio_eval + idx, value=titulo)
        
        #AÑADIR COLUMNAS DESGLOSADAS CDP y RP (SIPSE)
        for i in range(2, ultima_fila_analisis + 1):
            # 🔹 COLUMNA 3 - RP NUMERO (SIPSE)
            f3 = f'=IFERROR(TRIM(RIGHT(SUBSTITUTE(TRIM(LEFT(AI{i}, FIND("Fecha", AI{i})-1)), " ", REPT(" ", 100)), 100)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval, value=f3)
            # 🔹 COLUMNA 4 - RP VALOR (SIPSE)
            f4 = f'=IFERROR(TRIM(MID(AI{i}, FIND("Valor ", AI{i}) + 6, IFERROR(FIND(";", AI{i}), LEN(AI{i})+1) - FIND("Valor ", AI{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 1, value=f4)
            # 🔹 COLUMNA 5 - RP FECHA (SIPSE)
            f5 = f'=IFERROR(TRIM(MID(AI{i}, FIND("Fecha ", AI{i}) + 6, FIND(" Valor", AI{i}) - FIND("Fecha ", AI{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 2, value=f5)
            # 🔹 COLUMNA 6 - CDP NUMERO (SIPSE)
            f6 = f'=IFERROR(TRIM(RIGHT(SUBSTITUTE(TRIM(LEFT(AH{i}, FIND("Fecha", AH{i})-1)), " ", REPT(" ", 100)), 100)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 3, value=f6)
            # 🔹 COLUMNA 7 - CDP VALOR (SIPSE)
            f7 = f'=IFERROR(TRIM(MID(AH{i}, FIND("Valor ", AH{i}) + 6, IFERROR(FIND(";", AH{i}), LEN(AH{i})+1) - FIND("Valor ", AH{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 4, value=f7)
            # 🔹 COLUMNA 8 - CDP FECHA (SIPSE)
            f8 = f'=IFERROR(TRIM(MID(AH{i}, FIND("Fecha ", AH{i}) + 6, FIND(" Valor", AH{i}) - FIND("Fecha ", AH{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 5, value=f8)


        # --- AÑADIR COLUMNAS DESGLOSADAS CDP y RP (BOGDATA) ---
        idx_col_cedula_cj = column_index_from_string("CJ")
        letra_ev_nro_contrato = get_column_letter(col_inicio_eval + 12) # Esto equivale a la columna 'FJ'
        
        # Limpiamos el nombre del archivo actual (Ej: "SANTA FE" -> "SANTAFE")
        localidad_actual_norm = normalizar_texto_localidad(nombre_base)
        
        for i in range(2, ultima_fila_analisis + 1):
            # 1. Leer cédula de la hoja en construcción (Columna CJ)
            cedula_actual_raw = ws_analisis.cell(row=i, column=idx_col_cedula_cj).value
            cedula_busqueda = normalizar_cedula(cedula_actual_raw)
            
            # 2. Buscar en RP (Trae una lista con todos los RPs de la persona)
            lista_rps = dict_rp.get(cedula_busqueda, [])
            rp_data_elegido = ("", "", "", "", "")
            
            if len(lista_rps) == 1:
                # Si solo tiene un contrato, lo elegimos sin pensarlo
                rp_data_elegido = lista_rps[0][:5]
            elif len(lista_rps) > 1:
                # Si tiene varios, buscamos el que coincida con la localidad
                for rp in lista_rps:
                    localidad_rp_norm = normalizar_texto_localidad(rp[5]) # El 6to dato es Columna Z
                    # Si "SANTAFE" está dentro de "PMRFDLDESANTAFE", hace match
                    if localidad_actual_norm in localidad_rp_norm:
                        rp_data_elegido = rp[:5]
                        break

            rp_num, rp_val, rp_fec, cdp_num_from_rp, cdp_cod_from_rp = rp_data_elegido
            
            # 3. Buscar en CDP (Usando las 2 llaves que trajo el RP: Num y Cod Interno)
            cdp_data = dict_cdp.get((cdp_num_from_rp, cdp_cod_from_rp), ("", "", ""))
            cdp_num_out, cdp_val_out, cdp_fec_out = cdp_data

            # 4. Enviar a Excel usando fórmula =SI() vinculada a la coincidencia de contrato (FJ)
            def inyectar_condicional(valor):
                val_str = str(valor).replace('"', '""') if valor else ""
                # Si en FJ dice 'Numero contrato coincide' o 'Contrato coincide sin SIPSE', muestra el dato; de lo contrario, deja la celda vacía.
                return f'=IF(OR({letra_ev_nro_contrato}{i}="Numero contrato coincide", {letra_ev_nro_contrato}{i}="Contrato coincide sin SIPSE"), "{val_str}", "")'

            # 🔹 COLUMNA 9 - RP # (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 6, value=inyectar_condicional(rp_num))
            # 🔹 COLUMNA 10 - RP VALOR (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 7, value=inyectar_condicional(rp_val))
            # 🔹 COLUMNA 11 - RP FECHA (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 8, value=inyectar_condicional(rp_fec))
            
            # 🔹 COLUMNA 12 - CDP # (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 9, value=inyectar_condicional(cdp_num_out))
            # 🔹 COLUMNA 13 - CDP VALOR (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 10, value=inyectar_condicional(cdp_val_out))
            # 🔹 COLUMNA 14 - CDP FECHA (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 11, value=inyectar_condicional(cdp_fec_out))
        # -------------------------------------------------------------

        for i in range(2, ultima_fila_analisis + 1):
            COL_SOLICITUD = "B"
            # 🔹 EV. NO CONTRATO
            # Se añade SUBSTITUTE(UPPER(...), "SIPSE", "") para omitir la palabra si la escriben por error.
            #f15 = f'''=IF(COUNTIF($FK$2:FK{i},FK{i})>1,"",IF(AND(D{i}<>"",EV{i}<>""),IF(EW{i}="","Contrato coincide sin SIPSE",IF(TRIM({COL_SOLICITUD}{i})=TRIM(EW{i}),"Numero contrato coincide","Numero contrato NO coincide")),IF(D{i}<>"","Contrato no existe en SECOP",IF(EV{i}<>"","Contrato no existe en SIPSE","No existe"))))'''
            f15 = f'''=IF(COUNTIF($FK$2:FK{i},FK{i})>1,"",
                IF(AND(D{i}<>"",EV{i}<>""),
                    IF(EW{i}="","Contrato coincide sin SIPSE",
                        IF(TRIM(SUBSTITUTE(UPPER({COL_SOLICITUD}{i}),"SIPSE",""))=TRIM(SUBSTITUTE(UPPER(EW{i}),"SIPSE","")),"Numero contrato coincide","Numero contrato NO coincide")
                    ),
                    IF(D{i}<>"","Contrato no existe en SECOP",
                        IF(EV{i}<>"","Contrato no existe en SIPSE","No existe")
                    )
                )
            )'''
            # Eliminamos los saltos de línea para que Excel lo lea como una sola fórmula (opcional, pero más seguro en Python)
            f15 = f15.replace('\n', '').replace('    ', '')
            ws_analisis.cell(row=i, column=col_inicio_eval + 12, value=f15)

            # 🔹 AUX NRO CONTRATO (EX) → llave única
            f16 = f'=IF(D{i}<>"",D{i},EV{i})'
            ws_analisis.cell(row=i, column=col_inicio_eval + 13, value=f16)

            # 🔹 CÉDULA
            f17 = f'=IF(AND(BY{i}="",M{i}=""),"",IF(OR(BY{i}="",M{i}=""),"",IF(BY{i}="-","Sin cedula en SECOP",IF(M{i}="-","Sin cedula en SIPSE",IF(VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(CJ{i}),"CC",""),"NIT",""),"TI",""))=VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(M{i}),"CC",""),"NIT",""),"TI","")),"Cedula coincide","Cedula NO coincide")))))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 14, value=f17)

            # 🔹 REPETICIONES SIPSE
            f18 = f'=IF(FJ{i}="","",COUNTIF(D$2:D${ultima_fila_analisis},D{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 15, value=f18)

            # 🔹 REPETICIONES SECOP
            f19 = f'=IF(FJ{i}="","",COUNTIF(EV$2:EV${ultima_fila_analisis},EV{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 16, value=f19)

            # 🔹 FECHA SUSCRIPCION
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CE{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CE{i}), DAY(CE{i}), MONTH(CE{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f20 = f'''=IF(AND(H{i}="",CE{i}=""),"",IF(AND(H{i}="-",CE{i}="-"),"Ambos sin fechas",IF(H{i}="-","sin fecha SIPSE",IF(CE{i}="-","sin fecha SECOP",IF(OR(H{i}="",CE{i}=""),"",IF(INT(VALUE(SUBSTITUTE(H{i},"-","/"))) =INT(IF(ISNUMBER(CE{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 17, value=f20)

            # 🔹 FECHA ACTA DE INICIO
            cf_clean = f'SUBSTITUTE(SUBSTITUTE(CF{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CF{i}), DAY(CF{i}), MONTH(CF{i}))'
            parte_es_texto = f'DATE(RIGHT({cf_clean}, 4), LEFT({cf_clean}, FIND("/", {cf_clean}) - 1), MID({cf_clean}, FIND("/", {cf_clean}) + 1, FIND("/", {cf_clean}, FIND("/", {cf_clean}) + 1) - FIND("/", {cf_clean}) - 1))'

            f21 = f'''=IF(AND(I{i}="",CF{i}=""),"",IF(AND(I{i}="-",CF{i}="-"),"Ambos sin fechas",IF(I{i}="-","sin fecha SIPSE",IF(CF{i}="-","sin fecha SECOP",IF(OR(I{i}="",CF{i}=""),"",IF(INT(VALUE(SUBSTITUTE(I{i},"-","/"))) =INT(IF(ISNUMBER(CF{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 18, value=f21)

            # 🔹 FECHA FIN
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CG{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CG{i}), DAY(CG{i}), MONTH(CG{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f22 = f'''=IF(AND(J{i}="",CG{i}=""),"",IF(AND(J{i}="-",CG{i}="-"),"Ambos sin fechas",IF(J{i}="-","sin fecha SIPSE",IF(CG{i}="-","sin fecha SECOP",IF(OR(J{i}="",CG{i}=""),"",IF(INT(VALUE(SUBSTITUTE(J{i},"-","/"))) =INT(IF(ISNUMBER(CG{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 19, value=f22)

            # 🔹 DURACIÓN
            f23 = f'''=IF(AND(R{i}="",EE{i}=""),"",IF(AND(R{i}="-",EE{i}="-"),"Ambos sin duracion",IF(R{i}="-","sin duracion en SECOP",IF(EE{i}="-","sin duracion en SIPSE",IF(OR(R{i}="",EE{i}=""),"",IF(TRIM(UPPER(R{i}))=TRIM(UPPER(EE{i})),"Duracion coincide","Duracion NO coincide"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 20, value=f23)

            # 🔹 VALOR CONTRATO (Comparación Robusta)
            f24 = f'''=IF(AND(P{i}="",CU{i}=""),"",
                IF(AND(P{i}="-",CU{i}="-"),"Ambos sin valor",
                IF(P{i}="-","Sin valor en SECOP",
                IF(CU{i}="-","Sin valor en SIPSE",
                IF(OR(P{i}="",CU{i}=""),"",
                IF(P{i}=CU{i},"Valor coincide","Valor NO coincide"))))))'''

            ws_analisis.cell(row=i, column=col_inicio_eval + 21, value=f24)

            c_rp_sip   = get_column_letter(col_inicio_eval)       # EX (# RP SIPSE)
            c_val_rp_s = get_column_letter(col_inicio_eval + 1)   # EY (Valor RP SIPSE)
            c_fec_rp_s = get_column_letter(col_inicio_eval + 2)   # EZ (Fecha RP SIPSE)
            c_cdp_sip  = get_column_letter(col_inicio_eval + 3)   # FA (# CDP SIPSE)
            c_val_cd_s = get_column_letter(col_inicio_eval + 4)   # FB (Valor CDP SIPSE)
            c_fec_cd_s = get_column_letter(col_inicio_eval + 5)   # FC (Fecha CDP SIPSE)
            
            c_rp_bog   = get_column_letter(col_inicio_eval + 6)   # FD (# RP BOGDATA)
            c_val_rp_b = get_column_letter(col_inicio_eval + 7)   # FE (Valor RP BOGDATA)
            c_fec_rp_b = get_column_letter(col_inicio_eval + 8)   # FF (Fecha RP BOGDATA)
            c_cdp_bog  = get_column_letter(col_inicio_eval + 9)   # FG (# CDP BOGDATA)
            c_val_cd_b = get_column_letter(col_inicio_eval + 10)  # FH (Valor CDP BOGDATA)
            c_fec_cd_b = get_column_letter(col_inicio_eval + 11)  # FI (Fecha CDP BOGDATA)

            # 🔹 Ev. # RP
            f25 = (f'=IF(AND(TRIM({c_rp_sip}{i})="", TRIM({c_rp_bog}{i})=""), "Sin numeros RP", '
                   f'IF(TRIM({c_rp_sip}{i})="", "Sin numero RP en SIPSE", '
                   f'IF(TRIM({c_rp_bog}{i})="", "Sin numero RP en BOGDATA", '
                   f'IF(TRIM({c_rp_sip}{i})=TRIM({c_rp_bog}{i}), "El numero RP coincide", "El numero RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 22, value=f25)

            # 🔹 Ev. Valor RP (Limpiando comas para compatibilidad Web)
            f26 = (f'=IF(AND(TRIM({c_val_rp_s}{i})="", TRIM({c_val_rp_b}{i})=""), "Sin valores RP", '
                   f'IF(TRIM({c_val_rp_s}{i})="", "Sin valor RP en SIPSE", '
                   f'IF(TRIM({c_val_rp_b}{i})="", "Sin valor RP en BOGDATA", '
                   f'IF(IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_rp_s}{i}), ",", "")),TRIM({c_val_rp_s}{i}))=IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_rp_b}{i}), ",", "")),TRIM({c_val_rp_b}{i})), "El valor RP coincide", "El valor RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 23, value=f26)

            # 🔹 Ev. Fecha RP
            f27 = (f'=IF(AND(TRIM({c_fec_rp_s}{i})="", TRIM({c_fec_rp_b}{i})=""), "Sin fechas RP", '
                   f'IF(TRIM({c_fec_rp_s}{i})="", "Sin fecha RP en SIPSE", '
                   f'IF(TRIM({c_fec_rp_b}{i})="", "Sin fecha RP en BOGDATA", '
                   f'IF(TRIM({c_fec_rp_s}{i})=TRIM({c_fec_rp_b}{i}), "La fecha RP coincide", "La fecha RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 24, value=f27)

            # 🔹 Ev. # CDP
            f28 = (f'=IF(AND(TRIM({c_cdp_sip}{i})="", TRIM({c_cdp_bog}{i})=""), "Sin numeros CDP", '
                   f'IF(TRIM({c_cdp_sip}{i})="", "Sin numero CDP en SIPSE", '
                   f'IF(TRIM({c_cdp_bog}{i})="", "Sin numero CDP en BOGDATA", '
                   f'IF(TRIM({c_cdp_sip}{i})=TRIM({c_cdp_bog}{i}), "El numero CDP coincide", "El numero CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 25, value=f28)

            # 🔹 Ev. Valor CDP (Limpiando comas para compatibilidad Web)
            f29 = (f'=IF(AND(TRIM({c_val_cd_s}{i})="", TRIM({c_val_cd_b}{i})=""), "Sin valores CDP", '
                   f'IF(TRIM({c_val_cd_s}{i})="", "Sin valor CDP en SIPSE", '
                   f'IF(TRIM({c_val_cd_b}{i})="", "Sin valor CDP en BOGDATA", '
                   f'IF(IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_cd_s}{i}), ",", "")),TRIM({c_val_cd_s}{i}))=IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_cd_b}{i}), ",", "")),TRIM({c_val_cd_b}{i})), "El valor CDP coincide", "El valor CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 26, value=f29)

            # 🔹 Ev. Fecha CDP
            f30 = (f'=IF(AND(TRIM({c_fec_cd_s}{i})="", TRIM({c_fec_cd_b}{i})=""), "Sin fechas CDP", '
                   f'IF(TRIM({c_fec_cd_s}{i})="", "Sin fecha CDP en SIPSE", '
                   f'IF(TRIM({c_fec_cd_b}{i})="", "Sin fecha CDP en BOGDATA", '
                   f'IF(TRIM({c_fec_cd_s}{i})=TRIM({c_fec_cd_b}{i}), "La fecha CDP coincide", "La fecha CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 27, value=f30)

        # --- ESTILOS ANALISIS ---
        print("  -> Aplicando estilos y ocultando columnas en Analisis...")
        aplicar_estilo_ferrocarril(ws_analisis)
        aplicar_formato_condicional_alertas(ws_analisis, col_inicio_eval)
        
        # Ocultar columnas hoja analisis
        ocultar_columnas_por_lista(ws_analisis, RANGOS_OCULTAR_ANALISIS)

        # ==========================================
        # PARTE 5: Crear HOJA RESULTADOS
        # ==========================================
        print("  -> Creando hoja 'Resultados'...")
        if 'Resultados' in wb.sheetnames:
            del wb['Resultados']
        ws_res = wb.create_sheet("Resultados")
        nombre_limpio = os.path.splitext(archivo)[0]
        ws_res['B1'] = f"Métricas alcaldía local {nombre_limpio}"
        
        letra_ev_nro = get_column_letter(col_inicio_eval + 12)       
        letra_ev_ced = get_column_letter(col_inicio_eval + 14)   
        letra_rep_sipse = get_column_letter(col_inicio_eval + 15)
        letra_rep_secop = get_column_letter(col_inicio_eval + 16)
        letra_fecha_suscripcion = get_column_letter(col_inicio_eval + 17)
        letra_fecha_inicio = get_column_letter(col_inicio_eval + 18)
        letra_fecha_fin = get_column_letter(col_inicio_eval + 19)
        letra_duracion = get_column_letter(col_inicio_eval + 20)
        letra_valor = get_column_letter(col_inicio_eval + 21)

        letra_num_RP = get_column_letter(col_inicio_eval + 22)
        letra_val_RP = get_column_letter(col_inicio_eval + 23)
        letra_fec_RP = get_column_letter(col_inicio_eval + 24)
        letra_num_CDP = get_column_letter(col_inicio_eval + 25)
        letra_val_CDP = get_column_letter(col_inicio_eval + 26)
        letra_fec_CDP = get_column_letter(col_inicio_eval + 27)

        metricas = [
            #("Nro de contratos registrados en SIPSE",'=SUMPRODUCT((Analisis!A2:A10000<>"")/COUNTIF(Analisis!A2:A10000,Analisis!A2:A10000&""))'),
            #("Nro de contratos registrados SECOP",'=SUMPRODUCT((Analisis!EV2:EV10000<>"")/COUNTIF(Analisis!EV2:EV10000,Analisis!EV2:EV10000&""))'),
            ("Nro de contratos registrados SIPSE",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos registrados SECOP",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos registrados en SECOP y SIPSE con numero de contrato igual", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")"),
            ("Nro de contratos solo en SECOP y no en SIPSE", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos solo en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos diferentes en SIPSE-SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato NO coincide\")"),
            ("Nro de contratos Cédula diferente", f"=COUNTIF('Analisis'!{letra_ev_ced}:{letra_ev_ced}, \"Cedula NO coincide\")"),
            ("Nro de contratos dobles en SIPSE", f"=COUNTIF('Analisis'!{letra_rep_sipse}:{letra_rep_sipse}, 2)"),
            ("Nro de contratos dobles en SECOP", f"=COUNTIF('Analisis'!{letra_rep_secop}:{letra_rep_secop}, 2)"),
            ("Fecha de acta de suscriccpión Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"Coincide fecha\")"),
            ("Fecha de acta de suscriccpión no Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"No coinciden fechas\")"),
            ("Fecha de acta de inicio Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Coincide fecha\")"),
            ("Fecha de acta de inicio no coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"No coinciden fechas\")"),
            ("Fecha de Fin conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"Coincide fecha\")"),
            ("Fecha de Fin no conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"No coinciden fechas\")"),
            ("Contratos que coinciden en Duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion Coincide\")"),
            ("Contratos que tienen diferente duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion NO coincide\")"),
            ("Contratos que tienen Fecha de inicio en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Sin fecha SECOP\")"),
            ("Datos en la matriz de seguimiento", f"=C2+C3"),
            ("Valor contrato coinciden", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor coincide\")"),
            ("Valor contrato no coincidenn", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor NO coincide\")"),
            ("Numero RP no coinciden", f"=COUNTIF('Analisis'!{letra_num_RP}:{letra_num_RP}, \"El numero RP NO coincide\")"),
            ("Valor RP no coinciden", f"=COUNTIF('Analisis'!{letra_val_RP}:{letra_val_RP}, \"El valor RP NO coincide\")"),
            ("Fecha RP no coinciden", f"=COUNTIF('Analisis'!{letra_fec_RP}:{letra_fec_RP}, \"La fecha RP NO coincide\")"),
            ("Numero CDP no coinciden", f"=COUNTIF('Analisis'!{letra_num_CDP}:{letra_num_CDP}, \"El numero CDP NO coincide\")"),
            ("Valor CDP no coinciden", f"=COUNTIF('Analisis'!{letra_val_CDP}:{letra_val_CDP}, \"El valor CDP NO coincide\")"),
            ("Fecha CDP no coinciden", f"=COUNTIF('Analisis'!{letra_fec_CDP}:{letra_fec_CDP}, \"La fecha CDP NO coincide\")")
        ]


        for idx, (texto, formula) in enumerate(metricas, start=2):
            ws_res.cell(row=idx, column=1, value=idx-1)
            ws_res.cell(row=idx, column=2, value=texto)
            ws_res.cell(row=idx, column=3, value=formula)
        
        # Pintar B2 de amarillo
        fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
        ws_res['B1'].fill = fill_amarillo
        ws_res.column_dimensions['B'].width = 100
        
        ws_analisis.column_dimensions['FJ'].width = 29
        ws_analisis.column_dimensions['FK'].width = 10
        ws_analisis.column_dimensions['FL'].width = 20
        ws_analisis.column_dimensions['FM'].width = 15
        ws_analisis.column_dimensions['FN'].width = 15
        ws_analisis.column_dimensions['FO'].width = 15
        ws_analisis.column_dimensions['FP'].width = 20
        ws_analisis.column_dimensions['FQ'].width = 20
        ws_analisis.column_dimensions['FR'].width = 18
        ws_analisis.column_dimensions['FS'].width = 14
        ws_analisis.column_dimensions['FT'].width = 25
        ws_analisis.column_dimensions['FU'].width = 25
        ws_analisis.column_dimensions['FV'].width = 25
        ws_analisis.column_dimensions['FW'].width = 25
        ws_analisis.column_dimensions['FX'].width = 25
        ws_analisis.column_dimensions['FY'].width = 25

        dejar_columna_blanca(ws_analisis, 64)# columna 64 es la columna "BL"

        #Ajustar el texto en la fila 1 de las hojas aux y Analisis
        ajustar_texto_fila(ws_secop_aux, 1)
        ajustar_texto_fila(ws_analisis, 1)

        # ==========================================
        # PARTE 6: Crear HOJA 'Para MEMO'
        # ==========================================
        print("  -> Creando hoja 'Para MEMO'...")
        if 'Para MEMO' in wb.sheetnames:
            del wb['Para MEMO']
        ws_memo = wb.create_sheet("Para MEMO")
        
        datos_memo = []
        
        # 1. Escanear fila por fila extrayendo los datos directos
        for i in range(2, ultima_fila_analisis + 1):
            val_b = ws_analisis.cell(row=i, column=2).value # B: SIPSE (Solicitud)
            val_d = ws_analisis.cell(row=i, column=4).value # D: No. Contrato
            
            # Extraer celdas origen según tu matriz (SIPSE y SECOP)
            susc_sipse = ws_analisis.cell(row=i, column=8).value  # H
            susc_secop = ws_analisis.cell(row=i, column=83).value # CE
            ini_sipse  = ws_analisis.cell(row=i, column=9).value  # I
            ini_secop  = ws_analisis.cell(row=i, column=84).value # CF
            fin_sipse  = ws_analisis.cell(row=i, column=10).value # J
            fin_secop  = ws_analisis.cell(row=i, column=85).value # CG
            dur_sipse  = ws_analisis.cell(row=i, column=18).value # R
            dur_secop  = ws_analisis.cell(row=i, column=135).value# EE
            
            # Evaluarlas en Python
            err_susc = es_error_fecha(susc_sipse, susc_secop)
            err_ini  = es_error_fecha(ini_sipse, ini_secop)
            err_fin  = es_error_fecha(fin_sipse, fin_secop)
            err_dur  = es_error_duracion(dur_sipse, dur_secop)
            
            # Si tiene al menos 1 error, lo enlistamos
            if err_susc or err_ini or err_fin or err_dur:
                llave_orden = (err_susc, err_ini, err_fin, err_dur)
                datos_memo.append([
                    val_b, val_d,
                    "X" if err_susc else "",
                    "X" if err_ini else "",
                    "X" if err_fin else "",
                    "X" if err_dur else "",
                    llave_orden # Llave secreta para ordenar
                ])
                
        # 2. Ordenar datos para que queden agrupados por color como en tu imagen
        datos_memo.sort(key=lambda x: x[6], reverse=True)
        
        # 3. Diseño de la Cabecera
        relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
        fuente_blanca = Font(color="FFFFFF", bold=True)
        ws_memo.merge_cells("A1:A2")
        ws_memo['A1'] = "SIPSE"
        ws_memo.merge_cells("B1:B2")
        ws_memo['B1'] = "No. Contrato"
        ws_memo.merge_cells("C1:F1")
        ws_memo['C1'] = "TIPOLOGÍA DE LA INCONSISTENCIA"
        
        ws_memo['C2'] = "Fecha suscripción"
        ws_memo['D2'] = "Fecha Inicio"
        ws_memo['E2'] = "Fecha Fin"
        ws_memo['F2'] = "Plazo"
        ws_memo['A1'].fill = relleno_rojo
        ws_memo['B1'].fill = relleno_rojo
        ws_memo['C1'].fill = relleno_rojo
        ws_memo['C2'].fill = relleno_rojo
        ws_memo['D2'].fill = relleno_rojo
        ws_memo['E2'].fill = relleno_rojo
        ws_memo['F2'].fill = relleno_rojo
        
        align_header = Alignment(horizontal="center", vertical="center")
        borde_delgado = Border(left=Side(style='thin', color='000000'), right=Side(style='thin', color='000000'), top=Side(style='thin', color='000000'), bottom=Side(style='thin', color='000000'))
        
        for fila_h in ws_memo['A1:F2']:
            for celda in fila_h:
                celda.font = fuente_blanca
                celda.alignment = align_header
                celda.border = borde_delgado
                
        # 4. Volcar datos extraídos
        fila_actual = 3
        for fila_datos in datos_memo:
            for col_idx in range(1, 7):
                celda = ws_memo.cell(row=fila_actual, column=col_idx, value=fila_datos[col_idx-1])
                celda.border = borde_delgado
                celda.alignment = align_header # Centrado para todo
                
            fila_actual += 1
            
        # 5. Ajustar anchos
        ws_memo.column_dimensions['A'].width = 15
        ws_memo.column_dimensions['B'].width = 15
        ws_memo.column_dimensions['C'].width = 20
        ws_memo.column_dimensions['D'].width = 15
        ws_memo.column_dimensions['E'].width = 15
        ws_memo.column_dimensions['F'].width = 15

        wb.save(ruta_dest)
        print("  -> Procesado exitosamente.")

if __name__ == "__main__":
    procesar_archivos()


--- SELECCIÓN BOGDATA ---
1. Por favor selecciona el archivo de CDP (Excel)...
2. Por favor selecciona el archivo de RP (Excel)...
  -> Procesando archivo RP...
  -> Procesando archivo CDP...
Se encontraron 1 archivos de Excel.
Procesando: RAFAEL URIBE.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 'Para MEMO'...
  -> Procesado exitosamente.


In [3]:
import os
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
import re
import unicodedata

# ================= CONFIGURACIÓN =================
#CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1' 
#ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1\Consolidado\Consolidado_Localidades.xlsx"
CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 02 ABRIL\Diagnóstico' # Dell
ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 02 ABRIL\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx" #Dell
#CARPETA_ORIGEN = r'D:\Trabajo\2026\SecretariaDeGobierno\BasesDeDatosSipseSecop\2026_03_06\DatosPartidos\Diagnóstico' # PRedator
#ARCHIVO_DESTINO = r"D:\Trabajo\2026\SecretariaDeGobierno\BasesDeDatosSipseSecop\2026_03_06\DatosPartidos\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx" #Predator

nombres_hojas = [
    "1-Usaquén", "2-Chapinero", "3-Santa Fe", "4-San Cristóbal",
    "5-Usme", "6-Tunjuelito", "7-Bosa", "8-Kennedy", "9-Fontibón",
    "10-Engativá", "11-Suba", "12-Barrios Unidos", "13-Teusaquillo",
    "14-Los Mártires", "15-Antonio Nariño", "16-Puente Aranda",
    "17-Rafael Uribe", "18-La Candelaria", "19-Ciudad Bolívar", "20-Sumapaz"
]

#RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']
# Aquí puedes añadir todas las columnas a las que quieras fijarles un ancho específico
ANCHOS_COLUMNAS = {
    'FM': 29,
    'FN': 10,
    'FO': 20,
    'FP': 15,
    'FQ': 15,
    'FR': 15,
    'FS': 20,
    'FT': 20,
    'FU': 18,
    'FV': 14,
    'FW': 25,
    'FX': 25,
    'FY': 25,
    'FZ': 25,
    'GA': 25,
    'GB': 25
}

# ================= FUNCIONES AUXILIARES =================
def normalizar_texto(texto):
    texto = texto.lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^\d+[-_ ]?', '', texto)
    texto = re.sub(r'\b(la|el|los|las)\b', '', texto).strip()
    return texto

def encontrar_hoja_destino(nombre_archivo, hojas_destino):
    archivo_norm = normalizar_texto(nombre_archivo)
    for hoja in hojas_destino:
        if normalizar_texto(hoja) in archivo_norm:
            return hoja
    return None

def ocultar_columnas_por_lista(ws, lista_rangos, offset=3):
    for item in lista_rangos:
        item = item.upper().strip()
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio) + offset
            idx_fin = column_index_from_string(fin) + offset
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                ws.column_dimensions[letra].hidden = True 
        else:
            idx = column_index_from_string(item) + offset
            letra = get_column_letter(idx)
            ws.column_dimensions[letra].hidden = True

def adaptar_formulas_resultados(formula):
    pattern = re.compile(r"'?Analisis'?!([A-Za-z]+)(\d*)(?::([A-Za-z]+)(\d*))?")
    def replacer(match):
        col1, row1, col2, row2 = match.groups()
        nueva_col1 = get_column_letter(column_index_from_string(col1) + 3)
        nueva_row1 = str(int(row1) + 21) if row1 else ""
        resultado = f"{nueva_col1}{nueva_row1}"
        if col2:
            nueva_col2 = get_column_letter(column_index_from_string(col2) + 3)
            nueva_row2 = str(int(row2) + 21) if row2 else ""
            resultado += f":{nueva_col2}{nueva_row2}"
        return resultado
    return pattern.sub(replacer, formula)

def desplazar_formulas_analisis(formula):
    formula = re.sub(r"'?Analisis'?!", "", formula, flags=re.IGNORECASE)
    pattern = re.compile(r'(\$?)([A-Za-z]{1,3})(\$?)(\d+)(?!\s*\()')
    def replacer(match):
        lock_col, col_str, lock_row, row_str = match.groups()
        nueva_col = get_column_letter(column_index_from_string(col_str) + 3)
        nueva_fila = str(int(row_str) + len(metricas)+2)
        return f"{lock_col}{nueva_col}{lock_row}{nueva_fila}"
    return pattern.sub(replacer, formula)

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# ================= FUNCIONES DE DISEÑO =================
def aplicar_estilo_ferrocarril(ws, start_row, max_row):
    """Aplica formato estilo ferrocarril solo a columnas E:U y CA:FH desde start_row."""
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    
    # Rango 1: E (5) a AL (38) | Rango 2: CA (79) a GB (185)
    def es_columna_objetivo(c):
        return (5 <= c <= 38) or (79 <= c <= 185)

    # 1. Encabezados (Fila 22)
    for col_idx in range(1, ws.max_column + 1):
        if es_columna_objetivo(col_idx):
            cell = ws.cell(row=start_row, column=col_idx)
            cell.fill = fill_header
            cell.font = font_header
            cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas de Datos (Patrón Ferrocarril)
    for i_fila, r_idx in enumerate(range(start_row + 1, max_row + 1)):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF": # Evitar procesar blanco para ahorrar memoria
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for col_idx in range(1, ws.max_column + 1):
                if es_columna_objetivo(col_idx):
                    ws.cell(row=r_idx, column=col_idx).fill = relleno

def aplicar_formato_condicional_alertas(ws, start_row, max_row):
    """Aplica reglas condicionales desde la columna EZ."""
    col_inicio_eval = column_index_from_string('FA')
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    
    reglas = [
        (12, 'equal', '"Numero contrato NO coincide"'),
        (14, 'equal', '"Cedula NO coincide"'),
        (15, 'greaterThan', '1'),
        (16, 'greaterThan', '1'),
        (17, 'equal', '"No coinciden fechas"'),
        (18, 'equal', '"No coinciden fechas"'),
        (19, 'equal', '"No coinciden fechas"'),
        (20, 'equal', '"Duracion NO coincide"'),
        (21, 'equal', '"Valor NO coincide"'),
        (22, 'equal', '"El numero RP NO coincide"'),
        (23, 'equal', '"El valor RP NO coincide"'),
        (24, 'equal', '"La fecha RP NO coincide"'),
        (25, 'equal', '"El numero CDP NO coincide"'),
        (26, 'equal', '"El valor CDP NO coincide"'),
        (27, 'equal', '"La fecha CDP NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        # Se aplica desde la fila siguiente al encabezado (23 en adelante)
        rango = f"{letra_col}{start_row + 1}:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

# ================= PROGRAMA PRINCIPAL =================
def consolidar_excels():
    global metricas
    wb_nuevo = openpyxl.Workbook()
    
    # 1. Configurar Hoja General
    ws_general = wb_nuevo.active
    ws_general.title = "General"
    ws_general.cell(row=1, column=1, value="Nro").font = Font(bold=True)
    ws_general.cell(row=1, column=2, value="Alcaldía local").font = Font(bold=True)
    
    # Pre-llenar las localidades en General con Hipervínculos
    for i, hoja in enumerate(nombres_hojas):
        num, nombre = hoja.split("-", 1)
        row_idx = i + 2
        ws_general.cell(row=row_idx, column=1, value=int(num))
        
        celda_loc = ws_general.cell(row=row_idx, column=2, value=nombre.strip())
        celda_loc.hyperlink = f"#'{hoja}'!A1"
        celda_loc.font = Font(color="0563C1", underline="single") 

    for nombre in nombres_hojas:
        wb_nuevo.create_sheet(title=nombre)

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith('.xlsx') and not f.startswith('~')]
    metricas_extraidas = False 

    for archivo in archivos:
        hoja_destino_nombre = encontrar_hoja_destino(archivo, nombres_hojas)
        
        if not hoja_destino_nombre:
            continue
            
        print(f"-> Procesando '{archivo}' -> Destino: '{hoja_destino_nombre}'")
        
        ruta_completa = os.path.join(CARPETA_ORIGEN, archivo)
        wb_origen = openpyxl.load_workbook(ruta_completa)
        ws_destino = wb_nuevo[hoja_destino_nombre]

        # --- TAREA A: Copiar de 'Resultados' ---
        if "Resultados" in wb_origen.sheetnames:
            ws_resultados = wb_origen["Resultados"]
            row_gen = nombres_hojas.index(hoja_destino_nombre) + 2
            col_gen = 3 
            
            for r_idx in range(0, len(metricas)): 
                titulo_metrica = ws_resultados.cell(row=2 + r_idx, column=2).value 
                
                if not metricas_extraidas:
                    ws_general.cell(row=1, column=col_gen, value=titulo_metrica).font = Font(bold=True)
                
                ws_general.cell(row=row_gen, column=col_gen, value=f"='{hoja_destino_nombre}'!C{r_idx+2}")
                col_gen += 1
                
            metricas_extraidas = True 
            
            for row in ws_resultados['A1':f'C{len(metricas) + 1}']:
                for cell in row:
                    dest_coord = cell.coordinate
                    valor = cell.value
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = adaptar_formulas_resultados(valor)
                    else:
                        ws_destino[dest_coord].value = valor
        else:
            print(f"  - Faltó hoja 'Resultados' en {archivo}")

        # --- TAREA B: Copiar de 'Analisis' a la fila 22 ---
        if "Analisis" in wb_origen.sheetnames:
            ws_analisis = wb_origen["Analisis"]
            ultima_fila = ws_analisis.max_row
            max_col_origen = min(ws_analisis.max_column, column_index_from_string('FY'))
            
            ws_destino.merge_cells(f"D{len(metricas) + 2}:BN{len(metricas) + 2}")
            celda_sipse = ws_destino[f"D{len(metricas) + 2}"]
            celda_sipse.value = "SIPSE"
            celda_sipse.fill = PatternFill(start_color="0070C0", end_color="0070C0", fill_type="solid") 
            celda_sipse.font = Font(color="FFFFFF", bold=True)
            celda_sipse.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells(f"BP{len(metricas) + 2}:EX{len(metricas) + 2}")
            celda_secop = ws_destino[f"BP{len(metricas) + 2}"]
            celda_secop.value = "SECOP"
            celda_secop.fill = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid") 
            celda_secop.font = Font(color="FFFFFF", bold=True)
            celda_secop.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells(f"EY{len(metricas) + 2}:GB{len(metricas) + 2}")
            celda_comparacion = ws_destino[f"EY{len(metricas) + 2}"]
            celda_comparacion.value = "COMPARACIÓN"
            celda_comparacion.fill = PatternFill(start_color="7e350e", end_color="7e350e", fill_type="solid") 
            celda_comparacion.font = Font(color="FFFFFF", bold=True)
            celda_comparacion.alignment = Alignment(horizontal="center", vertical="center")

            # Pegar los datos desde la fila 22
            for r_idx, row_origen in enumerate(ws_analisis.iter_rows(min_row=1, max_row=ultima_fila, min_col=1, max_col=max_col_origen), start=len(metricas) + 3):
                for c_idx, cell in enumerate(row_origen, start=4):
                    dest_coord = f"{get_column_letter(c_idx)}{r_idx}"
                    valor = cell.value
                    
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = desplazar_formulas_analisis(valor)
                    else:
                        ws_destino[dest_coord].value = valor
            
            # 1. Ocultar columnas
            ocultar_columnas_por_lista(ws_destino, RANGOS_OCULTAR_ANALISIS, offset=3)
            # 2. Aplicar anchos de columnas
            for letra_col, ancho in ANCHOS_COLUMNAS.items():
                ws_destino.column_dimensions[letra_col].width = ancho
            # 2.1 Ajustar el alto del texto en la fila de encabezados de las hojas de cada localidad
            ajustar_texto_fila(ws_destino, len(metricas) + 3)
            # 3. Aplicar estilo Ferrocarril (La tabla empieza en la 22)
            aplicar_estilo_ferrocarril(ws_destino, start_row=len(metricas) + 3, max_row=ultima_fila + len(metricas) + 2)
            # 4. Aplicar alertas condicionales
            aplicar_formato_condicional_alertas(ws_destino, start_row=len(metricas) + 3, max_row=ultima_fila + len(metricas) + 2)
            # 5 Pintar y ajustar tabla de cada alcaldía
            fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
            ws_destino['B1'].fill = fill_amarillo
            ws_destino.column_dimensions['B'].width = 100
            
            # --- NUEVO: BOTÓN DE REGRESO A GENERAL ---
            celda_volver = ws_destino['A1']
            celda_volver.value = "General"
            celda_volver.hyperlink = "#'General'!A1"
            celda_volver.font = Font(color="0563C1", underline="single", bold=True)
            celda_volver.fill = PatternFill(start_color="E7E6E6", end_color="E7E6E6", fill_type="solid") # Gris claro
            celda_volver.alignment = Alignment(horizontal="center", vertical="center")

        else:
            print(f"  - Faltó hoja 'Analisis' en {archivo}")
            
        wb_origen.close()

    # --- DISEÑO FINAL HOJA GENERAL ---
    # 1. Fondo Rojo para la fila de títulos A1:AC1
    relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
    fuente_blanca = Font(color="FFFFFF", bold=True)
    for col in range(1, 30): # De la A (1) a la AC (30)
        celda = ws_general.cell(row=1, column=col)
        celda.fill = relleno_rojo
        celda.font = fuente_blanca
        celda.alignment = Alignment(horizontal="center", vertical="center")

    # 2. Centrar los valores de C1:AC1
    alineacion_centro = Alignment(horizontal="center", vertical="center")
    for r in range(2, 22): # Filas 2 a 21
        for c in range(3, 29): # Columnas C (3) a AC (30)
            ws_general.cell(row=r, column=c).alignment = alineacion_centro
            
    # Ajustar ligeramente el ancho de General para que se vea bien
    ws_general.column_dimensions['A'].width = 5
    ws_general.column_dimensions['B'].width = 25
    for c in range(3, 22):
        ws_general.column_dimensions[get_column_letter(c)].width = 15
    
    # Ajustar el texto en la fila 1 de las hoja General
    ajustar_texto_fila(ws_general, 1)

    # --- NUEVO: APLICAR GRILLA DELGADA (A1:U21) ---
    borde_delgado = Border(
        left=Side(style='thin', color='000000'),
        right=Side(style='thin', color='000000'),
        top=Side(style='thin', color='000000'),
        bottom=Side(style='thin', color='000000')
    )
    
    for r in range(1, 22):      # Filas de la 1 a la 21
        for c in range(1, 30):  # Columnas de la A(1) a la AC(29)
            ws_general.cell(row=r, column=c).border = borde_delgado

    wb_nuevo.save(ARCHIVO_DESTINO)
    print(f"\n🚀 ¡Proceso completado! Archivo guardado en: {ARCHIVO_DESTINO}")

if __name__ == "__main__":
    consolidar_excels()

-> Procesando 'ANTONIO NARINO.xlsx' -> Destino: '15-Antonio Nariño'
-> Procesando 'BARRIOS UNIDOS.xlsx' -> Destino: '12-Barrios Unidos'
-> Procesando 'BOSA.xlsx' -> Destino: '7-Bosa'
-> Procesando 'CANDELARIA.xlsx' -> Destino: '18-La Candelaria'
-> Procesando 'CHAPINERO.xlsx' -> Destino: '2-Chapinero'
-> Procesando 'CIUDAD BOLIVAR.xlsx' -> Destino: '19-Ciudad Bolívar'
-> Procesando 'ENGATIVA.xlsx' -> Destino: '10-Engativá'
-> Procesando 'FONTIBON.xlsx' -> Destino: '9-Fontibón'
-> Procesando 'KENNEDY.xlsx' -> Destino: '8-Kennedy'
-> Procesando 'MARTIRES.xlsx' -> Destino: '14-Los Mártires'
-> Procesando 'PUENTE ARANDA.xlsx' -> Destino: '16-Puente Aranda'
-> Procesando 'RAFAEL URIBE.xlsx' -> Destino: '17-Rafael Uribe'
-> Procesando 'SAN CRISTOBAL.xlsx' -> Destino: '4-San Cristóbal'
-> Procesando 'SANTA FE.xlsx' -> Destino: '3-Santa Fe'
-> Procesando 'SUBA.xlsx' -> Destino: '11-Suba'
-> Procesando 'SUMAPAZ.xlsx' -> Destino: '20-Sumapaz'
-> Procesando 'TEUSAQUILLO.xlsx' -> Destino: '13-Teus

#### Entrega 27 04 2026

In [1]:
import pandas as pd
import unicodedata
import re
import difflib
import os
import tkinter as tk
from tkinter import filedialog, messagebox

# ==========================================
# 0. DEFINICIÓN DE ESTRUCTURA ESPERADA
# ==========================================
# Reemplaza los elementos de estas listas con los nombres EXACTOS 
# y en el ORDEN que necesitas para cada hoja.
COLUMNAS_ESPERADAS_SIPSE = [
    "CONTRATO", "SOLICITUD", "VIGENCIA", "NUMERO_CONTRATO", "TIPO", "TIPO_PROCESO", "OBJETO", "FECHA_SUSCRIPCION", "FECHA_INICIO", "FECHA_TERMINACION", "TERMINA31_DICIEMBRE", "CONTRATISTA", "DOC_CONTRATISTA", "VALOR_INICIAL", "ADICIONES", "VALOR_TOTAL", "VALOR_MENSUAL", "PLAZO", "PRORROGAS", "PROYECTO", "DEPENDENCIA", "SUPERVISION", "ESTADO", "NAT", "PAGOS", "AVANCE", "FECHA_ACTA_LIQUIDACION", "PRESUPUESTO", "ANULACION", "DIRECCION_CONTRATISTA", "TELEFONO_CONTRATISTA", "CODIGO_PROYECTO", "NOMBRE_PROYECTO", "CDPS", "RPS", "ESTADO_RP", "MODIFICACION", "NUMERO_OC", "TIPO_SOCIEDAD", "TIPOLOGIA", "TIPO_PAGO", "NIVELACADEMICO", "PROFEXION", "ESPECIALIZ", "POLIZA", "PROCESO_CONTRATO", "ANTIGUEDAD", "SALDO_FAVOR_ENTIDAD", "FECHA_NACIMIENTO", "OBLIGACIONES", "TIPO_IDENTIFICACION", "CORREO_ELECTRONICO", "NOMBRE", "APELLIDO", "CELULAR", "GARANTIAS", "INICIO_AMPARO", "FIN_AMPARO", "IDEN_REP_LEGAL", "NOM_REP_LEGAL", "ENTIDAD", "DESCRIPCION_ENTIDAD", "GENERO"
]

COLUMNAS_ESPERADAS_SECOP = [
    "Nombre Entidad", "Nit Entidad", "Departamento", "Ciudad", "LocalizaciÃ³n", "Orden", "Sector", "Rama", "Entidad Centralizada", "Proceso de Compra", "ID Contrato", "Referencia del Contrato", "Estado Contrato", "Codigo de Categoria Principal", "Descripcion del Proceso", "Tipo de Contrato", "Modalidad de Contratacion", "Justificacion Modalidad de Contratacion", "Fecha de Firma", "Fecha de Inicio del Contrato", "Fecha de Fin del Contrato", "Condiciones de Entrega", "TipoDocProveedor", "Documento Proveedor", "Proveedor Adjudicado", "Es Grupo", "Es Pyme", "Habilita Pago Adelantado", "LiquidaciÃ³n", "ObligaciÃ³n Ambiental", "Obligaciones Postconsumo", "Reversion", "Origen de los Recursos", "Destino Gasto", "Valor del Contrato", "Valor de pago adelantado", "Valor Facturado", "Valor Pendiente de Pago", "Valor Pagado", "Valor Amortizado", "Valor Pendiente de Amortizacion", "Valor Pendiente de Ejecucion", "Estado BPIN", "CÃ³digo BPIN", "Anno BPIN", "Saldo CDP", "Saldo Vigencia", "EsPostConflicto", "Dias adicionados", "Puntos del Acuerdo", "Pilares del Acuerdo", "URLProceso", "Nombre Representante Legal", "Nacionalidad Representante Legal", "Domicilio Representante Legal", "Tipo de IdentificaciÃ³n Representante Legal", "IdentificaciÃ³n Representante Legal", "GÃ©nero Representante Legal", "Presupuesto General de la Nacion â€“ PGN", "Sistema General de Participaciones", "Sistema General de RegalÃ­as", "Recursos Propios (AlcaldÃ­as, Gobernaciones y Resguardos IndÃ­genas)", "Recursos de Credito", "Recursos Propios", "Ultima Actualizacion", "Codigo Entidad", "Codigo Proveedor", "Fecha Inicio Liquidacion", "Fecha Fin Liquidacion", "Objeto del Contrato", "DuraciÃ³n del contrato", "Nombre del banco", "Tipo de cuenta", "NÃºmero de cuenta", "El contrato puede ser prorrogado", "Fecha de notificaciÃ³n de prorrogaciÃ³n", "Nombre ordenador del gasto", "Tipo de documento Ordenador del gasto", "NÃºmero de documento Ordenador del gasto", "Nombre supervisor", "Tipo de documento supervisor", "NÃºmero de documento supervisor", "Nombre Ordenador de Pago", "Tipo de documento Ordenador de Pago", "NÃºmero de documento Ordenador de Pago", "Documentos Tipo", "Descripcion Documentos Tipo"
]

# ==========================================
# 1. FUNCIONES DE LIMPIEZA, COMPARACIÓN Y ESTRUCTURA
# ==========================================

def normalizar_texto(texto):
    """Limpia el texto: quita nulos, espacios, pasa a mayúsculas, quita tildes y artículos"""
    if pd.isna(texto):
        return ""
    texto = str(texto).upper().strip()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^(EL|LA|LOS|LAS)\s+', '', texto)
    return texto

def unificar_localidades(lista_localidades, similitud_minima=0.85):
    """Agrupa nombres similares bajo un mismo nombre 'canónico'"""
    nombres_canonicos = {}
    for loc in lista_localidades:
        if not loc: continue
        coincidencias = difflib.get_close_matches(loc, nombres_canonicos.keys(), n=1, cutoff=similitud_minima)
        if coincidencias:
            nombres_canonicos[loc] = coincidencias[0]
        else:
            nombres_canonicos[loc] = loc
    return nombres_canonicos

def alinear_columnas(df, columnas_esperadas):
    """
    Compara las columnas del df con la lista esperada.
    Crea las faltantes en blanco y reordena el dataframe.
    """
    if not columnas_esperadas:
        return df

    for col in columnas_esperadas:
        if col not in df.columns:
            df[col] = "" 
            
    columnas_extra = [col for col in df.columns if col not in columnas_esperadas]
    orden_final = columnas_esperadas + columnas_extra
    
    return df[orden_final]

# ==========================================
# 2. SELECCIÓN DE ARCHIVOS Y CARPETAS
# ==========================================

root = tk.Tk()
root.withdraw()

print("Por favor, selecciona el archivo Excel de origen...")
archivo_origen = filedialog.askopenfilename(
    title="Selecciona el archivo Excel (.xlsx o .xls)",
    filetypes=[("Archivos de Excel", "*.xlsx *.xls *")]
)

if not archivo_origen:
    print("No se seleccionó ningún archivo. Cancelando proceso.")
    exit()

print("Por favor, selecciona la carpeta de destino...")
carpeta_destino = filedialog.askdirectory(title="Selecciona la carpeta de destino")

if not carpeta_destino:
    print("No se seleccionó ninguna carpeta. Cancelando proceso.")
    exit()

# ==========================================
# 3. LECTURA Y PROCESAMIENTO DEL EXCEL
# ==========================================

try:
    print(f"\nLeyendo el archivo: {archivo_origen}")
    xls = pd.ExcelFile(archivo_origen)
    nombres_hojas = xls.sheet_names
    
    hoja_sipse = next((h for h in nombres_hojas if h.upper() == "SIPSE"), None)
    hoja_secop = next((h for h in nombres_hojas if h.upper() == "SECOP II"), None)
    
    if not hoja_sipse or not hoja_secop:
        raise ValueError("El archivo no contiene las hojas 'SIPSE' y/o 'SECOP II'.")

    df_sipse = pd.read_excel(xls, sheet_name=hoja_sipse)
    df_secop = pd.read_excel(xls, sheet_name=hoja_secop)
    
    if len(df_sipse.columns) <= 61:
        raise ValueError("La hoja SIPSE no tiene suficientes columnas para llegar a la columna BJ (índice 61).")
        
    # Identificar nombres de columnas clave
    col_sipse_nombre = df_sipse.columns[61] 
    col_secop_nombre = df_secop.columns[0]

    # --- NUEVO CAMBIO: REEMPLAZO ESPECÍFICO DE RAFAEL URIBE ---
    print("Ejecutando reemplazo de nombres de entidad específicos...")
    
    # Reemplazo en SECOP II (Columna A)
    df_secop[col_secop_nombre] = df_secop[col_secop_nombre].replace("RAFAEL URIBE URIBE", "RAFAEL URIBE")
    
    # Reemplazo en SIPSE (Columna BJ / índice 61)
    df_sipse[col_sipse_nombre] = df_sipse[col_sipse_nombre].replace("RAFAEL URIBE URIBE", "RAFAEL URIBE")
    # ---------------------------------------------------------

    print("Alineando y estandarizando columnas según listas esperadas...")
    df_sipse = alinear_columnas(df_sipse, COLUMNAS_ESPERADAS_SIPSE)
    df_secop = alinear_columnas(df_secop, COLUMNAS_ESPERADAS_SECOP)
    
    print("Normalizando nombres de localidades...")
    df_sipse['Loc_Limpia'] = df_sipse[col_sipse_nombre].apply(normalizar_texto)
    df_secop['Loc_Limpia'] = df_secop[col_secop_nombre].apply(normalizar_texto)
    
    todas_localidades = list(set(df_sipse['Loc_Limpia'].unique()) | set(df_secop['Loc_Limpia'].unique()))
    mapa_canonico = unificar_localidades(todas_localidades)
    
    df_sipse['Loc_Final'] = df_sipse['Loc_Limpia'].map(mapa_canonico)
    df_secop['Loc_Final'] = df_secop['Loc_Limpia'].map(mapa_canonico)
    
    localidades_finales = set(mapa_canonico.values())
    
    print(f"Se detectaron {len(localidades_finales)} localidades únicas. Generando archivos...")
    
    archivos_creados = 0
    for loc in localidades_finales:
        if not loc: continue 
        
        filtro_sipse = df_sipse[df_sipse['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        filtro_secop = df_secop[df_secop['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        
        if not filtro_sipse.empty or not filtro_secop.empty:
            nombre_archivo_seguro = re.sub(r'[\\/*?:"<>|]', "", loc)
            ruta_salida = os.path.join(carpeta_destino, f"{nombre_archivo_seguro}.xlsx")
            
            with pd.ExcelWriter(ruta_salida, engine='openpyxl') as writer:
                filtro_sipse.to_excel(writer, sheet_name="SIPSE", index=False)
                filtro_secop.to_excel(writer, sheet_name="SECOP II", index=False)
                
            archivos_creados += 1
            print(f" -> Creado: {nombre_archivo_seguro}.xlsx")

    print(f"\n¡Proceso Finalizado Exitosamente! Se crearon {archivos_creados} archivos.")
    messagebox.showinfo("Éxito", f"Se generaron {archivos_creados} archivos correctamente.")

except Exception as e:
    print(f"\nOcurrió un error: {e}")
    messagebox.showerror("Error", f"Ocurrió un error procesando el archivo:\n{e}")

finally:
    root.destroy()

Por favor, selecciona el archivo Excel de origen...
Por favor, selecciona la carpeta de destino...

Leyendo el archivo: C:/Users/Predator/Desktop/META 7 - FDL 03 MAYO/Datos Virgenes/ContratosVigencia_2026 (17).xlsx
Ejecutando reemplazo de nombres de entidad específicos...
Alineando y estandarizando columnas según listas esperadas...
Normalizando nombres de localidades...
Se detectaron 21 localidades únicas. Generando archivos...
 -> Creado: BOSA.xlsx
 -> Creado: CIUDAD BOLIVAR.xlsx
 -> Creado: KENNEDY.xlsx
 -> Creado: CHAPINERO.xlsx
 -> Creado: MARTIRES.xlsx
 -> Creado: RAFAEL URIBE.xlsx
 -> Creado: ANTONIO NARINO.xlsx
 -> Creado: USME.xlsx
 -> Creado: SANTAFE.xlsx
 -> Creado: TUNJUELITO.xlsx
 -> Creado: SUBA.xlsx
 -> Creado: USAQUEN.xlsx
 -> Creado: PUENTE ARANDA.xlsx
 -> Creado: SDG - NIVEL CENTRAL.xlsx
 -> Creado: BARRIOS UNIDOS.xlsx
 -> Creado: FONTIBON.xlsx
 -> Creado: CANDELARIA.xlsx
 -> Creado: SUMAPAZ.xlsx
 -> Creado: ENGATIVA.xlsx
 -> Creado: TEUSAQUILLO.xlsx
 -> Creado: SAN C

In [ ]:
import os
import shutil
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
from collections import defaultdict
import re
import pandas as pd
import difflib # para busqueda aproximada
import datetime

#Para abrir explorador de archivos
import tkinter as tk
from tkinter import filedialog
import math
import unicodedata

# --- CONFIGURACIÓN DE CARPETAS ---
#CARPETA_ORIGEN = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO'
#CARPETA_DESTINO = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\DiagnóstivoV2_1'
#CARPETA_ORIGEN = r'C:\\Users\\Pc\\Desktop\\META 7 - FDL 03 MAYO' #Dell
#CARPETA_DESTINO = r"C:\\Users\\Pc\\Desktop\\META 7 - FDL 03 MAYO\\Diagnóstico" #Dell
CARPETA_ORIGEN = r'D:\\Años\\2026\\SecretariaDeGobierno\\ConsolidadoAlcaldias\\META 7 - FDL 03 MAYO' #Predatos
CARPETA_DESTINO = r"D:\\Años\\2026\\SecretariaDeGobierno\\ConsolidadoAlcaldias\\META 7 - FDL 03 MAYO\\Diagnóstico" #Predator
#CARPETA_ORIGEN = r'D:\\Trabajo\\2026\\SecretariaDeGobierno\\BasesDeDatosSipseSecop\\2026_03_06\\DatosPartidos' #Predator
#CARPETA_DESTINO = r"D:\\Trabajo\\2026\\SecretariaDeGobierno\\BasesDeDatosSipseSecop\\2026_03_06\\DatosPartidos\\Diagnóstico" #PRedator

# --- RANGOS A OCULTAR (CORREGIDOS PARA USAR HIDDEN=TRUE) ---
# Nota: He ajustado 'A-AH' a 'Z-AH' en SECOP para NO ocultar las columnas L, M, T, X, Y que necesitas ver.
RANGOS_OCULTAR_SECOP = ['A-K', 'N-S', 'U-W', 'Z-AH', 'AJ-BR', 'BT-CI']
RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-AG','AJ-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']

# --- DEFINICIÓN DE FUNCIONES ---

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def ocultar_columnas_por_lista(ws, lista_rangos):
    """
    Recibe una lista de cadenas como ['A-K', 'M'] y establece la propiedad HIDDEN = True.
    """
    for item in lista_rangos:
        item = item.upper().strip()
        
        # Caso 1: Es un rango (ej: "A-K")
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio)
            idx_fin = column_index_from_string(fin)
            
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                # Esta es la forma correcta de ocultar en Excel
                ws.column_dimensions[letra].hidden = True 
                
        # Caso 2: Es una columna única (ej: "A")
        else:
            ws.column_dimensions[item].hidden = True

def aplicar_estilo_ferrocarril(ws):
    """
    Encabezados Azul Oscuro + Letra Blanca.
    Filas con patrón intercalado.
    """
    # 1. Encabezados
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    
    for cell in ws[1]:
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas (Patrón Ferrocarril)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    max_col = ws.max_column
    
    for i_fila, row in enumerate(ws.iter_rows(min_row=2, max_col=max_col), start=0):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF":
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for cell in row:
                cell.fill = relleno

def dejar_columna_blanca(ws, indice_columna):
    relleno_blanco = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    
    # Recorremos todas las celdas de la columna indicada
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=indice_columna, max_col=indice_columna):
        for cell in row:
            cell.fill = relleno_blanco

def aplicar_formato_condicional_alertas(ws, col_inicio_eval):
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    max_row = ws.max_row
    
    reglas = [
        (12, 'equal', '"Numero contrato NO coincide"'),
        (14, 'equal', '"Cedula NO coincide"'),
        (15, 'greaterThan', '1'),
        (16, 'greaterThan', '1'),
        (17, 'equal', '"No coinciden fechas"'),
        (18, 'equal', '"No coinciden fechas"'),
        (19, 'equal', '"No coinciden fechas"'),
        (20, 'equal', '"Duracion NO coincide"'),
        (21, 'equal', '"Valor NO coincide"'),
        (22, 'equal', '"El numero RP NO coincide"'),
        (23, 'equal', '"El valor RP NO coincide"'),
        (24, 'equal', '"La fecha RP NO coincide"'),
        (25, 'equal', '"El numero CDP NO coincide"'),
        (26, 'equal', '"El valor CDP NO coincide"'),
        (27, 'equal', '"La fecha CDP NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        rango = f"{letra_col}2:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

def normalizar_numero_contrato(valor):
    if valor is None: return 0
    texto = str(valor).strip()
    # Busca el primer grupo de números en la cadena
    match = re.search(r'\d+', texto)
    if match:
        return int(match.group())
    return 0

#Funciones Hoja "Para MEMO"
def es_error_fecha(val_sipse, val_secop):
    """Replica la fórmula de Excel de fechas para detectar discrepancias imitando tu cruce de día/mes."""
    s1 = str(val_sipse).strip() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip() if pd.notna(val_secop) and val_secop is not None else ""
    
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    
    # Parsear SIPSE 
    dt_sipse = None
    if isinstance(val_sipse, (datetime.datetime, datetime.date)):
        dt_sipse = datetime.date(val_sipse.year, val_sipse.month, val_sipse.day)
    else:
        try:
            dt_sipse = pd.to_datetime(s1.replace("-", "/"), dayfirst=True).date()
        except: pass

    # Parsear SECOP imitando la fórmula FECHA(AÑO, DIA, MES)
    dt_secop = None
    if isinstance(val_secop, (datetime.datetime, datetime.date)):
        try:
            dt_secop = datetime.date(val_secop.year, val_secop.day, val_secop.month)
        except: pass
    else:
        partes = s2.replace("-", "/").replace(" ", "/").split("/")
        if len(partes) >= 3:
            try:
                # Tu fórmula usa IZQUIERDA para el mes y EXTRAE para el día
                y = int(partes[2][-4:])
                m = int(partes[0])
                d = int(partes[1])
                dt_secop = datetime.date(y, m, d)
            except: pass
                
    if dt_sipse is not None and dt_secop is not None:
        return dt_sipse != dt_secop
        
    return s1 != s2 # Fallback a comparación de texto

def es_error_duracion(val_sipse, val_secop):
    """Replica la lógica de Duración."""
    s1 = str(val_sipse).strip().upper() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip().upper() if pd.notna(val_secop) and val_secop is not None else ""
    
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    return s1 != s2

def color_memo(err_tuple):
    """Devuelve los colores agrupados tipo pastel idénticos a los de tu imagen."""
    if err_tuple[0]: return "E4DFEC" # Beige/Grisáceo claro (Suscripción)
    if err_tuple[1]: return "B4C6E7" # Azul claro (Inicio)
    if err_tuple[2]: return "DDEBF7" # Gris claro (Fin)
    if err_tuple[3]: return "F4CCCC" # Rosa claro (Plazo)
    return "FFFFFF"

def normalizar_alfanumerico(valor):
    """Limpia códigos (como los de CDP) para evitar que pandas los lea como '1234.0'"""
    if pd.isna(valor) or valor is None:
        return ""
    texto = str(valor).strip().upper()
    if texto.endswith(".0"):
        texto = texto[:-2]
    return texto

def limpiar_dato(val):
    """Limpia el dato y elimina el '.0' que Pandas añade por defecto a los números."""
    if pd.isna(val) or val is None:
        return ""
    if isinstance(val, (datetime.datetime, datetime.date)):
        return val.strftime('%d/%m/%Y')
    
    val_str = str(val).strip()
    if val_str.endswith(".0"):
        val_str = val_str[:-2]
    return val_str

def normalizar_cedula(valor):
    """Limpia la cédula quitando letras y asegurando que sea un número exacto."""
    val_str = limpiar_dato(valor)
    return re.sub(r'\D', '', val_str)

def obtener_valor_seguro(row, indice):
    """Evita errores si el archivo de Excel viene con columnas faltantes."""
    try:
        return limpiar_dato(row.iloc[indice])
    except IndexError:
        return ""

def cargar_datos_bogdata():
    """Abre el explorador y procesa RP (por cédula) y CDP (por Nro + Cod Interno)."""
    root = tk.Tk()
    root.withdraw() 
    root.attributes('-topmost', True) 
    
    print("\n--- SELECCIÓN BOGDATA ---")
    print("1. Por favor selecciona el archivo de CDP (Excel)...")
    ruta_cdp = filedialog.askopenfilename(title="Seleccionar archivo CDP", filetypes=[("Excel files", "*.xlsx *.xls")])
    
    print("2. Por favor selecciona el archivo de RP (Excel)...")
    ruta_rp = filedialog.askopenfilename(title="Seleccionar archivo RP", filetypes=[("Excel files", "*.xlsx *.xls")])
         
    dict_cdp = {}
    dict_rp = defaultdict(list) # <--- Ahora es un diccionario que guarda listas

    # --- PROCESAR RP ---
    if ruta_rp:
        print(f"  -> Procesando archivo RP...")
        df_rp = pd.read_excel(ruta_rp, sheet_name="Data")
        # Índices: F=5(Fec), O=14(NumCDP), P=15(NumRP), R=17(Filtro1), Z=25(Localidad), AE=30(Ced), AL=37(Filtro2), AN=39(ValRP), AS=44(CodIntCDP)
        for _, row in df_rp.iterrows():
            val_r = obtener_valor_seguro(row, 17)
            digitos_r = re.sub(r'\D', '', val_r)
            
            # 1. Filtro 1: Columna R >= 7 dígitos
            if len(digitos_r) >= 7:
                
                # 2. Filtro 2: Columna AL NO debe ser mayor a 0
                val_al = obtener_valor_seguro(row, 37) # Columna AL
                try:
                    # Convertimos a número (quitando comas si las hay)
                    num_al = float(str(val_al).replace(',', ''))
                except ValueError:
                    # Si la celda está vacía o tiene letras, asumimos 0 para que no falle
                    num_al = 0.0 
                
                # Solo continuamos si AL es menor o igual a 0
                if num_al == 0:
                    cedula_raw = obtener_valor_seguro(row, 30)
                    llave_cedula = normalizar_cedula(cedula_raw)
                    
                    # Guardamos TODOS los RP válidos que encuentre de esa persona
                    if llave_cedula:
                        rp_num = obtener_valor_seguro(row, 15) # P
                        rp_val = obtener_valor_seguro(row, 39) # AN
                        rp_fec = obtener_valor_seguro(row, 5)  # F
                        cdp_num = obtener_valor_seguro(row, 14) # O
                        cdp_cod = obtener_valor_seguro(row, 44) # AS
                        
                        # Capturamos la localidad (Columna Z)
                        localidad_col_z = obtener_valor_seguro(row, 25) # Z
                        
                        dict_rp[llave_cedula].append((rp_num, rp_val, rp_fec, cdp_num, cdp_cod, localidad_col_z))

    # --- PROCESAR CDP ---
    if ruta_cdp:
        print(f"  -> Procesando archivo CDP...")
        df_cdp = pd.read_excel(ruta_cdp, sheet_name="Data")
        # Índices: H=7(NumCDP), Q=16(ValCDP), U=20(CodIntCDP), AC=28(FecCDP)
        for _, row in df_cdp.iterrows():
            cdp_num = obtener_valor_seguro(row, 7)  # H
            cdp_cod = obtener_valor_seguro(row, 20) # U
            
            # La llave maestra del CDP es la combinación de Número (H) + Código interno (U)
            llave_cdp = (cdp_num, cdp_cod)
            
            if llave_cdp != ("", "") and llave_cdp not in dict_cdp:
                cdp_val = obtener_valor_seguro(row, 16) # Q
                cdp_fec = obtener_valor_seguro(row, 28) # AC
                
                dict_cdp[llave_cdp] = (cdp_num, cdp_val, cdp_fec)

    return dict_cdp, dict_rp

def normalizar_texto_localidad(texto):
    """Elimina espacios, tildes y pasa a mayúsculas para un match perfecto."""
    if pd.isna(texto) or texto is None:
        return ""
    t = str(texto).upper().replace(" ", "")
    # Quitar tildes (ej: NARIÑO -> NARINO, BOGOTÁ -> BOGOTA)
    t = ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')
    return t

# --- LÓGICA PRINCIPAL ---
global dict_cdp
global dict_rp
global metricas
def procesar_archivos():
    global metricas
    if not os.path.exists(CARPETA_DESTINO):
        os.makedirs(CARPETA_DESTINO)

    #Cargar Bogdata en memoria antes del ciclo principal
    global dict_cdp
    global dict_rp
    dict_cdp, dict_rp = cargar_datos_bogdata()

    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('ANTONIO NARINO.xlsx', 'ANTONIO NARINO.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('SANTAFE.xlsx', 'SANTAFE.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('CANDELARIA.xlsx', 'CANDELARIA.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('RAFAEL URIBE.xlsx', 'RAFAEL URIBE.xls'))]
    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('.xlsx', '.xls'))]
    print(f"Se encontraron {len(archivos)} archivos de Excel.")

    for archivo in archivos:
        ruta_orig = os.path.join(CARPETA_ORIGEN, archivo)
        
        # Definimos la ruta de destino siempre con extensión .xlsx
        nombre_base = os.path.splitext(archivo)[0]
        ruta_dest = os.path.join(CARPETA_DESTINO, nombre_base + ".xlsx")
        
        print(f"Procesando: {archivo}...")

        # --- PASO DE CONVERSIÓN SI ES .XLS ---
        if archivo.lower().endswith('.xls'):
            try:
                # Leemos el .xls y lo guardamos inmediatamente como .xlsx
                # sheet_name=None lee todas las pestañas
                conversion = pd.read_excel(ruta_orig, sheet_name=None, engine='xlrd')
                with pd.ExcelWriter(ruta_dest, engine='openpyxl') as writer:
                    for nombre_hoja, datos in conversion.items():
                        datos.to_excel(writer, sheet_name=nombre_hoja, index=False)
                print(f"  -> Convertido de .xls a .xlsx con éxito.")
            except Exception as e:
                print(f"  Error convirtiendo {archivo}: {e}")
                continue
        else:
            # Si ya es .xlsx, solo lo copiamos como hacías antes
            shutil.copy2(ruta_orig, ruta_dest)
        # -------------------------------------

        try:
            # Ahora openpyxl siempre abrirá un archivo .xlsx (el copiado o el convertido)
            wb = openpyxl.load_workbook(ruta_dest)
        except Exception as e:
            print(f"   Error al abrir el archivo con openpyxl: {e}")
            continue

        # ==========================================
        # PARTE 1: Procesar SECOP II (Crear aux1)
        # ==========================================
        nombre_real_secop = next(n for n in wb.sheetnames if n.lower() == 'secop ii')
        ws_origen = wb[nombre_real_secop]
        
        if "SECOP II aux1" in wb.sheetnames:
            del wb["SECOP II aux1"]
            
        ws_secop_aux = wb.copy_worksheet(ws_origen)
        ws_secop_aux.title = "SECOP II aux1"

        filas = []
        for row in ws_secop_aux.iter_rows(min_row=2, values_only=True):
            filas.append(list(row))
        
        filas_ordenadas = sorted(filas, key=lambda x: normalizar_numero_contrato(x[11] if len(x) > 11 else 0))

        for row in ws_secop_aux.iter_rows(min_row=2):
            for cell in row:
                cell.value = None

        for i_fila, datos_fila in enumerate(filas_ordenadas, start=2):
            for i_col, valor in enumerate(datos_fila, start=1):
                ws_secop_aux.cell(row=i_fila, column=i_col, value=valor)

        # Para crear columnas "Columna1" a "Columna8" en la hoja "SECOP II aux1"
        max_col_secop = ws_secop_aux.max_column
        ws_secop_aux.cell(row=1, column=max_col_secop + 1, value="Columna1")
        ws_secop_aux.cell(row=1, column=max_col_secop + 2, value="Columna2")
        col_L = get_column_letter(12)
        
        for i in range(2, ws_secop_aux.max_row + 1):
            # Fórmula Columna 1
            f1 = (
                f'=IFERROR(MID({col_L}{i}, '
                f'MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), '
                f'MIN(IFERROR(FIND(" ", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999), '
                f'IFERROR(FIND("-", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999)) '
                f'- MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), "")'
            )
            ws_secop_aux.cell(row=i, column=max_col_secop + 1, value=f1)

            # Fórmula Columna 2 (Extrae número entre paréntesis)
            f2 = f'=IFERROR(MID({col_L}{i}, FIND("(", {col_L}{i}) + 1, FIND(")", {col_L}{i}) - FIND("(", {col_L}{i}) - 1), "")'
            ws_secop_aux.cell(row=i, column=max_col_secop + 2, value=f2)

        # ESTILOS SECOP
        print("  -> Aplicando estilos y ocultando columnas en SECOP II aux1...")
        aplicar_estilo_ferrocarril(ws_secop_aux)
        ocultar_columnas_por_lista(ws_secop_aux, RANGOS_OCULTAR_SECOP)

        # ==========================================
        # PARTE 2 y 3: ALINEACIÓN EN 'ANALISIS'
        # ==========================================
        print("  -> Generando hoja 'Analisis'...")

        nombre_real_sipse = next(n for n in wb.sheetnames if n.lower() == 'sipse')
        ws_sipse = wb[nombre_real_sipse]
        
        if 'Analisis' in wb.sheetnames:
            del wb['Analisis']
        ws_analisis = wb.create_sheet("Analisis")
        
        dict_sipse = defaultdict(list)
        dict_secop = defaultdict(list)
        
        filas_sipse = list(ws_sipse.values)
        header_sipse = filas_sipse[0] if filas_sipse else []
        for fila in filas_sipse[1:]:
            contrato = normalizar_numero_contrato(fila[3] if len(fila) > 3 else 0)
            dict_sipse[contrato].append(fila)

        filas_secop = list(ws_secop_aux.values)
        header_secop = filas_secop[0] if filas_secop else []
        for fila in filas_secop[1:]:
            contrato = normalizar_numero_contrato(fila[11] if len(fila) > 11 else 0)
            dict_secop[contrato].append(fila)

        ws_analisis.append(header_sipse) 
        col_inicio_secop = column_index_from_string("BM")
        for i, val in enumerate(header_secop):
            ws_analisis.cell(row=1, column=col_inicio_secop + i, value=val)

        todos_contratos = sorted(list(set(dict_sipse.keys()) | set(dict_secop.keys())))
        current_row = 2 
        
        for contrato in todos_contratos:
            filas_de_sipse = dict_sipse.get(contrato, [])
            filas_de_secop = dict_secop.get(contrato, [])
            max_filas = max(len(filas_de_sipse), len(filas_de_secop))
            
            for i in range(max_filas):
                if i < len(filas_de_sipse):
                    datos = filas_de_sipse[i]
                    for c_idx, valor in enumerate(datos, start=1):
                        ws_analisis.cell(row=current_row, column=c_idx, value=valor)
                
                if i < len(filas_de_secop):
                    datos = filas_de_secop[i]
                    for c_idx, valor in enumerate(datos):
                        col_excel = col_inicio_secop + c_idx
                        if c_idx == len(datos) - 2 and str(valor).startswith('='):
                             # Pega la fórmula de Columna1 (Ahora es la penúltima)
                             nueva_formula1 = (
                                f'=IFERROR(MID(BX{current_row}, '
                                f'MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), '
                                f'MIN(IFERROR(FIND(" ", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999), '
                                f'IFERROR(FIND("-", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999)) '
                                f'- MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), "")'
                            )
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula1)
                        elif c_idx == len(datos) - 1 and str(valor).startswith('='):
                             # Pega la fórmula de Columna2 (Extrae entre paréntesis)
                             nueva_formula2 = f'=IFERROR(MID(BX{current_row}, FIND("(", BX{current_row}) + 1, FIND(")", BX{current_row}) - FIND("(", BX{current_row}) - 1), "")'
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula2)
                        else:
                            ws_analisis.cell(row=current_row, column=col_excel, value=valor)
                current_row += 1

        # ==========================================
        # PARTE 4: Columnas de Evaluación
        # ==========================================
        print("  -> Agregando columnas de evaluación y estilos...")
        
        ultima_col_actual = ws_analisis.max_column
        col_inicio_eval = ultima_col_actual + 1
        ultima_fila_analisis = ws_analisis.max_row
        
        headers_nuevos = [
            "Columna3\n# RP\n(SIPSE)", "Columna4\nValor RP\n(SIPSE)", "Columna5\nFecha RP\n(SIPSE)", "Columna6\n# CDP\n(SIPSE)", "Columna7\nValor CDP\n(SIPSE)", "Columna8\nFecha CDP\n(SIPSE)", "Col.9\n# RP\n(BOGD)", "Col.10\nValor RP\n(BOGD)", "Col.11\nFecha RP\n(BOGD)", "Col.12\n# CDP\n(BOGD)", "Col.13\nValor CDP\n(BOGD)", "Col.14\nFecha CDP\n(BOGD)",
            "Ev. Nro contrato", "Aux Nro contrato", "Ev. Cédula", "Repeticiones contrato SIPSE", "Repeticiones contrato SECOP", "Fecha suscripción", "Fecha inicio", "Fecha fin", "Duración contratos", "Valor contratos", "Ev. # RP", "Ev. Valor RP", "Ev. Fecha RP", "Ev. # CDP", "Ev. Valor CDP", "Ev. Fecha CDP"
        ]

        #ESCRIBIR LOS TITULOS EN LOS HEADERS
        for idx, titulo in enumerate(headers_nuevos):
            ws_analisis.cell(row=1, column=col_inicio_eval + idx, value=titulo)
        
        #AÑADIR COLUMNAS DESGLOSADAS CDP y RP (SIPSE)
        for i in range(2, ultima_fila_analisis + 1):
            # 🔹 COLUMNA 3 - RP NUMERO (SIPSE)
            f3 = f'=IFERROR(TRIM(RIGHT(SUBSTITUTE(TRIM(LEFT(AI{i}, FIND("Fecha", AI{i})-1)), " ", REPT(" ", 100)), 100)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval, value=f3)
            # 🔹 COLUMNA 4 - RP VALOR (SIPSE)
            f4 = f'=IFERROR(TRIM(MID(AI{i}, FIND("Valor ", AI{i}) + 6, IFERROR(FIND(";", AI{i}), LEN(AI{i})+1) - FIND("Valor ", AI{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 1, value=f4)
            # 🔹 COLUMNA 5 - RP FECHA (SIPSE)
            f5 = f'=IFERROR(TRIM(MID(AI{i}, FIND("Fecha ", AI{i}) + 6, FIND(" Valor", AI{i}) - FIND("Fecha ", AI{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 2, value=f5)
            # 🔹 COLUMNA 6 - CDP NUMERO (SIPSE)
            f6 = f'=IFERROR(TRIM(RIGHT(SUBSTITUTE(TRIM(LEFT(AH{i}, FIND("Fecha", AH{i})-1)), " ", REPT(" ", 100)), 100)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 3, value=f6)
            # 🔹 COLUMNA 7 - CDP VALOR (SIPSE)
            f7 = f'=IFERROR(TRIM(MID(AH{i}, FIND("Valor ", AH{i}) + 6, IFERROR(FIND(";", AH{i}), LEN(AH{i})+1) - FIND("Valor ", AH{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 4, value=f7)
            # 🔹 COLUMNA 8 - CDP FECHA (SIPSE)
            f8 = f'=IFERROR(TRIM(MID(AH{i}, FIND("Fecha ", AH{i}) + 6, FIND(" Valor", AH{i}) - FIND("Fecha ", AH{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 5, value=f8)


        # --- AÑADIR COLUMNAS DESGLOSADAS CDP y RP (BOGDATA) ---
        idx_col_cedula_cj = column_index_from_string("CJ")
        letra_ev_nro_contrato = get_column_letter(col_inicio_eval + 12) # Esto equivale a la columna 'FJ'
        letra_ev_nro_cedula = get_column_letter(col_inicio_eval + 14) # Esto equivale a la columna 'FL'
        
        # Limpiamos el nombre del archivo actual (Ej: "SANTA FE" -> "SANTAFE")
        localidad_actual_norm = normalizar_texto_localidad(nombre_base)
        
        for i in range(2, ultima_fila_analisis + 1):
            # 1. Leer cédula de la hoja en construcción (Columna CJ)
            cedula_actual_raw = ws_analisis.cell(row=i, column=idx_col_cedula_cj).value
            cedula_busqueda = normalizar_cedula(cedula_actual_raw)
            
            # 2. Buscar en RP (Trae una lista con todos los RPs de la persona)
            lista_rps = dict_rp.get(cedula_busqueda, [])
            rp_data_elegido = ("", "", "", "", "")
            
            if len(lista_rps) == 1:
                # Si solo tiene un contrato, lo elegimos sin pensarlo
                rp_data_elegido = lista_rps[0][:5]
            elif len(lista_rps) > 1:
                # Si tiene varios, buscamos el que coincida con la localidad
                for rp in lista_rps:
                    localidad_rp_norm = normalizar_texto_localidad(rp[5]) # El 6to dato es Columna Z
                    # Si "SANTAFE" está dentro de "PMRFDLDESANTAFE", hace match
                    if localidad_actual_norm in localidad_rp_norm:
                        rp_data_elegido = rp[:5]
                        break

            rp_num, rp_val, rp_fec, cdp_num_from_rp, cdp_cod_from_rp = rp_data_elegido
            
            # 3. Buscar en CDP (Usando las 2 llaves que trajo el RP: Num y Cod Interno)
            cdp_data = dict_cdp.get((cdp_num_from_rp, cdp_cod_from_rp), ("", "", ""))
            cdp_num_out, cdp_val_out, cdp_fec_out = cdp_data

            # 4. Enviar a Excel usando fórmula =SI() vinculada a la coincidencia de contrato (FJ)
            def inyectar_condicional(valor):
                val_str = str(valor).replace('"', '""') if valor else ""
                # Si dice 'Numero contrato coincide', 'Contrato coincide sin SIPSE', o 'Cedula coincide', muestra el dato; de lo contrario, vacía.
                #return f'=IF(OR({letra_ev_nro_contrato}{i}="Numero contrato coincide", {letra_ev_nro_contrato}{i}="Contrato coincide sin SIPSE"), "{val_str}", "")'                
                return f'=IF(OR({letra_ev_nro_contrato}{i}="Numero contrato coincide", {letra_ev_nro_contrato}{i}="Contrato coincide sin SIPSE", {letra_ev_nro_cedula}{i}="Cedula coincide"), "{val_str}", "")'

            # 🔹 COLUMNA 9 - RP # (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 6, value=inyectar_condicional(rp_num))
            # 🔹 COLUMNA 10 - RP VALOR (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 7, value=inyectar_condicional(rp_val))
            # 🔹 COLUMNA 11 - RP FECHA (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 8, value=inyectar_condicional(rp_fec))
            
            # 🔹 COLUMNA 12 - CDP # (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 9, value=inyectar_condicional(cdp_num_out))
            # 🔹 COLUMNA 13 - CDP VALOR (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 10, value=inyectar_condicional(cdp_val_out))
            # 🔹 COLUMNA 14 - CDP FECHA (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 11, value=inyectar_condicional(cdp_fec_out))
        # -------------------------------------------------------------

        for i in range(2, ultima_fila_analisis + 1):
            COL_SOLICITUD = "B"
            # 🔹 EV. NO CONTRATO
            # Se añade SUBSTITUTE(UPPER(...), "SIPSE", "") para omitir la palabra si la escriben por error.
            #f15 = f'''=IF(COUNTIF($FK$2:FK{i},FK{i})>1,"",IF(AND(D{i}<>"",EV{i}<>""),IF(EW{i}="","Contrato coincide sin SIPSE",IF(TRIM({COL_SOLICITUD}{i})=TRIM(EW{i}),"Numero contrato coincide","Numero contrato NO coincide")),IF(D{i}<>"","Contrato no existe en SECOP",IF(EV{i}<>"","Contrato no existe en SIPSE","No existe"))))'''
            f15 = f'''=IF(COUNTIF($FK$2:FK{i},FK{i})>1,"",
                IF(AND(D{i}<>"",EV{i}<>""),
                    IF(EW{i}="","Contrato coincide sin SIPSE",
                        IF(TRIM(SUBSTITUTE(UPPER({COL_SOLICITUD}{i}),"SIPSE",""))=TRIM(SUBSTITUTE(UPPER(EW{i}),"SIPSE","")),"Numero contrato coincide","Numero contrato NO coincide")
                    ),
                    IF(D{i}<>"","Contrato no existe en SECOP",
                        IF(EV{i}<>"","Contrato no existe en SIPSE","No existe")
                    )
                )
            )'''
            # Eliminamos los saltos de línea para que Excel lo lea como una sola fórmula (opcional, pero más seguro en Python)
            f15 = f15.replace('\n', '').replace('    ', '')
            ws_analisis.cell(row=i, column=col_inicio_eval + 12, value=f15)

            # 🔹 AUX NRO CONTRATO (EX) → llave única
            f16 = f'=IF(D{i}<>"",D{i},EV{i})'
            ws_analisis.cell(row=i, column=col_inicio_eval + 13, value=f16)

            # 🔹 CÉDULA
            f17 = f'=IF(AND(BY{i}="",M{i}=""),"",IF(OR(BY{i}="",M{i}=""),"",IF(BY{i}="-","Sin cedula en SECOP",IF(M{i}="-","Sin cedula en SIPSE",IF(VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(CJ{i}),"CC",""),"NIT",""),"TI",""))=VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(M{i}),"CC",""),"NIT",""),"TI","")),"Cedula coincide","Cedula NO coincide")))))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 14, value=f17)

            # 🔹 REPETICIONES SIPSE
            f18 = f'=IF(FJ{i}="","",COUNTIF(D$2:D${ultima_fila_analisis},D{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 15, value=f18)

            # 🔹 REPETICIONES SECOP
            f19 = f'=IF(FJ{i}="","",COUNTIF(EV$2:EV${ultima_fila_analisis},EV{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 16, value=f19)

            # 🔹 FECHA SUSCRIPCION
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CE{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CE{i}), DAY(CE{i}), MONTH(CE{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f20 = f'''=IF(AND(H{i}="",CE{i}=""),"",IF(AND(H{i}="-",CE{i}="-"),"Ambos sin fechas",IF(H{i}="-","sin fecha SIPSE",IF(CE{i}="-","sin fecha SECOP",IF(OR(H{i}="",CE{i}=""),"",IF(INT(VALUE(SUBSTITUTE(H{i},"-","/"))) =INT(IF(ISNUMBER(CE{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 17, value=f20)

            # 🔹 FECHA ACTA DE INICIO
            cf_clean = f'SUBSTITUTE(SUBSTITUTE(CF{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CF{i}), DAY(CF{i}), MONTH(CF{i}))'
            parte_es_texto = f'DATE(RIGHT({cf_clean}, 4), LEFT({cf_clean}, FIND("/", {cf_clean}) - 1), MID({cf_clean}, FIND("/", {cf_clean}) + 1, FIND("/", {cf_clean}, FIND("/", {cf_clean}) + 1) - FIND("/", {cf_clean}) - 1))'

            f21 = f'''=IF(AND(I{i}="",CF{i}=""),"",IF(AND(I{i}="-",CF{i}="-"),"Ambos sin fechas",IF(I{i}="-","sin fecha SIPSE",IF(CF{i}="-","sin fecha SECOP",IF(OR(I{i}="",CF{i}=""),"",IF(INT(VALUE(SUBSTITUTE(I{i},"-","/"))) =INT(IF(ISNUMBER(CF{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 18, value=f21)

            # 🔹 FECHA FIN
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CG{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CG{i}), DAY(CG{i}), MONTH(CG{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f22 = f'''=IF(AND(J{i}="",CG{i}=""),"",IF(AND(J{i}="-",CG{i}="-"),"Ambos sin fechas",IF(J{i}="-","sin fecha SIPSE",IF(CG{i}="-","sin fecha SECOP",IF(OR(J{i}="",CG{i}=""),"",IF(INT(VALUE(SUBSTITUTE(J{i},"-","/"))) =INT(IF(ISNUMBER(CG{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 19, value=f22)

            # 🔹 DURACIÓN
            f23 = f'''=IF(AND(R{i}="",EE{i}=""),"",IF(AND(R{i}="-",EE{i}="-"),"Ambos sin duracion",IF(R{i}="-","sin duracion en SECOP",IF(EE{i}="-","sin duracion en SIPSE",IF(OR(R{i}="",EE{i}=""),"",IF(TRIM(UPPER(R{i}))=TRIM(UPPER(EE{i})),"Duracion coincide","Duracion NO coincide"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 20, value=f23)

            # 🔹 VALOR CONTRATO (Comparación Robusta)
            f24 = f'''=IF(AND(P{i}="",CU{i}=""),"",
                IF(AND(P{i}="-",CU{i}="-"),"Ambos sin valor",
                IF(P{i}="-","Sin valor en SECOP",
                IF(CU{i}="-","Sin valor en SIPSE",
                IF(OR(P{i}="",CU{i}=""),"",
                IF(P{i}=CU{i},"Valor coincide","Valor NO coincide"))))))'''

            ws_analisis.cell(row=i, column=col_inicio_eval + 21, value=f24)

            c_rp_sip   = get_column_letter(col_inicio_eval)       # EX (# RP SIPSE)
            c_val_rp_s = get_column_letter(col_inicio_eval + 1)   # EY (Valor RP SIPSE)
            c_fec_rp_s = get_column_letter(col_inicio_eval + 2)   # EZ (Fecha RP SIPSE)
            c_cdp_sip  = get_column_letter(col_inicio_eval + 3)   # FA (# CDP SIPSE)
            c_val_cd_s = get_column_letter(col_inicio_eval + 4)   # FB (Valor CDP SIPSE)
            c_fec_cd_s = get_column_letter(col_inicio_eval + 5)   # FC (Fecha CDP SIPSE)
            
            c_rp_bog   = get_column_letter(col_inicio_eval + 6)   # FD (# RP BOGDATA)
            c_val_rp_b = get_column_letter(col_inicio_eval + 7)   # FE (Valor RP BOGDATA)
            c_fec_rp_b = get_column_letter(col_inicio_eval + 8)   # FF (Fecha RP BOGDATA)
            c_cdp_bog  = get_column_letter(col_inicio_eval + 9)   # FG (# CDP BOGDATA)
            c_val_cd_b = get_column_letter(col_inicio_eval + 10)  # FH (Valor CDP BOGDATA)
            c_fec_cd_b = get_column_letter(col_inicio_eval + 11)  # FI (Fecha CDP BOGDATA)

            # 🔹 Ev. # RP
            f25 = (f'=IF(AND(TRIM({c_rp_sip}{i})="", TRIM({c_rp_bog}{i})=""), "Sin numeros RP", '
                   f'IF(TRIM({c_rp_sip}{i})="", "Sin numero RP en SIPSE", '
                   f'IF(TRIM({c_rp_bog}{i})="", "Sin numero RP en BOGDATA", '
                   f'IF(TRIM({c_rp_sip}{i})=TRIM({c_rp_bog}{i}), "El numero RP coincide", "El numero RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 22, value=f25)

            # 🔹 Ev. Valor RP (Limpiando comas para compatibilidad Web)
            f26 = (f'=IF(AND(TRIM({c_val_rp_s}{i})="", TRIM({c_val_rp_b}{i})=""), "Sin valores RP", '
                   f'IF(TRIM({c_val_rp_s}{i})="", "Sin valor RP en SIPSE", '
                   f'IF(TRIM({c_val_rp_b}{i})="", "Sin valor RP en BOGDATA", '
                   f'IF(IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_rp_s}{i}), ",", "")),TRIM({c_val_rp_s}{i}))=IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_rp_b}{i}), ",", "")),TRIM({c_val_rp_b}{i})), "El valor RP coincide", "El valor RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 23, value=f26)

            # 🔹 Ev. Fecha RP
            f27 = (f'=IF(AND(TRIM({c_fec_rp_s}{i})="", TRIM({c_fec_rp_b}{i})=""), "Sin fechas RP", '
                   f'IF(TRIM({c_fec_rp_s}{i})="", "Sin fecha RP en SIPSE", '
                   f'IF(TRIM({c_fec_rp_b}{i})="", "Sin fecha RP en BOGDATA", '
                   f'IF(TRIM({c_fec_rp_s}{i})=TRIM({c_fec_rp_b}{i}), "La fecha RP coincide", "La fecha RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 24, value=f27)

            # 🔹 Ev. # CDP
            f28 = (f'=IF(AND(TRIM({c_cdp_sip}{i})="", TRIM({c_cdp_bog}{i})=""), "Sin numeros CDP", '
                   f'IF(TRIM({c_cdp_sip}{i})="", "Sin numero CDP en SIPSE", '
                   f'IF(TRIM({c_cdp_bog}{i})="", "Sin numero CDP en BOGDATA", '
                   f'IF(TRIM({c_cdp_sip}{i})=TRIM({c_cdp_bog}{i}), "El numero CDP coincide", "El numero CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 25, value=f28)

            # 🔹 Ev. Valor CDP (Limpiando comas para compatibilidad Web)
            f29 = (f'=IF(AND(TRIM({c_val_cd_s}{i})="", TRIM({c_val_cd_b}{i})=""), "Sin valores CDP", '
                   f'IF(TRIM({c_val_cd_s}{i})="", "Sin valor CDP en SIPSE", '
                   f'IF(TRIM({c_val_cd_b}{i})="", "Sin valor CDP en BOGDATA", '
                   f'IF(IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_cd_s}{i}), ",", "")),TRIM({c_val_cd_s}{i}))=IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_cd_b}{i}), ",", "")),TRIM({c_val_cd_b}{i})), "El valor CDP coincide", "El valor CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 26, value=f29)

            # 🔹 Ev. Fecha CDP
            f30 = (f'=IF(AND(TRIM({c_fec_cd_s}{i})="", TRIM({c_fec_cd_b}{i})=""), "Sin fechas CDP", '
                   f'IF(TRIM({c_fec_cd_s}{i})="", "Sin fecha CDP en SIPSE", '
                   f'IF(TRIM({c_fec_cd_b}{i})="", "Sin fecha CDP en BOGDATA", '
                   f'IF(TRIM({c_fec_cd_s}{i})=TRIM({c_fec_cd_b}{i}), "La fecha CDP coincide", "La fecha CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 27, value=f30)

        # --- ESTILOS ANALISIS ---
        print("  -> Aplicando estilos y ocultando columnas en Analisis...")
        aplicar_estilo_ferrocarril(ws_analisis)
        aplicar_formato_condicional_alertas(ws_analisis, col_inicio_eval)
        
        # Ocultar columnas hoja analisis
        ocultar_columnas_por_lista(ws_analisis, RANGOS_OCULTAR_ANALISIS)

        # ==========================================
        # PARTE 5: Crear HOJA RESULTADOS
        # ==========================================
        print("  -> Creando hoja 'Resultados'...")
        if 'Resultados' in wb.sheetnames:
            del wb['Resultados']
        ws_res = wb.create_sheet("Resultados")
        nombre_limpio = os.path.splitext(archivo)[0]
        ws_res['B1'] = f"Métricas alcaldía local {nombre_limpio}"
        
        letra_ev_nro = get_column_letter(col_inicio_eval + 12)       
        letra_ev_ced = get_column_letter(col_inicio_eval + 14)   
        letra_rep_sipse = get_column_letter(col_inicio_eval + 15)
        letra_rep_secop = get_column_letter(col_inicio_eval + 16)
        letra_fecha_suscripcion = get_column_letter(col_inicio_eval + 17)
        letra_fecha_inicio = get_column_letter(col_inicio_eval + 18)
        letra_fecha_fin = get_column_letter(col_inicio_eval + 19)
        letra_duracion = get_column_letter(col_inicio_eval + 20)
        letra_valor = get_column_letter(col_inicio_eval + 21)

        letra_num_RP = get_column_letter(col_inicio_eval + 22)
        letra_val_RP = get_column_letter(col_inicio_eval + 23)
        letra_fec_RP = get_column_letter(col_inicio_eval + 24)
        letra_num_CDP = get_column_letter(col_inicio_eval + 25)
        letra_val_CDP = get_column_letter(col_inicio_eval + 26)
        letra_fec_CDP = get_column_letter(col_inicio_eval + 27)

        metricas = [
            #("Nro de contratos registrados en SIPSE",'=SUMPRODUCT((Analisis!A2:A10000<>"")/COUNTIF(Analisis!A2:A10000,Analisis!A2:A10000&""))'),
            #("Nro de contratos registrados SECOP",'=SUMPRODUCT((Analisis!EV2:EV10000<>"")/COUNTIF(Analisis!EV2:EV10000,Analisis!EV2:EV10000&""))'),
            ("Nro de contratos registrados SIPSE",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos registrados SECOP",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos registrados en SECOP y SIPSE con numero de contrato igual", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")"),
            ("Nro de contratos solo en SECOP y no en SIPSE", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos solo en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos diferentes en SIPSE-SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato NO coincide\")"),
            ("Nro de contratos Cédula diferente", f"=COUNTIF('Analisis'!{letra_ev_ced}:{letra_ev_ced}, \"Cedula NO coincide\")"),
            ("Nro de contratos dobles en SIPSE", f"=COUNTIF('Analisis'!{letra_rep_sipse}:{letra_rep_sipse}, 2)"),
            ("Nro de contratos dobles en SECOP", f"=COUNTIF('Analisis'!{letra_rep_secop}:{letra_rep_secop}, 2)"),
            ("Fecha de acta de suscriccpión Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"Coincide fecha\")"),
            ("Fecha de acta de suscriccpión no Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"No coinciden fechas\")"),
            ("Fecha de acta de inicio Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Coincide fecha\")"),
            ("Fecha de acta de inicio no coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"No coinciden fechas\")"),
            ("Fecha de Fin conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"Coincide fecha\")"),
            ("Fecha de Fin no conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"No coinciden fechas\")"),
            ("Contratos que coinciden en Duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion Coincide\")"),
            ("Contratos que tienen diferente duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion NO coincide\")"),
            ("Contratos que tienen Fecha de inicio en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Sin fecha SECOP\")"),
            ("Datos en la matriz de seguimiento", f"=C2+C3"),
            ("Valor contrato coinciden", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor coincide\")"),
            ("Valor contrato no coincidenn", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor NO coincide\")"),
            ("Numero RP no coinciden", f"=COUNTIF('Analisis'!{letra_num_RP}:{letra_num_RP}, \"El numero RP NO coincide\")"),
            ("Valor RP no coinciden", f"=COUNTIF('Analisis'!{letra_val_RP}:{letra_val_RP}, \"El valor RP NO coincide\")"),
            ("Fecha RP no coinciden", f"=COUNTIF('Analisis'!{letra_fec_RP}:{letra_fec_RP}, \"La fecha RP NO coincide\")"),
            ("Numero CDP no coinciden", f"=COUNTIF('Analisis'!{letra_num_CDP}:{letra_num_CDP}, \"El numero CDP NO coincide\")"),
            ("Valor CDP no coinciden", f"=COUNTIF('Analisis'!{letra_val_CDP}:{letra_val_CDP}, \"El valor CDP NO coincide\")"),
            ("Fecha CDP no coinciden", f"=COUNTIF('Analisis'!{letra_fec_CDP}:{letra_fec_CDP}, \"La fecha CDP NO coincide\")")
        ]


        for idx, (texto, formula) in enumerate(metricas, start=2):
            ws_res.cell(row=idx, column=1, value=idx-1)
            ws_res.cell(row=idx, column=2, value=texto)
            ws_res.cell(row=idx, column=3, value=formula)
        
        # Pintar B2 de amarillo
        fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
        ws_res['B1'].fill = fill_amarillo
        ws_res.column_dimensions['B'].width = 100
        
        ws_analisis.column_dimensions['FJ'].width = 29
        ws_analisis.column_dimensions['FK'].width = 10
        ws_analisis.column_dimensions['FL'].width = 20
        ws_analisis.column_dimensions['FM'].width = 15
        ws_analisis.column_dimensions['FN'].width = 15
        ws_analisis.column_dimensions['FO'].width = 15
        ws_analisis.column_dimensions['FP'].width = 20
        ws_analisis.column_dimensions['FQ'].width = 20
        ws_analisis.column_dimensions['FR'].width = 18
        ws_analisis.column_dimensions['FS'].width = 14
        ws_analisis.column_dimensions['FT'].width = 25
        ws_analisis.column_dimensions['FU'].width = 25
        ws_analisis.column_dimensions['FV'].width = 25
        ws_analisis.column_dimensions['FW'].width = 25
        ws_analisis.column_dimensions['FX'].width = 25
        ws_analisis.column_dimensions['FY'].width = 25

        dejar_columna_blanca(ws_analisis, 64)# columna 64 es la columna "BL"

        #Ajustar el texto en la fila 1 de las hojas aux y Analisis
        ajustar_texto_fila(ws_secop_aux, 1)
        ajustar_texto_fila(ws_analisis, 1)

        # ==========================================
        # PARTE 6: Crear HOJA 'Para MEMO'
        # ==========================================
        print("  -> Creando hoja 'Para MEMO'...")
        if 'Para MEMO' in wb.sheetnames:
            del wb['Para MEMO']
        ws_memo = wb.create_sheet("Para MEMO")
        
        datos_memo = []
        
        # 1. Escanear fila por fila extrayendo los datos directos
        for i in range(2, ultima_fila_analisis + 1):
            val_b = ws_analisis.cell(row=i, column=2).value # B: SIPSE (Solicitud)
            val_d = ws_analisis.cell(row=i, column=4).value # D: No. Contrato
            
            # Extraer celdas origen según tu matriz (SIPSE y SECOP)
            susc_sipse = ws_analisis.cell(row=i, column=8).value  # H
            susc_secop = ws_analisis.cell(row=i, column=83).value # CE
            ini_sipse  = ws_analisis.cell(row=i, column=9).value  # I
            ini_secop  = ws_analisis.cell(row=i, column=84).value # CF
            fin_sipse  = ws_analisis.cell(row=i, column=10).value # J
            fin_secop  = ws_analisis.cell(row=i, column=85).value # CG
            dur_sipse  = ws_analisis.cell(row=i, column=18).value # R
            dur_secop  = ws_analisis.cell(row=i, column=135).value# EE
            
            # Evaluarlas en Python
            err_susc = es_error_fecha(susc_sipse, susc_secop)
            err_ini  = es_error_fecha(ini_sipse, ini_secop)
            err_fin  = es_error_fecha(fin_sipse, fin_secop)
            err_dur  = es_error_duracion(dur_sipse, dur_secop)
            
            # Si tiene al menos 1 error, lo enlistamos
            if err_susc or err_ini or err_fin or err_dur:
                llave_orden = (err_susc, err_ini, err_fin, err_dur)
                datos_memo.append([
                    val_b, val_d,
                    "X" if err_susc else "",
                    "X" if err_ini else "",
                    "X" if err_fin else "",
                    "X" if err_dur else "",
                    llave_orden # Llave secreta para ordenar
                ])
                
        # 2. Ordenar datos para que queden agrupados por color como en tu imagen
        datos_memo.sort(key=lambda x: x[6], reverse=True)
        
        # 3. Diseño de la Cabecera
        relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
        fuente_blanca = Font(color="FFFFFF", bold=True)
        ws_memo.merge_cells("A1:A2")
        ws_memo['A1'] = "SIPSE"
        ws_memo.merge_cells("B1:B2")
        ws_memo['B1'] = "No. Contrato"
        ws_memo.merge_cells("C1:F1")
        ws_memo['C1'] = "TIPOLOGÍA DE LA INCONSISTENCIA"
        
        ws_memo['C2'] = "Fecha suscripción"
        ws_memo['D2'] = "Fecha Inicio"
        ws_memo['E2'] = "Fecha Fin"
        ws_memo['F2'] = "Plazo"
        ws_memo['A1'].fill = relleno_rojo
        ws_memo['B1'].fill = relleno_rojo
        ws_memo['C1'].fill = relleno_rojo
        ws_memo['C2'].fill = relleno_rojo
        ws_memo['D2'].fill = relleno_rojo
        ws_memo['E2'].fill = relleno_rojo
        ws_memo['F2'].fill = relleno_rojo
        
        align_header = Alignment(horizontal="center", vertical="center")
        borde_delgado = Border(left=Side(style='thin', color='000000'), right=Side(style='thin', color='000000'), top=Side(style='thin', color='000000'), bottom=Side(style='thin', color='000000'))
        
        for fila_h in ws_memo['A1:F2']:
            for celda in fila_h:
                celda.font = fuente_blanca
                celda.alignment = align_header
                celda.border = borde_delgado
                
        # 4. Volcar datos extraídos
        fila_actual = 3
        for fila_datos in datos_memo:
            for col_idx in range(1, 7):
                celda = ws_memo.cell(row=fila_actual, column=col_idx, value=fila_datos[col_idx-1])
                celda.border = borde_delgado
                celda.alignment = align_header # Centrado para todo
                
            fila_actual += 1
            
        # 5. Ajustar anchos
        ws_memo.column_dimensions['A'].width = 15
        ws_memo.column_dimensions['B'].width = 15
        ws_memo.column_dimensions['C'].width = 20
        ws_memo.column_dimensions['D'].width = 15
        ws_memo.column_dimensions['E'].width = 15
        ws_memo.column_dimensions['F'].width = 15

        wb.save(ruta_dest)
        print("  -> Procesado exitosamente.")

if __name__ == "__main__":
    procesar_archivos()


--- SELECCIÓN BOGDATA ---
1. Por favor selecciona el archivo de CDP (Excel)...
2. Por favor selecciona el archivo de RP (Excel)...
  -> Procesando archivo RP...
  -> Procesando archivo CDP...
Se encontraron 21 archivos de Excel.
Procesando: ANTONIO NARINO.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 'Para MEMO'...
  -> Procesado exitosamente.
Procesando: BARRIOS UNIDOS.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 'Para MEMO'...
  -> Procesado exitosamente.
Procesando: BOSA.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -

In [3]:
import os
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
import re
import unicodedata

# ================= CONFIGURACIÓN =================
#CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1' 
#ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1\Consolidado\Consolidado_Localidades.xlsx"
#CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 03 MAYO\Diagnóstico' # Dell
#ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 03 MAYO\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx" #Dell
CARPETA_ORIGEN = r'D:\\Años\\2026\\SecretariaDeGobierno\\ConsolidadoAlcaldias\\META 7 - FDL 03 MAYO\\Diagnóstico' #Predatos
ARCHIVO_DESTINO = r"D:\\Años\\2026\\SecretariaDeGobierno\\ConsolidadoAlcaldias\\META 7 - FDL 03 MAYO\\Diagnóstico\\Consolidado\\Consolidado_Localidades.xlsx" #Predator

nombres_hojas = [
    "1-Usaquén", "2-Chapinero", "3-Santa Fe", "4-San Cristóbal",
    "5-Usme", "6-Tunjuelito", "7-Bosa", "8-Kennedy", "9-Fontibón",
    "10-Engativá", "11-Suba", "12-Barrios Unidos", "13-Teusaquillo",
    "14-Los Mártires", "15-Antonio Nariño", "16-Puente Aranda",
    "17-Rafael Uribe", "18-La Candelaria", "19-Ciudad Bolívar", "20-Sumapaz"
]

#RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']
# Aquí puedes añadir todas las columnas a las que quieras fijarles un ancho específico
ANCHOS_COLUMNAS = {
    'FM': 29,
    'FN': 10,
    'FO': 20,
    'FP': 15,
    'FQ': 15,
    'FR': 15,
    'FS': 20,
    'FT': 20,
    'FU': 18,
    'FV': 14,
    'FW': 25,
    'FX': 25,
    'FY': 25,
    'FZ': 25,
    'GA': 25,
    'GB': 25
}

# ================= FUNCIONES AUXILIARES =================
def normalizar_texto(texto):
    texto = texto.lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^\d+[-_ ]?', '', texto)
    texto = re.sub(r'\b(la|el|los|las)\b', '', texto).strip()
    return texto

def encontrar_hoja_destino(nombre_archivo, hojas_destino):
    archivo_norm = normalizar_texto(nombre_archivo)
    for hoja in hojas_destino:
        if normalizar_texto(hoja) in archivo_norm:
            return hoja
    return None

def ocultar_columnas_por_lista(ws, lista_rangos, offset=3):
    for item in lista_rangos:
        item = item.upper().strip()
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio) + offset
            idx_fin = column_index_from_string(fin) + offset
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                ws.column_dimensions[letra].hidden = True 
        else:
            idx = column_index_from_string(item) + offset
            letra = get_column_letter(idx)
            ws.column_dimensions[letra].hidden = True

def adaptar_formulas_resultados(formula):
    pattern = re.compile(r"'?Analisis'?!([A-Za-z]+)(\d*)(?::([A-Za-z]+)(\d*))?")
    def replacer(match):
        col1, row1, col2, row2 = match.groups()
        nueva_col1 = get_column_letter(column_index_from_string(col1) + 3)
        nueva_row1 = str(int(row1) + 21) if row1 else ""
        resultado = f"{nueva_col1}{nueva_row1}"
        if col2:
            nueva_col2 = get_column_letter(column_index_from_string(col2) + 3)
            nueva_row2 = str(int(row2) + 21) if row2 else ""
            resultado += f":{nueva_col2}{nueva_row2}"
        return resultado
    return pattern.sub(replacer, formula)

def desplazar_formulas_analisis(formula):
    formula = re.sub(r"'?Analisis'?!", "", formula, flags=re.IGNORECASE)
    pattern = re.compile(r'(\$?)([A-Za-z]{1,3})(\$?)(\d+)(?!\s*\()')
    def replacer(match):
        lock_col, col_str, lock_row, row_str = match.groups()
        nueva_col = get_column_letter(column_index_from_string(col_str) + 3)
        nueva_fila = str(int(row_str) + len(metricas)+2)
        return f"{lock_col}{nueva_col}{lock_row}{nueva_fila}"
    return pattern.sub(replacer, formula)

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# ================= FUNCIONES DE DISEÑO =================
def aplicar_estilo_ferrocarril(ws, start_row, max_row):
    """Aplica formato estilo ferrocarril solo a columnas E:U y CA:FH desde start_row."""
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    
    # Rango 1: E (5) a AL (38) | Rango 2: CA (79) a GB (185)
    def es_columna_objetivo(c):
        return (5 <= c <= 38) or (79 <= c <= 185)

    # 1. Encabezados (Fila 22)
    for col_idx in range(1, ws.max_column + 1):
        if es_columna_objetivo(col_idx):
            cell = ws.cell(row=start_row, column=col_idx)
            cell.fill = fill_header
            cell.font = font_header
            cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas de Datos (Patrón Ferrocarril)
    for i_fila, r_idx in enumerate(range(start_row + 1, max_row + 1)):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF": # Evitar procesar blanco para ahorrar memoria
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for col_idx in range(1, ws.max_column + 1):
                if es_columna_objetivo(col_idx):
                    ws.cell(row=r_idx, column=col_idx).fill = relleno

def aplicar_formato_condicional_alertas(ws, start_row, max_row):
    """Aplica reglas condicionales desde la columna EZ."""
    col_inicio_eval = column_index_from_string('FA')
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    
    reglas = [
        (12, 'equal', '"Numero contrato NO coincide"'),
        (14, 'equal', '"Cedula NO coincide"'),
        (15, 'greaterThan', '1'),
        (16, 'greaterThan', '1'),
        (17, 'equal', '"No coinciden fechas"'),
        (18, 'equal', '"No coinciden fechas"'),
        (19, 'equal', '"No coinciden fechas"'),
        (20, 'equal', '"Duracion NO coincide"'),
        (21, 'equal', '"Valor NO coincide"'),
        (22, 'equal', '"El numero RP NO coincide"'),
        (23, 'equal', '"El valor RP NO coincide"'),
        (24, 'equal', '"La fecha RP NO coincide"'),
        (25, 'equal', '"El numero CDP NO coincide"'),
        (26, 'equal', '"El valor CDP NO coincide"'),
        (27, 'equal', '"La fecha CDP NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        # Se aplica desde la fila siguiente al encabezado (23 en adelante)
        rango = f"{letra_col}{start_row + 1}:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

# ================= PROGRAMA PRINCIPAL =================
def consolidar_excels():
    global metricas
    wb_nuevo = openpyxl.Workbook()
    
    # 1. Configurar Hoja General
    ws_general = wb_nuevo.active
    ws_general.title = "General"
    ws_general.cell(row=1, column=1, value="Nro").font = Font(bold=True)
    ws_general.cell(row=1, column=2, value="Alcaldía local").font = Font(bold=True)
    
    # Pre-llenar las localidades en General con Hipervínculos
    for i, hoja in enumerate(nombres_hojas):
        num, nombre = hoja.split("-", 1)
        row_idx = i + 2
        ws_general.cell(row=row_idx, column=1, value=int(num))
        
        celda_loc = ws_general.cell(row=row_idx, column=2, value=nombre.strip())
        celda_loc.hyperlink = f"#'{hoja}'!A1"
        celda_loc.font = Font(color="0563C1", underline="single") 

    for nombre in nombres_hojas:
        wb_nuevo.create_sheet(title=nombre)

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith('.xlsx') and not f.startswith('~')]
    metricas_extraidas = False 

    for archivo in archivos:
        hoja_destino_nombre = encontrar_hoja_destino(archivo, nombres_hojas)
        
        if not hoja_destino_nombre:
            continue
            
        print(f"-> Procesando '{archivo}' -> Destino: '{hoja_destino_nombre}'")
        
        ruta_completa = os.path.join(CARPETA_ORIGEN, archivo)
        wb_origen = openpyxl.load_workbook(ruta_completa)
        ws_destino = wb_nuevo[hoja_destino_nombre]

        # --- TAREA A: Copiar de 'Resultados' ---
        if "Resultados" in wb_origen.sheetnames:
            ws_resultados = wb_origen["Resultados"]
            row_gen = nombres_hojas.index(hoja_destino_nombre) + 2
            col_gen = 3 
            
            for r_idx in range(0, len(metricas)): 
                titulo_metrica = ws_resultados.cell(row=2 + r_idx, column=2).value 
                
                if not metricas_extraidas:
                    ws_general.cell(row=1, column=col_gen, value=titulo_metrica).font = Font(bold=True)
                
                ws_general.cell(row=row_gen, column=col_gen, value=f"='{hoja_destino_nombre}'!C{r_idx+2}")
                col_gen += 1
                
            metricas_extraidas = True 
            
            for row in ws_resultados['A1':f'C{len(metricas) + 1}']:
                for cell in row:
                    dest_coord = cell.coordinate
                    valor = cell.value
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = adaptar_formulas_resultados(valor)
                    else:
                        ws_destino[dest_coord].value = valor
        else:
            print(f"  - Faltó hoja 'Resultados' en {archivo}")

        # --- TAREA B: Copiar de 'Analisis' a la fila 22 ---
        if "Analisis" in wb_origen.sheetnames:
            ws_analisis = wb_origen["Analisis"]
            ultima_fila = ws_analisis.max_row
            max_col_origen = min(ws_analisis.max_column, column_index_from_string('FY'))
            
            ws_destino.merge_cells(f"D{len(metricas) + 2}:BN{len(metricas) + 2}")
            celda_sipse = ws_destino[f"D{len(metricas) + 2}"]
            celda_sipse.value = "SIPSE"
            celda_sipse.fill = PatternFill(start_color="0070C0", end_color="0070C0", fill_type="solid") 
            celda_sipse.font = Font(color="FFFFFF", bold=True)
            celda_sipse.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells(f"BP{len(metricas) + 2}:EX{len(metricas) + 2}")
            celda_secop = ws_destino[f"BP{len(metricas) + 2}"]
            celda_secop.value = "SECOP"
            celda_secop.fill = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid") 
            celda_secop.font = Font(color="FFFFFF", bold=True)
            celda_secop.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells(f"EY{len(metricas) + 2}:GB{len(metricas) + 2}")
            celda_comparacion = ws_destino[f"EY{len(metricas) + 2}"]
            celda_comparacion.value = "COMPARACIÓN"
            celda_comparacion.fill = PatternFill(start_color="7e350e", end_color="7e350e", fill_type="solid") 
            celda_comparacion.font = Font(color="FFFFFF", bold=True)
            celda_comparacion.alignment = Alignment(horizontal="center", vertical="center")

            # Pegar los datos desde la fila 22
            for r_idx, row_origen in enumerate(ws_analisis.iter_rows(min_row=1, max_row=ultima_fila, min_col=1, max_col=max_col_origen), start=len(metricas) + 3):
                for c_idx, cell in enumerate(row_origen, start=4):
                    dest_coord = f"{get_column_letter(c_idx)}{r_idx}"
                    valor = cell.value
                    
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = desplazar_formulas_analisis(valor)
                    else:
                        ws_destino[dest_coord].value = valor
            
            # 1. Ocultar columnas
            ocultar_columnas_por_lista(ws_destino, RANGOS_OCULTAR_ANALISIS, offset=3)
            # 2. Aplicar anchos de columnas
            for letra_col, ancho in ANCHOS_COLUMNAS.items():
                ws_destino.column_dimensions[letra_col].width = ancho
            # 2.1 Ajustar el alto del texto en la fila de encabezados de las hojas de cada localidad
            ajustar_texto_fila(ws_destino, len(metricas) + 3)
            # 3. Aplicar estilo Ferrocarril (La tabla empieza en la 22)
            aplicar_estilo_ferrocarril(ws_destino, start_row=len(metricas) + 3, max_row=ultima_fila + len(metricas) + 2)
            # 4. Aplicar alertas condicionales
            aplicar_formato_condicional_alertas(ws_destino, start_row=len(metricas) + 3, max_row=ultima_fila + len(metricas) + 2)
            # 5 Pintar y ajustar tabla de cada alcaldía
            fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
            ws_destino['B1'].fill = fill_amarillo
            ws_destino.column_dimensions['B'].width = 100
            
            # --- NUEVO: BOTÓN DE REGRESO A GENERAL ---
            celda_volver = ws_destino['A1']
            celda_volver.value = "General"
            celda_volver.hyperlink = "#'General'!A1"
            celda_volver.font = Font(color="0563C1", underline="single", bold=True)
            celda_volver.fill = PatternFill(start_color="E7E6E6", end_color="E7E6E6", fill_type="solid") # Gris claro
            celda_volver.alignment = Alignment(horizontal="center", vertical="center")

        else:
            print(f"  - Faltó hoja 'Analisis' en {archivo}")
            
        wb_origen.close()

    # --- DISEÑO FINAL HOJA GENERAL ---
    # 1. Fondo Rojo para la fila de títulos A1:AC1
    relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
    fuente_blanca = Font(color="FFFFFF", bold=True)
    for col in range(1, 30): # De la A (1) a la AC (30)
        celda = ws_general.cell(row=1, column=col)
        celda.fill = relleno_rojo
        celda.font = fuente_blanca
        celda.alignment = Alignment(horizontal="center", vertical="center")

    # 2. Centrar los valores de C1:AC1
    alineacion_centro = Alignment(horizontal="center", vertical="center")
    for r in range(2, 22): # Filas 2 a 21
        for c in range(3, 29): # Columnas C (3) a AC (30)
            ws_general.cell(row=r, column=c).alignment = alineacion_centro
            
    # Ajustar ligeramente el ancho de General para que se vea bien
    ws_general.column_dimensions['A'].width = 5
    ws_general.column_dimensions['B'].width = 25
    for c in range(3, 22):
        ws_general.column_dimensions[get_column_letter(c)].width = 15
    
    # Ajustar el texto en la fila 1 de las hoja General
    ajustar_texto_fila(ws_general, 1)

    # --- NUEVO: APLICAR GRILLA DELGADA (A1:U21) ---
    borde_delgado = Border(
        left=Side(style='thin', color='000000'),
        right=Side(style='thin', color='000000'),
        top=Side(style='thin', color='000000'),
        bottom=Side(style='thin', color='000000')
    )
    
    for r in range(1, 22):      # Filas de la 1 a la 21
        for c in range(1, 30):  # Columnas de la A(1) a la AC(29)
            ws_general.cell(row=r, column=c).border = borde_delgado
    
    # CARPETA 'CONSOLIDADO' SI NO EXISTE ---
    carpeta_destino = os.path.dirname(ARCHIVO_DESTINO) # Extrae la ruta de la carpeta
    os.makedirs(carpeta_destino, exist_ok=True)        # Crea la carpeta (y subcarpetas si faltan) sin dar error si ya existe

    wb_nuevo.save(ARCHIVO_DESTINO)
    print(f"\n🚀 ¡Proceso completado! Archivo guardado en: {ARCHIVO_DESTINO}")

if __name__ == "__main__":
    consolidar_excels()

PermissionError: [WinError 21] El dispositivo no está listo: 'D:\\\\Años\\\\2026\\\\SecretariaDeGobierno\\\\ConsolidadoAlcaldias\\\\META 7 - FDL 03 MAYO\\\\Diagnóstico'

#### Entrega 8 06 2026

In [1]:
import pandas as pd
import unicodedata
import re
import difflib
import os
import tkinter as tk
from tkinter import filedialog, messagebox

# ==========================================
# 0. DEFINICIÓN DE ESTRUCTURA ESPERADA
# ==========================================
# Reemplaza los elementos de estas listas con los nombres EXACTOS 
# y en el ORDEN que necesitas para cada hoja.
COLUMNAS_ESPERADAS_SIPSE = [
    "CONTRATO", "SOLICITUD", "VIGENCIA", "NUMERO_CONTRATO", "TIPO", "TIPO_PROCESO", "OBJETO", "FECHA_SUSCRIPCION", "FECHA_INICIO", "FECHA_TERMINACION", "TERMINA31_DICIEMBRE", "CONTRATISTA", "DOC_CONTRATISTA", "VALOR_INICIAL", "ADICIONES", "VALOR_TOTAL", "VALOR_MENSUAL", "PLAZO", "PRORROGAS", "PROYECTO", "DEPENDENCIA", "SUPERVISION", "ESTADO", "NAT", "PAGOS", "AVANCE", "FECHA_ACTA_LIQUIDACION", "PRESUPUESTO", "ANULACION", "DIRECCION_CONTRATISTA", "TELEFONO_CONTRATISTA", "CODIGO_PROYECTO", "NOMBRE_PROYECTO", "CDPS", "RPS", "ESTADO_RP", "MODIFICACION", "NUMERO_OC", "TIPO_SOCIEDAD", "TIPOLOGIA", "TIPO_PAGO", "NIVELACADEMICO", "PROFEXION", "ESPECIALIZ", "POLIZA", "PROCESO_CONTRATO", "ANTIGUEDAD", "SALDO_FAVOR_ENTIDAD", "FECHA_NACIMIENTO", "OBLIGACIONES", "TIPO_IDENTIFICACION", "CORREO_ELECTRONICO", "NOMBRE", "APELLIDO", "CELULAR", "GARANTIAS", "INICIO_AMPARO", "FIN_AMPARO", "IDEN_REP_LEGAL", "NOM_REP_LEGAL", "ENTIDAD", "DESCRIPCION_ENTIDAD", "GENERO"
]

COLUMNAS_ESPERADAS_SECOP = [
    "Nombre Entidad", "Nit Entidad", "Departamento", "Ciudad", "LocalizaciÃ³n", "Orden", "Sector", "Rama", "Entidad Centralizada", "Proceso de Compra", "ID Contrato", "Referencia del Contrato", "Estado Contrato", "Codigo de Categoria Principal", "Descripcion del Proceso", "Tipo de Contrato", "Modalidad de Contratacion", "Justificacion Modalidad de Contratacion", "Fecha de Firma", "Fecha de Inicio del Contrato", "Fecha de Fin del Contrato", "Condiciones de Entrega", "TipoDocProveedor", "Documento Proveedor", "Proveedor Adjudicado", "Es Grupo", "Es Pyme", "Habilita Pago Adelantado", "LiquidaciÃ³n", "ObligaciÃ³n Ambiental", "Obligaciones Postconsumo", "Reversion", "Origen de los Recursos", "Destino Gasto", "Valor del Contrato", "Valor de pago adelantado", "Valor Facturado", "Valor Pendiente de Pago", "Valor Pagado", "Valor Amortizado", "Valor Pendiente de Amortizacion", "Valor Pendiente de Ejecucion", "Estado BPIN", "CÃ³digo BPIN", "Anno BPIN", "Saldo CDP", "Saldo Vigencia", "EsPostConflicto", "Dias adicionados", "Puntos del Acuerdo", "Pilares del Acuerdo", "URLProceso", "Nombre Representante Legal", "Nacionalidad Representante Legal", "Domicilio Representante Legal", "Tipo de IdentificaciÃ³n Representante Legal", "IdentificaciÃ³n Representante Legal", "GÃ©nero Representante Legal", "Presupuesto General de la Nacion â€“ PGN", "Sistema General de Participaciones", "Sistema General de RegalÃ­as", "Recursos Propios (AlcaldÃ­as, Gobernaciones y Resguardos IndÃ­genas)", "Recursos de Credito", "Recursos Propios", "Ultima Actualizacion", "Codigo Entidad", "Codigo Proveedor", "Fecha Inicio Liquidacion", "Fecha Fin Liquidacion", "Objeto del Contrato", "DuraciÃ³n del contrato", "Nombre del banco", "Tipo de cuenta", "NÃºmero de cuenta", "El contrato puede ser prorrogado", "Fecha de notificaciÃ³n de prorrogaciÃ³n", "Nombre ordenador del gasto", "Tipo de documento Ordenador del gasto", "NÃºmero de documento Ordenador del gasto", "Nombre supervisor", "Tipo de documento supervisor", "NÃºmero de documento supervisor", "Nombre Ordenador de Pago", "Tipo de documento Ordenador de Pago", "NÃºmero de documento Ordenador de Pago", "Documentos Tipo", "Descripcion Documentos Tipo"
]

# ==========================================
# 1. FUNCIONES DE LIMPIEZA, COMPARACIÓN Y ESTRUCTURA
# ==========================================

def normalizar_texto(texto):
    """Limpia el texto: quita nulos, espacios, pasa a mayúsculas, quita tildes y artículos"""
    if pd.isna(texto):
        return ""
    texto = str(texto).upper().strip()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^(EL|LA|LOS|LAS)\s+', '', texto)
    return texto

def unificar_localidades(lista_localidades, similitud_minima=0.85):
    """Agrupa nombres similares bajo un mismo nombre 'canónico'"""
    nombres_canonicos = {}
    for loc in lista_localidades:
        if not loc: continue
        coincidencias = difflib.get_close_matches(loc, nombres_canonicos.keys(), n=1, cutoff=similitud_minima)
        if coincidencias:
            nombres_canonicos[loc] = coincidencias[0]
        else:
            nombres_canonicos[loc] = loc
    return nombres_canonicos

def alinear_columnas(df, columnas_esperadas):
    """
    Compara las columnas del df con la lista esperada.
    Crea las faltantes en blanco y reordena el dataframe.
    """
    if not columnas_esperadas:
        return df

    for col in columnas_esperadas:
        if col not in df.columns:
            df[col] = "" 
            
    columnas_extra = [col for col in df.columns if col not in columnas_esperadas]
    orden_final = columnas_esperadas + columnas_extra
    
    return df[orden_final]

# ==========================================
# 2. SELECCIÓN DE ARCHIVOS Y CARPETAS
# ==========================================

root = tk.Tk()
root.withdraw()

print("Por favor, selecciona el archivo Excel de origen...")
archivo_origen = filedialog.askopenfilename(
    title="Selecciona el archivo Excel (.xlsx o .xls)",
    filetypes=[("Archivos de Excel", "*.xlsx *.xls *")]
)

if not archivo_origen:
    print("No se seleccionó ningún archivo. Cancelando proceso.")
    exit()

print("Por favor, selecciona la carpeta de destino...")
carpeta_destino = filedialog.askdirectory(title="Selecciona la carpeta de destino")

if not carpeta_destino:
    print("No se seleccionó ninguna carpeta. Cancelando proceso.")
    exit()

# ==========================================
# 3. LECTURA Y PROCESAMIENTO DEL EXCEL
# ==========================================

try:
    print(f"\nLeyendo el archivo: {archivo_origen}")
    xls = pd.ExcelFile(archivo_origen)
    nombres_hojas = xls.sheet_names
    
    hoja_sipse = next((h for h in nombres_hojas if h.upper() == "SIPSE"), None)
    hoja_secop = next((h for h in nombres_hojas if h.upper() == "SECOP II"), None)
    
    if not hoja_sipse or not hoja_secop:
        raise ValueError("El archivo no contiene las hojas 'SIPSE' y/o 'SECOP II'.")

    df_sipse = pd.read_excel(xls, sheet_name=hoja_sipse)
    df_secop = pd.read_excel(xls, sheet_name=hoja_secop)
    
    if len(df_sipse.columns) <= 61:
        raise ValueError("La hoja SIPSE no tiene suficientes columnas para llegar a la columna BJ (índice 61).")
        
    # Identificar nombres de columnas clave
    col_sipse_nombre = df_sipse.columns[61] 
    col_secop_nombre = df_secop.columns[0]

    # --- NUEVO CAMBIO: REEMPLAZO ESPECÍFICO DE RAFAEL URIBE ---
    print("Ejecutando reemplazo de nombres de entidad específicos...")
    
    # Reemplazo en SECOP II (Columna A)
    df_secop[col_secop_nombre] = df_secop[col_secop_nombre].replace("RAFAEL URIBE URIBE", "RAFAEL URIBE")
    
    # Reemplazo en SIPSE (Columna BJ / índice 61)
    df_sipse[col_sipse_nombre] = df_sipse[col_sipse_nombre].replace("RAFAEL URIBE URIBE", "RAFAEL URIBE")
    # ---------------------------------------------------------

    print("Alineando y estandarizando columnas según listas esperadas...")
    df_sipse = alinear_columnas(df_sipse, COLUMNAS_ESPERADAS_SIPSE)
    df_secop = alinear_columnas(df_secop, COLUMNAS_ESPERADAS_SECOP)
    
    print("Normalizando nombres de localidades...")
    df_sipse['Loc_Limpia'] = df_sipse[col_sipse_nombre].apply(normalizar_texto)
    df_secop['Loc_Limpia'] = df_secop[col_secop_nombre].apply(normalizar_texto)
    
    todas_localidades = list(set(df_sipse['Loc_Limpia'].unique()) | set(df_secop['Loc_Limpia'].unique()))
    mapa_canonico = unificar_localidades(todas_localidades)
    
    df_sipse['Loc_Final'] = df_sipse['Loc_Limpia'].map(mapa_canonico)
    df_secop['Loc_Final'] = df_secop['Loc_Limpia'].map(mapa_canonico)
    
    localidades_finales = set(mapa_canonico.values())
    
    print(f"Se detectaron {len(localidades_finales)} localidades únicas. Generando archivos...")
    
    archivos_creados = 0
    for loc in localidades_finales:
        if not loc: continue 
        
        filtro_sipse = df_sipse[df_sipse['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        filtro_secop = df_secop[df_secop['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        
        if not filtro_sipse.empty or not filtro_secop.empty:
            nombre_archivo_seguro = re.sub(r'[\\/*?:"<>|]', "", loc)
            ruta_salida = os.path.join(carpeta_destino, f"{nombre_archivo_seguro}.xlsx")
            
            with pd.ExcelWriter(ruta_salida, engine='openpyxl') as writer:
                filtro_sipse.to_excel(writer, sheet_name="SIPSE", index=False)
                filtro_secop.to_excel(writer, sheet_name="SECOP II", index=False)
                
            archivos_creados += 1
            print(f" -> Creado: {nombre_archivo_seguro}.xlsx")

    print(f"\n¡Proceso Finalizado Exitosamente! Se crearon {archivos_creados} archivos.")
    messagebox.showinfo("Éxito", f"Se generaron {archivos_creados} archivos correctamente.")

except Exception as e:
    print(f"\nOcurrió un error: {e}")
    messagebox.showerror("Error", f"Ocurrió un error procesando el archivo:\n{e}")

finally:
    root.destroy()

Por favor, selecciona el archivo Excel de origen...
Por favor, selecciona la carpeta de destino...

Leyendo el archivo: C:/Users/Pc/Desktop/META 7/Datos Virgenes/Sipse vs Secop II 01 08 indicadores de calidad.xlsx
Ejecutando reemplazo de nombres de entidad específicos...
Alineando y estandarizando columnas según listas esperadas...
Normalizando nombres de localidades...
Se detectaron 21 localidades únicas. Generando archivos...
 -> Creado: TEUSAQUILLO.xlsx
 -> Creado: SAN CRISTOBAL.xlsx
 -> Creado: CIUDAD BOLIVAR.xlsx
 -> Creado: SUMAPAZ.xlsx
 -> Creado: SANTA FE.xlsx
 -> Creado: MARTIRES.xlsx
 -> Creado: SUBA.xlsx
 -> Creado: ANTONIO NARINO.xlsx
 -> Creado: BOSA.xlsx
 -> Creado: BARRIOS UNIDOS.xlsx
 -> Creado: USAQUEN.xlsx
 -> Creado: CANDELARIA.xlsx
 -> Creado: ENGATIVA.xlsx
 -> Creado: FONTIBON.xlsx
 -> Creado: USME.xlsx
 -> Creado: CHAPINERO.xlsx
 -> Creado: PUENTE ARANDA.xlsx
 -> Creado: SDG - NIVEL CENTRAL.xlsx
 -> Creado: TUNJUELITO.xlsx
 -> Creado: KENNEDY.xlsx
 -> Creado: RAFA

In [ ]:
import os
import shutil
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
from collections import defaultdict
import re
import pandas as pd
import difflib # para busqueda aproximada
from rapidfuzz import fuzz, process
import datetime

#Para abrir explorador de archivos
import tkinter as tk
from tkinter import filedialog
import math
import unicodedata

# --- CONFIGURACIÓN DE CARPETAS ---
#CARPETA_ORIGEN = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO'
#CARPETA_DESTINO = 'C:\\Users\\Pc\\Desktop\\META 7 - FDL 01 FEBRERO\\DiagnóstivoV2_1'
CARPETA_ORIGEN = r'C:\\Users\\Pc\\Desktop\\META 7' #Dell
CARPETA_DESTINO = r"C:\\Users\\Pc\\Desktop\\META 7\\Diagnóstico" #Dell
#CARPETA_ORIGEN = r'D:\\Años\\2026\\SecretariaDeGobierno\\ConsolidadoAlcaldias\\META 7' #Predator
#CARPETA_DESTINO = r"D:\\Años\\2026\\SecretariaDeGobierno\\ConsolidadoAlcaldias\\META 7\\Diagnóstico" #Predator
#CARPETA_ORIGEN = r'D:\\Trabajo\\2026\\SecretariaDeGobierno\\BasesDeDatosSipseSecop\\2026_03_06\\DatosPartidos' #Predator
#CARPETA_DESTINO = r"D:\\Trabajo\\2026\\SecretariaDeGobierno\\BasesDeDatosSipseSecop\\2026_03_06\\DatosPartidos\\Diagnóstico" #PRedator

# --- RANGOS A OCULTAR (CORREGIDOS PARA USAR HIDDEN=TRUE) ---
# Nota: He ajustado 'A-AH' a 'Z-AH' en SECOP para NO ocultar las columnas L, M, T, X, Y que necesitas ver.
RANGOS_OCULTAR_SECOP = ['A-K', 'N-S', 'U-W', 'Z-AH', 'AJ-BR', 'BT-CI']
RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-AG','AJ-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']

# --- DEFINICIÓN DE FUNCIONES ---

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def ocultar_columnas_por_lista(ws, lista_rangos):
    """
    Recibe una lista de cadenas como ['A-K', 'M'] y establece la propiedad HIDDEN = True.
    """
    for item in lista_rangos:
        item = item.upper().strip()
        
        # Caso 1: Es un rango (ej: "A-K")
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio)
            idx_fin = column_index_from_string(fin)
            
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                # Esta es la forma correcta de ocultar en Excel
                ws.column_dimensions[letra].hidden = True 
                
        # Caso 2: Es una columna única (ej: "A")
        else:
            ws.column_dimensions[item].hidden = True

def aplicar_estilo_ferrocarril(ws):
    """
    Encabezados Azul Oscuro + Letra Blanca.
    Filas con patrón intercalado.
    """
    # 1. Encabezados
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    
    for cell in ws[1]:
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas (Patrón Ferrocarril)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    max_col = ws.max_column
    
    for i_fila, row in enumerate(ws.iter_rows(min_row=2, max_col=max_col), start=0):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF":
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for cell in row:
                cell.fill = relleno

def dejar_columna_blanca(ws, indice_columna):
    relleno_blanco = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    
    # Recorremos todas las celdas de la columna indicada
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=indice_columna, max_col=indice_columna):
        for cell in row:
            cell.fill = relleno_blanco

def aplicar_formato_condicional_alertas(ws, col_inicio_eval):
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    max_row = ws.max_row
    
    reglas = [
        (12, 'equal', '"Numero contrato NO coincide"'),
        (14, 'equal', '"Cedula NO coincide"'),
        (15, 'greaterThan', '1'),
        (16, 'greaterThan', '1'),
        (17, 'equal', '"No coinciden fechas"'),
        (18, 'equal', '"No coinciden fechas"'),
        (19, 'equal', '"No coinciden fechas"'),
        (20, 'equal', '"Duracion NO coincide"'),
        (21, 'equal', '"Valor NO coincide"'),
        (22, 'equal', '"El numero RP NO coincide"'),
        (23, 'equal', '"El valor RP NO coincide"'),
        (24, 'equal', '"La fecha RP NO coincide"'),
        (25, 'equal', '"El numero CDP NO coincide"'),
        (26, 'equal', '"El valor CDP NO coincide"'),
        (27, 'equal', '"La fecha CDP NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        rango = f"{letra_col}2:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

def normalizar_numero_contrato(valor):
    if valor is None: return 0
    texto = str(valor).strip()
    # Busca el primer grupo de números en la cadena
    match = re.search(r'\d+', texto)
    if match:
        return int(match.group())
    return 0

#Funciones Hoja "Para MEMO"
def es_error_fecha(val_sipse, val_secop):
    """Replica la fórmula de Excel de fechas para detectar discrepancias imitando tu cruce de día/mes."""
    s1 = str(val_sipse).strip() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip() if pd.notna(val_secop) and val_secop is not None else ""
    
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    
    # Parsear SIPSE 
    dt_sipse = None
    if isinstance(val_sipse, (datetime.datetime, datetime.date)):
        dt_sipse = datetime.date(val_sipse.year, val_sipse.month, val_sipse.day)
    else:
        try:
            dt_sipse = pd.to_datetime(s1.replace("-", "/"), dayfirst=True).date()
        except: pass

    # Parsear SECOP imitando la fórmula FECHA(AÑO, DIA, MES)
    dt_secop = None
    if isinstance(val_secop, (datetime.datetime, datetime.date)):
        try:
            dt_secop = datetime.date(val_secop.year, val_secop.day, val_secop.month)
        except: pass
    else:
        partes = s2.replace("-", "/").replace(" ", "/").split("/")
        if len(partes) >= 3:
            try:
                # Tu fórmula usa IZQUIERDA para el mes y EXTRAE para el día
                y = int(partes[2][-4:])
                m = int(partes[0])
                d = int(partes[1])
                dt_secop = datetime.date(y, m, d)
            except: pass
                
    if dt_sipse is not None and dt_secop is not None:
        return dt_sipse != dt_secop
        
    return s1 != s2 # Fallback a comparación de texto

def es_error_duracion(val_sipse, val_secop):
    """Replica la lógica de Duración."""
    s1 = str(val_sipse).strip().upper() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip().upper() if pd.notna(val_secop) and val_secop is not None else ""
    
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    return s1 != s2

def color_memo(err_tuple):
    """Devuelve los colores agrupados tipo pastel idénticos a los de tu imagen."""
    if err_tuple[0]: return "E4DFEC" # Beige/Grisáceo claro (Suscripción)
    if err_tuple[1]: return "B4C6E7" # Azul claro (Inicio)
    if err_tuple[2]: return "DDEBF7" # Gris claro (Fin)
    if err_tuple[3]: return "F4CCCC" # Rosa claro (Plazo)
    return "FFFFFF"

def normalizar_alfanumerico(valor):
    """Limpia códigos (como los de CDP) para evitar que pandas los lea como '1234.0'"""
    if pd.isna(valor) or valor is None:
        return ""
    texto = str(valor).strip().upper()
    if texto.endswith(".0"):
        texto = texto[:-2]
    return texto

def limpiar_dato(val):
    """Limpia el dato y elimina el '.0' que Pandas añade por defecto a los números."""
    if pd.isna(val) or val is None:
        return ""
    if isinstance(val, (datetime.datetime, datetime.date)):
        return val.strftime('%d/%m/%Y')
    
    val_str = str(val).strip()
    if val_str.endswith(".0"):
        val_str = val_str[:-2]
    return val_str

def normalizar_cedula(valor):
    """Limpia la cédula quitando letras y asegurando que sea un número exacto."""
    val_str = limpiar_dato(valor)
    return re.sub(r'\D', '', val_str)

def obtener_valor_seguro(row, indice):
    """Evita errores si el archivo de Excel viene con columnas faltantes."""
    try:
        return limpiar_dato(row.iloc[indice])
    except IndexError:
        return ""

def mapear_columnas(columnas_df, columnas_buscadas, umbral=80):
    """
    Busca el índice (0-based) de la columna que más se parezca al nombre deseado.
    columnas_df: lista de columnas del dataframe (ej. df.columns.tolist())
    columnas_buscadas: diccionario { 'clave_interna': 'Nombre Deseado Excel' }
    Retorna: diccionario { 'clave_interna': indice_numerico }
    """
    mapa_indices = {}
    for clave, nombre_ideal in columnas_buscadas.items():
        # extractOne retorna una tupla: (mejor_coincidencia, puntaje, indice_original)
        resultado = process.extractOne(nombre_ideal, columnas_df, scorer=fuzz.token_sort_ratio)
        
        if resultado and resultado[1] >= umbral:
            mapa_indices[clave] = resultado[2]
        else:
            print(f"⚠️ Advertencia: No se encontró la columna para '{nombre_ideal}'.")
            # Podrías lanzar una excepción aquí si la columna es estrictamente obligatoria
            mapa_indices[clave] = None 
            
    return mapa_indices

def cargar_datos_bogdata():
    """Abre el explorador y procesa RP (por cédula) y CDP (por Nro + Cod Interno)."""
    root = tk.Tk()
    root.withdraw() 
    root.attributes('-topmost', True) 
    
    print("\n--- SELECCIÓN BOGDATA ---")
    print("1. Por favor selecciona el archivo de CDP (Excel)...")
    ruta_cdp = filedialog.askopenfilename(title="Seleccionar archivo CDP", filetypes=[("Excel files", "*.xlsx *.xls")])
    
    print("2. Por favor selecciona el archivo de RP (Excel)...")
    ruta_rp = filedialog.askopenfilename(title="Seleccionar archivo RP", filetypes=[("Excel files", "*.xlsx *.xls")])
         
    dict_cdp = {}
    dict_rp = defaultdict(list) # <--- Ahora es un diccionario que guarda listas

    # --- PROCESAR RP ---
    if ruta_rp:
        print(f"  -> Procesando archivo RP...")

        try:
            # 1. Leer los nombres de todas las hojas sin cargar los datos aún
            excel = pd.ExcelFile(ruta_rp)
            hojas_disponibles = excel.sheet_names

            # 2. Definir las palabras clave que buscas (siempre en minúsculas)
            palabras_clave = ["data", "crp"]

            # 3. Buscar la hoja correcta ignorando mayúsculas, minúsculas y espacios
            hoja_correcta = next(
                (
                    hoja
                    for hoja in hojas_disponibles
                    if hoja.strip().lower() in palabras_clave
                ),
                None,
            )

            # 4. Validar si se encontró la hoja y cargar el DataFrame
            if hoja_correcta:
                df_rp = pd.read_excel(excel, sheet_name=hoja_correcta)
                print(f"     Hoja detectada y cargada: '{hoja_correcta}'")
            else:
                print(
                    f"  [Error] No se encontró ninguna hoja 'Data' o 'CRP' en el archivo."
                )
                df_rp = None

        except Exception as e:
            print(f"  [Error] No se pudo leer el archivo Excel: {e}")
            df_rp = None

        nombres_ideales_rp = {
            'fecha': 'Fecha Registro',           # Lo que antes era F=5
            'num_cdp': 'Número de CDP',    # Lo que antes era O=14
            'num_rp': 'Número de CRP',      # Lo que antes era P=15
            'filtro1': 'Rubro',     # Lo que antes era R=17 (Pon el nombre real de la cabecera)
            'localidad': 'Texto Id Proyecto',   # Lo que antes era Z=25
            'cedula': 'Número Doc. BP Beneficiario',         # Lo que antes era AE=30
            'filtro2': 'Anulaciones',    # Lo que antes era AL=37 (Pon el nombre real de la cabecera)
            'val_rp': 'Valor Neto',          # Lo que antes era AN=39
            'cod_int': 'N° Interno CDP' # Lo que antes era AS=44
        }

        idx_rp = mapear_columnas(df_rp.columns.tolist(), nombres_ideales_rp)

        # Índices: F=5(Fec), O=14(NumCDP), P=15(NumRP), R=17(Filtro1), Z=25(Localidad), AE=30(Ced), AL=37(Filtro2), AN=39(ValRP), AS=44(CodIntCDP)
        for _, row in df_rp.iterrows():
            val_r = obtener_valor_seguro(row, idx_rp['filtro1'])
            digitos_r = re.sub(r'\D', '', val_r)
            
            # 1. Filtro 1: Columna R >= 7 dígitos
            if len(digitos_r) >= 7:
                
                # 2. Filtro 2: Columna AL NO debe ser mayor a 0
                val_al = obtener_valor_seguro(row, idx_rp['filtro2']) # Columna AL
                try:
                    # Convertimos a número (quitando comas si las hay)
                    num_al = float(str(val_al).replace(',', ''))
                except ValueError:
                    # Si la celda está vacía o tiene letras, asumimos 0 para que no falle
                    num_al = 0.0 
                
                # Solo continuamos si AL es menor o igual a 0
                if num_al == 0:
                    cedula_raw = obtener_valor_seguro(row, idx_rp['cedula'])
                    llave_cedula = normalizar_cedula(cedula_raw)
                    
                    # Guardamos TODOS los RP válidos que encuentre de esa persona
                    if llave_cedula:
                        rp_num = obtener_valor_seguro(row, idx_rp['num_rp']) # P
                        rp_val = obtener_valor_seguro(row, idx_rp['val_rp']) # AN
                        rp_fec = obtener_valor_seguro(row, idx_rp['fecha'])  # F
                        cdp_num = obtener_valor_seguro(row, idx_rp['num_cdp']) # O
                        cdp_cod = obtener_valor_seguro(row, idx_rp['cod_int']) # AS
                        
                        # Capturamos la localidad (Columna Z)
                        localidad_col_z = obtener_valor_seguro(row, idx_rp['localidad']) # Z
                        
                        dict_rp[llave_cedula].append((rp_num, rp_val, rp_fec, cdp_num, cdp_cod, localidad_col_z))

    # --- PROCESAR CDP ---
    if ruta_cdp:
        print(f"  -> Procesando archivo CDP...")

        try:
            # 1. Leer los nombres de todas las hojas sin cargar los datos aún
            excel = pd.ExcelFile(ruta_cdp)
            hojas_disponibles = excel.sheet_names

            # 2. Definir las palabras clave que buscas (siempre en minúsculas)
            palabras_clave = ["data", "cdp"]

            # 3. Buscar la hoja correcta ignorando mayúsculas, minúsculas y espacios
            hoja_correcta = next(
                (
                    hoja
                    for hoja in hojas_disponibles
                    if hoja.strip().lower() in palabras_clave
                ),
                None,
            )

            # 4. Validar si se encontró la hoja y cargar el DataFrame
            if hoja_correcta:
                df_cdp = pd.read_excel(excel, sheet_name=hoja_correcta)
                print(f"     Hoja detectada y cargada: '{hoja_correcta}'")
            else:
                print(
                    f"  [Error] No se encontró ninguna hoja 'Data' o 'CDP' en el archivo."
                )
                df_cdp = None

        except Exception as e:
            print(f"  [Error] No se pudo leer el archivo Excel: {e}")
            df_cdp = None

        nombres_ideales_cdp = {
            'num_cdp': 'No. CDP',       # Lo que antes era H=7
            'val_cdp': 'Valor',        # Lo que antes era Q=16
            'cod_int': 'No.Interno CDP',   # Lo que antes era U=20
            'fec_cdp': 'Fecha Registro'         # Lo que antes era AC=28
        }
        idx_cdp = mapear_columnas(df_cdp.columns.tolist(), nombres_ideales_cdp)
        
        # Índices: H=7(NumCDP), Q=16(ValCDP), U=20(CodIntCDP), AC=28(FecCDP)
        for _, row in df_cdp.iterrows():
            cdp_num = obtener_valor_seguro(row, idx_cdp['num_cdp'])  # H
            cdp_cod = obtener_valor_seguro(row, idx_cdp['cod_int']) # U
            
            # La llave maestra del CDP es la combinación de Número (H) + Código interno (U)
            llave_cdp = (cdp_num, cdp_cod)
            
            if llave_cdp != ("", "") and llave_cdp not in dict_cdp:
                cdp_val = obtener_valor_seguro(row, idx_cdp['val_cdp']) # Q
                cdp_fec = obtener_valor_seguro(row, idx_cdp['fec_cdp']) # AC
                
                dict_cdp[llave_cdp] = (cdp_num, cdp_val, cdp_fec)

    return dict_cdp, dict_rp

def normalizar_texto_localidad(texto):
    """Elimina tildes, pasa a mayúsculas y estandariza espacios."""
    if pd.isna(texto) or texto is None:
        return ""
    
    # split() y join() elimina espacios dobles, saltos de línea y espacios al inicio/final
    t = " ".join(str(texto).upper().split())
    
    # Quitar tildes (ej: NARIÑO -> NARINO, BOGOTÁ -> BOGOTA)
    t = ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')
    return t

# --- LÓGICA PRINCIPAL ---
global dict_cdp
global dict_rp
global metricas
def procesar_archivos():
    global metricas
    if not os.path.exists(CARPETA_DESTINO):
        os.makedirs(CARPETA_DESTINO)

    #Cargar Bogdata en memoria antes del ciclo principal
    global dict_cdp
    global dict_rp
    dict_cdp, dict_rp = cargar_datos_bogdata()

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('ANTONIO NARINO.xlsx', 'ANTONIO NARINO.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('SANTAFE.xlsx', 'SANTAFE.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('CANDELARIA.xlsx', 'CANDELARIA.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('RAFAEL URIBE.xlsx', 'RAFAEL URIBE.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('.xlsx', '.xls'))]
    print(f"Se encontraron {len(archivos)} archivos de Excel.")

    for archivo in archivos:
        ruta_orig = os.path.join(CARPETA_ORIGEN, archivo)
        
        # Definimos la ruta de destino siempre con extensión .xlsx
        nombre_base = os.path.splitext(archivo)[0]
        ruta_dest = os.path.join(CARPETA_DESTINO, nombre_base + ".xlsx")
        
        print(f"Procesando: {archivo}...")

        # --- PASO DE CONVERSIÓN SI ES .XLS ---
        if archivo.lower().endswith('.xls'):
            try:
                # Leemos el .xls y lo guardamos inmediatamente como .xlsx
                # sheet_name=None lee todas las pestañas
                conversion = pd.read_excel(ruta_orig, sheet_name=None, engine='xlrd')
                with pd.ExcelWriter(ruta_dest, engine='openpyxl') as writer:
                    for nombre_hoja, datos in conversion.items():
                        datos.to_excel(writer, sheet_name=nombre_hoja, index=False)
                print(f"  -> Convertido de .xls a .xlsx con éxito.")
            except Exception as e:
                print(f"  Error convirtiendo {archivo}: {e}")
                continue
        else:
            # Si ya es .xlsx, solo lo copiamos como hacías antes
            shutil.copy2(ruta_orig, ruta_dest)
        # -------------------------------------

        try:
            # Ahora openpyxl siempre abrirá un archivo .xlsx (el copiado o el convertido)
            wb = openpyxl.load_workbook(ruta_dest)
        except Exception as e:
            print(f"   Error al abrir el archivo con openpyxl: {e}")
            continue

        # ==========================================
        # PARTE 1: Procesar SECOP II (Crear aux1)
        # ==========================================
        nombre_real_secop = next(n for n in wb.sheetnames if n.lower() == 'secop ii')
        ws_origen = wb[nombre_real_secop]
        
        if "SECOP II aux1" in wb.sheetnames:
            del wb["SECOP II aux1"]
            
        ws_secop_aux = wb.copy_worksheet(ws_origen)
        ws_secop_aux.title = "SECOP II aux1"

        filas = []
        for row in ws_secop_aux.iter_rows(min_row=2, values_only=True):
            filas.append(list(row))
        
        filas_ordenadas = sorted(filas, key=lambda x: normalizar_numero_contrato(x[11] if len(x) > 11 else 0))

        for row in ws_secop_aux.iter_rows(min_row=2):
            for cell in row:
                cell.value = None

        for i_fila, datos_fila in enumerate(filas_ordenadas, start=2):
            for i_col, valor in enumerate(datos_fila, start=1):
                ws_secop_aux.cell(row=i_fila, column=i_col, value=valor)

        # Para crear columnas "Columna1" a "Columna8" en la hoja "SECOP II aux1"
        max_col_secop = ws_secop_aux.max_column
        ws_secop_aux.cell(row=1, column=max_col_secop + 1, value="Columna1")
        ws_secop_aux.cell(row=1, column=max_col_secop + 2, value="Columna2")
        col_L = get_column_letter(12)
        
        for i in range(2, ws_secop_aux.max_row + 1):
            # Fórmula Columna 1
            f1 = (
                f'=IFERROR(MID({col_L}{i}, '
                f'MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), '
                f'MIN(IFERROR(FIND(" ", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999), '
                f'IFERROR(FIND("-", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999)) '
                f'- MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), "")'
            )
            ws_secop_aux.cell(row=i, column=max_col_secop + 1, value=f1)

            # Fórmula Columna 2 (Extrae número entre paréntesis)
            f2 = f'=IFERROR(MID({col_L}{i}, FIND("(", {col_L}{i}) + 1, FIND(")", {col_L}{i}) - FIND("(", {col_L}{i}) - 1), "")'
            ws_secop_aux.cell(row=i, column=max_col_secop + 2, value=f2)

        # ESTILOS SECOP
        print("  -> Aplicando estilos y ocultando columnas en SECOP II aux1...")
        aplicar_estilo_ferrocarril(ws_secop_aux)
        ocultar_columnas_por_lista(ws_secop_aux, RANGOS_OCULTAR_SECOP)

        # ==========================================
        # PARTE 2 y 3: ALINEACIÓN EN 'ANALISIS'
        # ==========================================
        print("  -> Generando hoja 'Analisis'...")

        nombre_real_sipse = next(n for n in wb.sheetnames if n.lower() == 'sipse')
        ws_sipse = wb[nombre_real_sipse]
        
        if 'Analisis' in wb.sheetnames:
            del wb['Analisis']
        ws_analisis = wb.create_sheet("Analisis")
        
        dict_sipse = defaultdict(list)
        dict_secop = defaultdict(list)
        
        filas_sipse = list(ws_sipse.values)
        header_sipse = filas_sipse[0] if filas_sipse else []
        for fila in filas_sipse[1:]:
            contrato = normalizar_numero_contrato(fila[3] if len(fila) > 3 else 0)
            dict_sipse[contrato].append(fila)

        filas_secop = list(ws_secop_aux.values)
        header_secop = filas_secop[0] if filas_secop else []
        for fila in filas_secop[1:]:
            contrato = normalizar_numero_contrato(fila[11] if len(fila) > 11 else 0)
            dict_secop[contrato].append(fila)

        ws_analisis.append(header_sipse) 
        col_inicio_secop = column_index_from_string("BM")
        for i, val in enumerate(header_secop):
            ws_analisis.cell(row=1, column=col_inicio_secop + i, value=val)

        todos_contratos = sorted(list(set(dict_sipse.keys()) | set(dict_secop.keys())))
        current_row = 2 
        
        for contrato in todos_contratos:
            filas_de_sipse = dict_sipse.get(contrato, [])
            filas_de_secop = dict_secop.get(contrato, [])
            max_filas = max(len(filas_de_sipse), len(filas_de_secop))
            
            for i in range(max_filas):
                if i < len(filas_de_sipse):
                    datos = filas_de_sipse[i]
                    for c_idx, valor in enumerate(datos, start=1):
                        ws_analisis.cell(row=current_row, column=c_idx, value=valor)
                
                if i < len(filas_de_secop):
                    datos = filas_de_secop[i]
                    for c_idx, valor in enumerate(datos):
                        col_excel = col_inicio_secop + c_idx
                        if c_idx == len(datos) - 2 and str(valor).startswith('='):
                             # Pega la fórmula de Columna1 (Ahora es la penúltima)
                             nueva_formula1 = (
                                f'=IFERROR(MID(BX{current_row}, '
                                f'MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), '
                                f'MIN(IFERROR(FIND(" ", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999), '
                                f'IFERROR(FIND("-", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999)) '
                                f'- MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), "")'
                            )
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula1)
                        elif c_idx == len(datos) - 1 and str(valor).startswith('='):
                             # Pega la fórmula de Columna2 (Extrae entre paréntesis)
                             nueva_formula2 = f'=IFERROR(MID(BX{current_row}, FIND("(", BX{current_row}) + 1, FIND(")", BX{current_row}) - FIND("(", BX{current_row}) - 1), "")'
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula2)
                        else:
                            ws_analisis.cell(row=current_row, column=col_excel, value=valor)
                current_row += 1

        # ==========================================
        # PARTE 4: Columnas de Evaluación
        # ==========================================
        print("  -> Agregando columnas de evaluación y estilos...")
        
        ultima_col_actual = ws_analisis.max_column
        col_inicio_eval = ultima_col_actual + 1
        ultima_fila_analisis = ws_analisis.max_row
        
        headers_nuevos = [
            "Columna3\n# RP\n(SIPSE)", "Columna4\nValor RP\n(SIPSE)", "Columna5\nFecha RP\n(SIPSE)", "Columna6\n# CDP\n(SIPSE)", "Columna7\nValor CDP\n(SIPSE)", "Columna8\nFecha CDP\n(SIPSE)", "Col.9\n# RP\n(BOGD)", "Col.10\nValor RP\n(BOGD)", "Col.11\nFecha RP\n(BOGD)", "Col.12\n# CDP\n(BOGD)", "Col.13\nValor CDP\n(BOGD)", "Col.14\nFecha CDP\n(BOGD)",
            "Ev. Nro contrato", "Aux Nro contrato", "Ev. Cédula", "Repeticiones contrato SIPSE", "Repeticiones contrato SECOP", "Fecha suscripción", "Fecha inicio", "Fecha fin", "Duración contratos", "Valor contratos", "Ev. # RP", "Ev. Valor RP", "Ev. Fecha RP", "Ev. # CDP", "Ev. Valor CDP", "Ev. Fecha CDP"
        ]

        #ESCRIBIR LOS TITULOS EN LOS HEADERS
        for idx, titulo in enumerate(headers_nuevos):
            ws_analisis.cell(row=1, column=col_inicio_eval + idx, value=titulo)
        
        #AÑADIR COLUMNAS DESGLOSADAS CDP y RP (SIPSE)
        for i in range(2, ultima_fila_analisis + 1):
            # 🔹 COLUMNA 3 - RP NUMERO (SIPSE)
            f3 = f'=IFERROR(TRIM(RIGHT(SUBSTITUTE(TRIM(LEFT(AI{i}, FIND("Fecha", AI{i})-1)), " ", REPT(" ", 100)), 100)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval, value=f3)
            # 🔹 COLUMNA 4 - RP VALOR (SIPSE)
            f4 = f'=IFERROR(TRIM(MID(AI{i}, FIND("Valor ", AI{i}) + 6, IFERROR(FIND(";", AI{i}), LEN(AI{i})+1) - FIND("Valor ", AI{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 1, value=f4)
            # 🔹 COLUMNA 5 - RP FECHA (SIPSE)
            f5 = f'=IFERROR(TRIM(MID(AI{i}, FIND("Fecha ", AI{i}) + 6, FIND(" Valor", AI{i}) - FIND("Fecha ", AI{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 2, value=f5)
            # 🔹 COLUMNA 6 - CDP NUMERO (SIPSE)
            f6 = f'=IFERROR(TRIM(RIGHT(SUBSTITUTE(TRIM(LEFT(AH{i}, FIND("Fecha", AH{i})-1)), " ", REPT(" ", 100)), 100)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 3, value=f6)
            # 🔹 COLUMNA 7 - CDP VALOR (SIPSE)
            f7 = f'=IFERROR(TRIM(MID(AH{i}, FIND("Valor ", AH{i}) + 6, IFERROR(FIND(";", AH{i}), LEN(AH{i})+1) - FIND("Valor ", AH{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 4, value=f7)
            # 🔹 COLUMNA 8 - CDP FECHA (SIPSE)
            f8 = f'=IFERROR(TRIM(MID(AH{i}, FIND("Fecha ", AH{i}) + 6, FIND(" Valor", AH{i}) - FIND("Fecha ", AH{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 5, value=f8)


        # --- AÑADIR COLUMNAS DESGLOSADAS CDP y RP (BOGDATA) ---
        idx_col_cedula_cj = column_index_from_string("CJ")
        letra_ev_nro_contrato = get_column_letter(col_inicio_eval + 12) # Esto equivale a la columna 'FJ'
        letra_ev_nro_cedula = get_column_letter(col_inicio_eval + 14) # Esto equivale a la columna 'FL'
        
        # Limpiamos el nombre del archivo actual (Ej: "SANTA FE" -> "SANTAFE")
        localidad_actual_norm = normalizar_texto_localidad(nombre_base)
        
        for i in range(2, ultima_fila_analisis + 1):
            # 1. Leer cédula de la hoja en construcción (Columna CJ)
            cedula_actual_raw = ws_analisis.cell(row=i, column=idx_col_cedula_cj).value
            cedula_busqueda = normalizar_cedula(cedula_actual_raw)
            
            # 2. Buscar en RP (Trae una lista con todos los RPs de la persona)
            lista_rps = dict_rp.get(cedula_busqueda, [])
            rp_data_elegido = ("", "", "", "", "")
            
            if len(lista_rps) == 1:
                # Si solo tiene un contrato, lo elegimos sin pensarlo
                rp_data_elegido = lista_rps[0][:5]
            elif len(lista_rps) > 1:
                # Si tiene varios, buscamos el que coincida con la localidad
                for rp in lista_rps:
                    localidad_rp_norm = normalizar_texto_localidad(rp[5]) # El 6to dato es Columna Z
                    # Si "SANTAFE" está dentro de "PMRFDLDESANTAFE", hace match
                    if fuzz.partial_ratio(localidad_actual_norm.lower(), localidad_rp_norm.lower()) > 85:# Ese umbral es un hiperparámetro
                        rp_data_elegido = rp[:5]
                        break

            rp_num, rp_val, rp_fec, cdp_num_from_rp, cdp_cod_from_rp = rp_data_elegido
            
            # 3. Buscar en CDP (Usando las 2 llaves que trajo el RP: Num y Cod Interno)
            cdp_data = dict_cdp.get((cdp_num_from_rp, cdp_cod_from_rp), ("", "", ""))
            cdp_num_out, cdp_val_out, cdp_fec_out = cdp_data

            # 4. Enviar a Excel usando fórmula =SI() vinculada a la coincidencia de contrato (FJ)
            def inyectar_condicional(valor):
                val_str = str(valor).replace('"', '""') if valor else ""
                # Si dice 'Numero contrato coincide', 'Contrato coincide sin SIPSE', o 'Cedula coincide', muestra el dato; de lo contrario, vacía.
                #return f'=IF(OR({letra_ev_nro_contrato}{i}="Numero contrato coincide", {letra_ev_nro_contrato}{i}="Contrato coincide sin SIPSE"), "{val_str}", "")'                
                return f'=IF(OR({letra_ev_nro_contrato}{i}="Numero contrato coincide", {letra_ev_nro_contrato}{i}="Contrato coincide sin SIPSE", {letra_ev_nro_cedula}{i}="Cedula coincide"), "{val_str}", "")'

            # 🔹 COLUMNA 9 - RP # (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 6, value=inyectar_condicional(rp_num))
            # 🔹 COLUMNA 10 - RP VALOR (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 7, value=inyectar_condicional(rp_val))
            # 🔹 COLUMNA 11 - RP FECHA (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 8, value=inyectar_condicional(rp_fec))
            
            # 🔹 COLUMNA 12 - CDP # (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 9, value=inyectar_condicional(cdp_num_out))
            # 🔹 COLUMNA 13 - CDP VALOR (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 10, value=inyectar_condicional(cdp_val_out))
            # 🔹 COLUMNA 14 - CDP FECHA (BOGDATA)
            ws_analisis.cell(row=i, column=col_inicio_eval + 11, value=inyectar_condicional(cdp_fec_out))
        # -------------------------------------------------------------

        for i in range(2, ultima_fila_analisis + 1):
            COL_SOLICITUD = "B"
            # 🔹 EV. NO CONTRATO
            # Se añade SUBSTITUTE(UPPER(...), "SIPSE", "") para omitir la palabra si la escriben por error.
            f15 = f'''=IF(COUNTIF($FK$2:FK{i},FK{i})>1,"",
                IF(AND(D{i}<>"",EV{i}<>""),
                    IF(EW{i}="","Contrato coincide sin SIPSE",
                        IF(TRIM(SUBSTITUTE(UPPER({COL_SOLICITUD}{i}),"SIPSE",""))=TRIM(SUBSTITUTE(UPPER(EW{i}),"SIPSE","")),"Numero contrato coincide","Numero contrato NO coincide")
                    ),
                    IF(D{i}<>"","Contrato no existe en SECOP",
                        IF(EV{i}<>"","Contrato no existe en SIPSE","No existe")
                    )
                )
            )'''
            # Eliminamos los saltos de línea para que Excel lo lea como una sola fórmula (opcional, pero más seguro en Python)
            f15 = f15.replace('\n', '').replace('    ', '')
            ws_analisis.cell(row=i, column=col_inicio_eval + 12, value=f15)

            # 🔹 AUX NRO CONTRATO (EX) → llave única
            f16 = f'=IF(D{i}<>"",D{i},EV{i})'
            ws_analisis.cell(row=i, column=col_inicio_eval + 13, value=f16)

            # 🔹 CÉDULA
            f17 = f'=IF(AND(BY{i}="",M{i}=""),"",IF(OR(BY{i}="",M{i}=""),"",IF(BY{i}="-","Sin cedula en SECOP",IF(M{i}="-","Sin cedula en SIPSE",IF(VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(CJ{i}),"CC",""),"NIT",""),"TI",""))=VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(M{i}),"CC",""),"NIT",""),"TI","")),"Cedula coincide","Cedula NO coincide")))))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 14, value=f17)

            # 🔹 REPETICIONES SIPSE
            f18 = f'=IF(FJ{i}="","",COUNTIF(D$2:D${ultima_fila_analisis},D{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 15, value=f18)

            # 🔹 REPETICIONES SECOP
            f19 = f'=IF(FJ{i}="","",COUNTIF(EV$2:EV${ultima_fila_analisis},EV{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 16, value=f19)

            # 🔹 FECHA SUSCRIPCION
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CE{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CE{i}), DAY(CE{i}), MONTH(CE{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f20 = f'''=IF(AND(H{i}="",CE{i}=""),"",IF(AND(H{i}="-",CE{i}="-"),"Ambos sin fechas",IF(H{i}="-","sin fecha SIPSE",IF(CE{i}="-","sin fecha SECOP",IF(OR(H{i}="",CE{i}=""),"",IF(INT(VALUE(SUBSTITUTE(H{i},"-","/"))) =INT(IF(ISNUMBER(CE{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 17, value=f20)

            # 🔹 FECHA ACTA DE INICIO
            cf_clean = f'SUBSTITUTE(SUBSTITUTE(CF{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CF{i}), DAY(CF{i}), MONTH(CF{i}))'
            parte_es_texto = f'DATE(RIGHT({cf_clean}, 4), LEFT({cf_clean}, FIND("/", {cf_clean}) - 1), MID({cf_clean}, FIND("/", {cf_clean}) + 1, FIND("/", {cf_clean}, FIND("/", {cf_clean}) + 1) - FIND("/", {cf_clean}) - 1))'

            f21 = f'''=IF(AND(I{i}="",CF{i}=""),"",IF(AND(I{i}="-",CF{i}="-"),"Ambos sin fechas",IF(I{i}="-","sin fecha SIPSE",IF(CF{i}="-","sin fecha SECOP",IF(OR(I{i}="",CF{i}=""),"",IF(INT(VALUE(SUBSTITUTE(I{i},"-","/"))) =INT(IF(ISNUMBER(CF{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 18, value=f21)

            # 🔹 FECHA FIN
            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CG{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CG{i}), DAY(CG{i}), MONTH(CG{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f22 = f'''=IF(AND(J{i}="",CG{i}=""),"",IF(AND(J{i}="-",CG{i}="-"),"Ambos sin fechas",IF(J{i}="-","sin fecha SIPSE",IF(CG{i}="-","sin fecha SECOP",IF(OR(J{i}="",CG{i}=""),"",IF(INT(VALUE(SUBSTITUTE(J{i},"-","/"))) =INT(IF(ISNUMBER(CG{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 19, value=f22)

            # 🔹 DURACIÓN
            f23 = f'''=IF(AND(R{i}="",EE{i}=""),"",IF(AND(R{i}="-",EE{i}="-"),"Ambos sin duracion",IF(R{i}="-","sin duracion en SECOP",IF(EE{i}="-","sin duracion en SIPSE",IF(OR(R{i}="",EE{i}=""),"",IF(TRIM(UPPER(R{i}))=TRIM(UPPER(EE{i})),"Duracion coincide","Duracion NO coincide"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 20, value=f23)

            # 🔹 VALOR CONTRATO (Comparación Robusta)
            f24 = f'''=IF(AND(P{i}="",CU{i}=""),"",
                IF(AND(P{i}="-",CU{i}="-"),"Ambos sin valor",
                IF(P{i}="-","Sin valor en SECOP",
                IF(CU{i}="-","Sin valor en SIPSE",
                IF(OR(P{i}="",CU{i}=""),"",
                IF(P{i}=CU{i},"Valor coincide","Valor NO coincide"))))))'''

            ws_analisis.cell(row=i, column=col_inicio_eval + 21, value=f24)

            c_rp_sip   = get_column_letter(col_inicio_eval)       # EX (# RP SIPSE)
            c_val_rp_s = get_column_letter(col_inicio_eval + 1)   # EY (Valor RP SIPSE)
            c_fec_rp_s = get_column_letter(col_inicio_eval + 2)   # EZ (Fecha RP SIPSE)
            c_cdp_sip  = get_column_letter(col_inicio_eval + 3)   # FA (# CDP SIPSE)
            c_val_cd_s = get_column_letter(col_inicio_eval + 4)   # FB (Valor CDP SIPSE)
            c_fec_cd_s = get_column_letter(col_inicio_eval + 5)   # FC (Fecha CDP SIPSE)
            
            c_rp_bog   = get_column_letter(col_inicio_eval + 6)   # FD (# RP BOGDATA)
            c_val_rp_b = get_column_letter(col_inicio_eval + 7)   # FE (Valor RP BOGDATA)
            c_fec_rp_b = get_column_letter(col_inicio_eval + 8)   # FF (Fecha RP BOGDATA)
            c_cdp_bog  = get_column_letter(col_inicio_eval + 9)   # FG (# CDP BOGDATA)
            c_val_cd_b = get_column_letter(col_inicio_eval + 10)  # FH (Valor CDP BOGDATA)
            c_fec_cd_b = get_column_letter(col_inicio_eval + 11)  # FI (Fecha CDP BOGDATA)

            # 🔹 Ev. # RP
            f25 = (f'=IF(AND(TRIM({c_rp_sip}{i})="", TRIM({c_rp_bog}{i})=""), "Sin numeros RP", '
                   f'IF(TRIM({c_rp_sip}{i})="", "Sin numero RP en SIPSE", '
                   f'IF(TRIM({c_rp_bog}{i})="", "Sin numero RP en BOGDATA", '
                   f'IF(TRIM({c_rp_sip}{i})=TRIM({c_rp_bog}{i}), "El numero RP coincide", "El numero RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 22, value=f25)

            # 🔹 Ev. Valor RP (Limpiando comas para compatibilidad Web)
            f26 = (f'=IF(AND(TRIM({c_val_rp_s}{i})="", TRIM({c_val_rp_b}{i})=""), "Sin valores RP", '
                   f'IF(TRIM({c_val_rp_s}{i})="", "Sin valor RP en SIPSE", '
                   f'IF(TRIM({c_val_rp_b}{i})="", "Sin valor RP en BOGDATA", '
                   f'IF(IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_rp_s}{i}), ",", "")),TRIM({c_val_rp_s}{i}))=IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_rp_b}{i}), ",", "")),TRIM({c_val_rp_b}{i})), "El valor RP coincide", "El valor RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 23, value=f26)

            # 🔹 Ev. Fecha RP
            f27 = (f'=IF(AND(TRIM({c_fec_rp_s}{i})="", TRIM({c_fec_rp_b}{i})=""), "Sin fechas RP", '
                   f'IF(TRIM({c_fec_rp_s}{i})="", "Sin fecha RP en SIPSE", '
                   f'IF(TRIM({c_fec_rp_b}{i})="", "Sin fecha RP en BOGDATA", '
                   f'IF(TRIM({c_fec_rp_s}{i})=TRIM({c_fec_rp_b}{i}), "La fecha RP coincide", "La fecha RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 24, value=f27)

            # 🔹 Ev. # CDP
            f28 = (f'=IF(AND(TRIM({c_cdp_sip}{i})="", TRIM({c_cdp_bog}{i})=""), "Sin numeros CDP", '
                   f'IF(TRIM({c_cdp_sip}{i})="", "Sin numero CDP en SIPSE", '
                   f'IF(TRIM({c_cdp_bog}{i})="", "Sin numero CDP en BOGDATA", '
                   f'IF(TRIM({c_cdp_sip}{i})=TRIM({c_cdp_bog}{i}), "El numero CDP coincide", "El numero CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 25, value=f28)

            # 🔹 Ev. Valor CDP (Limpiando comas para compatibilidad Web)
            f29 = (f'=IF(AND(TRIM({c_val_cd_s}{i})="", TRIM({c_val_cd_b}{i})=""), "Sin valores CDP", '
                   f'IF(TRIM({c_val_cd_s}{i})="", "Sin valor CDP en SIPSE", '
                   f'IF(TRIM({c_val_cd_b}{i})="", "Sin valor CDP en BOGDATA", '
                   f'IF(IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_cd_s}{i}), ",", "")),TRIM({c_val_cd_s}{i}))=IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_cd_b}{i}), ",", "")),TRIM({c_val_cd_b}{i})), "El valor CDP coincide", "El valor CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 26, value=f29)

            # 🔹 Ev. Fecha CDP
            f30 = (f'=IF(AND(TRIM({c_fec_cd_s}{i})="", TRIM({c_fec_cd_b}{i})=""), "Sin fechas CDP", '
                   f'IF(TRIM({c_fec_cd_s}{i})="", "Sin fecha CDP en SIPSE", '
                   f'IF(TRIM({c_fec_cd_b}{i})="", "Sin fecha CDP en BOGDATA", '
                   f'IF(TRIM({c_fec_cd_s}{i})=TRIM({c_fec_cd_b}{i}), "La fecha CDP coincide", "La fecha CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 27, value=f30)

        # --- ESTILOS ANALISIS ---
        print("  -> Aplicando estilos y ocultando columnas en Analisis...")
        aplicar_estilo_ferrocarril(ws_analisis)
        aplicar_formato_condicional_alertas(ws_analisis, col_inicio_eval)
        
        # Ocultar columnas hoja analisis
        ocultar_columnas_por_lista(ws_analisis, RANGOS_OCULTAR_ANALISIS)

        # ==========================================
        # PARTE 5: Crear HOJA RESULTADOS
        # ==========================================
        print("  -> Creando hoja 'Resultados'...")
        if 'Resultados' in wb.sheetnames:
            del wb['Resultados']
        ws_res = wb.create_sheet("Resultados")
        nombre_limpio = os.path.splitext(archivo)[0]
        ws_res['B1'] = f"Métricas alcaldía local {nombre_limpio}"
        
        letra_ev_nro = get_column_letter(col_inicio_eval + 12)       
        letra_ev_ced = get_column_letter(col_inicio_eval + 14)   
        letra_rep_sipse = get_column_letter(col_inicio_eval + 15)
        letra_rep_secop = get_column_letter(col_inicio_eval + 16)
        letra_fecha_suscripcion = get_column_letter(col_inicio_eval + 17)
        letra_fecha_inicio = get_column_letter(col_inicio_eval + 18)
        letra_fecha_fin = get_column_letter(col_inicio_eval + 19)
        letra_duracion = get_column_letter(col_inicio_eval + 20)
        letra_valor = get_column_letter(col_inicio_eval + 21)

        letra_num_RP = get_column_letter(col_inicio_eval + 22)
        letra_val_RP = get_column_letter(col_inicio_eval + 23)
        letra_fec_RP = get_column_letter(col_inicio_eval + 24)
        letra_num_CDP = get_column_letter(col_inicio_eval + 25)
        letra_val_CDP = get_column_letter(col_inicio_eval + 26)
        letra_fec_CDP = get_column_letter(col_inicio_eval + 27)

        metricas = [
            #("Nro de contratos registrados en SIPSE",'=SUMPRODUCT((Analisis!A2:A10000<>"")/COUNTIF(Analisis!A2:A10000,Analisis!A2:A10000&""))'),
            #("Nro de contratos registrados SECOP",'=SUMPRODUCT((Analisis!EV2:EV10000<>"")/COUNTIF(Analisis!EV2:EV10000,Analisis!EV2:EV10000&""))'),
            ("Nro de contratos registrados SIPSE",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos registrados SECOP",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos registrados en SECOP y SIPSE con numero de contrato igual", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")"),
            ("Nro de contratos solo en SECOP y no en SIPSE", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos solo en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos diferentes en SIPSE-SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato NO coincide\")"),
            ("Nro de contratos Cédula diferente", f"=COUNTIF('Analisis'!{letra_ev_ced}:{letra_ev_ced}, \"Cedula NO coincide\")"),
            ("Nro de contratos dobles en SIPSE", f"=COUNTIF('Analisis'!{letra_rep_sipse}:{letra_rep_sipse}, 2)"),
            ("Nro de contratos dobles en SECOP", f"=COUNTIF('Analisis'!{letra_rep_secop}:{letra_rep_secop}, 2)"),
            ("Fecha de acta de suscriccpión Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"Coincide fecha\")"),
            ("Fecha de acta de suscriccpión no Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"No coinciden fechas\")"),
            ("Fecha de acta de inicio Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Coincide fecha\")"),
            ("Fecha de acta de inicio no coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"No coinciden fechas\")"),
            ("Fecha de Fin conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"Coincide fecha\")"),
            ("Fecha de Fin no conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"No coinciden fechas\")"),
            ("Contratos que coinciden en Duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion Coincide\")"),
            ("Contratos que tienen diferente duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion NO coincide\")"),
            ("Contratos que tienen Fecha de inicio en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Sin fecha SECOP\")"),
            ("Datos en la matriz de seguimiento", f"=C2+C3"),
            ("Valor contrato coinciden", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor coincide\")"),
            ("Valor contrato no coincidenn", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor NO coincide\")"),
            ("Numero RP no coinciden", f"=COUNTIF('Analisis'!{letra_num_RP}:{letra_num_RP}, \"El numero RP NO coincide\")"),
            ("Valor RP no coinciden", f"=COUNTIF('Analisis'!{letra_val_RP}:{letra_val_RP}, \"El valor RP NO coincide\")"),
            ("Fecha RP no coinciden", f"=COUNTIF('Analisis'!{letra_fec_RP}:{letra_fec_RP}, \"La fecha RP NO coincide\")"),
            ("Numero CDP no coinciden", f"=COUNTIF('Analisis'!{letra_num_CDP}:{letra_num_CDP}, \"El numero CDP NO coincide\")"),
            ("Valor CDP no coinciden", f"=COUNTIF('Analisis'!{letra_val_CDP}:{letra_val_CDP}, \"El valor CDP NO coincide\")"),
            ("Fecha CDP no coinciden", f"=COUNTIF('Analisis'!{letra_fec_CDP}:{letra_fec_CDP}, \"La fecha CDP NO coincide\")")
        ]


        for idx, (texto, formula) in enumerate(metricas, start=2):
            ws_res.cell(row=idx, column=1, value=idx-1)
            ws_res.cell(row=idx, column=2, value=texto)
            ws_res.cell(row=idx, column=3, value=formula)
        
        # Pintar B2 de amarillo
        fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
        ws_res['B1'].fill = fill_amarillo
        ws_res.column_dimensions['B'].width = 100
        
        ws_analisis.column_dimensions['FJ'].width = 29
        ws_analisis.column_dimensions['FK'].width = 10
        ws_analisis.column_dimensions['FL'].width = 20
        ws_analisis.column_dimensions['FM'].width = 15
        ws_analisis.column_dimensions['FN'].width = 15
        ws_analisis.column_dimensions['FO'].width = 15
        ws_analisis.column_dimensions['FP'].width = 20
        ws_analisis.column_dimensions['FQ'].width = 20
        ws_analisis.column_dimensions['FR'].width = 18
        ws_analisis.column_dimensions['FS'].width = 14
        ws_analisis.column_dimensions['FT'].width = 25
        ws_analisis.column_dimensions['FU'].width = 25
        ws_analisis.column_dimensions['FV'].width = 25
        ws_analisis.column_dimensions['FW'].width = 25
        ws_analisis.column_dimensions['FX'].width = 25
        ws_analisis.column_dimensions['FY'].width = 25

        dejar_columna_blanca(ws_analisis, 64)# columna 64 es la columna "BL"

        #Ajustar el texto en la fila 1 de las hojas aux y Analisis
        ajustar_texto_fila(ws_secop_aux, 1)
        ajustar_texto_fila(ws_analisis, 1)

        # ==========================================
        # PARTE 6: Crear HOJA 'Para MEMO'
        # ==========================================
        print("  -> Creando hoja 'Para MEMO'...")
        if 'Para MEMO' in wb.sheetnames:
            del wb['Para MEMO']
        ws_memo = wb.create_sheet("Para MEMO")
        
        datos_memo = []
        
        # 1. Escanear fila por fila extrayendo los datos directos
        for i in range(2, ultima_fila_analisis + 1):
            val_b = ws_analisis.cell(row=i, column=2).value # B: SIPSE (Solicitud)
            val_d = ws_analisis.cell(row=i, column=4).value # D: No. Contrato
            
            # Extraer celdas origen según tu matriz (SIPSE y SECOP)
            susc_sipse = ws_analisis.cell(row=i, column=8).value  # H
            susc_secop = ws_analisis.cell(row=i, column=83).value # CE
            ini_sipse  = ws_analisis.cell(row=i, column=9).value  # I
            ini_secop  = ws_analisis.cell(row=i, column=84).value # CF
            fin_sipse  = ws_analisis.cell(row=i, column=10).value # J
            fin_secop  = ws_analisis.cell(row=i, column=85).value # CG
            dur_sipse  = ws_analisis.cell(row=i, column=18).value # R
            dur_secop  = ws_analisis.cell(row=i, column=135).value# EE
            
            # Evaluarlas en Python
            err_susc = es_error_fecha(susc_sipse, susc_secop)
            err_ini  = es_error_fecha(ini_sipse, ini_secop)
            err_fin  = es_error_fecha(fin_sipse, fin_secop)
            err_dur  = es_error_duracion(dur_sipse, dur_secop)
            
            # Si tiene al menos 1 error, lo enlistamos
            if err_susc or err_ini or err_fin or err_dur:
                llave_orden = (err_susc, err_ini, err_fin, err_dur)
                datos_memo.append([
                    val_b, val_d,
                    "X" if err_susc else "",
                    "X" if err_ini else "",
                    "X" if err_fin else "",
                    "X" if err_dur else "",
                    llave_orden # Llave secreta para ordenar
                ])
                
        # 2. Ordenar datos para que queden agrupados por color como en tu imagen
        datos_memo.sort(key=lambda x: x[6], reverse=True)
        
        # 3. Diseño de la Cabecera
        relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
        fuente_blanca = Font(color="FFFFFF", bold=True)
        ws_memo.merge_cells("A1:A2")
        ws_memo['A1'] = "SIPSE"
        ws_memo.merge_cells("B1:B2")
        ws_memo['B1'] = "No. Contrato"
        ws_memo.merge_cells("C1:F1")
        ws_memo['C1'] = "TIPOLOGÍA DE LA INCONSISTENCIA"
        
        ws_memo['C2'] = "Fecha suscripción"
        ws_memo['D2'] = "Fecha Inicio"
        ws_memo['E2'] = "Fecha Fin"
        ws_memo['F2'] = "Plazo"
        ws_memo['A1'].fill = relleno_rojo
        ws_memo['B1'].fill = relleno_rojo
        ws_memo['C1'].fill = relleno_rojo
        ws_memo['C2'].fill = relleno_rojo
        ws_memo['D2'].fill = relleno_rojo
        ws_memo['E2'].fill = relleno_rojo
        ws_memo['F2'].fill = relleno_rojo
        
        align_header = Alignment(horizontal="center", vertical="center")
        borde_delgado = Border(left=Side(style='thin', color='000000'), right=Side(style='thin', color='000000'), top=Side(style='thin', color='000000'), bottom=Side(style='thin', color='000000'))
        
        for fila_h in ws_memo['A1:F2']:
            for celda in fila_h:
                celda.font = fuente_blanca
                celda.alignment = align_header
                celda.border = borde_delgado
                
        # 4. Volcar datos extraídos
        fila_actual = 3
        for fila_datos in datos_memo:
            for col_idx in range(1, 7):
                celda = ws_memo.cell(row=fila_actual, column=col_idx, value=fila_datos[col_idx-1])
                celda.border = borde_delgado
                celda.alignment = align_header # Centrado para todo
                
            fila_actual += 1
            
        # 5. Ajustar anchos
        ws_memo.column_dimensions['A'].width = 15
        ws_memo.column_dimensions['B'].width = 15
        ws_memo.column_dimensions['C'].width = 20
        ws_memo.column_dimensions['D'].width = 15
        ws_memo.column_dimensions['E'].width = 15
        ws_memo.column_dimensions['F'].width = 15

        wb.save(ruta_dest)
        print("  -> Procesado exitosamente.")

if __name__ == "__main__":
    procesar_archivos()


--- SELECCIÓN BOGDATA ---
1. Por favor selecciona el archivo de CDP (Excel)...
2. Por favor selecciona el archivo de RP (Excel)...
  -> Procesando archivo RP...
     Hoja detectada y cargada: 'CRP'
  -> Procesando archivo CDP...
     Hoja detectada y cargada: 'Data'
Se encontraron 1 archivos de Excel.
Procesando: ANTONIO NARINO.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 'Para MEMO'...
  -> Procesado exitosamente.


In [33]:
fuzz.partial_ratio("Martires".lower(), "PMR LOS MARTIRES".lower())

100.0

In [32]:
"Pmr MARTIRES".lower()

'pmr martires'

In [11]:
import os
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
import re
import unicodedata

# ================= CONFIGURACIÓN =================
#CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1' 
#ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1\Consolidado\Consolidado_Localidades.xlsx"
CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7\Diagnóstico' # Dell
ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx" #Dell
#CARPETA_ORIGEN = r'D:\\Años\\2026\\SecretariaDeGobierno\\ConsolidadoAlcaldias\\META 7\\Diagnóstico' #Predator
#ARCHIVO_DESTINO = r"D:\\Años\\2026\\SecretariaDeGobierno\\ConsolidadoAlcaldias\\META 7\\Diagnóstico\\Consolidado\\Consolidado_Localidades.xlsx" #Predator

nombres_hojas = [
    "1-Usaquén", "2-Chapinero", "3-Santa Fe", "4-San Cristóbal",
    "5-Usme", "6-Tunjuelito", "7-Bosa", "8-Kennedy", "9-Fontibón",
    "10-Engativá", "11-Suba", "12-Barrios Unidos", "13-Teusaquillo",
    "14-Los Mártires", "15-Antonio Nariño", "16-Puente Aranda",
    "17-Rafael Uribe", "18-La Candelaria", "19-Ciudad Bolívar", "20-Sumapaz"
]

#RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']
# Aquí puedes añadir todas las columnas a las que quieras fijarles un ancho específico
ANCHOS_COLUMNAS = {
    'FM': 29,
    'FN': 10,
    'FO': 20,
    'FP': 15,
    'FQ': 15,
    'FR': 15,
    'FS': 20,
    'FT': 20,
    'FU': 18,
    'FV': 14,
    'FW': 25,
    'FX': 25,
    'FY': 25,
    'FZ': 25,
    'GA': 25,
    'GB': 25
}

# ================= FUNCIONES AUXILIARES =================
def normalizar_texto(texto):
    texto = texto.lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^\d+[-_ ]?', '', texto)
    texto = re.sub(r'\b(la|el|los|las)\b', '', texto).strip()
    return texto

def encontrar_hoja_destino(nombre_archivo, hojas_destino):
    archivo_norm = normalizar_texto(nombre_archivo)
    for hoja in hojas_destino:
        if normalizar_texto(hoja) in archivo_norm:
            return hoja
    return None

def ocultar_columnas_por_lista(ws, lista_rangos, offset=3):
    for item in lista_rangos:
        item = item.upper().strip()
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio) + offset
            idx_fin = column_index_from_string(fin) + offset
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                ws.column_dimensions[letra].hidden = True 
        else:
            idx = column_index_from_string(item) + offset
            letra = get_column_letter(idx)
            ws.column_dimensions[letra].hidden = True

def adaptar_formulas_resultados(formula):
    pattern = re.compile(r"'?Analisis'?!([A-Za-z]+)(\d*)(?::([A-Za-z]+)(\d*))?")
    def replacer(match):
        col1, row1, col2, row2 = match.groups()
        nueva_col1 = get_column_letter(column_index_from_string(col1) + 3)
        nueva_row1 = str(int(row1) + 21) if row1 else ""
        resultado = f"{nueva_col1}{nueva_row1}"
        if col2:
            nueva_col2 = get_column_letter(column_index_from_string(col2) + 3)
            nueva_row2 = str(int(row2) + 21) if row2 else ""
            resultado += f":{nueva_col2}{nueva_row2}"
        return resultado
    return pattern.sub(replacer, formula)

def desplazar_formulas_analisis(formula):
    formula = re.sub(r"'?Analisis'?!", "", formula, flags=re.IGNORECASE)
    pattern = re.compile(r'(\$?)([A-Za-z]{1,3})(\$?)(\d+)(?!\s*\()')
    def replacer(match):
        lock_col, col_str, lock_row, row_str = match.groups()
        nueva_col = get_column_letter(column_index_from_string(col_str) + 3)
        nueva_fila = str(int(row_str) + len(metricas)+2)
        return f"{lock_col}{nueva_col}{lock_row}{nueva_fila}"
    return pattern.sub(replacer, formula)

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# ================= FUNCIONES DE DISEÑO =================
def aplicar_estilo_ferrocarril(ws, start_row, max_row):
    """Aplica formato estilo ferrocarril solo a columnas E:U y CA:FH desde start_row."""
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    
    # Rango 1: E (5) a AL (38) | Rango 2: CA (79) a GB (185)
    def es_columna_objetivo(c):
        return (5 <= c <= 38) or (79 <= c <= 185)

    # 1. Encabezados (Fila 22)
    for col_idx in range(1, ws.max_column + 1):
        if es_columna_objetivo(col_idx):
            cell = ws.cell(row=start_row, column=col_idx)
            cell.fill = fill_header
            cell.font = font_header
            cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas de Datos (Patrón Ferrocarril)
    for i_fila, r_idx in enumerate(range(start_row + 1, max_row + 1)):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF": # Evitar procesar blanco para ahorrar memoria
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for col_idx in range(1, ws.max_column + 1):
                if es_columna_objetivo(col_idx):
                    ws.cell(row=r_idx, column=col_idx).fill = relleno

def aplicar_formato_condicional_alertas(ws, start_row, max_row):
    """Aplica reglas condicionales desde la columna EZ."""
    col_inicio_eval = column_index_from_string('FA')
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    
    reglas = [
        (12, 'equal', '"Numero contrato NO coincide"'),
        (14, 'equal', '"Cedula NO coincide"'),
        (15, 'greaterThan', '1'),
        (16, 'greaterThan', '1'),
        (17, 'equal', '"No coinciden fechas"'),
        (18, 'equal', '"No coinciden fechas"'),
        (19, 'equal', '"No coinciden fechas"'),
        (20, 'equal', '"Duracion NO coincide"'),
        (21, 'equal', '"Valor NO coincide"'),
        (22, 'equal', '"El numero RP NO coincide"'),
        (23, 'equal', '"El valor RP NO coincide"'),
        (24, 'equal', '"La fecha RP NO coincide"'),
        (25, 'equal', '"El numero CDP NO coincide"'),
        (26, 'equal', '"El valor CDP NO coincide"'),
        (27, 'equal', '"La fecha CDP NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        # Se aplica desde la fila siguiente al encabezado (23 en adelante)
        rango = f"{letra_col}{start_row + 1}:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

# ================= PROGRAMA PRINCIPAL =================
def consolidar_excels():
    global metricas
    wb_nuevo = openpyxl.Workbook()
    
    # 1. Configurar Hoja General
    ws_general = wb_nuevo.active
    ws_general.title = "General"
    ws_general.cell(row=1, column=1, value="Nro").font = Font(bold=True)
    ws_general.cell(row=1, column=2, value="Alcaldía local").font = Font(bold=True)
    
    # Pre-llenar las localidades en General con Hipervínculos
    for i, hoja in enumerate(nombres_hojas):
        num, nombre = hoja.split("-", 1)
        row_idx = i + 2
        ws_general.cell(row=row_idx, column=1, value=int(num))
        
        celda_loc = ws_general.cell(row=row_idx, column=2, value=nombre.strip())
        celda_loc.hyperlink = f"#'{hoja}'!A1"
        celda_loc.font = Font(color="0563C1", underline="single") 

    for nombre in nombres_hojas:
        wb_nuevo.create_sheet(title=nombre)

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith('.xlsx') and not f.startswith('~')]
    metricas_extraidas = False 

    for archivo in archivos:
        hoja_destino_nombre = encontrar_hoja_destino(archivo, nombres_hojas)
        
        if not hoja_destino_nombre:
            continue
            
        print(f"-> Procesando '{archivo}' -> Destino: '{hoja_destino_nombre}'")
        
        ruta_completa = os.path.join(CARPETA_ORIGEN, archivo)
        wb_origen = openpyxl.load_workbook(ruta_completa)
        ws_destino = wb_nuevo[hoja_destino_nombre]

        # --- TAREA A: Copiar de 'Resultados' ---
        if "Resultados" in wb_origen.sheetnames:
            ws_resultados = wb_origen["Resultados"]
            row_gen = nombres_hojas.index(hoja_destino_nombre) + 2
            col_gen = 3 
            
            for r_idx in range(0, len(metricas)): 
                titulo_metrica = ws_resultados.cell(row=2 + r_idx, column=2).value 
                
                if not metricas_extraidas:
                    ws_general.cell(row=1, column=col_gen, value=titulo_metrica).font = Font(bold=True)
                
                ws_general.cell(row=row_gen, column=col_gen, value=f"='{hoja_destino_nombre}'!C{r_idx+2}")
                col_gen += 1
                
            metricas_extraidas = True 
            
            for row in ws_resultados['A1':f'C{len(metricas) + 1}']:
                for cell in row:
                    dest_coord = cell.coordinate
                    valor = cell.value
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = adaptar_formulas_resultados(valor)
                    else:
                        ws_destino[dest_coord].value = valor
        else:
            print(f"  - Faltó hoja 'Resultados' en {archivo}")

        # --- TAREA B: Copiar de 'Analisis' a la fila 22 ---
        if "Analisis" in wb_origen.sheetnames:
            ws_analisis = wb_origen["Analisis"]
            ultima_fila = ws_analisis.max_row
            max_col_origen = min(ws_analisis.max_column, column_index_from_string('FY'))
            
            ws_destino.merge_cells(f"D{len(metricas) + 2}:BN{len(metricas) + 2}")
            celda_sipse = ws_destino[f"D{len(metricas) + 2}"]
            celda_sipse.value = "SIPSE"
            celda_sipse.fill = PatternFill(start_color="0070C0", end_color="0070C0", fill_type="solid") 
            celda_sipse.font = Font(color="FFFFFF", bold=True)
            celda_sipse.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells(f"BP{len(metricas) + 2}:EX{len(metricas) + 2}")
            celda_secop = ws_destino[f"BP{len(metricas) + 2}"]
            celda_secop.value = "SECOP"
            celda_secop.fill = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid") 
            celda_secop.font = Font(color="FFFFFF", bold=True)
            celda_secop.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells(f"EY{len(metricas) + 2}:GB{len(metricas) + 2}")
            celda_comparacion = ws_destino[f"EY{len(metricas) + 2}"]
            celda_comparacion.value = "COMPARACIÓN"
            celda_comparacion.fill = PatternFill(start_color="7e350e", end_color="7e350e", fill_type="solid") 
            celda_comparacion.font = Font(color="FFFFFF", bold=True)
            celda_comparacion.alignment = Alignment(horizontal="center", vertical="center")

            # Pegar los datos desde la fila 22
            for r_idx, row_origen in enumerate(ws_analisis.iter_rows(min_row=1, max_row=ultima_fila, min_col=1, max_col=max_col_origen), start=len(metricas) + 3):
                for c_idx, cell in enumerate(row_origen, start=4):
                    dest_coord = f"{get_column_letter(c_idx)}{r_idx}"
                    valor = cell.value
                    
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = desplazar_formulas_analisis(valor)
                    else:
                        ws_destino[dest_coord].value = valor
            
            # 1. Ocultar columnas
            ocultar_columnas_por_lista(ws_destino, RANGOS_OCULTAR_ANALISIS, offset=3)
            # 2. Aplicar anchos de columnas
            for letra_col, ancho in ANCHOS_COLUMNAS.items():
                ws_destino.column_dimensions[letra_col].width = ancho
            # 2.1 Ajustar el alto del texto en la fila de encabezados de las hojas de cada localidad
            ajustar_texto_fila(ws_destino, len(metricas) + 3)
            # 3. Aplicar estilo Ferrocarril (La tabla empieza en la 22)
            aplicar_estilo_ferrocarril(ws_destino, start_row=len(metricas) + 3, max_row=ultima_fila + len(metricas) + 2)
            # 4. Aplicar alertas condicionales
            aplicar_formato_condicional_alertas(ws_destino, start_row=len(metricas) + 3, max_row=ultima_fila + len(metricas) + 2)
            # 5 Pintar y ajustar tabla de cada alcaldía
            fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
            ws_destino['B1'].fill = fill_amarillo
            ws_destino.column_dimensions['B'].width = 100
            
            # --- NUEVO: BOTÓN DE REGRESO A GENERAL ---
            celda_volver = ws_destino['A1']
            celda_volver.value = "General"
            celda_volver.hyperlink = "#'General'!A1"
            celda_volver.font = Font(color="0563C1", underline="single", bold=True)
            celda_volver.fill = PatternFill(start_color="E7E6E6", end_color="E7E6E6", fill_type="solid") # Gris claro
            celda_volver.alignment = Alignment(horizontal="center", vertical="center")

        else:
            print(f"  - Faltó hoja 'Analisis' en {archivo}")
            
        wb_origen.close()

    # --- DISEÑO FINAL HOJA GENERAL ---
    # 1. Fondo Rojo para la fila de títulos A1:AC1
    relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
    fuente_blanca = Font(color="FFFFFF", bold=True)
    for col in range(1, 30): # De la A (1) a la AC (30)
        celda = ws_general.cell(row=1, column=col)
        celda.fill = relleno_rojo
        celda.font = fuente_blanca
        celda.alignment = Alignment(horizontal="center", vertical="center")

    # 2. Centrar los valores de C1:AC1
    alineacion_centro = Alignment(horizontal="center", vertical="center")
    for r in range(2, 22): # Filas 2 a 21
        for c in range(3, 29): # Columnas C (3) a AC (30)
            ws_general.cell(row=r, column=c).alignment = alineacion_centro
            
    # Ajustar ligeramente el ancho de General para que se vea bien
    ws_general.column_dimensions['A'].width = 5
    ws_general.column_dimensions['B'].width = 25
    for c in range(3, 22):
        ws_general.column_dimensions[get_column_letter(c)].width = 15
    
    # Ajustar el texto en la fila 1 de las hoja General
    ajustar_texto_fila(ws_general, 1)

    # --- NUEVO: APLICAR GRILLA DELGADA (A1:U21) ---
    borde_delgado = Border(
        left=Side(style='thin', color='000000'),
        right=Side(style='thin', color='000000'),
        top=Side(style='thin', color='000000'),
        bottom=Side(style='thin', color='000000')
    )
    
    for r in range(1, 22):      # Filas de la 1 a la 21
        for c in range(1, 30):  # Columnas de la A(1) a la AC(29)
            ws_general.cell(row=r, column=c).border = borde_delgado
    
    # CARPETA 'CONSOLIDADO' SI NO EXISTE ---
    carpeta_destino = os.path.dirname(ARCHIVO_DESTINO) # Extrae la ruta de la carpeta
    os.makedirs(carpeta_destino, exist_ok=True)        # Crea la carpeta (y subcarpetas si faltan) sin dar error si ya existe

    wb_nuevo.save(ARCHIVO_DESTINO)
    print(f"\n🚀 ¡Proceso completado! Archivo guardado en: {ARCHIVO_DESTINO}")

if __name__ == "__main__":
    consolidar_excels()

-> Procesando 'ANTONIO NARINO.xlsx' -> Destino: '15-Antonio Nariño'


-> Procesando 'BARRIOS UNIDOS.xlsx' -> Destino: '12-Barrios Unidos'
-> Procesando 'BOSA.xlsx' -> Destino: '7-Bosa'
-> Procesando 'CANDELARIA.xlsx' -> Destino: '18-La Candelaria'
-> Procesando 'CHAPINERO.xlsx' -> Destino: '2-Chapinero'
-> Procesando 'CIUDAD BOLIVAR.xlsx' -> Destino: '19-Ciudad Bolívar'
-> Procesando 'ENGATIVA.xlsx' -> Destino: '10-Engativá'
-> Procesando 'FONTIBON.xlsx' -> Destino: '9-Fontibón'
-> Procesando 'KENNEDY.xlsx' -> Destino: '8-Kennedy'
-> Procesando 'MARTIRES.xlsx' -> Destino: '14-Los Mártires'
-> Procesando 'PUENTE ARANDA.xlsx' -> Destino: '16-Puente Aranda'
-> Procesando 'RAFAEL URIBE.xlsx' -> Destino: '17-Rafael Uribe'
-> Procesando 'SAN CRISTOBAL.xlsx' -> Destino: '4-San Cristóbal'
-> Procesando 'SANTA FE.xlsx' -> Destino: '3-Santa Fe'
-> Procesando 'SUBA.xlsx' -> Destino: '11-Suba'
-> Procesando 'SUMAPAZ.xlsx' -> Destino: '20-Sumapaz'
-> Procesando 'TEUSAQUILLO.xlsx' -> Destino: '13-Teusaquillo'
-> Procesando 'TUNJUELITO.xlsx' -> Destino: '6-Tunjuelito'


#### Entrega 4 Ago 2026

In [2]:
import pandas as pd
import unicodedata
import re
import difflib
import os
import tkinter as tk
from tkinter import filedialog, messagebox

# ==========================================
# 0. DEFINICIÓN DE ESTRUCTURA ESPERADA
# ==========================================
# Reemplaza los elementos de estas listas con los nombres EXACTOS 
# y en el ORDEN que necesitas para cada hoja.
COLUMNAS_ESPERADAS_SIPSE = [
    "CONTRATO", "SOLICITUD", "VIGENCIA", "NUMERO_CONTRATO", "TIPO", "TIPO_PROCESO", "OBJETO", "FECHA_SUSCRIPCION", "FECHA_INICIO", "FECHA_TERMINACION", "TERMINA31_DICIEMBRE", "CONTRATISTA", "DOC_CONTRATISTA", "VALOR_INICIAL", "ADICIONES", "VALOR_TOTAL", "VALOR_MENSUAL", "PLAZO", "PRORROGAS", "PROYECTO", "DEPENDENCIA", "SUPERVISION", "ESTADO", "NAT", "PAGOS", "AVANCE", "FECHA_ACTA_LIQUIDACION", "PRESUPUESTO", "ANULACION", "DIRECCION_CONTRATISTA", "TELEFONO_CONTRATISTA", "CODIGO_PROYECTO", "NOMBRE_PROYECTO", "CDPS", "RPS", "ESTADO_RP", "MODIFICACION", "NUMERO_OC", "TIPO_SOCIEDAD", "TIPOLOGIA", "TIPO_PAGO", "NIVELACADEMICO", "PROFEXION", "ESPECIALIZ", "POLIZA", "PROCESO_CONTRATO", "ANTIGUEDAD", "SALDO_FAVOR_ENTIDAD", "FECHA_NACIMIENTO", "OBLIGACIONES", "TIPO_IDENTIFICACION", "CORREO_ELECTRONICO", "NOMBRE", "APELLIDO", "CELULAR", "GARANTIAS", "INICIO_AMPARO", "FIN_AMPARO", "IDEN_REP_LEGAL", "NOM_REP_LEGAL", "ENTIDAD", "DESCRIPCION_ENTIDAD", "GENERO"
]

COLUMNAS_ESPERADAS_SECOP = [
    "Nombre Entidad", "Nit Entidad", "Departamento", "Ciudad", "LocalizaciÃ³n", "Orden", "Sector", "Rama", "Entidad Centralizada", "Proceso de Compra", "ID Contrato", "Referencia del Contrato", "Estado Contrato", "Codigo de Categoria Principal", "Descripcion del Proceso", "Tipo de Contrato", "Modalidad de Contratacion", "Justificacion Modalidad de Contratacion", "Fecha de Firma", "Fecha de Inicio del Contrato", "Fecha de Fin del Contrato", "Condiciones de Entrega", "TipoDocProveedor", "Documento Proveedor", "Proveedor Adjudicado", "Es Grupo", "Es Pyme", "Habilita Pago Adelantado", "LiquidaciÃ³n", "ObligaciÃ³n Ambiental", "Obligaciones Postconsumo", "Reversion", "Origen de los Recursos", "Destino Gasto", "Valor del Contrato", "Valor de pago adelantado", "Valor Facturado", "Valor Pendiente de Pago", "Valor Pagado", "Valor Amortizado", "Valor Pendiente de Amortizacion", "Valor Pendiente de Ejecucion", "Estado BPIN", "CÃ³digo BPIN", "Anno BPIN", "Saldo CDP", "Saldo Vigencia", "EsPostConflicto", "Dias adicionados", "Puntos del Acuerdo", "Pilares del Acuerdo", "URLProceso", "Nombre Representante Legal", "Nacionalidad Representante Legal", "Domicilio Representante Legal", "Tipo de IdentificaciÃ³n Representante Legal", "IdentificaciÃ³n Representante Legal", "GÃ©nero Representante Legal", "Presupuesto General de la Nacion â€“ PGN", "Sistema General de Participaciones", "Sistema General de RegalÃ­as", "Recursos Propios (AlcaldÃ­as, Gobernaciones y Resguardos IndÃ­genas)", "Recursos de Credito", "Recursos Propios", "Ultima Actualizacion", "Codigo Entidad", "Codigo Proveedor", "Fecha Inicio Liquidacion", "Fecha Fin Liquidacion", "Objeto del Contrato", "DuraciÃ³n del contrato", "Nombre del banco", "Tipo de cuenta", "NÃºmero de cuenta", "El contrato puede ser prorrogado", "Fecha de notificaciÃ³n de prorrogaciÃ³n", "Nombre ordenador del gasto", "Tipo de documento Ordenador del gasto", "NÃºmero de documento Ordenador del gasto", "Nombre supervisor", "Tipo de documento supervisor", "NÃºmero de documento supervisor", "Nombre Ordenador de Pago", "Tipo de documento Ordenador de Pago", "NÃºmero de documento Ordenador de Pago", "Documentos Tipo", "Descripcion Documentos Tipo"
]

# ==========================================
# 1. FUNCIONES DE LIMPIEZA, COMPARACIÓN Y ESTRUCTURA
# ==========================================

def normalizar_texto(texto):
    """Limpia el texto: quita nulos, espacios, pasa a mayúsculas, quita tildes y artículos"""
    if pd.isna(texto):
        return ""
    texto = str(texto).upper().strip()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^(EL|LA|LOS|LAS)\s+', '', texto)
    return texto

def unificar_localidades(lista_localidades, similitud_minima=0.85):
    """Agrupa nombres similares bajo un mismo nombre 'canónico'"""
    nombres_canonicos = {}
    for loc in lista_localidades:
        if not loc: continue
        coincidencias = difflib.get_close_matches(loc, nombres_canonicos.keys(), n=1, cutoff=similitud_minima)
        if coincidencias:
            nombres_canonicos[loc] = coincidencias[0]
        else:
            nombres_canonicos[loc] = loc
    return nombres_canonicos

def alinear_columnas(df, columnas_esperadas):
    """
    Compara las columnas del df con la lista esperada.
    Crea las faltantes en blanco y ELIMINA las que sobran.
    """
    if not columnas_esperadas:
        return df

    # 1. Crear las columnas que faltan en el DataFrame original
    for col in columnas_esperadas:
        if col not in df.columns:
            df[col] = "" 
            
    # 2. Retornar el DataFrame filtrado SOLAMENTE con las columnas esperadas y en el orden exacto
    return df[columnas_esperadas]

# ==========================================
# 2. SELECCIÓN DE ARCHIVOS Y CARPETAS
# ==========================================

root = tk.Tk()
root.withdraw()

print("Por favor, selecciona el archivo Excel de origen...")
archivo_origen = filedialog.askopenfilename(
    title="Selecciona el archivo Excel (.xlsx o .xls)",
    filetypes=[("Archivos de Excel", "*.xlsx *.xls *")]
)

if not archivo_origen:
    print("No se seleccionó ningún archivo. Cancelando proceso.")
    exit()

print("Por favor, selecciona la carpeta de destino...")
carpeta_destino = filedialog.askdirectory(title="Selecciona la carpeta de destino")

if not carpeta_destino:
    print("No se seleccionó ninguna carpeta. Cancelando proceso.")
    exit()

# ==========================================
# 3. LECTURA Y PROCESAMIENTO DEL EXCEL
# ==========================================

try:
    print(f"\nLeyendo el archivo: {archivo_origen}")
    xls = pd.ExcelFile(archivo_origen)
    nombres_hojas = xls.sheet_names
    
    hoja_sipse = next((h for h in nombres_hojas if h.upper() == "SIPSE"), None)
    hoja_secop = next((h for h in nombres_hojas if h.upper() == "SECOP II"), None)
    
    if not hoja_sipse or not hoja_secop:
        raise ValueError("El archivo no contiene las hojas 'SIPSE' y/o 'SECOP II'.")

    df_sipse = pd.read_excel(xls, sheet_name=hoja_sipse)
    df_secop = pd.read_excel(xls, sheet_name=hoja_secop)
    
    if len(df_sipse.columns) <= 61:
        raise ValueError("La hoja SIPSE no tiene suficientes columnas para llegar a la columna BJ (índice 61).")
        
    # Identificar nombres de columnas clave
    col_sipse_nombre = df_sipse.columns[61] 
    col_secop_nombre = df_secop.columns[0]

    # --- NUEVO CAMBIO: REEMPLAZO ESPECÍFICO DE RAFAEL URIBE ---
    print("Ejecutando reemplazo de nombres de entidad específicos...")
    
    # Reemplazo en SECOP II (Columna A)
    df_secop[col_secop_nombre] = df_secop[col_secop_nombre].replace("RAFAEL URIBE URIBE", "RAFAEL URIBE")
    
    # Reemplazo en SIPSE (Columna BJ / índice 61)
    df_sipse[col_sipse_nombre] = df_sipse[col_sipse_nombre].replace("RAFAEL URIBE URIBE", "RAFAEL URIBE")
    # ---------------------------------------------------------

    print("Alineando y estandarizando columnas según listas esperadas (eliminando sobrantes)...")
    df_sipse = alinear_columnas(df_sipse, COLUMNAS_ESPERADAS_SIPSE)
    df_secop = alinear_columnas(df_secop, COLUMNAS_ESPERADAS_SECOP)
    
    print("Normalizando nombres de localidades...")
    # Como ya alineamos las columnas, garantizamos que existen, pero usamos el nombre extraído previamente
    df_sipse['Loc_Limpia'] = df_sipse[col_sipse_nombre].apply(normalizar_texto)
    df_secop['Loc_Limpia'] = df_secop[col_secop_nombre].apply(normalizar_texto)
    
    todas_localidades = list(set(df_sipse['Loc_Limpia'].unique()) | set(df_secop['Loc_Limpia'].unique()))
    mapa_canonico = unificar_localidades(todas_localidades)
    
    df_sipse['Loc_Final'] = df_sipse['Loc_Limpia'].map(mapa_canonico)
    df_secop['Loc_Final'] = df_secop['Loc_Limpia'].map(mapa_canonico)
    
    localidades_finales = set(mapa_canonico.values())
    
    print(f"Se detectaron {len(localidades_finales)} localidades únicas. Generando archivos...")
    
    archivos_creados = 0
    for loc in localidades_finales:
        if not loc: continue 
        
        # Filtramos y eliminamos las dos columnas temporales (Loc_Limpia y Loc_Final) que usamos para segmentar
        filtro_sipse = df_sipse[df_sipse['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        filtro_secop = df_secop[df_secop['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        
        if not filtro_sipse.empty or not filtro_secop.empty:
            nombre_archivo_seguro = re.sub(r'[\\/*?:"<>|]', "", loc)
            ruta_salida = os.path.join(carpeta_destino, f"{nombre_archivo_seguro}.xlsx")
            
            with pd.ExcelWriter(ruta_salida, engine='openpyxl') as writer:
                filtro_sipse.to_excel(writer, sheet_name="SIPSE", index=False)
                filtro_secop.to_excel(writer, sheet_name="SECOP II", index=False)
                
            archivos_creados += 1
            print(f" -> Creado: {nombre_archivo_seguro}.xlsx")

    print(f"\n¡Proceso Finalizado Exitosamente! Se crearon {archivos_creados} archivos.")
    messagebox.showinfo("Éxito", f"Se generaron {archivos_creados} archivos correctamente únicamente con las columnas especificadas.")

except Exception as e:
    print(f"\nOcurrió un error: {e}")
    messagebox.showerror("Error", f"Ocurrió un error procesando el archivo:\n{e}")

finally:
    root.destroy()

Por favor, selecciona el archivo Excel de origen...
Por favor, selecciona la carpeta de destino...

Leyendo el archivo: C:/Users/Pc/Desktop/META 7/Datos Virgenes/sipse + secop2.xlsx
Ejecutando reemplazo de nombres de entidad específicos...
Alineando y estandarizando columnas según listas esperadas (eliminando sobrantes)...
Normalizando nombres de localidades...
Se detectaron 20 localidades únicas. Generando archivos...
 -> Creado: FONTIBON.xlsx
 -> Creado: SUMAPAZ.xlsx
 -> Creado: CHAPINERO.xlsx
 -> Creado: BOSA.xlsx
 -> Creado: ENGATIVA.xlsx
 -> Creado: KENNEDY.xlsx
 -> Creado: RAFAEL URIBE.xlsx
 -> Creado: CANDELARIA.xlsx
 -> Creado: PUENTE ARANDA.xlsx
 -> Creado: TEUSAQUILLO.xlsx
 -> Creado: SUBA.xlsx
 -> Creado: ANTONIO NARINO.xlsx
 -> Creado: MARTIRES.xlsx
 -> Creado: SAN CRISTOBAL.xlsx
 -> Creado: TUNJUELITO.xlsx
 -> Creado: USAQUEN.xlsx
 -> Creado: SANTAFE.xlsx
 -> Creado: BARRIOS UNIDOS.xlsx
 -> Creado: USME.xlsx
 -> Creado: CIUDAD BOLIVAR.xlsx

¡Proceso Finalizado Exitosamente

In [12]:
#Celda 2.1
import os
import shutil
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
from collections import defaultdict
import re
import pandas as pd
import difflib # para busqueda aproximada
from rapidfuzz import fuzz, process
import datetime

#Para abrir explorador de archivos
import tkinter as tk
from tkinter import filedialog
import math
import unicodedata

# --- CONFIGURACIÓN DE CARPETAS ---
CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7' #Dell
CARPETA_DESTINO = r"C:\Users\Pc\Desktop\META 7\Diagnóstico" #Dell

# --- RANGOS A OCULTAR ---
RANGOS_OCULTAR_SECOP = ['A-K', 'N-S', 'U-W', 'Z-AH', 'AJ-BR', 'BT-CI']
RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-AG','AJ-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']

# --- DEFINICIÓN DE FUNCIONES ---
def ajustar_texto_fila(ws, numero_fila):
    for cell in ws[numero_fila]:
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def ocultar_columnas_por_lista(ws, lista_rangos):
    for item in lista_rangos:
        item = item.upper().strip()
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio)
            idx_fin = column_index_from_string(fin)
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                ws.column_dimensions[letra].hidden = True 
        else:
            ws.column_dimensions[item].hidden = True

def aplicar_estilo_ferrocarril(ws):
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    
    for cell in ws[1]:
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal='center', vertical='center')

    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    max_col = ws.max_column
    for i_fila, row in enumerate(ws.iter_rows(min_row=2, max_col=max_col), start=0):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF":
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for cell in row:
                cell.fill = relleno

def dejar_columna_blanca(ws, indice_columna):
    relleno_blanco = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=indice_columna, max_col=indice_columna):
        for cell in row:
            cell.fill = relleno_blanco

def aplicar_formato_condicional_alertas(ws, col_inicio_eval):
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    max_row = ws.max_row
    
    reglas = [
        (12, 'equal', '"Numero contrato NO coincide"'),
        (14, 'equal', '"Cedula NO coincide"'),
        (15, 'greaterThan', '1'),
        (16, 'greaterThan', '1'),
        (17, 'equal', '"No coinciden fechas"'),
        (18, 'equal', '"No coinciden fechas"'),
        (19, 'equal', '"No coinciden fechas"'),
        (20, 'equal', '"Duracion NO coincide"'),
        (21, 'equal', '"Valor NO coincide"'),
        (22, 'equal', '"El numero RP NO coincide"'),
        (23, 'equal', '"El valor RP NO coincide"'),
        (24, 'equal', '"La fecha RP NO coincide"'),
        (25, 'equal', '"El numero CDP NO coincide"'),
        (26, 'equal', '"El valor CDP NO coincide"'),
        (27, 'equal', '"La fecha CDP NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        rango = f"{letra_col}2:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

def normalizar_numero_contrato(valor):
    if valor is None: return 0
    texto = str(valor).strip()
    match = re.search(r'\d+', texto)
    if match:
        return int(match.group())
    return 0

def es_error_fecha(val_sipse, val_secop):
    s1 = str(val_sipse).strip() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip() if pd.notna(val_secop) and val_secop is not None else ""
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    
    dt_sipse = None
    if isinstance(val_sipse, (datetime.datetime, datetime.date)):
        dt_sipse = datetime.date(val_sipse.year, val_sipse.month, val_sipse.day)
    else:
        try: dt_sipse = pd.to_datetime(s1.replace("-", "/"), dayfirst=True).date()
        except: pass

    dt_secop = None
    if isinstance(val_secop, (datetime.datetime, datetime.date)):
        try: dt_secop = datetime.date(val_secop.year, val_secop.day, val_secop.month)
        except: pass
    else:
        partes = s2.replace("-", "/").replace(" ", "/").split("/")
        if len(partes) >= 3:
            try:
                y = int(partes[2][-4:])
                m = int(partes[0])
                d = int(partes[1])
                dt_secop = datetime.date(y, m, d)
            except: pass
                
    if dt_sipse is not None and dt_secop is not None:
        return dt_sipse != dt_secop
    return s1 != s2 

def es_error_duracion(val_sipse, val_secop):
    s1 = str(val_sipse).strip().upper() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip().upper() if pd.notna(val_secop) and val_secop is not None else ""
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    return s1 != s2

def color_memo(err_tuple):
    if err_tuple[0]: return "E4DFEC"
    if err_tuple[1]: return "B4C6E7"
    if err_tuple[2]: return "DDEBF7"
    if err_tuple[3]: return "F4CCCC"
    return "FFFFFF"

def normalizar_alfanumerico(valor):
    if pd.isna(valor) or valor is None: return ""
    texto = str(valor).strip().upper()
    if texto.endswith(".0"): texto = texto[:-2]
    return texto

def limpiar_dato(val):
    if pd.isna(val) or val is None: return ""
    if isinstance(val, (datetime.datetime, datetime.date)):
        return val.strftime('%d/%m/%Y')
    val_str = str(val).strip()
    if val_str.endswith(".0"): val_str = val_str[:-2]
    return val_str

def normalizar_cedula(valor):
    val_str = limpiar_dato(valor)
    return re.sub(r'\D', '', val_str)

def obtener_valor_seguro(row, indice):
    try: return limpiar_dato(row.iloc[indice])
    except IndexError: return ""

def mapear_columnas(columnas_df, columnas_buscadas, umbral=80):
    mapa_indices = {}
    for clave, nombre_ideal in columnas_buscadas.items():
        resultado = process.extractOne(nombre_ideal, columnas_df, scorer=fuzz.token_sort_ratio)
        if resultado and resultado[1] >= umbral:
            mapa_indices[clave] = resultado[2]
        else:
            print(f"⚠️ Advertencia: No se encontró la columna para '{nombre_ideal}'.")
            mapa_indices[clave] = None 
    return mapa_indices

def normalizar_texto_localidad(texto):
    if pd.isna(texto) or texto is None: return ""
    t = " ".join(str(texto).upper().split())
    t = ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')
    return t

def cargar_datos_bogdata():
    """Abre el explorador y procesa RP y CDP. Retorna los diccionarios."""
    root = tk.Tk()
    root.withdraw() 
    root.attributes('-topmost', True) 
    
    print("\n--- SELECCIÓN BOGDATA ---")
    print("1. Por favor selecciona el archivo de CDP (Excel)...")
    ruta_cdp = filedialog.askopenfilename(title="Seleccionar archivo CDP", filetypes=[("Excel files", "*.xlsx *.xls")])
    
    print("2. Por favor selecciona el archivo de RP (Excel)...")
    ruta_rp = filedialog.askopenfilename(title="Seleccionar archivo RP", filetypes=[("Excel files", "*.xlsx *.xls")])
         
    dict_cdp = {}
    dict_rp = defaultdict(list)

    # --- PROCESAR RP ---
    if ruta_rp:
        print(f"  -> Procesando archivo RP...")
        try:
            excel = pd.ExcelFile(ruta_rp)
            hojas_disponibles = excel.sheet_names
            palabras_clave = ["data", "crp"]
            hoja_correcta = next((hoja for hoja in hojas_disponibles if hoja.strip().lower() in palabras_clave), None)

            if hoja_correcta:
                df_rp = pd.read_excel(excel, sheet_name=hoja_correcta)
                print(f"     Hoja detectada y cargada: '{hoja_correcta}'")
            else:
                print(f"  [Error] No se encontró ninguna hoja 'Data' o 'CRP' en el archivo.")
                df_rp = None
        except Exception as e:
            print(f"  [Error] No se pudo leer el archivo Excel: {e}")
            df_rp = None

        if df_rp is not None:
            nombres_ideales_rp = {
                'fecha': 'Fecha Registro', 'num_cdp': 'Número de CDP', 'num_rp': 'Número de CRP',
                'filtro1': 'Rubro', 'localidad': 'Texto Id Proyecto', 'cedula': 'Número Doc. BP Beneficiario',
                'filtro2': 'Anulaciones', 'val_rp': 'Valor Neto', 'cod_int': 'N° Interno CDP'
            }
            idx_rp = mapear_columnas(df_rp.columns.tolist(), nombres_ideales_rp)
            for _, row in df_rp.iterrows():
                val_r = obtener_valor_seguro(row, idx_rp['filtro1'])
                digitos_r = re.sub(r'\D', '', val_r)
                if len(digitos_r) >= 7:
                    val_al = obtener_valor_seguro(row, idx_rp['filtro2'])
                    try: num_al = float(str(val_al).replace(',', ''))
                    except ValueError: num_al = 0.0 
                    
                    if num_al == 0:
                        cedula_raw = obtener_valor_seguro(row, idx_rp['cedula'])
                        llave_cedula = normalizar_cedula(cedula_raw)
                        if llave_cedula:
                            rp_num = obtener_valor_seguro(row, idx_rp['num_rp']) 
                            rp_val = obtener_valor_seguro(row, idx_rp['val_rp']) 
                            rp_fec = obtener_valor_seguro(row, idx_rp['fecha'])  
                            cdp_num = obtener_valor_seguro(row, idx_rp['num_cdp']) 
                            cdp_cod = obtener_valor_seguro(row, idx_rp['cod_int']) 
                            localidad_col_z = obtener_valor_seguro(row, idx_rp['localidad']) 
                            dict_rp[llave_cedula].append((rp_num, rp_val, rp_fec, cdp_num, cdp_cod, localidad_col_z))

    # --- PROCESAR CDP ---
    if ruta_cdp:
        print(f"  -> Procesando archivo CDP...")
        try:
            excel = pd.ExcelFile(ruta_cdp)
            hojas_disponibles = excel.sheet_names
            palabras_clave = ["data", "cdp"]
            hoja_correcta = next((hoja for hoja in hojas_disponibles if hoja.strip().lower() in palabras_clave), None)

            if hoja_correcta:
                df_cdp = pd.read_excel(excel, sheet_name=hoja_correcta)
                print(f"     Hoja detectada y cargada: '{hoja_correcta}'")
            else:
                print(f"  [Error] No se encontró ninguna hoja 'Data' o 'CDP' en el archivo.")
                df_cdp = None
        except Exception as e:
            print(f"  [Error] No se pudo leer el archivo Excel: {e}")
            df_cdp = None

        if df_cdp is not None:
            nombres_ideales_cdp = {
                'num_cdp': 'No. CDP', 'val_cdp': 'Valor', 'cod_int': 'No.Interno CDP', 'fec_cdp': 'Fecha Registro'
            }
            idx_cdp = mapear_columnas(df_cdp.columns.tolist(), nombres_ideales_cdp)
            for _, row in df_cdp.iterrows():
                cdp_num = obtener_valor_seguro(row, idx_cdp['num_cdp'])  
                cdp_cod = obtener_valor_seguro(row, idx_cdp['cod_int']) 
                llave_cdp = (cdp_num, cdp_cod)
                if llave_cdp != ("", "") and llave_cdp not in dict_cdp:
                    cdp_val = obtener_valor_seguro(row, idx_cdp['val_cdp']) 
                    cdp_fec = obtener_valor_seguro(row, idx_cdp['fec_cdp']) 
                    dict_cdp[llave_cdp] = (cdp_num, cdp_val, cdp_fec)

    return dict_cdp, dict_rp

# =======================================================
# EJECUCIÓN DE LA CELDA 1 (Almacenar en memoria)
# =======================================================
print("Iniciando la carga de diccionarios (esto tomará tiempo)...")
dict_cdp_global, dict_rp_global = cargar_datos_bogdata()
print("¡Carga finalizada con éxito! Los diccionarios están en memoria. Ya puedes correr la Celda 2.")

Iniciando la carga de diccionarios (esto tomará tiempo)...

--- SELECCIÓN BOGDATA ---
1. Por favor selecciona el archivo de CDP (Excel)...
2. Por favor selecciona el archivo de RP (Excel)...
  -> Procesando archivo RP...
     Hoja detectada y cargada: 'Data'
  -> Procesando archivo CDP...
     Hoja detectada y cargada: 'Data'
¡Carga finalizada con éxito! Los diccionarios están en memoria. Ya puedes correr la Celda 2.


In [ ]:
#Celda 2.2
global metricas

def procesar_archivos(dict_cdp_en_memoria, dict_rp_en_memoria):
    global metricas
    if not os.path.exists(CARPETA_DESTINO):
        os.makedirs(CARPETA_DESTINO)

    # 🔹 Aquí mapeamos los diccionarios pre-cargados a las variables locales
    dict_cdp = dict_cdp_en_memoria
    dict_rp = dict_rp_en_memoria

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('.xlsx', '.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('ENGATIVA.xlsx', 'ENGATIVA.xls'))]
    print(f"Se encontraron {len(archivos)} archivos de Excel.")

    for archivo in archivos:
        ruta_orig = os.path.join(CARPETA_ORIGEN, archivo)
        nombre_base = os.path.splitext(archivo)[0]
        ruta_dest = os.path.join(CARPETA_DESTINO, nombre_base + ".xlsx")
        
        print(f"Procesando: {archivo}...")

        # --- PASO DE CONVERSIÓN SI ES .XLS ---
        if archivo.lower().endswith('.xls'):
            try:
                conversion = pd.read_excel(ruta_orig, sheet_name=None, engine='xlrd')
                with pd.ExcelWriter(ruta_dest, engine='openpyxl') as writer:
                    for nombre_hoja, datos in conversion.items():
                        datos.to_excel(writer, sheet_name=nombre_hoja, index=False)
                print(f"  -> Convertido de .xls a .xlsx con éxito.")
            except Exception as e:
                print(f"  Error convirtiendo {archivo}: {e}")
                continue
        else:
            shutil.copy2(ruta_orig, ruta_dest)

        try:
            wb = openpyxl.load_workbook(ruta_dest)
        except Exception as e:
            print(f"   Error al abrir el archivo con openpyxl: {e}")
            continue

        # ==========================================
        # PARTE 1: Procesar SECOP II (Crear aux1)
        # ==========================================
        nombre_real_secop = next(n for n in wb.sheetnames if n.lower() == 'secop ii')
        ws_origen = wb[nombre_real_secop]
        
        if "SECOP II aux1" in wb.sheetnames:
            del wb["SECOP II aux1"]
            
        ws_secop_aux = wb.copy_worksheet(ws_origen)
        ws_secop_aux.title = "SECOP II aux1"

        filas = []
        for row in ws_secop_aux.iter_rows(min_row=2, values_only=True):
            filas.append(list(row))
        
        filas_ordenadas = sorted(filas, key=lambda x: normalizar_numero_contrato(x[11] if len(x) > 11 else 0))

        for row in ws_secop_aux.iter_rows(min_row=2):
            for cell in row:
                cell.value = None

        for i_fila, datos_fila in enumerate(filas_ordenadas, start=2):
            for i_col, valor in enumerate(datos_fila, start=1):
                ws_secop_aux.cell(row=i_fila, column=i_col, value=valor)

        max_col_secop = ws_secop_aux.max_column
        ws_secop_aux.cell(row=1, column=max_col_secop + 1, value="Columna1")
        ws_secop_aux.cell(row=1, column=max_col_secop + 2, value="Columna2")
        col_L = get_column_letter(12)
        
        for i in range(2, ws_secop_aux.max_row + 1):
            f1 = (
                f'=IFERROR(MID({col_L}{i}, '
                f'MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), '
                f'MIN(IFERROR(FIND(" ", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999), '
                f'IFERROR(FIND("-", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999)) '
                f'- MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), "")'
            )
            ws_secop_aux.cell(row=i, column=max_col_secop + 1, value=f1)

            f2 = f'=IFERROR(MID({col_L}{i}, FIND("(", {col_L}{i}) + 1, FIND(")", {col_L}{i}) - FIND("(", {col_L}{i}) - 1), "")'
            ws_secop_aux.cell(row=i, column=max_col_secop + 2, value=f2)

        print("  -> Aplicando estilos y ocultando columnas en SECOP II aux1...")
        aplicar_estilo_ferrocarril(ws_secop_aux)
        ocultar_columnas_por_lista(ws_secop_aux, RANGOS_OCULTAR_SECOP)

        # ==========================================
        # PARTE 2 y 3: ALINEACIÓN EN 'ANALISIS'
        # ==========================================
        print("  -> Generando hoja 'Analisis'...")

        nombre_real_sipse = next(n for n in wb.sheetnames if n.lower() == 'sipse')
        ws_sipse = wb[nombre_real_sipse]
        
        if 'Analisis' in wb.sheetnames:
            del wb['Analisis']
        ws_analisis = wb.create_sheet("Analisis")
        
        dict_sipse = defaultdict(list)
        dict_secop = defaultdict(list)
        
        filas_sipse = list(ws_sipse.values)
        header_sipse = filas_sipse[0] if filas_sipse else []
        for fila in filas_sipse[1:]:
            contrato = normalizar_numero_contrato(fila[3] if len(fila) > 3 else 0)
            dict_sipse[contrato].append(fila)

        filas_secop = list(ws_secop_aux.values)
        header_secop = filas_secop[0] if filas_secop else []
        for fila in filas_secop[1:]:
            contrato = normalizar_numero_contrato(fila[11] if len(fila) > 11 else 0)
            dict_secop[contrato].append(fila)

        ws_analisis.append(header_sipse) 
        col_inicio_secop = column_index_from_string("BM")
        for i, val in enumerate(header_secop):
            ws_analisis.cell(row=1, column=col_inicio_secop + i, value=val)

        todos_contratos = sorted(list(set(dict_sipse.keys()) | set(dict_secop.keys())))
        current_row = 2 
        
        for contrato in todos_contratos:
            filas_de_sipse = dict_sipse.get(contrato, [])
            filas_de_secop = dict_secop.get(contrato, [])
            max_filas = max(len(filas_de_sipse), len(filas_de_secop))
            
            for i in range(max_filas):
                if i < len(filas_de_sipse):
                    datos = filas_de_sipse[i]
                    for c_idx, valor in enumerate(datos, start=1):
                        ws_analisis.cell(row=current_row, column=c_idx, value=valor)
                
                if i < len(filas_de_secop):
                    datos = filas_de_secop[i]
                    for c_idx, valor in enumerate(datos):
                        col_excel = col_inicio_secop + c_idx
                        if c_idx == len(datos) - 2 and str(valor).startswith('='):
                             nueva_formula1 = (
                                f'=IFERROR(MID(BX{current_row}, '
                                f'MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), '
                                f'MIN(IFERROR(FIND(" ", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999), '
                                f'IFERROR(FIND("-", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999)) '
                                f'- MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), "")'
                            )
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula1)
                        elif c_idx == len(datos) - 1 and str(valor).startswith('='):
                             nueva_formula2 = f'=IFERROR(MID(BX{current_row}, FIND("(", BX{current_row}) + 1, FIND(")", BX{current_row}) - FIND("(", BX{current_row}) - 1), "")'
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula2)
                        else:
                            ws_analisis.cell(row=current_row, column=col_excel, value=valor)
                current_row += 1

        # ==========================================
        # PARTE 4: Columnas de Evaluación
        # ==========================================
        print("  -> Agregando columnas de evaluación y estilos...")
        
        ultima_col_actual = ws_analisis.max_column
        col_inicio_eval = ultima_col_actual + 1
        ultima_fila_analisis = ws_analisis.max_row
        
        headers_nuevos = [
            "Columna3\n# RP\n(SIPSE)", "Columna4\nValor RP\n(SIPSE)", "Columna5\nFecha RP\n(SIPSE)", "Columna6\n# CDP\n(SIPSE)", "Columna7\nValor CDP\n(SIPSE)", "Columna8\nFecha CDP\n(SIPSE)", "Col.9\n# RP\n(BOGD)", "Col.10\nValor RP\n(BOGD)", "Col.11\nFecha RP\n(BOGD)", "Col.12\n# CDP\n(BOGD)", "Col.13\nValor CDP\n(BOGD)", "Col.14\nFecha CDP\n(BOGD)",
            "Ev. Nro contrato", "Aux Nro contrato", "Ev. Cédula", "Repeticiones contrato SIPSE", "Repeticiones contrato SECOP", "Fecha suscripción", "Fecha inicio", "Fecha fin", "Duración contratos", "Valor contratos", "Ev. # RP", "Ev. Valor RP", "Ev. Fecha RP", "Ev. # CDP", "Ev. Valor CDP", "Ev. Fecha CDP",
            "Ev. Indicador 1", "Ev. Indicador 2"
        ]

        for idx, titulo in enumerate(headers_nuevos):
            ws_analisis.cell(row=1, column=col_inicio_eval + idx, value=titulo)
        
        for i in range(2, ultima_fila_analisis + 1):
            f3 = f'=IFERROR(TRIM(RIGHT(SUBSTITUTE(TRIM(LEFT(AI{i}, FIND("Fecha", AI{i})-1)), " ", REPT(" ", 100)), 100)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval, value=f3)
            f4 = f'=IFERROR(TRIM(MID(AI{i}, FIND("Valor ", AI{i}) + 6, IFERROR(FIND(";", AI{i}), LEN(AI{i})+1) - FIND("Valor ", AI{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 1, value=f4)
            f5 = f'=IFERROR(TRIM(MID(AI{i}, FIND("Fecha ", AI{i}) + 6, FIND(" Valor", AI{i}) - FIND("Fecha ", AI{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 2, value=f5)
            f6 = f'=IFERROR(TRIM(RIGHT(SUBSTITUTE(TRIM(LEFT(AH{i}, FIND("Fecha", AH{i})-1)), " ", REPT(" ", 100)), 100)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 3, value=f6)
            f7 = f'=IFERROR(TRIM(MID(AH{i}, FIND("Valor ", AH{i}) + 6, IFERROR(FIND(";", AH{i}), LEN(AH{i})+1) - FIND("Valor ", AH{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 4, value=f7)
            f8 = f'=IFERROR(TRIM(MID(AH{i}, FIND("Fecha ", AH{i}) + 6, FIND(" Valor", AH{i}) - FIND("Fecha ", AH{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 5, value=f8)

        idx_col_cedula_cj = column_index_from_string("CJ")
        letra_ev_nro_contrato = get_column_letter(col_inicio_eval + 12) 
        letra_aux_nro_contrato = get_column_letter(col_inicio_eval + 13) 
        letra_ev_nro_cedula = get_column_letter(col_inicio_eval + 14) 
        
        localidad_actual_norm = normalizar_texto_localidad(nombre_base)
        
        for i in range(2, ultima_fila_analisis + 1):
            cedula_actual_raw = ws_analisis.cell(row=i, column=idx_col_cedula_cj).value
            cedula_busqueda = normalizar_cedula(cedula_actual_raw)
            
            lista_rps = dict_rp.get(cedula_busqueda, [])
            rp_data_elegido = ("", "", "", "", "")
            
            if len(lista_rps) == 1:
                rp_data_elegido = lista_rps[0][:5]
            elif len(lista_rps) > 1:
                for rp in lista_rps:
                    localidad_rp_norm = normalizar_texto_localidad(rp[5]) 
                    if fuzz.partial_ratio(localidad_actual_norm.lower(), localidad_rp_norm.lower()) > 85:
                        rp_data_elegido = rp[:5]
                        break

            rp_num, rp_val, rp_fec, cdp_num_from_rp, cdp_cod_from_rp = rp_data_elegido
            cdp_data = dict_cdp.get((cdp_num_from_rp, cdp_cod_from_rp), ("", "", ""))
            cdp_num_out, cdp_val_out, cdp_fec_out = cdp_data

            def inyectar_condicional(valor):
                val_str = str(valor).replace('"', '""') if valor else ""
                return f'=IF(OR({letra_ev_nro_contrato}{i}="Numero contrato coincide", {letra_ev_nro_contrato}{i}="Contrato coincide sin SIPSE", {letra_ev_nro_cedula}{i}="Cedula coincide"), "{val_str}", "")'

            ws_analisis.cell(row=i, column=col_inicio_eval + 6, value=inyectar_condicional(rp_num))
            ws_analisis.cell(row=i, column=col_inicio_eval + 7, value=inyectar_condicional(rp_val))
            ws_analisis.cell(row=i, column=col_inicio_eval + 8, value=inyectar_condicional(rp_fec))
            ws_analisis.cell(row=i, column=col_inicio_eval + 9, value=inyectar_condicional(cdp_num_out))
            ws_analisis.cell(row=i, column=col_inicio_eval + 10, value=inyectar_condicional(cdp_val_out))
            ws_analisis.cell(row=i, column=col_inicio_eval + 11, value=inyectar_condicional(cdp_fec_out))

        for i in range(2, ultima_fila_analisis + 1):
            COL_SOLICITUD = "B"
            f15 = f'''=IF(COUNTIF(${letra_aux_nro_contrato}$2:{letra_aux_nro_contrato}{i},{letra_aux_nro_contrato}{i})>1,"",
                IF(AND(D{i}<>"",EV{i}<>""),
                    IF(EW{i}="","Contrato coincide sin SIPSE",
                        IF(TRIM(SUBSTITUTE(UPPER({COL_SOLICITUD}{i}),"SIPSE",""))=TRIM(SUBSTITUTE(UPPER(EW{i}),"SIPSE","")),"Numero contrato coincide","Numero contrato NO coincide")
                    ),
                    IF(D{i}<>"","Contrato no existe en SECOP",
                        IF(EV{i}<>"","Contrato no existe en SIPSE","No existe")
                    )
                )
            )'''
            f15 = f15.replace('\n', '').replace('    ', '')
            ws_analisis.cell(row=i, column=col_inicio_eval + 12, value=f15)

            f16 = f'=IF(D{i}<>"",D{i},EV{i})'
            ws_analisis.cell(row=i, column=col_inicio_eval + 13, value=f16)

            f17 = f'=IF(AND(BY{i}="",M{i}=""),"",IF(OR(BY{i}="",M{i}=""),"",IF(BY{i}="-","Sin cedula en SECOP",IF(M{i}="-","Sin cedula en SIPSE",IF(VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(CJ{i}),"CC",""),"NIT",""),"TI",""))=VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(M{i}),"CC",""),"NIT",""),"TI","")),"Cedula coincide","Cedula NO coincide")))))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 14, value=f17)

            f18 = f'=IF({letra_ev_nro_contrato}{i}="","",COUNTIF(D$2:D${ultima_fila_analisis},D{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 15, value=f18)

            f19 = f'=IF({letra_ev_nro_contrato}{i}="","",COUNTIF(EV$2:EV${ultima_fila_analisis},EV{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 16, value=f19)

            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CE{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CE{i}), DAY(CE{i}), MONTH(CE{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f20 = f'''=IF(AND(H{i}="",CE{i}=""),"",IF(AND(H{i}="-",CE{i}="-"),"Ambos sin fechas",IF(H{i}="-","sin fecha SIPSE",IF(CE{i}="-","sin fecha SECOP",IF(OR(H{i}="",CE{i}=""),"",IF(INT(VALUE(SUBSTITUTE(H{i},"-","/"))) =INT(IF(ISNUMBER(CE{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 17, value=f20)

            # 🔹 FECHA ACTA DE INICIO
            cf_clean = f'SUBSTITUTE(SUBSTITUTE(CF{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CF{i}), DAY(CF{i}), MONTH(CF{i}))'
            parte_es_texto = f'DATE(RIGHT({cf_clean}, 4), LEFT({cf_clean}, FIND("/", {cf_clean}) - 1), MID({cf_clean}, FIND("/", {cf_clean}) + 1, FIND("/", {cf_clean}, FIND("/", {cf_clean}) + 1) - FIND("/", {cf_clean}) - 1))'

            f21 = f'''=IF(AND(I{i}="",CF{i}=""),"",IF(AND(I{i}="-",CF{i}="-"),"Ambos sin fechas",IF(I{i}="-","sin fecha SIPSE",IF(CF{i}="-","sin fecha SECOP",IF(OR(I{i}="",CF{i}=""),"",IF(INT(VALUE(SUBSTITUTE(I{i},"-","/"))) =INT(IF(ISNUMBER(CF{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 18, value=f21)

            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CG{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CG{i}), DAY(CG{i}), MONTH(CG{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f22 = f'''=IF(AND(J{i}="",CG{i}=""),"",IF(AND(J{i}="-",CG{i}="-"),"Ambos sin fechas",IF(J{i}="-","sin fecha SIPSE",IF(CG{i}="-","sin fecha SECOP",IF(OR(J{i}="",CG{i}=""),"",IF(INT(VALUE(SUBSTITUTE(J{i},"-","/"))) =INT(IF(ISNUMBER(CG{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 19, value=f22)

            f23 = f'''=IF(AND(R{i}="",EE{i}=""),"",IF(AND(R{i}="-",EE{i}="-"),"Ambos sin duracion",IF(R{i}="-","sin duracion en SECOP",IF(EE{i}="-","sin duracion en SIPSE",IF(OR(R{i}="",EE{i}=""),"",IF(TRIM(UPPER(R{i}))=TRIM(UPPER(EE{i})),"Duracion coincide","Duracion NO coincide"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 20, value=f23)

            f24 = f'''=IF(AND(P{i}="",CU{i}=""),"",
                IF(AND(P{i}="-",CU{i}="-"),"Ambos sin valor",
                IF(P{i}="-","Sin valor en SECOP",
                IF(CU{i}="-","Sin valor en SIPSE",
                IF(OR(P{i}="",CU{i}=""),"",
                IF(P{i}=CU{i},"Valor coincide","Valor NO coincide"))))))'''

            ws_analisis.cell(row=i, column=col_inicio_eval + 21, value=f24)

            c_rp_sip   = get_column_letter(col_inicio_eval)       
            c_val_rp_s = get_column_letter(col_inicio_eval + 1)   
            c_fec_rp_s = get_column_letter(col_inicio_eval + 2)   
            c_cdp_sip  = get_column_letter(col_inicio_eval + 3)   
            c_val_cd_s = get_column_letter(col_inicio_eval + 4)   
            c_fec_cd_s = get_column_letter(col_inicio_eval + 5)   
            
            c_rp_bog   = get_column_letter(col_inicio_eval + 6)   
            c_val_rp_b = get_column_letter(col_inicio_eval + 7)   
            c_fec_rp_b = get_column_letter(col_inicio_eval + 8)   
            c_cdp_bog  = get_column_letter(col_inicio_eval + 9)   
            c_val_cd_b = get_column_letter(col_inicio_eval + 10)  
            c_fec_cd_b = get_column_letter(col_inicio_eval + 11)  

            f25 = (f'=IF(AND(TRIM({c_rp_sip}{i})="", TRIM({c_rp_bog}{i})=""), "Sin numeros RP", '
                   f'IF(TRIM({c_rp_sip}{i})="", "Sin numero RP en SIPSE", '
                   f'IF(TRIM({c_rp_bog}{i})="", "Sin numero RP en BOGDATA", '
                   f'IF(TRIM({c_rp_sip}{i})=TRIM({c_rp_bog}{i}), "El numero RP coincide", "El numero RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 22, value=f25)

            f26 = (f'=IF(AND(TRIM({c_val_rp_s}{i})="", TRIM({c_val_rp_b}{i})=""), "Sin valores RP", '
                   f'IF(TRIM({c_val_rp_s}{i})="", "Sin valor RP en SIPSE", '
                   f'IF(TRIM({c_val_rp_b}{i})="", "Sin valor RP en BOGDATA", '
                   f'IF(IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_rp_s}{i}), ",", "")),TRIM({c_val_rp_s}{i}))=IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_rp_b}{i}), ",", "")),TRIM({c_val_rp_b}{i})), "El valor RP coincide", "El valor RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 23, value=f26)

            f27 = (f'=IF(AND(TRIM({c_fec_rp_s}{i})="", TRIM({c_fec_rp_b}{i})=""), "Sin fechas RP", '
                   f'IF(TRIM({c_fec_rp_s}{i})="", "Sin fecha RP en SIPSE", '
                   f'IF(TRIM({c_fec_rp_b}{i})="", "Sin fecha RP en BOGDATA", '
                   f'IF(TRIM({c_fec_rp_s}{i})=TRIM({c_fec_rp_b}{i}), "La fecha RP coincide", "La fecha RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 24, value=f27)

            f28 = (f'=IF(AND(TRIM({c_cdp_sip}{i})="", TRIM({c_cdp_bog}{i})=""), "Sin numeros CDP", '
                   f'IF(TRIM({c_cdp_sip}{i})="", "Sin numero CDP en SIPSE", '
                   f'IF(TRIM({c_cdp_bog}{i})="", "Sin numero CDP en BOGDATA", '
                   f'IF(TRIM({c_cdp_sip}{i})=TRIM({c_cdp_bog}{i}), "El numero CDP coincide", "El numero CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 25, value=f28)

            f29 = (f'=IF(AND(TRIM({c_val_cd_s}{i})="", TRIM({c_val_cd_b}{i})=""), "Sin valores CDP", '
                   f'IF(TRIM({c_val_cd_s}{i})="", "Sin valor CDP en SIPSE", '
                   f'IF(TRIM({c_val_cd_b}{i})="", "Sin valor CDP en BOGDATA", '
                   f'IF(IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_cd_s}{i}), ",", "")),TRIM({c_val_cd_s}{i}))=IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_cd_b}{i}), ",", "")),TRIM({c_val_cd_b}{i})), "El valor CDP coincide", "El valor CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 26, value=f29)

            f30 = (f'=IF(AND(TRIM({c_fec_cd_s}{i})="", TRIM({c_fec_cd_b}{i})=""), "Sin fechas CDP", '
                   f'IF(TRIM({c_fec_cd_s}{i})="", "Sin fecha CDP en SIPSE", '
                   f'IF(TRIM({c_fec_cd_b}{i})="", "Sin fecha CDP en BOGDATA", '
                   f'IF(TRIM({c_fec_cd_s}{i})=TRIM({c_fec_cd_b}{i}), "La fecha CDP coincide", "La fecha CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 27, value=f30)

            # 🔹 Indicador 1 de Completitud
            f31 = f'=IF(AND(FP{i}="Coincide fecha", FL{i}="Cedula coincide"), "Completo", "No Completo")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 28, value=f31)

            # 🔹 Indicador 2 de Exactitud
            f32 = f'=IF(AND(FS{i}="Valor coincide", FR{i}="Duracion coincide", OR(FJ{i}="Numero contrato coincide", FJ{i}="Contrato coincide sin SIPSE"), FP{i}="Coincide fecha", FL{i}="Cedula coincide"), "Exacto", "No Exacto")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 29, value=f32)

        print("  -> Aplicando estilos y ocultando columnas en Analisis...")
        aplicar_estilo_ferrocarril(ws_analisis)
        aplicar_formato_condicional_alertas(ws_analisis, col_inicio_eval)
        ocultar_columnas_por_lista(ws_analisis, RANGOS_OCULTAR_ANALISIS)

        # ==========================================
        # PARTE 5: Crear HOJA RESULTADOS
        # ==========================================
        print("  -> Creando hoja 'Resultados'...")
        if 'Resultados' in wb.sheetnames:
            del wb['Resultados']
        ws_res = wb.create_sheet("Resultados")
        nombre_limpio = os.path.splitext(archivo)[0]
        ws_res['B1'] = f"Métricas alcaldía local {nombre_limpio}"
        
        letra_ev_nro = get_column_letter(col_inicio_eval + 12)       
        letra_ev_ced = get_column_letter(col_inicio_eval + 14)   
        letra_rep_sipse = get_column_letter(col_inicio_eval + 15)
        letra_rep_secop = get_column_letter(col_inicio_eval + 16)
        letra_fecha_suscripcion = get_column_letter(col_inicio_eval + 17)
        letra_fecha_inicio = get_column_letter(col_inicio_eval + 18)
        letra_fecha_fin = get_column_letter(col_inicio_eval + 19)
        letra_duracion = get_column_letter(col_inicio_eval + 20)
        letra_valor = get_column_letter(col_inicio_eval + 21)

        letra_num_RP = get_column_letter(col_inicio_eval + 22)
        letra_val_RP = get_column_letter(col_inicio_eval + 23)
        letra_fec_RP = get_column_letter(col_inicio_eval + 24)
        letra_num_CDP = get_column_letter(col_inicio_eval + 25)
        letra_val_CDP = get_column_letter(col_inicio_eval + 26)
        letra_fec_CDP = get_column_letter(col_inicio_eval + 27)

        letra_ind_1 = get_column_letter(col_inicio_eval + 28)
        letra_ind_2 = get_column_letter(col_inicio_eval + 29)

        metricas = [
            ("Nro de contratos registrados SIPSE",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos registrados SECOP",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos registrados en SECOP y SIPSE con numero de contrato igual", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")"),
            ("Nro de contratos solo en SECOP y no en SIPSE", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos solo en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos diferentes en SIPSE-SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato NO coincide\")"),
            ("Nro de contratos Cédula diferente", f"=COUNTIF('Analisis'!{letra_ev_ced}:{letra_ev_ced}, \"Cedula NO coincide\")"),
            ("Nro de contratos dobles en SIPSE", f"=COUNTIF('Analisis'!{letra_rep_sipse}:{letra_rep_sipse}, 2)"),
            ("Nro de contratos dobles en SECOP", f"=COUNTIF('Analisis'!{letra_rep_secop}:{letra_rep_secop}, 2)"),
            ("Fecha de acta de suscriccpión Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"Coincide fecha\")"),
            ("Fecha de acta de suscriccpión no Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"No coinciden fechas\")"),
            ("Fecha de acta de inicio Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Coincide fecha\")"),
            ("Fecha de acta de inicio no coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"No coinciden fechas\")"),
            ("Fecha de Fin conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"Coincide fecha\")"),
            ("Fecha de Fin no conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"No coinciden fechas\")"),
            ("Contratos que coinciden en Duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion Coincide\")"),
            ("Contratos que tienen diferente duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion NO coincide\")"),
            ("Contratos que tienen Fecha de inicio en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Sin fecha SECOP\")"),
            ("Datos en la matriz de seguimiento", f"=C2+C3"),
            ("Valor contrato coinciden", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor coincide\")"),
            ("Valor contrato no coincidenn", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor NO coincide\")"),
            ("Numero RP no coinciden", f"=COUNTIF('Analisis'!{letra_num_RP}:{letra_num_RP}, \"El numero RP NO coincide\")"),
            ("Valor RP no coinciden", f"=COUNTIF('Analisis'!{letra_val_RP}:{letra_val_RP}, \"El valor RP NO coincide\")"),
            ("Fecha RP no coinciden", f"=COUNTIF('Analisis'!{letra_fec_RP}:{letra_fec_RP}, \"La fecha RP NO coincide\")"),
            ("Numero CDP no coinciden", f"=COUNTIF('Analisis'!{letra_num_CDP}:{letra_num_CDP}, \"El numero CDP NO coincide\")"),
            ("Valor CDP no coinciden", f"=COUNTIF('Analisis'!{letra_val_CDP}:{letra_val_CDP}, \"El valor CDP NO coincide\")"),
            ("Fecha CDP no coinciden", f"=COUNTIF('Analisis'!{letra_fec_CDP}:{letra_fec_CDP}, \"La fecha CDP NO coincide\")"),
            ("Indicador completitud", f"=COUNTIF('Analisis'!{letra_ind_1}:{letra_ind_1}, \"Completo\") / (COUNTIF('Analisis'!{letra_ind_1}:{letra_ind_1}, \"Completo\") + COUNTIF('Analisis'!{letra_ind_1}:{letra_ind_1}, \"Incompleto\"))"),
            ("Indicador exactitud", f"=COUNTIF('Analisis'!{letra_ind_2}:{letra_ind_2}, \"Exacto\") / (COUNTIF('Analisis'!{letra_ind_2}:{letra_ind_2}, \"Exacto\") + COUNTIF('Analisis'!{letra_ind_2}:{letra_ind_2}, \"No Exacto\"))")
        ]

        for idx, (texto, formula) in enumerate(metricas, start=2):
            ws_res.cell(row=idx, column=1, value=idx-1)
            ws_res.cell(row=idx, column=2, value=texto)
            ws_res.cell(row=idx, column=3, value=formula)
        
        fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
        ws_res['B1'].fill = fill_amarillo
        ws_res.column_dimensions['B'].width = 100
        
        ws_analisis.column_dimensions['FJ'].width = 29
        ws_analisis.column_dimensions['FK'].width = 10
        ws_analisis.column_dimensions['FL'].width = 20
        ws_analisis.column_dimensions['FM'].width = 15
        ws_analisis.column_dimensions['FN'].width = 15
        ws_analisis.column_dimensions['FO'].width = 15
        ws_analisis.column_dimensions['FP'].width = 20
        ws_analisis.column_dimensions['FQ'].width = 20
        ws_analisis.column_dimensions['FR'].width = 18
        ws_analisis.column_dimensions['FS'].width = 14
        ws_analisis.column_dimensions['FT'].width = 25
        ws_analisis.column_dimensions['FU'].width = 25
        ws_analisis.column_dimensions['FV'].width = 25
        ws_analisis.column_dimensions['FW'].width = 25
        ws_analisis.column_dimensions['FX'].width = 25
        ws_analisis.column_dimensions['FY'].width = 25

        dejar_columna_blanca(ws_analisis, 64)

        ajustar_texto_fila(ws_secop_aux, 1)
        ajustar_texto_fila(ws_analisis, 1)

        # ==========================================
        # PARTE 6: Crear HOJA 'Para MEMO'
        # ==========================================
        print("  -> Creando hoja 'Para MEMO'...")
        if 'Para MEMO' in wb.sheetnames:
            del wb['Para MEMO']
        ws_memo = wb.create_sheet("Para MEMO")
        
        datos_memo = []
        
        for i in range(2, ultima_fila_analisis + 1):
            val_b = ws_analisis.cell(row=i, column=2).value 
            val_d = ws_analisis.cell(row=i, column=4).value 
            
            susc_sipse = ws_analisis.cell(row=i, column=8).value  
            susc_secop = ws_analisis.cell(row=i, column=83).value 
            ini_sipse  = ws_analisis.cell(row=i, column=9).value  
            ini_secop  = ws_analisis.cell(row=i, column=84).value 
            fin_sipse  = ws_analisis.cell(row=i, column=10).value 
            fin_secop  = ws_analisis.cell(row=i, column=85).value 
            dur_sipse  = ws_analisis.cell(row=i, column=18).value 
            dur_secop  = ws_analisis.cell(row=i, column=135).value
            
            err_susc = es_error_fecha(susc_sipse, susc_secop)
            err_ini  = es_error_fecha(ini_sipse, ini_secop)
            err_fin  = es_error_fecha(fin_sipse, fin_secop)
            err_dur  = es_error_duracion(dur_sipse, dur_secop)
            
            if err_susc or err_ini or err_fin or err_dur:
                llave_orden = (err_susc, err_ini, err_fin, err_dur)
                datos_memo.append([
                    val_b, val_d,
                    "X" if err_susc else "",
                    "X" if err_ini else "",
                    "X" if err_fin else "",
                    "X" if err_dur else "",
                    llave_orden 
                ])
                
        datos_memo.sort(key=lambda x: x[6], reverse=True)
        
        relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
        fuente_blanca = Font(color="FFFFFF", bold=True)
        ws_memo.merge_cells("A1:A2")
        ws_memo['A1'] = "SIPSE"
        ws_memo.merge_cells("B1:B2")
        ws_memo['B1'] = "No. Contrato"
        ws_memo.merge_cells("C1:F1")
        ws_memo['C1'] = "TIPOLOGÍA DE LA INCONSISTENCIA"
        
        ws_memo['C2'] = "Fecha suscripción"
        ws_memo['D2'] = "Fecha Inicio"
        ws_memo['E2'] = "Fecha Fin"
        ws_memo['F2'] = "Plazo"
        ws_memo['A1'].fill = relleno_rojo
        ws_memo['B1'].fill = relleno_rojo
        ws_memo['C1'].fill = relleno_rojo
        ws_memo['C2'].fill = relleno_rojo
        ws_memo['D2'].fill = relleno_rojo
        ws_memo['E2'].fill = relleno_rojo
        ws_memo['F2'].fill = relleno_rojo
        
        align_header = Alignment(horizontal="center", vertical="center")
        borde_delgado = Border(left=Side(style='thin', color='000000'), right=Side(style='thin', color='000000'), top=Side(style='thin', color='000000'), bottom=Side(style='thin', color='000000'))
        
        for fila_h in ws_memo['A1:F2']:
            for celda in fila_h:
                celda.font = fuente_blanca
                celda.alignment = align_header
                celda.border = borde_delgado
                
        fila_actual = 3
        for fila_datos in datos_memo:
            for col_idx in range(1, 7):
                celda = ws_memo.cell(row=fila_actual, column=col_idx, value=fila_datos[col_idx-1])
                celda.border = borde_delgado
                celda.alignment = align_header
                
            fila_actual += 1
            
        ws_memo.column_dimensions['A'].width = 15
        ws_memo.column_dimensions['B'].width = 15
        ws_memo.column_dimensions['C'].width = 20
        ws_memo.column_dimensions['D'].width = 15
        ws_memo.column_dimensions['E'].width = 15
        ws_memo.column_dimensions['F'].width = 15

        wb.save(ruta_dest)
        print("  -> Procesado exitosamente.")


# =======================================================
# EJECUCIÓN DE LA CELDA 2
# =======================================================
# Le pasamos a la función los diccionarios que ya quedaron cargados
# en la memoria de Jupyter desde la Celda 1.
procesar_archivos(dict_cdp_global, dict_rp_global)

Se encontraron 1 archivos de Excel.
Procesando: CANDELARIA.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 'Para MEMO'...
  -> Procesado exitosamente.


In [ ]:
import os
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
import re
import unicodedata

# ================= CONFIGURACIÓN =================
#CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1' 
#ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7 - FDL 01 FEBRERO\DiagnóstivoV2_1\Consolidado\Consolidado_Localidades.xlsx"
CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7\Diagnóstico' # Dell
ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx" #Dell
#CARPETA_ORIGEN = r'D:\\Años\\2026\\SecretariaDeGobierno\\ConsolidadoAlcaldias\\META 7\\Diagnóstico' #Predator
#ARCHIVO_DESTINO = r"D:\\Años\\2026\\SecretariaDeGobierno\\ConsolidadoAlcaldias\\META 7\\Diagnóstico\\Consolidado\\Consolidado_Localidades.xlsx" #Predator

nombres_hojas = [
    "1-Usaquén", "2-Chapinero", "3-Santa Fe", "4-San Cristóbal",
    "5-Usme", "6-Tunjuelito", "7-Bosa", "8-Kennedy", "9-Fontibón",
    "10-Engativá", "11-Suba", "12-Barrios Unidos", "13-Teusaquillo",
    "14-Los Mártires", "15-Antonio Nariño", "16-Puente Aranda",
    "17-Rafael Uribe", "18-La Candelaria", "19-Ciudad Bolívar", "20-Sumapaz"
]

#RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']
# Aquí puedes añadir todas las columnas a las que quieras fijarles un ancho específico
ANCHOS_COLUMNAS = {
    'FM': 29,
    'FN': 10,
    'FO': 20,
    'FP': 15,
    'FQ': 15,
    'FR': 15,
    'FS': 20,
    'FT': 20,
    'FU': 18,
    'FV': 14,
    'FW': 25,
    'FX': 25,
    'FY': 25,
    'FZ': 25,
    'GA': 25,
    'GB': 25
}

# ================= FUNCIONES AUXILIARES =================
def normalizar_texto(texto):
    texto = texto.lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^\d+[-_ ]?', '', texto)
    texto = re.sub(r'\b(la|el|los|las)\b', '', texto).strip()
    return texto

def encontrar_hoja_destino(nombre_archivo, hojas_destino):
    archivo_norm = normalizar_texto(nombre_archivo)
    for hoja in hojas_destino:
        if normalizar_texto(hoja) in archivo_norm:
            return hoja
    return None

def ocultar_columnas_por_lista(ws, lista_rangos, offset=3):
    for item in lista_rangos:
        item = item.upper().strip()
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio) + offset
            idx_fin = column_index_from_string(fin) + offset
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                ws.column_dimensions[letra].hidden = True 
        else:
            idx = column_index_from_string(item) + offset
            letra = get_column_letter(idx)
            ws.column_dimensions[letra].hidden = True

def adaptar_formulas_resultados(formula):
    pattern = re.compile(r"'?Analisis'?!([A-Za-z]+)(\d*)(?::([A-Za-z]+)(\d*))?")
    def replacer(match):
        col1, row1, col2, row2 = match.groups()
        nueva_col1 = get_column_letter(column_index_from_string(col1) + 3)
        nueva_row1 = str(int(row1) + 21) if row1 else ""
        resultado = f"{nueva_col1}{nueva_row1}"
        if col2:
            nueva_col2 = get_column_letter(column_index_from_string(col2) + 3)
            nueva_row2 = str(int(row2) + 21) if row2 else ""
            resultado += f":{nueva_col2}{nueva_row2}"
        return resultado
    return pattern.sub(replacer, formula)

def desplazar_formulas_analisis(formula):
    formula = re.sub(r"'?Analisis'?!", "", formula, flags=re.IGNORECASE)
    pattern = re.compile(r'(\$?)([A-Za-z]{1,3})(\$?)(\d+)(?!\s*\()')
    def replacer(match):
        lock_col, col_str, lock_row, row_str = match.groups()
        nueva_col = get_column_letter(column_index_from_string(col_str) + 3)
        nueva_fila = str(int(row_str) + len(metricas)+2)
        return f"{lock_col}{nueva_col}{lock_row}{nueva_fila}"
    return pattern.sub(replacer, formula)

def ajustar_texto_fila(ws, numero_fila):
    """
    Activa la propiedad 'Ajustar texto' (Wrap Text) para toda la fila indicada.
    Mantiene la alineación centrada.
    """
    for cell in ws[numero_fila]:
        # Creamos una nueva alineación conservando centro pero activando wrap_text
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# ================= FUNCIONES DE DISEÑO =================
def aplicar_estilo_ferrocarril(ws, start_row, max_row):
    """Aplica formato estilo ferrocarril solo a columnas E:U y CA:FH desde start_row."""
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    
    # Rango 1: E (5) a AL (38) | Rango 2: CA (79) a GB (185)
    def es_columna_objetivo(c):
        return (5 <= c <= 38) or (79 <= c <= 185)

    # 1. Encabezados (Fila 22)
    for col_idx in range(1, ws.max_column + 1):
        if es_columna_objetivo(col_idx):
            cell = ws.cell(row=start_row, column=col_idx)
            cell.fill = fill_header
            cell.font = font_header
            cell.alignment = Alignment(horizontal='center', vertical='center')

    # 2. Filas de Datos (Patrón Ferrocarril)
    for i_fila, r_idx in enumerate(range(start_row + 1, max_row + 1)):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF": # Evitar procesar blanco para ahorrar memoria
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for col_idx in range(1, ws.max_column + 1):
                if es_columna_objetivo(col_idx):
                    ws.cell(row=r_idx, column=col_idx).fill = relleno

def aplicar_formato_condicional_alertas(ws, start_row, max_row):
    """Aplica reglas condicionales desde la columna EZ."""
    col_inicio_eval = column_index_from_string('FA')
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    
    reglas = [
        (12, 'equal', '"Numero contrato NO coincide"'),
        (14, 'equal', '"Cedula NO coincide"'),
        (15, 'greaterThan', '1'),
        (16, 'greaterThan', '1'),
        (17, 'equal', '"No coinciden fechas"'),
        (18, 'equal', '"No coinciden fechas"'),
        (19, 'equal', '"No coinciden fechas"'),
        (20, 'equal', '"Duracion NO coincide"'),
        (21, 'equal', '"Valor NO coincide"'),
        (22, 'equal', '"El numero RP NO coincide"'),
        (23, 'equal', '"El valor RP NO coincide"'),
        (24, 'equal', '"La fecha RP NO coincide"'),
        (25, 'equal', '"El numero CDP NO coincide"'),
        (26, 'equal', '"El valor CDP NO coincide"'),
        (27, 'equal', '"La fecha CDP NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        # Se aplica desde la fila siguiente al encabezado (23 en adelante)
        rango = f"{letra_col}{start_row + 1}:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

# ================= PROGRAMA PRINCIPAL =================
def consolidar_excels():
    global metricas
    wb_nuevo = openpyxl.Workbook()
    
    # 1. Configurar Hoja General
    ws_general = wb_nuevo.active
    ws_general.title = "General"
    ws_general.cell(row=1, column=1, value="Nro").font = Font(bold=True)
    ws_general.cell(row=1, column=2, value="Alcaldía local").font = Font(bold=True)
    
    # Pre-llenar las localidades en General con Hipervínculos
    for i, hoja in enumerate(nombres_hojas):
        num, nombre = hoja.split("-", 1)
        row_idx = i + 2
        ws_general.cell(row=row_idx, column=1, value=int(num))
        
        celda_loc = ws_general.cell(row=row_idx, column=2, value=nombre.strip())
        celda_loc.hyperlink = f"#'{hoja}'!A1"
        celda_loc.font = Font(color="0563C1", underline="single") 

    for nombre in nombres_hojas:
        wb_nuevo.create_sheet(title=nombre)

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith('.xlsx') and not f.startswith('~')]
    metricas_extraidas = False 

    for archivo in archivos:
        hoja_destino_nombre = encontrar_hoja_destino(archivo, nombres_hojas)
        
        if not hoja_destino_nombre:
            continue
            
        print(f"-> Procesando '{archivo}' -> Destino: '{hoja_destino_nombre}'")
        
        ruta_completa = os.path.join(CARPETA_ORIGEN, archivo)
        wb_origen = openpyxl.load_workbook(ruta_completa)
        ws_destino = wb_nuevo[hoja_destino_nombre]

        # --- TAREA A: Copiar de 'Resultados' ---
        if "Resultados" in wb_origen.sheetnames:
            ws_resultados = wb_origen["Resultados"]
            row_gen = nombres_hojas.index(hoja_destino_nombre) + 2
            col_gen = 3 
            
            for r_idx in range(0, len(metricas)): 
                titulo_metrica = ws_resultados.cell(row=2 + r_idx, column=2).value 
                
                if not metricas_extraidas:
                    ws_general.cell(row=1, column=col_gen, value=titulo_metrica).font = Font(bold=True)
                
                ws_general.cell(row=row_gen, column=col_gen, value=f"='{hoja_destino_nombre}'!C{r_idx+2}")
                col_gen += 1
                
            metricas_extraidas = True 
            
            for row in ws_resultados['A1':f'C{len(metricas) + 1}']:
                for cell in row:
                    dest_coord = cell.coordinate
                    valor = cell.value
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = adaptar_formulas_resultados(valor)
                    else:
                        ws_destino[dest_coord].value = valor
        else:
            print(f"  - Faltó hoja 'Resultados' en {archivo}")

        # --- TAREA B: Copiar de 'Analisis' a la fila 22 ---
        if "Analisis" in wb_origen.sheetnames:
            ws_analisis = wb_origen["Analisis"]
            ultima_fila = ws_analisis.max_row
            max_col_origen = min(ws_analisis.max_column, column_index_from_string('FY'))
            
            ws_destino.merge_cells(f"D{len(metricas) + 2}:BN{len(metricas) + 2}")
            celda_sipse = ws_destino[f"D{len(metricas) + 2}"]
            celda_sipse.value = "SIPSE"
            celda_sipse.fill = PatternFill(start_color="0070C0", end_color="0070C0", fill_type="solid") 
            celda_sipse.font = Font(color="FFFFFF", bold=True)
            celda_sipse.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells(f"BP{len(metricas) + 2}:EX{len(metricas) + 2}")
            celda_secop = ws_destino[f"BP{len(metricas) + 2}"]
            celda_secop.value = "SECOP"
            celda_secop.fill = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid") 
            celda_secop.font = Font(color="FFFFFF", bold=True)
            celda_secop.alignment = Alignment(horizontal="center", vertical="center")

            ws_destino.merge_cells(f"EY{len(metricas) + 2}:GB{len(metricas) + 2}")
            celda_comparacion = ws_destino[f"EY{len(metricas) + 2}"]
            celda_comparacion.value = "COMPARACIÓN"
            celda_comparacion.fill = PatternFill(start_color="7e350e", end_color="7e350e", fill_type="solid") 
            celda_comparacion.font = Font(color="FFFFFF", bold=True)
            celda_comparacion.alignment = Alignment(horizontal="center", vertical="center")

            # Pegar los datos desde la fila 22
            for r_idx, row_origen in enumerate(ws_analisis.iter_rows(min_row=1, max_row=ultima_fila, min_col=1, max_col=max_col_origen), start=len(metricas) + 3):
                for c_idx, cell in enumerate(row_origen, start=4):
                    dest_coord = f"{get_column_letter(c_idx)}{r_idx}"
                    valor = cell.value
                    
                    if isinstance(valor, str) and valor.startswith('='):
                        ws_destino[dest_coord].value = desplazar_formulas_analisis(valor)
                    else:
                        ws_destino[dest_coord].value = valor
            
            # 1. Ocultar columnas
            ocultar_columnas_por_lista(ws_destino, RANGOS_OCULTAR_ANALISIS, offset=3)
            # 2. Aplicar anchos de columnas
            for letra_col, ancho in ANCHOS_COLUMNAS.items():
                ws_destino.column_dimensions[letra_col].width = ancho
            # 2.1 Ajustar el alto del texto en la fila de encabezados de las hojas de cada localidad
            ajustar_texto_fila(ws_destino, len(metricas) + 3)
            # 3. Aplicar estilo Ferrocarril (La tabla empieza en la 22)
            aplicar_estilo_ferrocarril(ws_destino, start_row=len(metricas) + 3, max_row=ultima_fila + len(metricas) + 2)
            # 4. Aplicar alertas condicionales
            aplicar_formato_condicional_alertas(ws_destino, start_row=len(metricas) + 3, max_row=ultima_fila + len(metricas) + 2)
            # 5 Pintar y ajustar tabla de cada alcaldía
            fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
            ws_destino['B1'].fill = fill_amarillo
            ws_destino.column_dimensions['B'].width = 100
            
            # --- NUEVO: BOTÓN DE REGRESO A GENERAL ---
            celda_volver = ws_destino['A1']
            celda_volver.value = "General"
            celda_volver.hyperlink = "#'General'!A1"
            celda_volver.font = Font(color="0563C1", underline="single", bold=True)
            celda_volver.fill = PatternFill(start_color="E7E6E6", end_color="E7E6E6", fill_type="solid") # Gris claro
            celda_volver.alignment = Alignment(horizontal="center", vertical="center")

        else:
            print(f"  - Faltó hoja 'Analisis' en {archivo}")
            
        wb_origen.close()

    # --- DISEÑO FINAL HOJA GENERAL ---
    # 1. Fondo Rojo para la fila de títulos A1:AC1
    relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
    fuente_blanca = Font(color="FFFFFF", bold=True)
    for col in range(1, 30): # De la A (1) a la AC (30)
        celda = ws_general.cell(row=1, column=col)
        celda.fill = relleno_rojo
        celda.font = fuente_blanca
        celda.alignment = Alignment(horizontal="center", vertical="center")

    # 2. Centrar los valores de C1:AC1
    alineacion_centro = Alignment(horizontal="center", vertical="center")
    for r in range(2, 22): # Filas 2 a 21
        for c in range(3, 29): # Columnas C (3) a AC (30)
            ws_general.cell(row=r, column=c).alignment = alineacion_centro
            
    # Ajustar ligeramente el ancho de General para que se vea bien
    ws_general.column_dimensions['A'].width = 5
    ws_general.column_dimensions['B'].width = 25
    for c in range(3, 22):
        ws_general.column_dimensions[get_column_letter(c)].width = 15
    
    # Ajustar el texto en la fila 1 de las hoja General
    ajustar_texto_fila(ws_general, 1)

    # --- NUEVO: APLICAR GRILLA DELGADA (A1:U21) ---
    borde_delgado = Border(
        left=Side(style='thin', color='000000'),
        right=Side(style='thin', color='000000'),
        top=Side(style='thin', color='000000'),
        bottom=Side(style='thin', color='000000')
    )
    
    for r in range(1, 22):      # Filas de la 1 a la 21
        for c in range(1, 30):  # Columnas de la A(1) a la AC(29)
            ws_general.cell(row=r, column=c).border = borde_delgado
    
    # CARPETA 'CONSOLIDADO' SI NO EXISTE ---
    carpeta_destino = os.path.dirname(ARCHIVO_DESTINO) # Extrae la ruta de la carpeta
    os.makedirs(carpeta_destino, exist_ok=True)        # Crea la carpeta (y subcarpetas si faltan) sin dar error si ya existe

    wb_nuevo.save(ARCHIVO_DESTINO)
    print(f"\n🚀 ¡Proceso completado! Archivo guardado en: {ARCHIVO_DESTINO}")

if __name__ == "__main__":
    consolidar_excels()

#### Entrega 7 Oct 2026

In [ ]:
import pandas as pd
import unicodedata
import re
import difflib
import os
import tkinter as tk
from tkinter import filedialog, messagebox

# ==========================================
# 0. DEFINICIÓN DE ESTRUCTURA ESPERADA
# ==========================================
# Reemplaza los elementos de estas listas con los nombres EXACTOS 
# y en el ORDEN que necesitas para cada hoja.
COLUMNAS_ESPERADAS_SIPSE = [
    "CONTRATO", "SOLICITUD", "VIGENCIA", "NUMERO_CONTRATO", "TIPO", "TIPO_PROCESO", "OBJETO", "FECHA_SUSCRIPCION", "FECHA_INICIO", "FECHA_TERMINACION", "TERMINA31_DICIEMBRE", "CONTRATISTA", "DOC_CONTRATISTA", "VALOR_INICIAL", "ADICIONES", "VALOR_TOTAL", "VALOR_MENSUAL", "PLAZO", "PRORROGAS", "PROYECTO", "DEPENDENCIA", "SUPERVISION", "ESTADO", "NAT", "PAGOS", "AVANCE", "FECHA_ACTA_LIQUIDACION", "PRESUPUESTO", "ANULACION", "DIRECCION_CONTRATISTA", "TELEFONO_CONTRATISTA", "CODIGO_PROYECTO", "NOMBRE_PROYECTO", "CDPS", "RPS", "ESTADO_RP", "MODIFICACION", "NUMERO_OC", "TIPO_SOCIEDAD", "TIPOLOGIA", "TIPO_PAGO", "NIVELACADEMICO", "PROFEXION", "ESPECIALIZ", "POLIZA", "PROCESO_CONTRATO", "ANTIGUEDAD", "SALDO_FAVOR_ENTIDAD", "FECHA_NACIMIENTO", "OBLIGACIONES", "TIPO_IDENTIFICACION", "CORREO_ELECTRONICO", "NOMBRE", "APELLIDO", "CELULAR", "GARANTIAS", "INICIO_AMPARO", "FIN_AMPARO", "IDEN_REP_LEGAL", "NOM_REP_LEGAL", "ENTIDAD", "DESCRIPCION_ENTIDAD", "GENERO"
]

COLUMNAS_ESPERADAS_SECOP = [
    "Nombre Entidad", "Nit Entidad", "Departamento", "Ciudad", "LocalizaciÃ³n", "Orden", "Sector", "Rama", "Entidad Centralizada", "Proceso de Compra", "ID Contrato", "Referencia del Contrato", "Estado Contrato", "Codigo de Categoria Principal", "Descripcion del Proceso", "Tipo de Contrato", "Modalidad de Contratacion", "Justificacion Modalidad de Contratacion", "Fecha de Firma", "Fecha de Inicio del Contrato", "Fecha de Fin del Contrato", "Condiciones de Entrega", "TipoDocProveedor", "Documento Proveedor", "Proveedor Adjudicado", "Es Grupo", "Es Pyme", "Habilita Pago Adelantado", "LiquidaciÃ³n", "ObligaciÃ³n Ambiental", "Obligaciones Postconsumo", "Reversion", "Origen de los Recursos", "Destino Gasto", "Valor del Contrato", "Valor de pago adelantado", "Valor Facturado", "Valor Pendiente de Pago", "Valor Pagado", "Valor Amortizado", "Valor Pendiente de Amortizacion", "Valor Pendiente de Ejecucion", "Estado BPIN", "CÃ³digo BPIN", "Anno BPIN", "Saldo CDP", "Saldo Vigencia", "EsPostConflicto", "Dias adicionados", "Puntos del Acuerdo", "Pilares del Acuerdo", "URLProceso", "Nombre Representante Legal", "Nacionalidad Representante Legal", "Domicilio Representante Legal", "Tipo de IdentificaciÃ³n Representante Legal", "IdentificaciÃ³n Representante Legal", "GÃ©nero Representante Legal", "Presupuesto General de la Nacion â€“ PGN", "Sistema General de Participaciones", "Sistema General de RegalÃ­as", "Recursos Propios (AlcaldÃ­as, Gobernaciones y Resguardos IndÃ­genas)", "Recursos de Credito", "Recursos Propios", "Ultima Actualizacion", "Codigo Entidad", "Codigo Proveedor", "Fecha Inicio Liquidacion", "Fecha Fin Liquidacion", "Objeto del Contrato", "DuraciÃ³n del contrato", "Nombre del banco", "Tipo de cuenta", "NÃºmero de cuenta", "El contrato puede ser prorrogado", "Fecha de notificaciÃ³n de prorrogaciÃ³n", "Nombre ordenador del gasto", "Tipo de documento Ordenador del gasto", "NÃºmero de documento Ordenador del gasto", "Nombre supervisor", "Tipo de documento supervisor", "NÃºmero de documento supervisor", "Nombre Ordenador de Pago", "Tipo de documento Ordenador de Pago", "NÃºmero de documento Ordenador de Pago", "Documentos Tipo", "Descripcion Documentos Tipo"
]

# ==========================================
# 1. FUNCIONES DE LIMPIEZA, COMPARACIÓN Y ESTRUCTURA
# ==========================================

def normalizar_texto(texto):
    """Limpia el texto: quita nulos, espacios, pasa a mayúsculas, quita tildes y artículos"""
    if pd.isna(texto):
        return ""
    texto = str(texto).upper().strip()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^(EL|LA|LOS|LAS)\s+', '', texto)
    return texto

def unificar_localidades(lista_localidades, similitud_minima=0.85):
    """Agrupa nombres similares bajo un mismo nombre 'canónico'"""
    nombres_canonicos = {}
    for loc in lista_localidades:
        if not loc: continue
        coincidencias = difflib.get_close_matches(loc, nombres_canonicos.keys(), n=1, cutoff=similitud_minima)
        if coincidencias:
            nombres_canonicos[loc] = coincidencias[0]
        else:
            nombres_canonicos[loc] = loc
    return nombres_canonicos

def alinear_columnas(df, columnas_esperadas):
    """
    Compara las columnas del df con la lista esperada.
    Crea las faltantes en blanco y ELIMINA las que sobran.
    """
    if not columnas_esperadas:
        return df

    # 1. Crear las columnas que faltan en el DataFrame original
    for col in columnas_esperadas:
        if col not in df.columns:
            df[col] = "" 
            
    # 2. Retornar el DataFrame filtrado SOLAMENTE con las columnas esperadas y en el orden exacto
    return df[columnas_esperadas]

# ==========================================
# 2. SELECCIÓN DE ARCHIVOS Y CARPETAS
# ==========================================

root = tk.Tk()
root.withdraw()

print("Por favor, selecciona el archivo Excel de origen...")
archivo_origen = filedialog.askopenfilename(
    title="Selecciona el archivo Excel (.xlsx o .xls)",
    filetypes=[("Archivos de Excel", "*.xlsx *.xls *")]
)

if not archivo_origen:
    print("No se seleccionó ningún archivo. Cancelando proceso.")
    exit()

print("Por favor, selecciona la carpeta de destino...")
carpeta_destino = filedialog.askdirectory(title="Selecciona la carpeta de destino")

if not carpeta_destino:
    print("No se seleccionó ninguna carpeta. Cancelando proceso.")
    exit()

# ==========================================
# 3. LECTURA Y PROCESAMIENTO DEL EXCEL
# ==========================================

try:
    print(f"\nLeyendo el archivo: {archivo_origen}")
    xls = pd.ExcelFile(archivo_origen)
    nombres_hojas = xls.sheet_names
    
    hoja_sipse = next((h for h in nombres_hojas if h.upper() == "SIPSE"), None)
    hoja_secop = next((h for h in nombres_hojas if h.upper() == "SECOP II"), None)
    
    if not hoja_sipse or not hoja_secop:
        raise ValueError("El archivo no contiene las hojas 'SIPSE' y/o 'SECOP II'.")

    df_sipse = pd.read_excel(xls, sheet_name=hoja_sipse)
    df_secop = pd.read_excel(xls, sheet_name=hoja_secop)
    
    if len(df_sipse.columns) <= 61:
        raise ValueError("La hoja SIPSE no tiene suficientes columnas para llegar a la columna BJ (índice 61).")
        
    # Identificar nombres de columnas clave
    col_sipse_nombre = df_sipse.columns[61] 
    col_secop_nombre = df_secop.columns[0]

    # --- NUEVO CAMBIO: REEMPLAZO ESPECÍFICO DE RAFAEL URIBE ---
    print("Ejecutando reemplazo de nombres de entidad específicos...")
    
    # Reemplazo en SECOP II (Columna A)
    df_secop[col_secop_nombre] = df_secop[col_secop_nombre].replace("RAFAEL URIBE URIBE", "RAFAEL URIBE")
    
    # Reemplazo en SIPSE (Columna BJ / índice 61)
    df_sipse[col_sipse_nombre] = df_sipse[col_sipse_nombre].replace("RAFAEL URIBE URIBE", "RAFAEL URIBE")
    # ---------------------------------------------------------

    print("Alineando y estandarizando columnas según listas esperadas (eliminando sobrantes)...")
    df_sipse = alinear_columnas(df_sipse, COLUMNAS_ESPERADAS_SIPSE)
    df_secop = alinear_columnas(df_secop, COLUMNAS_ESPERADAS_SECOP)
    
    print("Normalizando nombres de localidades...")
    # Como ya alineamos las columnas, garantizamos que existen, pero usamos el nombre extraído previamente
    df_sipse['Loc_Limpia'] = df_sipse[col_sipse_nombre].apply(normalizar_texto)
    df_secop['Loc_Limpia'] = df_secop[col_secop_nombre].apply(normalizar_texto)
    
    todas_localidades = list(set(df_sipse['Loc_Limpia'].unique()) | set(df_secop['Loc_Limpia'].unique()))
    mapa_canonico = unificar_localidades(todas_localidades)
    
    df_sipse['Loc_Final'] = df_sipse['Loc_Limpia'].map(mapa_canonico)
    df_secop['Loc_Final'] = df_secop['Loc_Limpia'].map(mapa_canonico)
    
    localidades_finales = set(mapa_canonico.values())
    
    print(f"Se detectaron {len(localidades_finales)} localidades únicas. Generando archivos...")
    
    archivos_creados = 0
    for loc in localidades_finales:
        if not loc: continue 
        
        # Filtramos y eliminamos las dos columnas temporales (Loc_Limpia y Loc_Final) que usamos para segmentar
        filtro_sipse = df_sipse[df_sipse['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        filtro_secop = df_secop[df_secop['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        
        if not filtro_sipse.empty or not filtro_secop.empty:
            nombre_archivo_seguro = re.sub(r'[\\/*?:"<>|]', "", loc)
            ruta_salida = os.path.join(carpeta_destino, f"{nombre_archivo_seguro}.xlsx")
            
            with pd.ExcelWriter(ruta_salida, engine='openpyxl') as writer:
                filtro_sipse.to_excel(writer, sheet_name="SIPSE", index=False)
                filtro_secop.to_excel(writer, sheet_name="SECOP II", index=False)
                
            archivos_creados += 1
            print(f" -> Creado: {nombre_archivo_seguro}.xlsx")

    print(f"\n¡Proceso Finalizado Exitosamente! Se crearon {archivos_creados} archivos.")
    messagebox.showinfo("Éxito", f"Se generaron {archivos_creados} archivos correctamente únicamente con las columnas especificadas.")

except Exception as e:
    print(f"\nOcurrió un error: {e}")
    messagebox.showerror("Error", f"Ocurrió un error procesando el archivo:\n{e}")

finally:
    root.destroy()

In [ ]:
#Celda 2.1
import os
import shutil
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
from collections import defaultdict
import re
import pandas as pd
import difflib # para busqueda aproximada
from rapidfuzz import fuzz, process
import datetime

#Para abrir explorador de archivos
import tkinter as tk
from tkinter import filedialog
import math
import unicodedata

# --- CONFIGURACIÓN DE CARPETAS ---
CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7' #Dell
CARPETA_DESTINO = r"C:\Users\Pc\Desktop\META 7\Diagnóstico" #Dell

# --- RANGOS A OCULTAR ---
RANGOS_OCULTAR_SECOP = ['A-K', 'N-S', 'U-W', 'Z-AH', 'AJ-BR', 'BT-CI']
RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-AG','AJ-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']

# --- DEFINICIÓN DE FUNCIONES ---
def ajustar_texto_fila(ws, numero_fila):
    for cell in ws[numero_fila]:
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def ocultar_columnas_por_lista(ws, lista_rangos):
    for item in lista_rangos:
        item = item.upper().strip()
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio)
            idx_fin = column_index_from_string(fin)
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                ws.column_dimensions[letra].hidden = True 
        else:
            ws.column_dimensions[item].hidden = True

def aplicar_estilo_ferrocarril(ws):
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    
    for cell in ws[1]:
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal='center', vertical='center')

    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    max_col = ws.max_column
    for i_fila, row in enumerate(ws.iter_rows(min_row=2, max_col=max_col), start=0):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF":
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for cell in row:
                cell.fill = relleno

def dejar_columna_blanca(ws, indice_columna):
    relleno_blanco = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=indice_columna, max_col=indice_columna):
        for cell in row:
            cell.fill = relleno_blanco

def aplicar_formato_condicional_alertas(ws, col_inicio_eval):
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    max_row = ws.max_row
    
    reglas = [
        (12, 'equal', '"Numero contrato NO coincide"'),
        (14, 'equal', '"Cedula NO coincide"'),
        (15, 'greaterThan', '1'),
        (16, 'greaterThan', '1'),
        (17, 'equal', '"No coinciden fechas"'),
        (18, 'equal', '"No coinciden fechas"'),
        (19, 'equal', '"No coinciden fechas"'),
        (20, 'equal', '"Duracion NO coincide"'),
        (21, 'equal', '"Valor NO coincide"'),
        (22, 'equal', '"El numero RP NO coincide"'),
        (23, 'equal', '"El valor RP NO coincide"'),
        (24, 'equal', '"La fecha RP NO coincide"'),
        (25, 'equal', '"El numero CDP NO coincide"'),
        (26, 'equal', '"El valor CDP NO coincide"'),
        (27, 'equal', '"La fecha CDP NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        rango = f"{letra_col}2:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

def normalizar_numero_contrato(valor):
    if valor is None: return 0
    texto = str(valor).strip()
    match = re.search(r'\d+', texto)
    if match:
        return int(match.group())
    return 0

def es_error_fecha(val_sipse, val_secop):
    s1 = str(val_sipse).strip() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip() if pd.notna(val_secop) and val_secop is not None else ""
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    
    dt_sipse = None
    if isinstance(val_sipse, (datetime.datetime, datetime.date)):
        dt_sipse = datetime.date(val_sipse.year, val_sipse.month, val_sipse.day)
    else:
        try: dt_sipse = pd.to_datetime(s1.replace("-", "/"), dayfirst=True).date()
        except: pass

    dt_secop = None
    if isinstance(val_secop, (datetime.datetime, datetime.date)):
        try: dt_secop = datetime.date(val_secop.year, val_secop.day, val_secop.month)
        except: pass
    else:
        partes = s2.replace("-", "/").replace(" ", "/").split("/")
        if len(partes) >= 3:
            try:
                y = int(partes[2][-4:])
                m = int(partes[0])
                d = int(partes[1])
                dt_secop = datetime.date(y, m, d)
            except: pass
                
    if dt_sipse is not None and dt_secop is not None:
        return dt_sipse != dt_secop
    return s1 != s2 

def es_error_duracion(val_sipse, val_secop):
    s1 = str(val_sipse).strip().upper() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip().upper() if pd.notna(val_secop) and val_secop is not None else ""
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    return s1 != s2

def color_memo(err_tuple):
    if err_tuple[0]: return "E4DFEC"
    if err_tuple[1]: return "B4C6E7"
    if err_tuple[2]: return "DDEBF7"
    if err_tuple[3]: return "F4CCCC"
    return "FFFFFF"

def normalizar_alfanumerico(valor):
    if pd.isna(valor) or valor is None: return ""
    texto = str(valor).strip().upper()
    if texto.endswith(".0"): texto = texto[:-2]
    return texto

def limpiar_dato(val):
    if pd.isna(val) or val is None: return ""
    if isinstance(val, (datetime.datetime, datetime.date)):
        return val.strftime('%d/%m/%Y')
    val_str = str(val).strip()
    if val_str.endswith(".0"): val_str = val_str[:-2]
    return val_str

def normalizar_cedula(valor):
    val_str = limpiar_dato(valor)
    return re.sub(r'\D', '', val_str)

def obtener_valor_seguro(row, indice):
    try: return limpiar_dato(row.iloc[indice])
    except IndexError: return ""

def mapear_columnas(columnas_df, columnas_buscadas, umbral=80):
    mapa_indices = {}
    for clave, nombre_ideal in columnas_buscadas.items():
        resultado = process.extractOne(nombre_ideal, columnas_df, scorer=fuzz.token_sort_ratio)
        if resultado and resultado[1] >= umbral:
            mapa_indices[clave] = resultado[2]
        else:
            print(f"⚠️ Advertencia: No se encontró la columna para '{nombre_ideal}'.")
            mapa_indices[clave] = None 
    return mapa_indices

def normalizar_texto_localidad(texto):
    if pd.isna(texto) or texto is None: return ""
    t = " ".join(str(texto).upper().split())
    t = ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')
    return t

def cargar_datos_bogdata():
    """Abre el explorador y procesa RP y CDP. Retorna los diccionarios."""
    root = tk.Tk()
    root.withdraw() 
    root.attributes('-topmost', True) 
    
    print("\n--- SELECCIÓN BOGDATA ---")
    print("1. Por favor selecciona el archivo de CDP (Excel)...")
    ruta_cdp = filedialog.askopenfilename(title="Seleccionar archivo CDP", filetypes=[("Excel files", "*.xlsx *.xls")])
    
    print("2. Por favor selecciona el archivo de RP (Excel)...")
    ruta_rp = filedialog.askopenfilename(title="Seleccionar archivo RP", filetypes=[("Excel files", "*.xlsx *.xls")])
         
    dict_cdp = {}
    dict_rp = defaultdict(list)

    # --- PROCESAR RP ---
    if ruta_rp:
        print(f"  -> Procesando archivo RP...")
        try:
            excel = pd.ExcelFile(ruta_rp)
            hojas_disponibles = excel.sheet_names
            palabras_clave = ["data", "crp"]
            hoja_correcta = next((hoja for hoja in hojas_disponibles if hoja.strip().lower() in palabras_clave), None)

            if hoja_correcta:
                df_rp = pd.read_excel(excel, sheet_name=hoja_correcta)
                print(f"     Hoja detectada y cargada: '{hoja_correcta}'")
            else:
                print(f"  [Error] No se encontró ninguna hoja 'Data' o 'CRP' en el archivo.")
                df_rp = None
        except Exception as e:
            print(f"  [Error] No se pudo leer el archivo Excel: {e}")
            df_rp = None

        if df_rp is not None:
            nombres_ideales_rp = {
                'fecha': 'Fecha Registro', 'num_cdp': 'Número de CDP', 'num_rp': 'Número de CRP',
                'filtro1': 'Rubro', 'localidad': 'Texto Id Proyecto', 'cedula': 'Número Doc. BP Beneficiario',
                'filtro2': 'Anulaciones', 'val_rp': 'Valor Neto', 'cod_int': 'N° Interno CDP'
            }
            idx_rp = mapear_columnas(df_rp.columns.tolist(), nombres_ideales_rp)
            for _, row in df_rp.iterrows():
                val_r = obtener_valor_seguro(row, idx_rp['filtro1'])
                digitos_r = re.sub(r'\D', '', val_r)
                if len(digitos_r) >= 7:
                    val_al = obtener_valor_seguro(row, idx_rp['filtro2'])
                    try: num_al = float(str(val_al).replace(',', ''))
                    except ValueError: num_al = 0.0 
                    
                    if num_al == 0:
                        cedula_raw = obtener_valor_seguro(row, idx_rp['cedula'])
                        llave_cedula = normalizar_cedula(cedula_raw)
                        if llave_cedula:
                            rp_num = obtener_valor_seguro(row, idx_rp['num_rp']) 
                            rp_val = obtener_valor_seguro(row, idx_rp['val_rp']) 
                            rp_fec = obtener_valor_seguro(row, idx_rp['fecha'])  
                            cdp_num = obtener_valor_seguro(row, idx_rp['num_cdp']) 
                            cdp_cod = obtener_valor_seguro(row, idx_rp['cod_int']) 
                            localidad_col_z = obtener_valor_seguro(row, idx_rp['localidad']) 
                            dict_rp[llave_cedula].append((rp_num, rp_val, rp_fec, cdp_num, cdp_cod, localidad_col_z))

    # --- PROCESAR CDP ---
    if ruta_cdp:
        print(f"  -> Procesando archivo CDP...")
        try:
            excel = pd.ExcelFile(ruta_cdp)
            hojas_disponibles = excel.sheet_names
            palabras_clave = ["data", "cdp"]
            hoja_correcta = next((hoja for hoja in hojas_disponibles if hoja.strip().lower() in palabras_clave), None)

            if hoja_correcta:
                df_cdp = pd.read_excel(excel, sheet_name=hoja_correcta)
                print(f"     Hoja detectada y cargada: '{hoja_correcta}'")
            else:
                print(f"  [Error] No se encontró ninguna hoja 'Data' o 'CDP' en el archivo.")
                df_cdp = None
        except Exception as e:
            print(f"  [Error] No se pudo leer el archivo Excel: {e}")
            df_cdp = None

        if df_cdp is not None:
            nombres_ideales_cdp = {
                'num_cdp': 'No. CDP', 'val_cdp': 'Valor', 'cod_int': 'No.Interno CDP', 'fec_cdp': 'Fecha Registro'
            }
            idx_cdp = mapear_columnas(df_cdp.columns.tolist(), nombres_ideales_cdp)
            for _, row in df_cdp.iterrows():
                cdp_num = obtener_valor_seguro(row, idx_cdp['num_cdp'])  
                cdp_cod = obtener_valor_seguro(row, idx_cdp['cod_int']) 
                llave_cdp = (cdp_num, cdp_cod)
                if llave_cdp != ("", "") and llave_cdp not in dict_cdp:
                    cdp_val = obtener_valor_seguro(row, idx_cdp['val_cdp']) 
                    cdp_fec = obtener_valor_seguro(row, idx_cdp['fec_cdp']) 
                    dict_cdp[llave_cdp] = (cdp_num, cdp_val, cdp_fec)

    return dict_cdp, dict_rp

# =======================================================
# EJECUCIÓN DE LA CELDA 1 (Almacenar en memoria)
# =======================================================
print("Iniciando la carga de diccionarios (esto tomará tiempo)...")
dict_cdp_global, dict_rp_global = cargar_datos_bogdata()
print("¡Carga finalizada con éxito! Los diccionarios están en memoria. Ya puedes correr la Celda 2.")
#3m 49s

In [ ]:
#Celda 2.2
global metricas

def procesar_archivos(dict_cdp_en_memoria, dict_rp_en_memoria):
    global metricas
    if not os.path.exists(CARPETA_DESTINO):
        os.makedirs(CARPETA_DESTINO)

    # 🔹 Aquí mapeamos los diccionarios pre-cargados a las variables locales
    dict_cdp = dict_cdp_en_memoria
    dict_rp = dict_rp_en_memoria

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('.xlsx', '.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('ENGATIVA.xlsx', 'ENGATIVA.xls'))]
    print(f"Se encontraron {len(archivos)} archivos de Excel.")

    for archivo in archivos:
        ruta_orig = os.path.join(CARPETA_ORIGEN, archivo)
        nombre_base = os.path.splitext(archivo)[0]
        ruta_dest = os.path.join(CARPETA_DESTINO, nombre_base + ".xlsx")
        
        print(f"Procesando: {archivo}...")

        # --- PASO DE CONVERSIÓN SI ES .XLS ---
        if archivo.lower().endswith('.xls'):
            try:
                conversion = pd.read_excel(ruta_orig, sheet_name=None, engine='xlrd')
                with pd.ExcelWriter(ruta_dest, engine='openpyxl') as writer:
                    for nombre_hoja, datos in conversion.items():
                        datos.to_excel(writer, sheet_name=nombre_hoja, index=False)
                print(f"  -> Convertido de .xls a .xlsx con éxito.")
            except Exception as e:
                print(f"  Error convirtiendo {archivo}: {e}")
                continue
        else:
            shutil.copy2(ruta_orig, ruta_dest)

        try:
            wb = openpyxl.load_workbook(ruta_dest)
        except Exception as e:
            print(f"   Error al abrir el archivo con openpyxl: {e}")
            continue

        # ==========================================
        # PARTE 1: Procesar SECOP II (Crear aux1)
        # ==========================================
        nombre_real_secop = next(n for n in wb.sheetnames if n.lower() == 'secop ii')
        ws_origen = wb[nombre_real_secop]
        
        if "SECOP II aux1" in wb.sheetnames:
            del wb["SECOP II aux1"]
            
        ws_secop_aux = wb.copy_worksheet(ws_origen)
        ws_secop_aux.title = "SECOP II aux1"

        filas = []
        for row in ws_secop_aux.iter_rows(min_row=2, values_only=True):
            filas.append(list(row))
        
        filas_ordenadas = sorted(filas, key=lambda x: normalizar_numero_contrato(x[11] if len(x) > 11 else 0))

        for row in ws_secop_aux.iter_rows(min_row=2):
            for cell in row:
                cell.value = None

        for i_fila, datos_fila in enumerate(filas_ordenadas, start=2):
            for i_col, valor in enumerate(datos_fila, start=1):
                ws_secop_aux.cell(row=i_fila, column=i_col, value=valor)

        max_col_secop = ws_secop_aux.max_column
        ws_secop_aux.cell(row=1, column=max_col_secop + 1, value="Columna1")
        ws_secop_aux.cell(row=1, column=max_col_secop + 2, value="Columna2")
        col_L = get_column_letter(12)
        
        for i in range(2, ws_secop_aux.max_row + 1):
            f1 = (
                f'=IFERROR(MID({col_L}{i}, '
                f'MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), '
                f'MIN(IFERROR(FIND(" ", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999), '
                f'IFERROR(FIND("-", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999)) '
                f'- MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), "")'
            )
            ws_secop_aux.cell(row=i, column=max_col_secop + 1, value=f1)

            f2 = f'=IFERROR(MID({col_L}{i}, FIND("(", {col_L}{i}) + 1, FIND(")", {col_L}{i}) - FIND("(", {col_L}{i}) - 1), "")'
            ws_secop_aux.cell(row=i, column=max_col_secop + 2, value=f2)

        print("  -> Aplicando estilos y ocultando columnas en SECOP II aux1...")
        aplicar_estilo_ferrocarril(ws_secop_aux)
        ocultar_columnas_por_lista(ws_secop_aux, RANGOS_OCULTAR_SECOP)

        # ==========================================
        # PARTE 2 y 3: ALINEACIÓN EN 'ANALISIS'
        # ==========================================
        print("  -> Generando hoja 'Analisis'...")

        nombre_real_sipse = next(n for n in wb.sheetnames if n.lower() == 'sipse')
        ws_sipse = wb[nombre_real_sipse]
        
        if 'Analisis' in wb.sheetnames:
            del wb['Analisis']
        ws_analisis = wb.create_sheet("Analisis")
        
        dict_sipse = defaultdict(list)
        dict_secop = defaultdict(list)
        
        filas_sipse = list(ws_sipse.values)
        header_sipse = filas_sipse[0] if filas_sipse else []
        for fila in filas_sipse[1:]:
            contrato = normalizar_numero_contrato(fila[3] if len(fila) > 3 else 0)
            dict_sipse[contrato].append(fila)

        filas_secop = list(ws_secop_aux.values)
        header_secop = filas_secop[0] if filas_secop else []
        for fila in filas_secop[1:]:
            contrato = normalizar_numero_contrato(fila[11] if len(fila) > 11 else 0)
            dict_secop[contrato].append(fila)

        ws_analisis.append(header_sipse) 
        col_inicio_secop = column_index_from_string("BM")
        for i, val in enumerate(header_secop):
            ws_analisis.cell(row=1, column=col_inicio_secop + i, value=val)

        todos_contratos = sorted(list(set(dict_sipse.keys()) | set(dict_secop.keys())))
        current_row = 2 
        
        for contrato in todos_contratos:
            filas_de_sipse = dict_sipse.get(contrato, [])
            filas_de_secop = dict_secop.get(contrato, [])
            max_filas = max(len(filas_de_sipse), len(filas_de_secop))
            
            for i in range(max_filas):
                if i < len(filas_de_sipse):
                    datos = filas_de_sipse[i]
                    for c_idx, valor in enumerate(datos, start=1):
                        ws_analisis.cell(row=current_row, column=c_idx, value=valor)
                
                if i < len(filas_de_secop):
                    datos = filas_de_secop[i]
                    for c_idx, valor in enumerate(datos):
                        col_excel = col_inicio_secop + c_idx
                        if c_idx == len(datos) - 2 and str(valor).startswith('='):
                             nueva_formula1 = (
                                f'=IFERROR(MID(BX{current_row}, '
                                f'MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), '
                                f'MIN(IFERROR(FIND(" ", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999), '
                                f'IFERROR(FIND("-", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999)) '
                                f'- MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), "")'
                            )
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula1)
                        elif c_idx == len(datos) - 1 and str(valor).startswith('='):
                             nueva_formula2 = f'=IFERROR(MID(BX{current_row}, FIND("(", BX{current_row}) + 1, FIND(")", BX{current_row}) - FIND("(", BX{current_row}) - 1), "")'
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula2)
                        else:
                            ws_analisis.cell(row=current_row, column=col_excel, value=valor)
                current_row += 1

        # ==========================================
        # PARTE 4: Columnas de Evaluación
        # ==========================================
        print("  -> Agregando columnas de evaluación y estilos...")
        
        ultima_col_actual = ws_analisis.max_column
        col_inicio_eval = ultima_col_actual + 1
        ultima_fila_analisis = ws_analisis.max_row
        
        headers_nuevos = [
            "Columna3\n# RP\n(SIPSE)", "Columna4\nValor RP\n(SIPSE)", "Columna5\nFecha RP\n(SIPSE)", "Columna6\n# CDP\n(SIPSE)", "Columna7\nValor CDP\n(SIPSE)", "Columna8\nFecha CDP\n(SIPSE)", "Col.9\n# RP\n(BOGD)", "Col.10\nValor RP\n(BOGD)", "Col.11\nFecha RP\n(BOGD)", "Col.12\n# CDP\n(BOGD)", "Col.13\nValor CDP\n(BOGD)", "Col.14\nFecha CDP\n(BOGD)",
            "Ev. Nro contrato", "Aux Nro contrato", "Ev. Cédula", "Repeticiones contrato SIPSE", "Repeticiones contrato SECOP", "Fecha suscripción", "Fecha inicio", "Fecha fin", "Duración contratos", "Valor contratos", "Ev. # RP", "Ev. Valor RP", "Ev. Fecha RP", "Ev. # CDP", "Ev. Valor CDP", "Ev. Fecha CDP",
            "Ev. Indicador 1", "Ev. Indicador 2"
        ]

        for idx, titulo in enumerate(headers_nuevos):
            ws_analisis.cell(row=1, column=col_inicio_eval + idx, value=titulo)
        
        for i in range(2, ultima_fila_analisis + 1):
            f3 = f'=IFERROR(TRIM(RIGHT(SUBSTITUTE(TRIM(LEFT(AI{i}, FIND("Fecha", AI{i})-1)), " ", REPT(" ", 100)), 100)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval, value=f3)
            f4 = f'=IFERROR(TRIM(MID(AI{i}, FIND("Valor ", AI{i}) + 6, IFERROR(FIND(";", AI{i}), LEN(AI{i})+1) - FIND("Valor ", AI{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 1, value=f4)
            f5 = f'=IFERROR(TRIM(MID(AI{i}, FIND("Fecha ", AI{i}) + 6, FIND(" Valor", AI{i}) - FIND("Fecha ", AI{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 2, value=f5)
            f6 = f'=IFERROR(TRIM(RIGHT(SUBSTITUTE(TRIM(LEFT(AH{i}, FIND("Fecha", AH{i})-1)), " ", REPT(" ", 100)), 100)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 3, value=f6)
            f7 = f'=IFERROR(TRIM(MID(AH{i}, FIND("Valor ", AH{i}) + 6, IFERROR(FIND(";", AH{i}), LEN(AH{i})+1) - FIND("Valor ", AH{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 4, value=f7)
            f8 = f'=IFERROR(TRIM(MID(AH{i}, FIND("Fecha ", AH{i}) + 6, FIND(" Valor", AH{i}) - FIND("Fecha ", AH{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 5, value=f8)

        idx_col_cedula_cj = column_index_from_string("CJ")
        letra_ev_nro_contrato = get_column_letter(col_inicio_eval + 12) 
        letra_aux_nro_contrato = get_column_letter(col_inicio_eval + 13) 
        letra_ev_nro_cedula = get_column_letter(col_inicio_eval + 14) 
        
        localidad_actual_norm = normalizar_texto_localidad(nombre_base)
        
        for i in range(2, ultima_fila_analisis + 1):
            cedula_actual_raw = ws_analisis.cell(row=i, column=idx_col_cedula_cj).value
            cedula_busqueda = normalizar_cedula(cedula_actual_raw)
            
            lista_rps = dict_rp.get(cedula_busqueda, [])
            rp_data_elegido = ("", "", "", "", "")
            
            if len(lista_rps) == 1:
                rp_data_elegido = lista_rps[0][:5]
            elif len(lista_rps) > 1:
                for rp in lista_rps:
                    localidad_rp_norm = normalizar_texto_localidad(rp[5]) 
                    if fuzz.partial_ratio(localidad_actual_norm.lower(), localidad_rp_norm.lower()) > 85:
                        rp_data_elegido = rp[:5]
                        break

            rp_num, rp_val, rp_fec, cdp_num_from_rp, cdp_cod_from_rp = rp_data_elegido
            cdp_data = dict_cdp.get((cdp_num_from_rp, cdp_cod_from_rp), ("", "", ""))
            cdp_num_out, cdp_val_out, cdp_fec_out = cdp_data

            def inyectar_condicional(valor):
                val_str = str(valor).replace('"', '""') if valor else ""
                return f'=IF(OR({letra_ev_nro_contrato}{i}="Numero contrato coincide", {letra_ev_nro_contrato}{i}="Contrato coincide sin SIPSE", {letra_ev_nro_cedula}{i}="Cedula coincide"), "{val_str}", "")'

            ws_analisis.cell(row=i, column=col_inicio_eval + 6, value=inyectar_condicional(rp_num))
            ws_analisis.cell(row=i, column=col_inicio_eval + 7, value=inyectar_condicional(rp_val))
            ws_analisis.cell(row=i, column=col_inicio_eval + 8, value=inyectar_condicional(rp_fec))
            ws_analisis.cell(row=i, column=col_inicio_eval + 9, value=inyectar_condicional(cdp_num_out))
            ws_analisis.cell(row=i, column=col_inicio_eval + 10, value=inyectar_condicional(cdp_val_out))
            ws_analisis.cell(row=i, column=col_inicio_eval + 11, value=inyectar_condicional(cdp_fec_out))

        for i in range(2, ultima_fila_analisis + 1):
            COL_SOLICITUD = "B"
            f15 = f'''=IF(COUNTIF(${letra_aux_nro_contrato}$2:{letra_aux_nro_contrato}{i},{letra_aux_nro_contrato}{i})>1,"",
                IF(AND(D{i}<>"",EV{i}<>""),
                    IF(EW{i}="","Contrato coincide sin SIPSE",
                        IF(TRIM(SUBSTITUTE(UPPER({COL_SOLICITUD}{i}),"SIPSE",""))=TRIM(SUBSTITUTE(UPPER(EW{i}),"SIPSE","")),"Numero contrato coincide","Numero contrato NO coincide")
                    ),
                    IF(D{i}<>"","Contrato no existe en SECOP",
                        IF(EV{i}<>"","Contrato no existe en SIPSE","No existe")
                    )
                )
            )'''
            f15 = f15.replace('\n', '').replace('    ', '')
            ws_analisis.cell(row=i, column=col_inicio_eval + 12, value=f15)

            f16 = f'=IF(D{i}<>"",D{i},EV{i})'
            ws_analisis.cell(row=i, column=col_inicio_eval + 13, value=f16)

            f17 = f'=IF(AND(BY{i}="",M{i}=""),"",IF(OR(BY{i}="",M{i}=""),"",IF(BY{i}="-","Sin cedula en SECOP",IF(M{i}="-","Sin cedula en SIPSE",IF(VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(CJ{i}),"CC",""),"NIT",""),"TI",""))=VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(M{i}),"CC",""),"NIT",""),"TI","")),"Cedula coincide","Cedula NO coincide")))))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 14, value=f17)

            f18 = f'=IF({letra_ev_nro_contrato}{i}="","",COUNTIF(D$2:D${ultima_fila_analisis},D{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 15, value=f18)

            f19 = f'=IF({letra_ev_nro_contrato}{i}="","",COUNTIF(EV$2:EV${ultima_fila_analisis},EV{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 16, value=f19)

            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CE{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CE{i}), DAY(CE{i}), MONTH(CE{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f20 = f'''=IF(AND(H{i}="",CE{i}=""),"",IF(AND(H{i}="-",CE{i}="-"),"Ambos sin fechas",IF(H{i}="-","sin fecha SIPSE",IF(CE{i}="-","sin fecha SECOP",IF(OR(H{i}="",CE{i}=""),"",IF(INT(VALUE(SUBSTITUTE(H{i},"-","/"))) =INT(IF(ISNUMBER(CE{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 17, value=f20)

            # 🔹 FECHA ACTA DE INICIO
            cf_clean = f'SUBSTITUTE(SUBSTITUTE(CF{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CF{i}), DAY(CF{i}), MONTH(CF{i}))'
            parte_es_texto = f'DATE(RIGHT({cf_clean}, 4), LEFT({cf_clean}, FIND("/", {cf_clean}) - 1), MID({cf_clean}, FIND("/", {cf_clean}) + 1, FIND("/", {cf_clean}, FIND("/", {cf_clean}) + 1) - FIND("/", {cf_clean}) - 1))'

            f21 = f'''=IF(AND(I{i}="",CF{i}=""),"",IF(AND(I{i}="-",CF{i}="-"),"Ambos sin fechas",IF(I{i}="-","sin fecha SIPSE",IF(CF{i}="-","sin fecha SECOP",IF(OR(I{i}="",CF{i}=""),"",IF(INT(VALUE(SUBSTITUTE(I{i},"-","/"))) =INT(IF(ISNUMBER(CF{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 18, value=f21)

            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CG{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CG{i}), DAY(CG{i}), MONTH(CG{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f22 = f'''=IF(AND(J{i}="",CG{i}=""),"",IF(AND(J{i}="-",CG{i}="-"),"Ambos sin fechas",IF(J{i}="-","sin fecha SIPSE",IF(CG{i}="-","sin fecha SECOP",IF(OR(J{i}="",CG{i}=""),"",IF(INT(VALUE(SUBSTITUTE(J{i},"-","/"))) =INT(IF(ISNUMBER(CG{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 19, value=f22)

            f23 = f'''=IF(AND(R{i}="",EE{i}=""),"",IF(AND(R{i}="-",EE{i}="-"),"Ambos sin duracion",IF(R{i}="-","sin duracion en SECOP",IF(EE{i}="-","sin duracion en SIPSE",IF(OR(R{i}="",EE{i}=""),"",IF(TRIM(UPPER(R{i}))=TRIM(UPPER(EE{i})),"Duracion coincide","Duracion NO coincide"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 20, value=f23)

            f24 = f'''=IF(AND(P{i}="",CU{i}=""),"",
                IF(AND(P{i}="-",CU{i}="-"),"Ambos sin valor",
                IF(P{i}="-","Sin valor en SECOP",
                IF(CU{i}="-","Sin valor en SIPSE",
                IF(OR(P{i}="",CU{i}=""),"",
                IF(P{i}=CU{i},"Valor coincide","Valor NO coincide"))))))'''

            ws_analisis.cell(row=i, column=col_inicio_eval + 21, value=f24)

            c_rp_sip   = get_column_letter(col_inicio_eval)       
            c_val_rp_s = get_column_letter(col_inicio_eval + 1)   
            c_fec_rp_s = get_column_letter(col_inicio_eval + 2)   
            c_cdp_sip  = get_column_letter(col_inicio_eval + 3)   
            c_val_cd_s = get_column_letter(col_inicio_eval + 4)   
            c_fec_cd_s = get_column_letter(col_inicio_eval + 5)   
            
            c_rp_bog   = get_column_letter(col_inicio_eval + 6)   
            c_val_rp_b = get_column_letter(col_inicio_eval + 7)   
            c_fec_rp_b = get_column_letter(col_inicio_eval + 8)   
            c_cdp_bog  = get_column_letter(col_inicio_eval + 9)   
            c_val_cd_b = get_column_letter(col_inicio_eval + 10)  
            c_fec_cd_b = get_column_letter(col_inicio_eval + 11)  

            f25 = (f'=IF(AND(TRIM({c_rp_sip}{i})="", TRIM({c_rp_bog}{i})=""), "Sin numeros RP", '
                   f'IF(TRIM({c_rp_sip}{i})="", "Sin numero RP en SIPSE", '
                   f'IF(TRIM({c_rp_bog}{i})="", "Sin numero RP en BOGDATA", '
                   f'IF(TRIM({c_rp_sip}{i})=TRIM({c_rp_bog}{i}), "El numero RP coincide", "El numero RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 22, value=f25)

            f26 = (f'=IF(AND(TRIM({c_val_rp_s}{i})="", TRIM({c_val_rp_b}{i})=""), "Sin valores RP", '
                   f'IF(TRIM({c_val_rp_s}{i})="", "Sin valor RP en SIPSE", '
                   f'IF(TRIM({c_val_rp_b}{i})="", "Sin valor RP en BOGDATA", '
                   f'IF(IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_rp_s}{i}), ",", "")),TRIM({c_val_rp_s}{i}))=IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_rp_b}{i}), ",", "")),TRIM({c_val_rp_b}{i})), "El valor RP coincide", "El valor RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 23, value=f26)

            f27 = (f'=IF(AND(TRIM({c_fec_rp_s}{i})="", TRIM({c_fec_rp_b}{i})=""), "Sin fechas RP", '
                   f'IF(TRIM({c_fec_rp_s}{i})="", "Sin fecha RP en SIPSE", '
                   f'IF(TRIM({c_fec_rp_b}{i})="", "Sin fecha RP en BOGDATA", '
                   f'IF(TRIM({c_fec_rp_s}{i})=TRIM({c_fec_rp_b}{i}), "La fecha RP coincide", "La fecha RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 24, value=f27)

            f28 = (f'=IF(AND(TRIM({c_cdp_sip}{i})="", TRIM({c_cdp_bog}{i})=""), "Sin numeros CDP", '
                   f'IF(TRIM({c_cdp_sip}{i})="", "Sin numero CDP en SIPSE", '
                   f'IF(TRIM({c_cdp_bog}{i})="", "Sin numero CDP en BOGDATA", '
                   f'IF(TRIM({c_cdp_sip}{i})=TRIM({c_cdp_bog}{i}), "El numero CDP coincide", "El numero CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 25, value=f28)

            f29 = (f'=IF(AND(TRIM({c_val_cd_s}{i})="", TRIM({c_val_cd_b}{i})=""), "Sin valores CDP", '
                   f'IF(TRIM({c_val_cd_s}{i})="", "Sin valor CDP en SIPSE", '
                   f'IF(TRIM({c_val_cd_b}{i})="", "Sin valor CDP en BOGDATA", '
                   f'IF(IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_cd_s}{i}), ",", "")),TRIM({c_val_cd_s}{i}))=IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_cd_b}{i}), ",", "")),TRIM({c_val_cd_b}{i})), "El valor CDP coincide", "El valor CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 26, value=f29)

            f30 = (f'=IF(AND(TRIM({c_fec_cd_s}{i})="", TRIM({c_fec_cd_b}{i})=""), "Sin fechas CDP", '
                   f'IF(TRIM({c_fec_cd_s}{i})="", "Sin fecha CDP en SIPSE", '
                   f'IF(TRIM({c_fec_cd_b}{i})="", "Sin fecha CDP en BOGDATA", '
                   f'IF(TRIM({c_fec_cd_s}{i})=TRIM({c_fec_cd_b}{i}), "La fecha CDP coincide", "La fecha CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 27, value=f30)

            # 🔹 Indicador 1 de Completitud
            f31 = f'=IF(B{i}="", "", (IF(I{i}<>"-", 1, 0) + IF(M{i}<>"-", 1, 0) + IF(V{i}<>"-", 1, 0) + IF(AP{i}<>"-", 1, 0)) / 4)'
            ws_analisis.cell(row=i, column=col_inicio_eval + 28, value=f31)

            # 🔹 Indicador 2 de Exactitud
            f32 = f'=IF(B{i}="", "", IF(AND(FS{i}="Valor coincide", FR{i}="Duracion coincide", OR(FJ{i}="Numero contrato coincide", FJ{i}="Contrato coincide sin SIPSE"), FP{i}="Coincide fecha", FL{i}="Cedula coincide"), "Exacto", "No Exacto"))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 29, value=f32)

        print("  -> Aplicando estilos y ocultando columnas en Analisis...")
        aplicar_estilo_ferrocarril(ws_analisis)
        aplicar_formato_condicional_alertas(ws_analisis, col_inicio_eval)
        ocultar_columnas_por_lista(ws_analisis, RANGOS_OCULTAR_ANALISIS)

        # ==========================================
        # PARTE 5: Crear HOJA RESULTADOS
        # ==========================================
        print("  -> Creando hoja 'Resultados'...")
        if 'Resultados' in wb.sheetnames:
            del wb['Resultados']
        ws_res = wb.create_sheet("Resultados")
        nombre_limpio = os.path.splitext(archivo)[0]
        ws_res['B1'] = f"Métricas alcaldía local {nombre_limpio}"
        
        letra_ev_nro = get_column_letter(col_inicio_eval + 12)       
        letra_ev_ced = get_column_letter(col_inicio_eval + 14)   
        letra_rep_sipse = get_column_letter(col_inicio_eval + 15)
        letra_rep_secop = get_column_letter(col_inicio_eval + 16)
        letra_fecha_suscripcion = get_column_letter(col_inicio_eval + 17)
        letra_fecha_inicio = get_column_letter(col_inicio_eval + 18)
        letra_fecha_fin = get_column_letter(col_inicio_eval + 19)
        letra_duracion = get_column_letter(col_inicio_eval + 20)
        letra_valor = get_column_letter(col_inicio_eval + 21)

        letra_num_RP = get_column_letter(col_inicio_eval + 22)
        letra_val_RP = get_column_letter(col_inicio_eval + 23)
        letra_fec_RP = get_column_letter(col_inicio_eval + 24)
        letra_num_CDP = get_column_letter(col_inicio_eval + 25)
        letra_val_CDP = get_column_letter(col_inicio_eval + 26)
        letra_fec_CDP = get_column_letter(col_inicio_eval + 27)

        letra_ind_1 = get_column_letter(col_inicio_eval + 28)
        letra_ind_2 = get_column_letter(col_inicio_eval + 29)

        metricas = [
            ("Nro de contratos registrados SIPSE",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos registrados SECOP",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos registrados en SECOP y SIPSE con numero de contrato igual", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")"),
            ("Nro de contratos solo en SECOP y no en SIPSE", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos solo en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos diferentes en SIPSE-SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato NO coincide\")"),
            ("Nro de contratos Cédula diferente", f"=COUNTIF('Analisis'!{letra_ev_ced}:{letra_ev_ced}, \"Cedula NO coincide\")"),
            ("Nro de contratos dobles en SIPSE", f"=COUNTIF('Analisis'!{letra_rep_sipse}:{letra_rep_sipse}, 2)"),
            ("Nro de contratos dobles en SECOP", f"=COUNTIF('Analisis'!{letra_rep_secop}:{letra_rep_secop}, 2)"),
            ("Fecha de acta de suscriccpión Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"Coincide fecha\")"),
            ("Fecha de acta de suscriccpión no Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"No coinciden fechas\")"),
            ("Fecha de acta de inicio Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Coincide fecha\")"),
            ("Fecha de acta de inicio no coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"No coinciden fechas\")"),
            ("Fecha de Fin conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"Coincide fecha\")"),
            ("Fecha de Fin no conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"No coinciden fechas\")"),
            ("Contratos que coinciden en Duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion Coincide\")"),
            ("Contratos que tienen diferente duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion NO coincide\")"),
            ("Contratos que tienen Fecha de inicio en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Sin fecha SECOP\")"),
            ("Datos en la matriz de seguimiento", f"=C2+C3"),
            ("Valor contrato coinciden", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor coincide\")"),
            ("Valor contrato no coincidenn", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor NO coincide\")"),
            ("Numero RP no coinciden", f"=COUNTIF('Analisis'!{letra_num_RP}:{letra_num_RP}, \"El numero RP NO coincide\")"),
            ("Valor RP no coinciden", f"=COUNTIF('Analisis'!{letra_val_RP}:{letra_val_RP}, \"El valor RP NO coincide\")"),
            ("Fecha RP no coinciden", f"=COUNTIF('Analisis'!{letra_fec_RP}:{letra_fec_RP}, \"La fecha RP NO coincide\")"),
            ("Numero CDP no coinciden", f"=COUNTIF('Analisis'!{letra_num_CDP}:{letra_num_CDP}, \"El numero CDP NO coincide\")"),
            ("Valor CDP no coinciden", f"=COUNTIF('Analisis'!{letra_val_CDP}:{letra_val_CDP}, \"El valor CDP NO coincide\")"),
            ("Fecha CDP no coinciden", f"=COUNTIF('Analisis'!{letra_fec_CDP}:{letra_fec_CDP}, \"La fecha CDP NO coincide\")"),
            ("Indicador completitud", f"=AVERAGE('Analisis'!{letra_ind_1}:{letra_ind_1})"),
            ("Indicador exactitud", f"=COUNTIF('Analisis'!{letra_ind_2}:{letra_ind_2}, \"Exacto\") / (COUNTIF('Analisis'!{letra_ind_2}:{letra_ind_2}, \"Exacto\") + COUNTIF('Analisis'!{letra_ind_2}:{letra_ind_2}, \"No Exacto\"))")
        ]

        for idx, (texto, formula) in enumerate(metricas, start=2):
            ws_res.cell(row=idx, column=1, value=idx-1)
            ws_res.cell(row=idx, column=2, value=texto)
            ws_res.cell(row=idx, column=3, value=formula)
        
        fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
        ws_res['B1'].fill = fill_amarillo
        ws_res.column_dimensions['B'].width = 100
        
        ws_analisis.column_dimensions['FJ'].width = 29
        ws_analisis.column_dimensions['FK'].width = 10
        ws_analisis.column_dimensions['FL'].width = 20
        ws_analisis.column_dimensions['FM'].width = 15
        ws_analisis.column_dimensions['FN'].width = 15
        ws_analisis.column_dimensions['FO'].width = 15
        ws_analisis.column_dimensions['FP'].width = 20
        ws_analisis.column_dimensions['FQ'].width = 20
        ws_analisis.column_dimensions['FR'].width = 18
        ws_analisis.column_dimensions['FS'].width = 14
        ws_analisis.column_dimensions['FT'].width = 25
        ws_analisis.column_dimensions['FU'].width = 25
        ws_analisis.column_dimensions['FV'].width = 25
        ws_analisis.column_dimensions['FW'].width = 25
        ws_analisis.column_dimensions['FX'].width = 25
        ws_analisis.column_dimensions['FY'].width = 25

        dejar_columna_blanca(ws_analisis, 64)

        ajustar_texto_fila(ws_secop_aux, 1)
        ajustar_texto_fila(ws_analisis, 1)

        # ==========================================
        # PARTE 6: Crear HOJA 'Para MEMO'
        # ==========================================
        print("  -> Creando hoja 'Para MEMO'...")
        if 'Para MEMO' in wb.sheetnames:
            del wb['Para MEMO']
        ws_memo = wb.create_sheet("Para MEMO")
        
        datos_memo = []
        
        for i in range(2, ultima_fila_analisis + 1):
            val_b = ws_analisis.cell(row=i, column=2).value 
            val_d = ws_analisis.cell(row=i, column=4).value 
            
            susc_sipse = ws_analisis.cell(row=i, column=8).value  
            susc_secop = ws_analisis.cell(row=i, column=83).value 
            ini_sipse  = ws_analisis.cell(row=i, column=9).value  
            ini_secop  = ws_analisis.cell(row=i, column=84).value 
            fin_sipse  = ws_analisis.cell(row=i, column=10).value 
            fin_secop  = ws_analisis.cell(row=i, column=85).value 
            dur_sipse  = ws_analisis.cell(row=i, column=18).value 
            dur_secop  = ws_analisis.cell(row=i, column=135).value
            
            err_susc = es_error_fecha(susc_sipse, susc_secop)
            err_ini  = es_error_fecha(ini_sipse, ini_secop)
            err_fin  = es_error_fecha(fin_sipse, fin_secop)
            err_dur  = es_error_duracion(dur_sipse, dur_secop)
            
            if err_susc or err_ini or err_fin or err_dur:
                llave_orden = (err_susc, err_ini, err_fin, err_dur)
                datos_memo.append([
                    val_b, val_d,
                    "X" if err_susc else "",
                    "X" if err_ini else "",
                    "X" if err_fin else "",
                    "X" if err_dur else "",
                    llave_orden 
                ])
                
        datos_memo.sort(key=lambda x: x[6], reverse=True)
        
        relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
        fuente_blanca = Font(color="FFFFFF", bold=True)
        ws_memo.merge_cells("A1:A2")
        ws_memo['A1'] = "SIPSE"
        ws_memo.merge_cells("B1:B2")
        ws_memo['B1'] = "No. Contrato"
        ws_memo.merge_cells("C1:F1")
        ws_memo['C1'] = "TIPOLOGÍA DE LA INCONSISTENCIA"
        
        ws_memo['C2'] = "Fecha suscripción"
        ws_memo['D2'] = "Fecha Inicio"
        ws_memo['E2'] = "Fecha Fin"
        ws_memo['F2'] = "Plazo"
        ws_memo['A1'].fill = relleno_rojo
        ws_memo['B1'].fill = relleno_rojo
        ws_memo['C1'].fill = relleno_rojo
        ws_memo['C2'].fill = relleno_rojo
        ws_memo['D2'].fill = relleno_rojo
        ws_memo['E2'].fill = relleno_rojo
        ws_memo['F2'].fill = relleno_rojo
        
        align_header = Alignment(horizontal="center", vertical="center")
        borde_delgado = Border(left=Side(style='thin', color='000000'), right=Side(style='thin', color='000000'), top=Side(style='thin', color='000000'), bottom=Side(style='thin', color='000000'))
        
        for fila_h in ws_memo['A1:F2']:
            for celda in fila_h:
                celda.font = fuente_blanca
                celda.alignment = align_header
                celda.border = borde_delgado
                
        fila_actual = 3
        for fila_datos in datos_memo:
            for col_idx in range(1, 7):
                celda = ws_memo.cell(row=fila_actual, column=col_idx, value=fila_datos[col_idx-1])
                celda.border = borde_delgado
                celda.alignment = align_header
                
            fila_actual += 1
            
        ws_memo.column_dimensions['A'].width = 15
        ws_memo.column_dimensions['B'].width = 15
        ws_memo.column_dimensions['C'].width = 20
        ws_memo.column_dimensions['D'].width = 15
        ws_memo.column_dimensions['E'].width = 15
        ws_memo.column_dimensions['F'].width = 15

        wb.save(ruta_dest)
        print("  -> Procesado exitosamente.")


# =======================================================
# EJECUCIÓN DE LA CELDA 2
# =======================================================
# Le pasamos a la función los diccionarios que ya quedaron cargados
# en la memoria de Jupyter desde la Celda 1.
procesar_archivos(dict_cdp_global, dict_rp_global)
#26m 30s

Se encontraron 20 archivos de Excel.
Procesando: ANTONIO NARINO.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 'Para MEMO'...
  -> Procesado exitosamente.
Procesando: BARRIOS UNIDOS.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 'Para MEMO'...
  -> Procesado exitosamente.
Procesando: BOSA.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 

In [21]:
import os
import win32com.client as win32
import re
import unicodedata

# ================= CONFIGURACIÓN =================
CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7\Diagnóstico'
ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx"

nombres_hojas = [
    "1-Usaquén", "2-Chapinero", "3-Santafe", "4-San Cristóbal",
    "5-Usme", "6-Tunjuelito", "7-Bosa", "8-Kennedy", "9-Fontibón",
    "10-Engativá", "11-Suba", "12-Barrios Unidos", "13-Teusaquillo",
    "14-Mártires", "15-Antonio Nariño", "16-Puente Aranda",
    "17-Rafael Uribe", "18-Candelaria", "19-Ciudad Bolívar", "20-Sumapaz"
]

# ================= FUNCIONES AUXILIARES =================
def normalizar_texto(texto):
    if not texto: return ""
    texto = str(texto).lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^\d+[-_ ]?', '', texto)
    # Elimina espacios, guiones y barras para unificar "Santa Fe" y "Santafe"
    texto = re.sub(r'[\s_-]+', '', texto)
    return texto

def encontrar_archivo_para_hoja(nombre_hoja, archivos):
    """Busca el archivo que coincida con la hoja específica respetando el orden"""
    for archivo in archivos:
        nombre_base = os.path.splitext(archivo)[0]
        if normalizar_texto(nombre_hoja) in normalizar_texto(nombre_base):
            return archivo
    return None

def color_rgb(hex_str):
    hex_str = hex_str.lstrip('#')
    r, g, b = tuple(int(hex_str[i:i+2], 16) for i in (0, 2, 4))
    return r + (g * 256) + (b * 65536)

# ================= PROGRAMA PRINCIPAL =================
def consolidar_resultados_fantasma():
    print("Iniciando Excel en segundo plano (Modo Fantasma)...")
    excel = win32.DispatchEx("Excel.Application")
    excel.Visible = False 
    excel.DisplayAlerts = False

    try:
        wb_nuevo = excel.Workbooks.Add()
        ws_general = wb_nuevo.Sheets(1)
        ws_general.Name = "General"

        # Títulos hoja General
        ws_general.Cells(1, 1).Value = "Nro"
        ws_general.Cells(1, 2).Value = "Alcaldía local"

        # Pre-llenar filas de General
        for i, hoja in enumerate(nombres_hojas, start=2):
            num, nombre = hoja.split("-", 1)
            ws_general.Cells(i, 1).Value = int(num)
            ws_general.Cells(i, 2).Value = nombre.strip()
            ws_general.Hyperlinks.Add(Anchor=ws_general.Cells(i, 2), Address="", SubAddress=f"'{hoja}'!A1")

        archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith('.xlsx') and not f.startswith('~')]
        metricas_extraidas = False

        # Iterar estrictamente en el orden ascendente de la lista nombres_hojas
        for hoja_destino_nombre in nombres_hojas:
            archivo = encontrar_archivo_para_hoja(hoja_destino_nombre, archivos)
            
            if not archivo:
                print(f"  - No se encontró archivo para la hoja: '{hoja_destino_nombre}'")
                continue

            print(f"-> Procesando métricas de: '{archivo}' -> Destino: '{hoja_destino_nombre}'")
            ruta_completa = os.path.join(CARPETA_ORIGEN, archivo)

            # Extraer números fijos
            wb_origen = excel.Workbooks.Open(ruta_completa, ReadOnly=True)
            try:
                ws_origen = wb_origen.Sheets("Resultados")
            except Exception:
                print(f"  - Faltó hoja 'Resultados' en {archivo}")
                wb_origen.Close(SaveChanges=False)
                continue

            ultima_fila = ws_origen.Cells(ws_origen.Rows.Count, "A").End(-4162).Row
            
            # Crear pestaña local en el orden correcto secuencial
            ws_destino = wb_nuevo.Sheets.Add(After=wb_nuevo.Sheets(wb_nuevo.Sheets.Count))
            ws_destino.Name = hoja_destino_nombre

            # Copiar y pegar tabla como VALORES
            ws_origen.Range(f"A1:C{ultima_fila}").Copy()
            ws_destino.Range("A1").PasteSpecial(Paste=-4163) # Valores
            ws_destino.Range("A1").PasteSpecial(Paste=-4122) # Formatos
            
            # Botón Volver en la celda A1
            celda_volver = ws_destino.Cells(1, 1)
            celda_volver.Value = "General"
            ws_destino.Hyperlinks.Add(Anchor=celda_volver, Address="", SubAddress="'General'!A1")
            celda_volver.Font.Color = color_rgb("0563C1")
            celda_volver.Font.Underline = 2 
            celda_volver.Font.Bold = True
            celda_volver.Interior.Color = color_rgb("E7E6E6") 
            celda_volver.HorizontalAlignment = -4108 
            celda_volver.VerticalAlignment = -4108
            
            ws_destino.Columns.AutoFit()

            # Extraer números fijos a la hoja General
            fila_gen = nombres_hojas.index(hoja_destino_nombre) + 2
            col_gen = 3
            
            for r in range(2, ultima_fila + 1):
                titulo_metrica = ws_destino.Cells(r, 2).Value
                valor_metrica = ws_destino.Cells(r, 3).Value
                
                if not metricas_extraidas and titulo_metrica:
                    ws_general.Cells(1, col_gen).Value = titulo_metrica
                
                ws_general.Cells(fila_gen, col_gen).Value = valor_metrica
                col_gen += 1
                
            metricas_extraidas = True
            wb_origen.Close(SaveChanges=False)

        # Formato, anchos y grilla en la Hoja General
        max_col = ws_general.Cells(1, ws_general.Columns.Count).End(-4159).Column 
        ultima_fila_general = len(nombres_hojas) + 1 
        
        rango_titulos = ws_general.Range(ws_general.Cells(1, 1), ws_general.Cells(1, max_col))
        rango_titulos.Interior.Color = color_rgb("C00000") 
        rango_titulos.Font.Color = color_rgb("FFFFFF") 
        rango_titulos.Font.Bold = True
        rango_titulos.VerticalAlignment = -4108 
        rango_titulos.HorizontalAlignment = -4108
        
        rango_titulos.WrapText = True
        ws_general.Rows(1).AutoFit()

        if max_col >= 3:
            alineacion_centro = ws_general.Range(ws_general.Cells(2, 3), ws_general.Cells(ultima_fila_general, max_col))
            alineacion_centro.HorizontalAlignment = -4108
            alineacion_centro.VerticalAlignment = -4108

        ws_general.Columns(1).ColumnWidth = 5
        ws_general.Columns(2).ColumnWidth = 25
        if max_col >= 3:
            ws_general.Range(ws_general.Cells(1, 3), ws_general.Cells(1, max_col)).EntireColumn.ColumnWidth = 15

        rango_tabla = ws_general.Range(ws_general.Cells(1, 1), ws_general.Cells(ultima_fila_general, max_col))
        rango_tabla.Borders.LineStyle = 1 
        rango_tabla.Borders.Weight = 2 

        # Asegurar que General es la primera y activa
        ws_general.Move(Before=wb_nuevo.Sheets(1))
        ws_general.Activate()
        ws_general.Cells(1, 1).Select()

        os.makedirs(os.path.dirname(ARCHIVO_DESTINO), exist_ok=True)
        if os.path.exists(ARCHIVO_DESTINO):
            os.remove(ARCHIVO_DESTINO) 

        wb_nuevo.SaveAs(ARCHIVO_DESTINO)
        print(f"\n🚀 ¡Consolidado ordenado y completado! Archivo guardado en:\n{ARCHIVO_DESTINO}")

    except Exception as e:
        print(f"Error durante el proceso: {e}")
    finally:
        try: wb_nuevo.Close(SaveChanges=False) 
        except: pass
        excel.Quit()

if __name__ == "__main__":
    consolidar_resultados_fantasma()
#8m 52s

Iniciando Excel en segundo plano (Modo Fantasma)...
-> Procesando métricas de: 'USAQUEN.xlsx' -> Destino: '1-Usaquén'
-> Procesando métricas de: 'CHAPINERO.xlsx' -> Destino: '2-Chapinero'
-> Procesando métricas de: 'SANTAFE.xlsx' -> Destino: '3-Santafe'
-> Procesando métricas de: 'SAN CRISTOBAL.xlsx' -> Destino: '4-San Cristóbal'
-> Procesando métricas de: 'USME.xlsx' -> Destino: '5-Usme'
-> Procesando métricas de: 'TUNJUELITO.xlsx' -> Destino: '6-Tunjuelito'
-> Procesando métricas de: 'BOSA.xlsx' -> Destino: '7-Bosa'
-> Procesando métricas de: 'KENNEDY.xlsx' -> Destino: '8-Kennedy'
-> Procesando métricas de: 'FONTIBON.xlsx' -> Destino: '9-Fontibón'
-> Procesando métricas de: 'ENGATIVA.xlsx' -> Destino: '10-Engativá'
-> Procesando métricas de: 'SUBA.xlsx' -> Destino: '11-Suba'
-> Procesando métricas de: 'BARRIOS UNIDOS.xlsx' -> Destino: '12-Barrios Unidos'
-> Procesando métricas de: 'TEUSAQUILLO.xlsx' -> Destino: '13-Teusaquillo'
-> Procesando métricas de: 'MARTIRES.xlsx' -> Destino: '

#### Código versionado en GIT

In [ ]:
import pandas as pd
import unicodedata
import re
import difflib
import os
import tkinter as tk
from tkinter import filedialog, messagebox

# ==========================================
# 0. DEFINICIÓN DE ESTRUCTURA ESPERADA
# ==========================================
# Reemplaza los elementos de estas listas con los nombres EXACTOS 
# y en el ORDEN que necesitas para cada hoja.
COLUMNAS_ESPERADAS_SIPSE = [
    "CONTRATO", "SOLICITUD", "VIGENCIA", "NUMERO_CONTRATO", "TIPO", "TIPO_PROCESO", "OBJETO", "FECHA_SUSCRIPCION", "FECHA_INICIO", "FECHA_TERMINACION", "TERMINA31_DICIEMBRE", "CONTRATISTA", "DOC_CONTRATISTA", "VALOR_INICIAL", "ADICIONES", "VALOR_TOTAL", "VALOR_MENSUAL", "PLAZO", "PRORROGAS", "PROYECTO", "DEPENDENCIA", "SUPERVISION", "ESTADO", "NAT", "PAGOS", "AVANCE", "FECHA_ACTA_LIQUIDACION", "PRESUPUESTO", "ANULACION", "DIRECCION_CONTRATISTA", "TELEFONO_CONTRATISTA", "CODIGO_PROYECTO", "NOMBRE_PROYECTO", "CDPS", "RPS", "ESTADO_RP", "MODIFICACION", "NUMERO_OC", "TIPO_SOCIEDAD", "TIPOLOGIA", "TIPO_PAGO", "NIVELACADEMICO", "PROFEXION", "ESPECIALIZ", "POLIZA", "PROCESO_CONTRATO", "ANTIGUEDAD", "SALDO_FAVOR_ENTIDAD", "FECHA_NACIMIENTO", "OBLIGACIONES", "TIPO_IDENTIFICACION", "CORREO_ELECTRONICO", "NOMBRE", "APELLIDO", "CELULAR", "GARANTIAS", "INICIO_AMPARO", "FIN_AMPARO", "IDEN_REP_LEGAL", "NOM_REP_LEGAL", "ENTIDAD", "DESCRIPCION_ENTIDAD", "GENERO"
]

COLUMNAS_ESPERADAS_SECOP = [
    "Nombre Entidad", "Nit Entidad", "Departamento", "Ciudad", "LocalizaciÃ³n", "Orden", "Sector", "Rama", "Entidad Centralizada", "Proceso de Compra", "ID Contrato", "Referencia del Contrato", "Estado Contrato", "Codigo de Categoria Principal", "Descripcion del Proceso", "Tipo de Contrato", "Modalidad de Contratacion", "Justificacion Modalidad de Contratacion", "Fecha de Firma", "Fecha de Inicio del Contrato", "Fecha de Fin del Contrato", "Condiciones de Entrega", "TipoDocProveedor", "Documento Proveedor", "Proveedor Adjudicado", "Es Grupo", "Es Pyme", "Habilita Pago Adelantado", "LiquidaciÃ³n", "ObligaciÃ³n Ambiental", "Obligaciones Postconsumo", "Reversion", "Origen de los Recursos", "Destino Gasto", "Valor del Contrato", "Valor de pago adelantado", "Valor Facturado", "Valor Pendiente de Pago", "Valor Pagado", "Valor Amortizado", "Valor Pendiente de Amortizacion", "Valor Pendiente de Ejecucion", "Estado BPIN", "CÃ³digo BPIN", "Anno BPIN", "Saldo CDP", "Saldo Vigencia", "EsPostConflicto", "Dias adicionados", "Puntos del Acuerdo", "Pilares del Acuerdo", "URLProceso", "Nombre Representante Legal", "Nacionalidad Representante Legal", "Domicilio Representante Legal", "Tipo de IdentificaciÃ³n Representante Legal", "IdentificaciÃ³n Representante Legal", "GÃ©nero Representante Legal", "Presupuesto General de la Nacion â€“ PGN", "Sistema General de Participaciones", "Sistema General de RegalÃ­as", "Recursos Propios (AlcaldÃ­as, Gobernaciones y Resguardos IndÃ­genas)", "Recursos de Credito", "Recursos Propios", "Ultima Actualizacion", "Codigo Entidad", "Codigo Proveedor", "Fecha Inicio Liquidacion", "Fecha Fin Liquidacion", "Objeto del Contrato", "DuraciÃ³n del contrato", "Nombre del banco", "Tipo de cuenta", "NÃºmero de cuenta", "El contrato puede ser prorrogado", "Fecha de notificaciÃ³n de prorrogaciÃ³n", "Nombre ordenador del gasto", "Tipo de documento Ordenador del gasto", "NÃºmero de documento Ordenador del gasto", "Nombre supervisor", "Tipo de documento supervisor", "NÃºmero de documento supervisor", "Nombre Ordenador de Pago", "Tipo de documento Ordenador de Pago", "NÃºmero de documento Ordenador de Pago", "Documentos Tipo", "Descripcion Documentos Tipo"
]

# ==========================================
# 1. FUNCIONES DE LIMPIEZA, COMPARACIÓN Y ESTRUCTURA
# ==========================================

def normalizar_texto(texto):
    """Limpia el texto: quita nulos, espacios, pasa a mayúsculas, quita tildes y artículos"""
    if pd.isna(texto):
        return ""
    texto = str(texto).upper().strip()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^(EL|LA|LOS|LAS)\s+', '', texto)
    return texto

def unificar_localidades(lista_localidades, similitud_minima=0.85):
    """Agrupa nombres similares bajo un mismo nombre 'canónico'"""
    nombres_canonicos = {}
    for loc in lista_localidades:
        if not loc: continue
        coincidencias = difflib.get_close_matches(loc, nombres_canonicos.keys(), n=1, cutoff=similitud_minima)
        if coincidencias:
            nombres_canonicos[loc] = coincidencias[0]
        else:
            nombres_canonicos[loc] = loc
    return nombres_canonicos

def alinear_columnas(df, columnas_esperadas):
    """
    Compara las columnas del df con la lista esperada.
    Crea las faltantes en blanco y ELIMINA las que sobran.
    """
    if not columnas_esperadas:
        return df

    # 1. Crear las columnas que faltan en el DataFrame original
    for col in columnas_esperadas:
        if col not in df.columns:
            df[col] = "" 
            
    # 2. Retornar el DataFrame filtrado SOLAMENTE con las columnas esperadas y en el orden exacto
    return df[columnas_esperadas]

# ==========================================
# 2. SELECCIÓN DE ARCHIVOS Y CARPETAS
# ==========================================

root = tk.Tk()
root.withdraw()

print("Por favor, selecciona el archivo Excel de origen...")
archivo_origen = filedialog.askopenfilename(
    title="Selecciona el archivo Excel (.xlsx o .xls)",
    filetypes=[("Archivos de Excel", "*.xlsx *.xls *")]
)

if not archivo_origen:
    print("No se seleccionó ningún archivo. Cancelando proceso.")
    exit()

print("Por favor, selecciona la carpeta de destino...")
carpeta_destino = filedialog.askdirectory(title="Selecciona la carpeta de destino")

if not carpeta_destino:
    print("No se seleccionó ninguna carpeta. Cancelando proceso.")
    exit()

# ==========================================
# 3. LECTURA Y PROCESAMIENTO DEL EXCEL
# ==========================================

try:
    print(f"\nLeyendo el archivo: {archivo_origen}")
    xls = pd.ExcelFile(archivo_origen)
    nombres_hojas = xls.sheet_names
    
    hoja_sipse = next((h for h in nombres_hojas if h.upper() == "SIPSE"), None)
    hoja_secop = next((h for h in nombres_hojas if h.upper() == "SECOP II"), None)
    
    if not hoja_sipse or not hoja_secop:
        raise ValueError("El archivo no contiene las hojas 'SIPSE' y/o 'SECOP II'.")

    df_sipse = pd.read_excel(xls, sheet_name=hoja_sipse)
    df_secop = pd.read_excel(xls, sheet_name=hoja_secop)
    
    if len(df_sipse.columns) <= 61:
        raise ValueError("La hoja SIPSE no tiene suficientes columnas para llegar a la columna BJ (índice 61).")
        
    # Identificar nombres de columnas clave
    col_sipse_nombre = df_sipse.columns[61] 
    col_secop_nombre = df_secop.columns[0]

    # --- NUEVO CAMBIO: REEMPLAZO ESPECÍFICO DE RAFAEL URIBE ---
    print("Ejecutando reemplazo de nombres de entidad específicos...")
    
    # Reemplazo en SECOP II (Columna A)
    df_secop[col_secop_nombre] = df_secop[col_secop_nombre].replace("RAFAEL URIBE URIBE", "RAFAEL URIBE")
    
    # Reemplazo en SIPSE (Columna BJ / índice 61)
    df_sipse[col_sipse_nombre] = df_sipse[col_sipse_nombre].replace("RAFAEL URIBE URIBE", "RAFAEL URIBE")
    # ---------------------------------------------------------

    print("Alineando y estandarizando columnas según listas esperadas (eliminando sobrantes)...")
    df_sipse = alinear_columnas(df_sipse, COLUMNAS_ESPERADAS_SIPSE)
    df_secop = alinear_columnas(df_secop, COLUMNAS_ESPERADAS_SECOP)
    
    print("Normalizando nombres de localidades...")
    # Como ya alineamos las columnas, garantizamos que existen, pero usamos el nombre extraído previamente
    df_sipse['Loc_Limpia'] = df_sipse[col_sipse_nombre].apply(normalizar_texto)
    df_secop['Loc_Limpia'] = df_secop[col_secop_nombre].apply(normalizar_texto)
    
    todas_localidades = list(set(df_sipse['Loc_Limpia'].unique()) | set(df_secop['Loc_Limpia'].unique()))
    mapa_canonico = unificar_localidades(todas_localidades)
    
    df_sipse['Loc_Final'] = df_sipse['Loc_Limpia'].map(mapa_canonico)
    df_secop['Loc_Final'] = df_secop['Loc_Limpia'].map(mapa_canonico)
    
    localidades_finales = set(mapa_canonico.values())
    
    print(f"Se detectaron {len(localidades_finales)} localidades únicas. Generando archivos...")
    
    archivos_creados = 0
    for loc in localidades_finales:
        if not loc: continue 
        
        # Filtramos y eliminamos las dos columnas temporales (Loc_Limpia y Loc_Final) que usamos para segmentar
        filtro_sipse = df_sipse[df_sipse['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        filtro_secop = df_secop[df_secop['Loc_Final'] == loc].drop(columns=['Loc_Limpia', 'Loc_Final'])
        
        if not filtro_sipse.empty or not filtro_secop.empty:
            nombre_archivo_seguro = re.sub(r'[\\/*?:"<>|]', "", loc)
            ruta_salida = os.path.join(carpeta_destino, f"{nombre_archivo_seguro}.xlsx")
            
            with pd.ExcelWriter(ruta_salida, engine='openpyxl') as writer:
                filtro_sipse.to_excel(writer, sheet_name="SIPSE", index=False)
                filtro_secop.to_excel(writer, sheet_name="SECOP II", index=False)
                
            archivos_creados += 1
            print(f" -> Creado: {nombre_archivo_seguro}.xlsx")

    print(f"\n¡Proceso Finalizado Exitosamente! Se crearon {archivos_creados} archivos.")
    messagebox.showinfo("Éxito", f"Se generaron {archivos_creados} archivos correctamente únicamente con las columnas especificadas.")

except Exception as e:
    print(f"\nOcurrió un error: {e}")
    messagebox.showerror("Error", f"Ocurrió un error procesando el archivo:\n{e}")

finally:
    root.destroy()

In [21]:
#Celda 2.1
import os
import shutil
import openpyxl
from openpyxl.utils import get_column_letter, column_index_from_string
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.formatting.rule import CellIsRule
from collections import defaultdict
import re
import pandas as pd
import difflib # para busqueda aproximada
from rapidfuzz import fuzz, process
import datetime

#Para abrir explorador de archivos
import tkinter as tk
from tkinter import filedialog
import math
import unicodedata

# --- CONFIGURACIÓN DE CARPETAS ---
CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7\08 AGOSTO' #Dell
CARPETA_DESTINO = r"C:\Users\Pc\Desktop\META 7\08 AGOSTO\Diagnóstico" #Dell

# --- RANGOS A OCULTAR ---
RANGOS_OCULTAR_SECOP = ['A-K', 'N-S', 'U-W', 'Z-AH', 'AJ-BR', 'BT-CI']
RANGOS_OCULTAR_ANALISIS = ['A', 'C', 'E-G', 'K', 'N', 'O', 'Q', 'S-AG','AJ-BK', 'BM-BW', 'BY-CD', 'CH-CI', 'CL-CT', 'CV-ED', 'EF-EU']

# --- DEFINICIÓN DE FUNCIONES ---
def ajustar_texto_fila(ws, numero_fila):
    for cell in ws[numero_fila]:
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def ocultar_columnas_por_lista(ws, lista_rangos):
    for item in lista_rangos:
        item = item.upper().strip()
        if "-" in item:
            inicio, fin = item.split("-")
            idx_inicio = column_index_from_string(inicio)
            idx_fin = column_index_from_string(fin)
            for col_idx in range(idx_inicio, idx_fin + 1):
                letra = get_column_letter(col_idx)
                ws.column_dimensions[letra].hidden = True 
        else:
            ws.column_dimensions[item].hidden = True

def aplicar_estilo_ferrocarril(ws):
    fill_header = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    font_header = Font(color="FFFFFF", bold=True, size=11)
    
    for cell in ws[1]:
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = Alignment(horizontal='center', vertical='center')

    colores_ciclo = ["BDD7EE", "DDEBF7", "FFFFFF", "DDEBF7"]
    max_col = ws.max_column
    for i_fila, row in enumerate(ws.iter_rows(min_row=2, max_col=max_col), start=0):
        color_hex = colores_ciclo[i_fila % 4]
        if color_hex != "FFFFFF":
            relleno = PatternFill(start_color=color_hex, end_color=color_hex, fill_type="solid")
            for cell in row:
                cell.fill = relleno

def dejar_columna_blanca(ws, indice_columna):
    relleno_blanco = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=indice_columna, max_col=indice_columna):
        for cell in row:
            cell.fill = relleno_blanco

def aplicar_formato_condicional_alertas(ws, col_inicio_eval):
    relleno_rojo = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    fuente_roja = Font(color="9C0006")
    max_row = ws.max_row
    
    reglas = [
        (12, 'equal', '"Numero contrato NO coincide"'),
        (14, 'equal', '"Cedula NO coincide"'),
        (15, 'greaterThan', '1'),
        (16, 'greaterThan', '1'),
        (17, 'equal', '"No coinciden fechas"'),
        (18, 'equal', '"No coinciden fechas"'),
        (19, 'equal', '"No coinciden fechas"'),
        (20, 'equal', '"Duracion NO coincide"'),
        (21, 'equal', '"Valor NO coincide"'),
        (22, 'equal', '"El numero RP NO coincide"'),
        (23, 'equal', '"El valor RP NO coincide"'),
        (24, 'equal', '"La fecha RP NO coincide"'),
        (25, 'equal', '"El numero CDP NO coincide"'),
        (26, 'equal', '"El valor CDP NO coincide"'),
        (27, 'equal', '"La fecha CDP NO coincide"')
    ]
    
    for offset, operador, valor in reglas:
        col_idx = col_inicio_eval + offset
        letra_col = get_column_letter(col_idx)
        rango = f"{letra_col}2:{letra_col}{max_row}"
        rule = CellIsRule(operator=operador, formula=[valor], stopIfTrue=True, fill=relleno_rojo, font=fuente_roja)
        ws.conditional_formatting.add(rango, rule)

def normalizar_numero_contrato(valor):
    if valor is None: return 0
    texto = str(valor).strip()
    match = re.search(r'\d+', texto)
    if match:
        return int(match.group())
    return 0

def es_error_fecha(val_sipse, val_secop):
    s1 = str(val_sipse).strip() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip() if pd.notna(val_secop) and val_secop is not None else ""
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    
    dt_sipse = None
    if isinstance(val_sipse, (datetime.datetime, datetime.date)):
        dt_sipse = datetime.date(val_sipse.year, val_sipse.month, val_sipse.day)
    else:
        try: dt_sipse = pd.to_datetime(s1.replace("-", "/"), dayfirst=True).date()
        except: pass

    dt_secop = None
    if isinstance(val_secop, (datetime.datetime, datetime.date)):
        try: dt_secop = datetime.date(val_secop.year, val_secop.day, val_secop.month)
        except: pass
    else:
        partes = s2.replace("-", "/").replace(" ", "/").split("/")
        if len(partes) >= 3:
            try:
                y = int(partes[2][-4:])
                m = int(partes[0])
                d = int(partes[1])
                dt_secop = datetime.date(y, m, d)
            except: pass
                
    if dt_sipse is not None and dt_secop is not None:
        return dt_sipse != dt_secop
    return s1 != s2 

def es_error_duracion(val_sipse, val_secop):
    s1 = str(val_sipse).strip().upper() if pd.notna(val_sipse) and val_sipse is not None else ""
    s2 = str(val_secop).strip().upper() if pd.notna(val_secop) and val_secop is not None else ""
    if s1 == "" or s2 == "": return False
    if s1 == "-" or s2 == "-": return False
    return s1 != s2

def color_memo(err_tuple):
    if err_tuple[0]: return "E4DFEC"
    if err_tuple[1]: return "B4C6E7"
    if err_tuple[2]: return "DDEBF7"
    if err_tuple[3]: return "F4CCCC"
    return "FFFFFF"

def normalizar_alfanumerico(valor):
    if pd.isna(valor) or valor is None: return ""
    texto = str(valor).strip().upper()
    if texto.endswith(".0"): texto = texto[:-2]
    return texto

def limpiar_dato(val):
    if pd.isna(val) or val is None: return ""
    if isinstance(val, (datetime.datetime, datetime.date)):
        return val.strftime('%d/%m/%Y')
    val_str = str(val).strip()
    if val_str.endswith(".0"): val_str = val_str[:-2]
    return val_str

def normalizar_cedula(valor):
    val_str = limpiar_dato(valor)
    return re.sub(r'\D', '', val_str)

def obtener_valor_seguro(row, indice):
    try: return limpiar_dato(row.iloc[indice])
    except IndexError: return ""

def mapear_columnas(columnas_df, columnas_buscadas, umbral=80):
    mapa_indices = {}
    for clave, nombre_ideal in columnas_buscadas.items():
        resultado = process.extractOne(nombre_ideal, columnas_df, scorer=fuzz.token_sort_ratio)
        if resultado and resultado[1] >= umbral:
            mapa_indices[clave] = resultado[2]
        else:
            print(f"⚠️ Advertencia: No se encontró la columna para '{nombre_ideal}'.")
            mapa_indices[clave] = None 
    return mapa_indices

def normalizar_texto_localidad(texto):
    if pd.isna(texto) or texto is None: return ""
    t = " ".join(str(texto).upper().split())
    t = ''.join(c for c in unicodedata.normalize('NFD', t) if unicodedata.category(c) != 'Mn')
    return t

def cargar_datos_bogdata():
    """Abre el explorador y procesa RP y CDP. Retorna los diccionarios."""
    root = tk.Tk()
    root.withdraw() 
    root.attributes('-topmost', True) 
    
    print("\n--- SELECCIÓN BOGDATA ---")
    print("1. Por favor selecciona el archivo de CDP (Excel)...")
    ruta_cdp = filedialog.askopenfilename(title="Seleccionar archivo CDP", filetypes=[("Excel files", "*.xlsx *.xls")])
    
    print("2. Por favor selecciona el archivo de RP (Excel)...")
    ruta_rp = filedialog.askopenfilename(title="Seleccionar archivo RP", filetypes=[("Excel files", "*.xlsx *.xls")])
         
    dict_cdp = {}
    dict_rp = defaultdict(list)

    # --- PROCESAR RP ---
    if ruta_rp:
        print(f"  -> Procesando archivo RP...")
        try:
            excel = pd.ExcelFile(ruta_rp)
            hojas_disponibles = excel.sheet_names
            palabras_clave = ["data", "crp"]
            hoja_correcta = next((hoja for hoja in hojas_disponibles if hoja.strip().lower() in palabras_clave), None)

            if hoja_correcta:
                df_rp = pd.read_excel(excel, sheet_name=hoja_correcta)
                print(f"     Hoja detectada y cargada: '{hoja_correcta}'")
            else:
                print(f"  [Error] No se encontró ninguna hoja 'Data' o 'CRP' en el archivo.")
                df_rp = None
        except Exception as e:
            print(f"  [Error] No se pudo leer el archivo Excel: {e}")
            df_rp = None

        if df_rp is not None:
            nombres_ideales_rp = {
                'fecha': 'Fecha Registro', 'num_cdp': 'Número de CDP', 'num_rp': 'Número de CRP',
                'filtro1': 'Rubro', 'localidad': 'Texto Id Proyecto', 'cedula': 'Número Doc. BP Beneficiario',
                'filtro2': 'Anulaciones', 'val_rp': 'Valor Neto', 'cod_int': 'N° Interno CDP'
            }
            idx_rp = mapear_columnas(df_rp.columns.tolist(), nombres_ideales_rp)
            for _, row in df_rp.iterrows():
                val_r = obtener_valor_seguro(row, idx_rp['filtro1'])
                digitos_r = re.sub(r'\D', '', val_r)
                if len(digitos_r) >= 7:
                    val_al = obtener_valor_seguro(row, idx_rp['filtro2'])
                    try: num_al = float(str(val_al).replace(',', ''))
                    except ValueError: num_al = 0.0 
                    
                    if num_al == 0:
                        cedula_raw = obtener_valor_seguro(row, idx_rp['cedula'])
                        llave_cedula = normalizar_cedula(cedula_raw)
                        if llave_cedula:
                            rp_num = obtener_valor_seguro(row, idx_rp['num_rp']) 
                            rp_val = obtener_valor_seguro(row, idx_rp['val_rp']) 
                            rp_fec = obtener_valor_seguro(row, idx_rp['fecha'])  
                            cdp_num = obtener_valor_seguro(row, idx_rp['num_cdp']) 
                            cdp_cod = obtener_valor_seguro(row, idx_rp['cod_int']) 
                            localidad_col_z = obtener_valor_seguro(row, idx_rp['localidad']) 
                            dict_rp[llave_cedula].append((rp_num, rp_val, rp_fec, cdp_num, cdp_cod, localidad_col_z))

    # --- PROCESAR CDP ---
    if ruta_cdp:
        print(f"  -> Procesando archivo CDP...")
        try:
            excel = pd.ExcelFile(ruta_cdp)
            hojas_disponibles = excel.sheet_names
            palabras_clave = ["data", "cdp"]
            hoja_correcta = next((hoja for hoja in hojas_disponibles if hoja.strip().lower() in palabras_clave), None)

            if hoja_correcta:
                df_cdp = pd.read_excel(excel, sheet_name=hoja_correcta)
                print(f"     Hoja detectada y cargada: '{hoja_correcta}'")
            else:
                print(f"  [Error] No se encontró ninguna hoja 'Data' o 'CDP' en el archivo.")
                df_cdp = None
        except Exception as e:
            print(f"  [Error] No se pudo leer el archivo Excel: {e}")
            df_cdp = None

        if df_cdp is not None:
            nombres_ideales_cdp = {
                'num_cdp': 'No. CDP', 'val_cdp': 'Valor', 'cod_int': 'No.Interno CDP', 'fec_cdp': 'Fecha Registro'
            }
            idx_cdp = mapear_columnas(df_cdp.columns.tolist(), nombres_ideales_cdp)
            for _, row in df_cdp.iterrows():
                cdp_num = obtener_valor_seguro(row, idx_cdp['num_cdp'])  
                cdp_cod = obtener_valor_seguro(row, idx_cdp['cod_int']) 
                llave_cdp = (cdp_num, cdp_cod)
                if llave_cdp != ("", "") and llave_cdp not in dict_cdp:
                    cdp_val = obtener_valor_seguro(row, idx_cdp['val_cdp']) 
                    cdp_fec = obtener_valor_seguro(row, idx_cdp['fec_cdp']) 
                    dict_cdp[llave_cdp] = (cdp_num, cdp_val, cdp_fec)

    return dict_cdp, dict_rp

# =======================================================
# EJECUCIÓN DE LA CELDA 1 (Almacenar en memoria)
# =======================================================
print("Iniciando la carga de diccionarios (esto tomará tiempo)...")
dict_cdp_global, dict_rp_global = cargar_datos_bogdata()
print("¡Carga finalizada con éxito! Los diccionarios están en memoria. Ya puedes correr la Celda 2.")
#3m 49s

Iniciando la carga de diccionarios (esto tomará tiempo)...

--- SELECCIÓN BOGDATA ---
1. Por favor selecciona el archivo de CDP (Excel)...
2. Por favor selecciona el archivo de RP (Excel)...
  -> Procesando archivo RP...
     Hoja detectada y cargada: 'Data'
  -> Procesando archivo CDP...
     Hoja detectada y cargada: 'cdp'
¡Carga finalizada con éxito! Los diccionarios están en memoria. Ya puedes correr la Celda 2.


In [22]:
#Celda 2.2
global metricas

def procesar_archivos(dict_cdp_en_memoria, dict_rp_en_memoria):
    global metricas
    if not os.path.exists(CARPETA_DESTINO):
        os.makedirs(CARPETA_DESTINO)

    # 🔹 Aquí mapeamos los diccionarios pre-cargados a las variables locales
    dict_cdp = dict_cdp_en_memoria
    dict_rp = dict_rp_en_memoria

    archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('.xlsx', '.xls'))]
    #archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith(('ENGATIVA.xlsx', 'ENGATIVA.xls'))]
    print(f"Se encontraron {len(archivos)} archivos de Excel.")

    for archivo in archivos:
        ruta_orig = os.path.join(CARPETA_ORIGEN, archivo)
        nombre_base = os.path.splitext(archivo)[0]
        ruta_dest = os.path.join(CARPETA_DESTINO, nombre_base + ".xlsx")
        
        print(f"Procesando: {archivo}...")

        # --- PASO DE CONVERSIÓN SI ES .XLS ---
        if archivo.lower().endswith('.xls'):
            try:
                conversion = pd.read_excel(ruta_orig, sheet_name=None, engine='xlrd')
                with pd.ExcelWriter(ruta_dest, engine='openpyxl') as writer:
                    for nombre_hoja, datos in conversion.items():
                        datos.to_excel(writer, sheet_name=nombre_hoja, index=False)
                print(f"  -> Convertido de .xls a .xlsx con éxito.")
            except Exception as e:
                print(f"  Error convirtiendo {archivo}: {e}")
                continue
        else:
            shutil.copy2(ruta_orig, ruta_dest)

        try:
            wb = openpyxl.load_workbook(ruta_dest)
        except Exception as e:
            print(f"   Error al abrir el archivo con openpyxl: {e}")
            continue

        # ==========================================
        # PARTE 1: Procesar SECOP II (Crear aux1)
        # ==========================================
        nombre_real_secop = next(n for n in wb.sheetnames if n.lower() == 'secop ii')
        ws_origen = wb[nombre_real_secop]
        
        if "SECOP II aux1" in wb.sheetnames:
            del wb["SECOP II aux1"]
            
        ws_secop_aux = wb.copy_worksheet(ws_origen)
        ws_secop_aux.title = "SECOP II aux1"

        filas = []
        for row in ws_secop_aux.iter_rows(min_row=2, values_only=True):
            filas.append(list(row))
        
        filas_ordenadas = sorted(filas, key=lambda x: normalizar_numero_contrato(x[11] if len(x) > 11 else 0))

        for row in ws_secop_aux.iter_rows(min_row=2):
            for cell in row:
                cell.value = None

        for i_fila, datos_fila in enumerate(filas_ordenadas, start=2):
            for i_col, valor in enumerate(datos_fila, start=1):
                ws_secop_aux.cell(row=i_fila, column=i_col, value=valor)

        max_col_secop = ws_secop_aux.max_column
        ws_secop_aux.cell(row=1, column=max_col_secop + 1, value="Columna1")
        ws_secop_aux.cell(row=1, column=max_col_secop + 2, value="Columna2")
        col_L = get_column_letter(12)
        
        for i in range(2, ws_secop_aux.max_row + 1):
            f1 = (
                f'=IFERROR(MID({col_L}{i}, '
                f'MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0), '
                f'MIN(IFERROR(FIND(" ", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999), '
                f'IFERROR(FIND("-", {col_L}{i}, MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), 999)) '
                f'- MATCH(1, INDEX(--ISNUMBER(--MID({col_L}{i}, ROW(INDIRECT("1:" & LEN({col_L}{i}))), 1)), 0), 0)), "")'
            )
            ws_secop_aux.cell(row=i, column=max_col_secop + 1, value=f1)

            f2 = f'=IFERROR(MID({col_L}{i}, FIND("(", {col_L}{i}) + 1, FIND(")", {col_L}{i}) - FIND("(", {col_L}{i}) - 1), "")'
            ws_secop_aux.cell(row=i, column=max_col_secop + 2, value=f2)

        print("  -> Aplicando estilos y ocultando columnas en SECOP II aux1...")
        aplicar_estilo_ferrocarril(ws_secop_aux)
        ocultar_columnas_por_lista(ws_secop_aux, RANGOS_OCULTAR_SECOP)

        # ==========================================
        # PARTE 2 y 3: ALINEACIÓN EN 'ANALISIS'
        # ==========================================
        print("  -> Generando hoja 'Analisis'...")

        nombre_real_sipse = next(n for n in wb.sheetnames if n.lower() == 'sipse')
        ws_sipse = wb[nombre_real_sipse]
        
        if 'Analisis' in wb.sheetnames:
            del wb['Analisis']
        ws_analisis = wb.create_sheet("Analisis")
        
        dict_sipse = defaultdict(list)
        dict_secop = defaultdict(list)
        
        filas_sipse = list(ws_sipse.values)
        header_sipse = filas_sipse[0] if filas_sipse else []
        for fila in filas_sipse[1:]:
            contrato = normalizar_numero_contrato(fila[3] if len(fila) > 3 else 0)
            dict_sipse[contrato].append(fila)

        filas_secop = list(ws_secop_aux.values)
        header_secop = filas_secop[0] if filas_secop else []
        for fila in filas_secop[1:]:
            contrato = normalizar_numero_contrato(fila[11] if len(fila) > 11 else 0)
            dict_secop[contrato].append(fila)

        ws_analisis.append(header_sipse) 
        col_inicio_secop = column_index_from_string("BM")
        for i, val in enumerate(header_secop):
            ws_analisis.cell(row=1, column=col_inicio_secop + i, value=val)

        todos_contratos = sorted(list(set(dict_sipse.keys()) | set(dict_secop.keys())))
        current_row = 2 
        
        for contrato in todos_contratos:
            filas_de_sipse = dict_sipse.get(contrato, [])
            filas_de_secop = dict_secop.get(contrato, [])
            max_filas = max(len(filas_de_sipse), len(filas_de_secop))
            
            for i in range(max_filas):
                if i < len(filas_de_sipse):
                    datos = filas_de_sipse[i]
                    for c_idx, valor in enumerate(datos, start=1):
                        ws_analisis.cell(row=current_row, column=c_idx, value=valor)
                
                if i < len(filas_de_secop):
                    datos = filas_de_secop[i]
                    for c_idx, valor in enumerate(datos):
                        col_excel = col_inicio_secop + c_idx
                        if c_idx == len(datos) - 2 and str(valor).startswith('='):
                             nueva_formula1 = (
                                f'=IFERROR(MID(BX{current_row}, '
                                f'MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0), '
                                f'MIN(IFERROR(FIND(" ", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999), '
                                f'IFERROR(FIND("-", BX{current_row}, MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), 999)) '
                                f'- MATCH(1, INDEX(--ISNUMBER(--MID(BX{current_row}, ROW(INDIRECT("1:" & LEN(BX{current_row}))), 1)), 0), 0)), "")'
                            )
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula1)
                        elif c_idx == len(datos) - 1 and str(valor).startswith('='):
                             nueva_formula2 = f'=IFERROR(MID(BX{current_row}, FIND("(", BX{current_row}) + 1, FIND(")", BX{current_row}) - FIND("(", BX{current_row}) - 1), "")'
                             ws_analisis.cell(row=current_row, column=col_excel, value=nueva_formula2)
                        else:
                            ws_analisis.cell(row=current_row, column=col_excel, value=valor)
                current_row += 1

        # ==========================================
        # PARTE 4: Columnas de Evaluación
        # ==========================================
        print("  -> Agregando columnas de evaluación y estilos...")
        
        ultima_col_actual = ws_analisis.max_column
        col_inicio_eval = ultima_col_actual + 1
        ultima_fila_analisis = ws_analisis.max_row
        
        headers_nuevos = [
            "Columna3\n# RP\n(SIPSE)", "Columna4\nValor RP\n(SIPSE)", "Columna5\nFecha RP\n(SIPSE)", "Columna6\n# CDP\n(SIPSE)", "Columna7\nValor CDP\n(SIPSE)", "Columna8\nFecha CDP\n(SIPSE)", "Col.9\n# RP\n(BOGD)", "Col.10\nValor RP\n(BOGD)", "Col.11\nFecha RP\n(BOGD)", "Col.12\n# CDP\n(BOGD)", "Col.13\nValor CDP\n(BOGD)", "Col.14\nFecha CDP\n(BOGD)",
            "Ev. Nro contrato", "Aux Nro contrato", "Ev. Cédula", "Repeticiones contrato SIPSE", "Repeticiones contrato SECOP", "Fecha suscripción", "Fecha inicio", "Fecha fin", "Duración contratos", "Valor contratos", "Ev. # RP", "Ev. Valor RP", "Ev. Fecha RP", "Ev. # CDP", "Ev. Valor CDP", "Ev. Fecha CDP",
            "Ev. Indicador 1", "Ev. Indicador 2", "Ev. indicador ATI 1"
        ]

        for idx, titulo in enumerate(headers_nuevos):
            ws_analisis.cell(row=1, column=col_inicio_eval + idx, value=titulo)

        # 🔹 Ubicar columnas para el indicador ATI 1 (búsqueda por nombre, con respaldo fijo)
        def buscar_columna(ws, nombre, col_min, col_max):
            objetivo = normalizar_texto_localidad(nombre)
            for c in range(col_min, col_max + 1):
                if normalizar_texto_localidad(ws.cell(row=1, column=c).value) == objetivo:
                    return get_column_letter(c)
            return None

        letra_modif_sipse = buscar_columna(ws_analisis, "MODIFICACION", 1, col_inicio_secop - 1) or "AK"
        letra_estado_secop = buscar_columna(ws_analisis, "Estado Contrato", col_inicio_secop, col_inicio_eval - 1) or "BY"
        print(f"  -> ATI 1: MODIFICACION en {letra_modif_sipse}, Estado Contrato en {letra_estado_secop}")
        
        for i in range(2, ultima_fila_analisis + 1):
            f3 = f'=IFERROR(TRIM(RIGHT(SUBSTITUTE(TRIM(LEFT(AI{i}, FIND("Fecha", AI{i})-1)), " ", REPT(" ", 100)), 100)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval, value=f3)
            f4 = f'=IFERROR(TRIM(MID(AI{i}, FIND("Valor ", AI{i}) + 6, IFERROR(FIND(";", AI{i}), LEN(AI{i})+1) - FIND("Valor ", AI{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 1, value=f4)
            f5 = f'=IFERROR(TRIM(MID(AI{i}, FIND("Fecha ", AI{i}) + 6, FIND(" Valor", AI{i}) - FIND("Fecha ", AI{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 2, value=f5)
            f6 = f'=IFERROR(TRIM(RIGHT(SUBSTITUTE(TRIM(LEFT(AH{i}, FIND("Fecha", AH{i})-1)), " ", REPT(" ", 100)), 100)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 3, value=f6)
            f7 = f'=IFERROR(TRIM(MID(AH{i}, FIND("Valor ", AH{i}) + 6, IFERROR(FIND(";", AH{i}), LEN(AH{i})+1) - FIND("Valor ", AH{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 4, value=f7)
            f8 = f'=IFERROR(TRIM(MID(AH{i}, FIND("Fecha ", AH{i}) + 6, FIND(" Valor", AH{i}) - FIND("Fecha ", AH{i}) - 6)), "")'
            ws_analisis.cell(row=i, column=col_inicio_eval + 5, value=f8)

        idx_col_cedula_cj = column_index_from_string("CJ")
        letra_ev_nro_contrato = get_column_letter(col_inicio_eval + 12) 
        letra_aux_nro_contrato = get_column_letter(col_inicio_eval + 13) 
        letra_ev_nro_cedula = get_column_letter(col_inicio_eval + 14) 
        
        localidad_actual_norm = normalizar_texto_localidad(nombre_base)
        
        for i in range(2, ultima_fila_analisis + 1):
            cedula_actual_raw = ws_analisis.cell(row=i, column=idx_col_cedula_cj).value
            cedula_busqueda = normalizar_cedula(cedula_actual_raw)
            
            lista_rps = dict_rp.get(cedula_busqueda, [])
            rp_data_elegido = ("", "", "", "", "")
            
            if len(lista_rps) == 1:
                rp_data_elegido = lista_rps[0][:5]
            elif len(lista_rps) > 1:
                for rp in lista_rps:
                    localidad_rp_norm = normalizar_texto_localidad(rp[5]) 
                    if fuzz.partial_ratio(localidad_actual_norm.lower(), localidad_rp_norm.lower()) > 85:
                        rp_data_elegido = rp[:5]
                        break

            rp_num, rp_val, rp_fec, cdp_num_from_rp, cdp_cod_from_rp = rp_data_elegido
            cdp_data = dict_cdp.get((cdp_num_from_rp, cdp_cod_from_rp), ("", "", ""))
            cdp_num_out, cdp_val_out, cdp_fec_out = cdp_data

            def inyectar_condicional(valor):
                val_str = str(valor).replace('"', '""') if valor else ""
                return f'=IF(OR({letra_ev_nro_contrato}{i}="Numero contrato coincide", {letra_ev_nro_contrato}{i}="Contrato coincide sin SIPSE", {letra_ev_nro_cedula}{i}="Cedula coincide"), "{val_str}", "")'

            ws_analisis.cell(row=i, column=col_inicio_eval + 6, value=inyectar_condicional(rp_num))
            ws_analisis.cell(row=i, column=col_inicio_eval + 7, value=inyectar_condicional(rp_val))
            ws_analisis.cell(row=i, column=col_inicio_eval + 8, value=inyectar_condicional(rp_fec))
            ws_analisis.cell(row=i, column=col_inicio_eval + 9, value=inyectar_condicional(cdp_num_out))
            ws_analisis.cell(row=i, column=col_inicio_eval + 10, value=inyectar_condicional(cdp_val_out))
            ws_analisis.cell(row=i, column=col_inicio_eval + 11, value=inyectar_condicional(cdp_fec_out))

        for i in range(2, ultima_fila_analisis + 1):
            COL_SOLICITUD = "B"
            f15 = f'''=IF(COUNTIF(${letra_aux_nro_contrato}$2:{letra_aux_nro_contrato}{i},{letra_aux_nro_contrato}{i})>1,"",
                IF(AND(D{i}<>"",EV{i}<>""),
                    IF(EW{i}="","Contrato coincide sin SIPSE",
                        IF(TRIM(SUBSTITUTE(UPPER({COL_SOLICITUD}{i}),"SIPSE",""))=TRIM(SUBSTITUTE(UPPER(EW{i}),"SIPSE","")),"Numero contrato coincide","Numero contrato NO coincide")
                    ),
                    IF(D{i}<>"","Contrato no existe en SECOP",
                        IF(EV{i}<>"","Contrato no existe en SIPSE","No existe")
                    )
                )
            )'''
            f15 = f15.replace('\n', '').replace('    ', '')
            ws_analisis.cell(row=i, column=col_inicio_eval + 12, value=f15)

            f16 = f'=IF(D{i}<>"",D{i},EV{i})'
            ws_analisis.cell(row=i, column=col_inicio_eval + 13, value=f16)

            f17 = f'=IF(AND(BY{i}="",M{i}=""),"",IF(OR(BY{i}="",M{i}=""),"",IF(BY{i}="-","Sin cedula en SECOP",IF(M{i}="-","Sin cedula en SIPSE",IF(VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(CJ{i}),"CC",""),"NIT",""),"TI",""))=VALUE(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(UPPER(M{i}),"CC",""),"NIT",""),"TI","")),"Cedula coincide","Cedula NO coincide")))))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 14, value=f17)

            f18 = f'=IF({letra_ev_nro_contrato}{i}="","",COUNTIF(D$2:D${ultima_fila_analisis},D{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 15, value=f18)

            f19 = f'=IF({letra_ev_nro_contrato}{i}="","",COUNTIF(EV$2:EV${ultima_fila_analisis},EV{i}))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 16, value=f19)

            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CE{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CE{i}), DAY(CE{i}), MONTH(CE{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f20 = f'''=IF(AND(H{i}="",CE{i}=""),"",IF(AND(H{i}="-",CE{i}="-"),"Ambos sin fechas",IF(H{i}="-","sin fecha SIPSE",IF(CE{i}="-","sin fecha SECOP",IF(OR(H{i}="",CE{i}=""),"",IF(INT(VALUE(SUBSTITUTE(H{i},"-","/"))) =INT(IF(ISNUMBER(CE{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 17, value=f20)

            # 🔹 FECHA ACTA DE INICIO
            cf_clean = f'SUBSTITUTE(SUBSTITUTE(CF{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CF{i}), DAY(CF{i}), MONTH(CF{i}))'
            parte_es_texto = f'DATE(RIGHT({cf_clean}, 4), LEFT({cf_clean}, FIND("/", {cf_clean}) - 1), MID({cf_clean}, FIND("/", {cf_clean}) + 1, FIND("/", {cf_clean}, FIND("/", {cf_clean}) + 1) - FIND("/", {cf_clean}) - 1))'

            f21 = f'''=IF(AND(I{i}="",CF{i}=""),"",IF(AND(I{i}="-",CF{i}="-"),"Ambos sin fechas",IF(I{i}="-","sin fecha SIPSE",IF(CF{i}="-","sin fecha SECOP",IF(OR(I{i}="",CF{i}=""),"",IF(INT(VALUE(SUBSTITUTE(I{i},"-","/"))) =INT(IF(ISNUMBER(CF{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 18, value=f21)

            cg_clean = f'SUBSTITUTE(SUBSTITUTE(CG{i}, "-", "/"), " ", "/")'
            parte_es_numero = f'DATE(YEAR(CG{i}), DAY(CG{i}), MONTH(CG{i}))'
            parte_es_texto = f'DATE(RIGHT({cg_clean}, 4), LEFT({cg_clean}, FIND("/", {cg_clean}) - 1), MID({cg_clean}, FIND("/", {cg_clean}) + 1, FIND("/", {cg_clean}, FIND("/", {cg_clean}) + 1) - FIND("/", {cg_clean}) - 1))'

            f22 = f'''=IF(AND(J{i}="",CG{i}=""),"",IF(AND(J{i}="-",CG{i}="-"),"Ambos sin fechas",IF(J{i}="-","sin fecha SIPSE",IF(CG{i}="-","sin fecha SECOP",IF(OR(J{i}="",CG{i}=""),"",IF(INT(VALUE(SUBSTITUTE(J{i},"-","/"))) =INT(IF(ISNUMBER(CG{i}),{parte_es_numero},{parte_es_texto})),"Coincide fecha","No coinciden fechas"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 19, value=f22)

            f23 = f'''=IF(AND(R{i}="",EE{i}=""),"",IF(AND(R{i}="-",EE{i}="-"),"Ambos sin duracion",IF(R{i}="-","sin duracion en SECOP",IF(EE{i}="-","sin duracion en SIPSE",IF(OR(R{i}="",EE{i}=""),"",IF(TRIM(UPPER(R{i}))=TRIM(UPPER(EE{i})),"Duracion coincide","Duracion NO coincide"))))))'''
            ws_analisis.cell(row=i, column=col_inicio_eval + 20, value=f23)

            f24 = f'''=IF(AND(P{i}="",CU{i}=""),"",
                IF(AND(P{i}="-",CU{i}="-"),"Ambos sin valor",
                IF(P{i}="-","Sin valor en SECOP",
                IF(CU{i}="-","Sin valor en SIPSE",
                IF(OR(P{i}="",CU{i}=""),"",
                IF(P{i}=CU{i},"Valor coincide","Valor NO coincide"))))))'''

            ws_analisis.cell(row=i, column=col_inicio_eval + 21, value=f24)

            c_rp_sip   = get_column_letter(col_inicio_eval)       
            c_val_rp_s = get_column_letter(col_inicio_eval + 1)   
            c_fec_rp_s = get_column_letter(col_inicio_eval + 2)   
            c_cdp_sip  = get_column_letter(col_inicio_eval + 3)   
            c_val_cd_s = get_column_letter(col_inicio_eval + 4)   
            c_fec_cd_s = get_column_letter(col_inicio_eval + 5)   
            
            c_rp_bog   = get_column_letter(col_inicio_eval + 6)   
            c_val_rp_b = get_column_letter(col_inicio_eval + 7)   
            c_fec_rp_b = get_column_letter(col_inicio_eval + 8)   
            c_cdp_bog  = get_column_letter(col_inicio_eval + 9)   
            c_val_cd_b = get_column_letter(col_inicio_eval + 10)  
            c_fec_cd_b = get_column_letter(col_inicio_eval + 11)  

            f25 = (f'=IF(AND(TRIM({c_rp_sip}{i})="", TRIM({c_rp_bog}{i})=""), "Sin numeros RP", '
                   f'IF(TRIM({c_rp_sip}{i})="", "Sin numero RP en SIPSE", '
                   f'IF(TRIM({c_rp_bog}{i})="", "Sin numero RP en BOGDATA", '
                   f'IF(TRIM({c_rp_sip}{i})=TRIM({c_rp_bog}{i}), "El numero RP coincide", "El numero RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 22, value=f25)

            f26 = (f'=IF(AND(TRIM({c_val_rp_s}{i})="", TRIM({c_val_rp_b}{i})=""), "Sin valores RP", '
                   f'IF(TRIM({c_val_rp_s}{i})="", "Sin valor RP en SIPSE", '
                   f'IF(TRIM({c_val_rp_b}{i})="", "Sin valor RP en BOGDATA", '
                   f'IF(IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_rp_s}{i}), ",", "")),TRIM({c_val_rp_s}{i}))=IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_rp_b}{i}), ",", "")),TRIM({c_val_rp_b}{i})), "El valor RP coincide", "El valor RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 23, value=f26)

            f27 = (f'=IF(AND(TRIM({c_fec_rp_s}{i})="", TRIM({c_fec_rp_b}{i})=""), "Sin fechas RP", '
                   f'IF(TRIM({c_fec_rp_s}{i})="", "Sin fecha RP en SIPSE", '
                   f'IF(TRIM({c_fec_rp_b}{i})="", "Sin fecha RP en BOGDATA", '
                   f'IF(TRIM({c_fec_rp_s}{i})=TRIM({c_fec_rp_b}{i}), "La fecha RP coincide", "La fecha RP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 24, value=f27)

            f28 = (f'=IF(AND(TRIM({c_cdp_sip}{i})="", TRIM({c_cdp_bog}{i})=""), "Sin numeros CDP", '
                   f'IF(TRIM({c_cdp_sip}{i})="", "Sin numero CDP en SIPSE", '
                   f'IF(TRIM({c_cdp_bog}{i})="", "Sin numero CDP en BOGDATA", '
                   f'IF(TRIM({c_cdp_sip}{i})=TRIM({c_cdp_bog}{i}), "El numero CDP coincide", "El numero CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 25, value=f28)

            f29 = (f'=IF(AND(TRIM({c_val_cd_s}{i})="", TRIM({c_val_cd_b}{i})=""), "Sin valores CDP", '
                   f'IF(TRIM({c_val_cd_s}{i})="", "Sin valor CDP en SIPSE", '
                   f'IF(TRIM({c_val_cd_b}{i})="", "Sin valor CDP en BOGDATA", '
                   f'IF(IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_cd_s}{i}), ",", "")),TRIM({c_val_cd_s}{i}))=IFERROR(VALUE(SUBSTITUTE(TRIM({c_val_cd_b}{i}), ",", "")),TRIM({c_val_cd_b}{i})), "El valor CDP coincide", "El valor CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 26, value=f29)

            f30 = (f'=IF(AND(TRIM({c_fec_cd_s}{i})="", TRIM({c_fec_cd_b}{i})=""), "Sin fechas CDP", '
                   f'IF(TRIM({c_fec_cd_s}{i})="", "Sin fecha CDP en SIPSE", '
                   f'IF(TRIM({c_fec_cd_b}{i})="", "Sin fecha CDP en BOGDATA", '
                   f'IF(TRIM({c_fec_cd_s}{i})=TRIM({c_fec_cd_b}{i}), "La fecha CDP coincide", "La fecha CDP NO coincide"))))')
            ws_analisis.cell(row=i, column=col_inicio_eval + 27, value=f30)

            # 🔹 Indicador 1 de Completitud
            f31 = f'=IF(B{i}="", "", (IF(I{i}<>"-", 1, 0) + IF(M{i}<>"-", 1, 0) + IF(V{i}<>"-", 1, 0) + IF(AP{i}<>"-", 1, 0)) / 4)'
            ws_analisis.cell(row=i, column=col_inicio_eval + 28, value=f31)

            # 🔹 Indicador 2 de Exactitud
            f32 = f'=IF(B{i}="", "", IF(AND(FS{i}="Valor coincide", FR{i}="Duracion coincide", OR(FJ{i}="Numero contrato coincide", FJ{i}="Contrato coincide sin SIPSE"), FP{i}="Coincide fecha", FL{i}="Cedula coincide"), "Exacto", "No Exacto"))'
            ws_analisis.cell(row=i, column=col_inicio_eval + 29, value=f32)

            # 🔹 Indicador ATI 1: hay modificación en SIPSE y SECOP dice cedido/modificado
            f33 = (f'=IF(AND(TRIM({letra_modif_sipse}{i})<>"", TRIM({letra_modif_sipse}{i})<>"-", '
                   f'OR(ISNUMBER(SEARCH("cedido",{letra_estado_secop}{i})), '
                   f'ISNUMBER(SEARCH("modificado",{letra_estado_secop}{i})))), 1, 0)')
            ws_analisis.cell(row=i, column=col_inicio_eval + 30, value=f33)

        print("  -> Aplicando estilos y ocultando columnas en Analisis...")
        aplicar_estilo_ferrocarril(ws_analisis)
        aplicar_formato_condicional_alertas(ws_analisis, col_inicio_eval)
        ocultar_columnas_por_lista(ws_analisis, RANGOS_OCULTAR_ANALISIS)

        # ==========================================
        # PARTE 5: Crear HOJA RESULTADOS
        # ==========================================
        print("  -> Creando hoja 'Resultados'...")
        if 'Resultados' in wb.sheetnames:
            del wb['Resultados']
        ws_res = wb.create_sheet("Resultados")
        nombre_limpio = os.path.splitext(archivo)[0]
        ws_res['B1'] = f"Métricas alcaldía local {nombre_limpio}"
        
        letra_ev_nro = get_column_letter(col_inicio_eval + 12)       
        letra_ev_ced = get_column_letter(col_inicio_eval + 14)   
        letra_rep_sipse = get_column_letter(col_inicio_eval + 15)
        letra_rep_secop = get_column_letter(col_inicio_eval + 16)
        letra_fecha_suscripcion = get_column_letter(col_inicio_eval + 17)
        letra_fecha_inicio = get_column_letter(col_inicio_eval + 18)
        letra_fecha_fin = get_column_letter(col_inicio_eval + 19)
        letra_duracion = get_column_letter(col_inicio_eval + 20)
        letra_valor = get_column_letter(col_inicio_eval + 21)

        letra_num_RP = get_column_letter(col_inicio_eval + 22)
        letra_val_RP = get_column_letter(col_inicio_eval + 23)
        letra_fec_RP = get_column_letter(col_inicio_eval + 24)
        letra_num_CDP = get_column_letter(col_inicio_eval + 25)
        letra_val_CDP = get_column_letter(col_inicio_eval + 26)
        letra_fec_CDP = get_column_letter(col_inicio_eval + 27)

        letra_ind_1 = get_column_letter(col_inicio_eval + 28)
        letra_ind_2 = get_column_letter(col_inicio_eval + 29)
        letra_ind_ati1 = get_column_letter(col_inicio_eval + 30)

        metricas = [
            ("Nro de contratos registrados SIPSE",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos registrados SECOP",f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos registrados en SECOP y SIPSE con numero de contrato igual", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato coincide\")+COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato coincide sin SIPSE\")"),
            ("Nro de contratos solo en SECOP y no en SIPSE", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SIPSE\")"),
            ("Nro de contratos solo en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Contrato no existe en SECOP\")"),
            ("Nro de contratos diferentes en SIPSE-SECOP", f"=COUNTIF('Analisis'!{letra_ev_nro}:{letra_ev_nro}, \"Numero contrato NO coincide\")"),
            ("Nro de contratos Cédula diferente", f"=COUNTIF('Analisis'!{letra_ev_ced}:{letra_ev_ced}, \"Cedula NO coincide\")"),
            ("Nro de contratos dobles en SIPSE", f"=COUNTIF('Analisis'!{letra_rep_sipse}:{letra_rep_sipse}, 2)"),
            ("Nro de contratos dobles en SECOP", f"=COUNTIF('Analisis'!{letra_rep_secop}:{letra_rep_secop}, 2)"),
            ("Fecha de acta de suscriccpión Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"Coincide fecha\")"),
            ("Fecha de acta de suscriccpión no Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_suscripcion}:{letra_fecha_suscripcion}, \"No coinciden fechas\")"),
            ("Fecha de acta de inicio Coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Coincide fecha\")"),
            ("Fecha de acta de inicio no coinciden", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"No coinciden fechas\")"),
            ("Fecha de Fin conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"Coincide fecha\")"),
            ("Fecha de Fin no conciden", f"=COUNTIF('Analisis'!{letra_fecha_fin}:{letra_fecha_fin}, \"No coinciden fechas\")"),
            ("Contratos que coinciden en Duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion Coincide\")"),
            ("Contratos que tienen diferente duración", f"=COUNTIF('Analisis'!{letra_duracion}:{letra_duracion}, \"Duracion NO coincide\")"),
            ("Contratos que tienen Fecha de inicio en SIPSE y no en SECOP", f"=COUNTIF('Analisis'!{letra_fecha_inicio}:{letra_fecha_inicio}, \"Sin fecha SECOP\")"),
            ("Datos en la matriz de seguimiento", f"=C2+C3"),
            ("Valor contrato coinciden", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor coincide\")"),
            ("Valor contrato no coincidenn", f"=COUNTIF('Analisis'!{letra_valor}:{letra_valor}, \"Valor NO coincide\")"),
            ("Numero RP no coinciden", f"=COUNTIF('Analisis'!{letra_num_RP}:{letra_num_RP}, \"El numero RP NO coincide\")"),
            ("Valor RP no coinciden", f"=COUNTIF('Analisis'!{letra_val_RP}:{letra_val_RP}, \"El valor RP NO coincide\")"),
            ("Fecha RP no coinciden", f"=COUNTIF('Analisis'!{letra_fec_RP}:{letra_fec_RP}, \"La fecha RP NO coincide\")"),
            ("Numero CDP no coinciden", f"=COUNTIF('Analisis'!{letra_num_CDP}:{letra_num_CDP}, \"El numero CDP NO coincide\")"),
            ("Valor CDP no coinciden", f"=COUNTIF('Analisis'!{letra_val_CDP}:{letra_val_CDP}, \"El valor CDP NO coincide\")"),
            ("Fecha CDP no coinciden", f"=COUNTIF('Analisis'!{letra_fec_CDP}:{letra_fec_CDP}, \"La fecha CDP NO coincide\")"),
            ("Indicador completitud", f"=AVERAGE('Analisis'!{letra_ind_1}:{letra_ind_1})"),
            ("Indicador exactitud", f"=COUNTIF('Analisis'!{letra_ind_2}:{letra_ind_2}, \"Exacto\") / (COUNTIF('Analisis'!{letra_ind_2}:{letra_ind_2}, \"Exacto\") + COUNTIF('Analisis'!{letra_ind_2}:{letra_ind_2}, \"No Exacto\"))"),
            ("Indicador Asistencia Técnica Integral 1", f"=IFERROR(COUNTIF('Analisis'!{letra_ind_ati1}:{letra_ind_ati1}, 1) / (COUNTIF('Analisis'!{letra_estado_secop}:{letra_estado_secop}, \"*cedido*\") + COUNTIF('Analisis'!{letra_estado_secop}:{letra_estado_secop}, \"*modificado*\")), \"Sin modificaciones en SECOP\")")
        ]

        for idx, (texto, formula) in enumerate(metricas, start=2):
            ws_res.cell(row=idx, column=1, value=idx-1)
            ws_res.cell(row=idx, column=2, value=texto)
            ws_res.cell(row=idx, column=3, value=formula)
        
        fill_amarillo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
        ws_res['B1'].fill = fill_amarillo
        ws_res.column_dimensions['B'].width = 100
        
        ws_analisis.column_dimensions['FJ'].width = 29
        ws_analisis.column_dimensions['FK'].width = 10
        ws_analisis.column_dimensions['FL'].width = 20
        ws_analisis.column_dimensions['FM'].width = 15
        ws_analisis.column_dimensions['FN'].width = 15
        ws_analisis.column_dimensions['FO'].width = 15
        ws_analisis.column_dimensions['FP'].width = 20
        ws_analisis.column_dimensions['FQ'].width = 20
        ws_analisis.column_dimensions['FR'].width = 18
        ws_analisis.column_dimensions['FS'].width = 14
        ws_analisis.column_dimensions['FT'].width = 25
        ws_analisis.column_dimensions['FU'].width = 25
        ws_analisis.column_dimensions['FV'].width = 25
        ws_analisis.column_dimensions['FW'].width = 25
        ws_analisis.column_dimensions['FX'].width = 25
        ws_analisis.column_dimensions['FY'].width = 25
        ws_analisis.column_dimensions[letra_ind_ati1].width = 14

        dejar_columna_blanca(ws_analisis, 64)

        ajustar_texto_fila(ws_secop_aux, 1)
        ajustar_texto_fila(ws_analisis, 1)

        # ==========================================
        # PARTE 6: Crear HOJA 'Para MEMO'
        # ==========================================
        print("  -> Creando hoja 'Para MEMO'...")
        if 'Para MEMO' in wb.sheetnames:
            del wb['Para MEMO']
        ws_memo = wb.create_sheet("Para MEMO")
        
        datos_memo = []
        
        for i in range(2, ultima_fila_analisis + 1):
            val_b = ws_analisis.cell(row=i, column=2).value 
            val_d = ws_analisis.cell(row=i, column=4).value 
            
            susc_sipse = ws_analisis.cell(row=i, column=8).value  
            susc_secop = ws_analisis.cell(row=i, column=83).value 
            ini_sipse  = ws_analisis.cell(row=i, column=9).value  
            ini_secop  = ws_analisis.cell(row=i, column=84).value 
            fin_sipse  = ws_analisis.cell(row=i, column=10).value 
            fin_secop  = ws_analisis.cell(row=i, column=85).value 
            dur_sipse  = ws_analisis.cell(row=i, column=18).value 
            dur_secop  = ws_analisis.cell(row=i, column=135).value
            
            err_susc = es_error_fecha(susc_sipse, susc_secop)
            err_ini  = es_error_fecha(ini_sipse, ini_secop)
            err_fin  = es_error_fecha(fin_sipse, fin_secop)
            err_dur  = es_error_duracion(dur_sipse, dur_secop)
            
            if err_susc or err_ini or err_fin or err_dur:
                llave_orden = (err_susc, err_ini, err_fin, err_dur)
                datos_memo.append([
                    val_b, val_d,
                    "X" if err_susc else "",
                    "X" if err_ini else "",
                    "X" if err_fin else "",
                    "X" if err_dur else "",
                    llave_orden 
                ])
                
        datos_memo.sort(key=lambda x: x[6], reverse=True)
        
        relleno_rojo = PatternFill(start_color="C00000", end_color="C00000", fill_type="solid")
        fuente_blanca = Font(color="FFFFFF", bold=True)
        ws_memo.merge_cells("A1:A2")
        ws_memo['A1'] = "SIPSE"
        ws_memo.merge_cells("B1:B2")
        ws_memo['B1'] = "No. Contrato"
        ws_memo.merge_cells("C1:F1")
        ws_memo['C1'] = "TIPOLOGÍA DE LA INCONSISTENCIA"
        
        ws_memo['C2'] = "Fecha suscripción"
        ws_memo['D2'] = "Fecha Inicio"
        ws_memo['E2'] = "Fecha Fin"
        ws_memo['F2'] = "Plazo"
        ws_memo['A1'].fill = relleno_rojo
        ws_memo['B1'].fill = relleno_rojo
        ws_memo['C1'].fill = relleno_rojo
        ws_memo['C2'].fill = relleno_rojo
        ws_memo['D2'].fill = relleno_rojo
        ws_memo['E2'].fill = relleno_rojo
        ws_memo['F2'].fill = relleno_rojo
        
        align_header = Alignment(horizontal="center", vertical="center")
        borde_delgado = Border(left=Side(style='thin', color='000000'), right=Side(style='thin', color='000000'), top=Side(style='thin', color='000000'), bottom=Side(style='thin', color='000000'))
        
        for fila_h in ws_memo['A1:F2']:
            for celda in fila_h:
                celda.font = fuente_blanca
                celda.alignment = align_header
                celda.border = borde_delgado
                
        fila_actual = 3
        for fila_datos in datos_memo:
            for col_idx in range(1, 7):
                celda = ws_memo.cell(row=fila_actual, column=col_idx, value=fila_datos[col_idx-1])
                celda.border = borde_delgado
                celda.alignment = align_header
                
            fila_actual += 1
            
        ws_memo.column_dimensions['A'].width = 15
        ws_memo.column_dimensions['B'].width = 15
        ws_memo.column_dimensions['C'].width = 20
        ws_memo.column_dimensions['D'].width = 15
        ws_memo.column_dimensions['E'].width = 15
        ws_memo.column_dimensions['F'].width = 15

        wb.save(ruta_dest)
        print("  -> Procesado exitosamente.")


# =======================================================
# EJECUCIÓN DE LA CELDA 2
# =======================================================
# Le pasamos a la función los diccionarios que ya quedaron cargados
# en la memoria de Jupyter desde la Celda 1.
procesar_archivos(dict_cdp_global, dict_rp_global)
#26m 30s

Se encontraron 21 archivos de Excel.
Procesando: ANTONIO NARINO.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> ATI 1: MODIFICACION en AK, Estado Contrato en BY
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 'Para MEMO'...
  -> Procesado exitosamente.
Procesando: BARRIOS UNIDOS.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -> ATI 1: MODIFICACION en AK, Estado Contrato en BY
  -> Aplicando estilos y ocultando columnas en Analisis...
  -> Creando hoja 'Resultados'...
  -> Creando hoja 'Para MEMO'...
  -> Procesado exitosamente.
Procesando: BOSA.xlsx...
  -> Aplicando estilos y ocultando columnas en SECOP II aux1...
  -> Generando hoja 'Analisis'...
  -> Agregando columnas de evaluación y estilos...
  -

In [ ]:
import os
import win32com.client as win32
import re
import unicodedata

# ================= CONFIGURACIÓN =================
CARPETA_ORIGEN = r'C:\Users\Pc\Desktop\META 7\09 SEPTIEMBRE\Diagnóstico'
ARCHIVO_DESTINO = r"C:\Users\Pc\Desktop\META 7\09 SEPTIEMBRE\Diagnóstico\Consolidado\Consolidado_Localidades.xlsx"

nombres_hojas = [
    "1-Usaquén", "2-Chapinero", "3-Santafe", "4-San Cristóbal",
    "5-Usme", "6-Tunjuelito", "7-Bosa", "8-Kennedy", "9-Fontibón",
    "10-Engativá", "11-Suba", "12-Barrios Unidos", "13-Teusaquillo",
    "14-Mártires", "15-Antonio Nariño", "16-Puente Aranda",
    "17-Rafael Uribe", "18-Candelaria", "19-Ciudad Bolívar", "20-Sumapaz"
]

# ================= FUNCIONES AUXILIARES =================
def normalizar_texto(texto):
    if not texto: return ""
    texto = str(texto).lower()
    texto = ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'^\d+[-_ ]?', '', texto)
    # Elimina espacios, guiones y barras para unificar "Santa Fe" y "Santafe"
    texto = re.sub(r'[\s_-]+', '', texto)
    return texto

def encontrar_archivo_para_hoja(nombre_hoja, archivos):
    """Busca el archivo que coincida con la hoja específica respetando el orden"""
    for archivo in archivos:
        nombre_base = os.path.splitext(archivo)[0]
        if normalizar_texto(nombre_hoja) in normalizar_texto(nombre_base):
            return archivo
    return None

def color_rgb(hex_str):
    hex_str = hex_str.lstrip('#')
    r, g, b = tuple(int(hex_str[i:i+2], 16) for i in (0, 2, 4))
    return r + (g * 256) + (b * 65536)

# ================= PROGRAMA PRINCIPAL =================
def consolidar_resultados_fantasma():
    print("Iniciando Excel en segundo plano (Modo Fantasma)...")
    excel = win32.DispatchEx("Excel.Application")
    excel.Visible = False 
    excel.DisplayAlerts = False

    try:
        wb_nuevo = excel.Workbooks.Add()
        ws_general = wb_nuevo.Sheets(1)
        ws_general.Name = "General"

        # Títulos hoja General
        ws_general.Cells(1, 1).Value = "Nro"
        ws_general.Cells(1, 2).Value = "Alcaldía local"

        # Pre-llenar filas de General
        for i, hoja in enumerate(nombres_hojas, start=2):
            num, nombre = hoja.split("-", 1)
            ws_general.Cells(i, 1).Value = int(num)
            ws_general.Cells(i, 2).Value = nombre.strip()
            ws_general.Hyperlinks.Add(Anchor=ws_general.Cells(i, 2), Address="", SubAddress=f"'{hoja}'!A1")

        archivos = [f for f in os.listdir(CARPETA_ORIGEN) if f.endswith('.xlsx') and not f.startswith('~')]
        metricas_extraidas = False

        # Iterar estrictamente en el orden ascendente de la lista nombres_hojas
        for hoja_destino_nombre in nombres_hojas:
            archivo = encontrar_archivo_para_hoja(hoja_destino_nombre, archivos)
            
            if not archivo:
                print(f"  - No se encontró archivo para la hoja: '{hoja_destino_nombre}'")
                continue

            print(f"-> Procesando métricas de: '{archivo}' -> Destino: '{hoja_destino_nombre}'")
            ruta_completa = os.path.join(CARPETA_ORIGEN, archivo)

            # Extraer números fijos
            wb_origen = excel.Workbooks.Open(ruta_completa, ReadOnly=True)
            try:
                ws_origen = wb_origen.Sheets("Resultados")
            except Exception:
                print(f"  - Faltó hoja 'Resultados' en {archivo}")
                wb_origen.Close(SaveChanges=False)
                continue

            ultima_fila = ws_origen.Cells(ws_origen.Rows.Count, "A").End(-4162).Row
            
            # Crear pestaña local en el orden correcto secuencial
            ws_destino = wb_nuevo.Sheets.Add(After=wb_nuevo.Sheets(wb_nuevo.Sheets.Count))
            ws_destino.Name = hoja_destino_nombre

            # Copiar y pegar tabla como VALORES
            ws_origen.Range(f"A1:C{ultima_fila}").Copy()
            ws_destino.Range("A1").PasteSpecial(Paste=-4163) # Valores
            ws_destino.Range("A1").PasteSpecial(Paste=-4122) # Formatos
            
            # Botón Volver en la celda A1
            celda_volver = ws_destino.Cells(1, 1)
            celda_volver.Value = "General"
            ws_destino.Hyperlinks.Add(Anchor=celda_volver, Address="", SubAddress="'General'!A1")
            celda_volver.Font.Color = color_rgb("0563C1")
            celda_volver.Font.Underline = 2 
            celda_volver.Font.Bold = True
            celda_volver.Interior.Color = color_rgb("E7E6E6") 
            celda_volver.HorizontalAlignment = -4108 
            celda_volver.VerticalAlignment = -4108
            
            ws_destino.Columns.AutoFit()

            # Extraer números fijos a la hoja General
            fila_gen = nombres_hojas.index(hoja_destino_nombre) + 2
            col_gen = 3
            
            for r in range(2, ultima_fila + 1):
                titulo_metrica = ws_destino.Cells(r, 2).Value
                valor_metrica = ws_destino.Cells(r, 3).Value
                
                if not metricas_extraidas and titulo_metrica:
                    ws_general.Cells(1, col_gen).Value = titulo_metrica
                
                ws_general.Cells(fila_gen, col_gen).Value = valor_metrica
                col_gen += 1
                
            metricas_extraidas = True
            wb_origen.Close(SaveChanges=False)

        # Formato, anchos y grilla en la Hoja General
        max_col = ws_general.Cells(1, ws_general.Columns.Count).End(-4159).Column 
        ultima_fila_general = len(nombres_hojas) + 1 
        
        rango_titulos = ws_general.Range(ws_general.Cells(1, 1), ws_general.Cells(1, max_col))
        rango_titulos.Interior.Color = color_rgb("C00000") 
        rango_titulos.Font.Color = color_rgb("FFFFFF") 
        rango_titulos.Font.Bold = True
        rango_titulos.VerticalAlignment = -4108 
        rango_titulos.HorizontalAlignment = -4108
        
        rango_titulos.WrapText = True
        ws_general.Rows(1).AutoFit()

        if max_col >= 3:
            alineacion_centro = ws_general.Range(ws_general.Cells(2, 3), ws_general.Cells(ultima_fila_general, max_col))
            alineacion_centro.HorizontalAlignment = -4108
            alineacion_centro.VerticalAlignment = -4108

        ws_general.Columns(1).ColumnWidth = 5
        ws_general.Columns(2).ColumnWidth = 25
        if max_col >= 3:
            ws_general.Range(ws_general.Cells(1, 3), ws_general.Cells(1, max_col)).EntireColumn.ColumnWidth = 15

        rango_tabla = ws_general.Range(ws_general.Cells(1, 1), ws_general.Cells(ultima_fila_general, max_col))
        rango_tabla.Borders.LineStyle = 1 
        rango_tabla.Borders.Weight = 2 

        # Asegurar que General es la primera y activa
        ws_general.Move(Before=wb_nuevo.Sheets(1))
        ws_general.Activate()
        ws_general.Cells(1, 1).Select()

        os.makedirs(os.path.dirname(ARCHIVO_DESTINO), exist_ok=True)
        if os.path.exists(ARCHIVO_DESTINO):
            os.remove(ARCHIVO_DESTINO) 

        wb_nuevo.SaveAs(ARCHIVO_DESTINO)
        print(f"\n🚀 ¡Consolidado ordenado y completado! Archivo guardado en:\n{ARCHIVO_DESTINO}")

    except Exception as e:
        print(f"Error durante el proceso: {e}")
    finally:
        try: wb_nuevo.Close(SaveChanges=False) 
        except: pass
        excel.Quit()

if __name__ == "__main__":
    consolidar_resultados_fantasma()
#8m 52s

Iniciando Excel en segundo plano (Modo Fantasma)...
-> Procesando métricas de: 'USAQUEN.xlsx' -> Destino: '1-Usaquén'
-> Procesando métricas de: 'CHAPINERO.xlsx' -> Destino: '2-Chapinero'
-> Procesando métricas de: 'SANTAFE.xlsx' -> Destino: '3-Santafe'
-> Procesando métricas de: 'SAN CRISTOBAL.xlsx' -> Destino: '4-San Cristóbal'
-> Procesando métricas de: 'USME.xlsx' -> Destino: '5-Usme'
-> Procesando métricas de: 'TUNJUELITO.xlsx' -> Destino: '6-Tunjuelito'
-> Procesando métricas de: 'BOSA.xlsx' -> Destino: '7-Bosa'
-> Procesando métricas de: 'KENNEDY.xlsx' -> Destino: '8-Kennedy'
-> Procesando métricas de: 'FONTIBON.xlsx' -> Destino: '9-Fontibón'
-> Procesando métricas de: 'ENGATIVA.xlsx' -> Destino: '10-Engativá'
-> Procesando métricas de: 'SUBA.xlsx' -> Destino: '11-Suba'
-> Procesando métricas de: 'BARRIOS UNIDOS.xlsx' -> Destino: '12-Barrios Unidos'
-> Procesando métricas de: 'TEUSAQUILLO.xlsx' -> Destino: '13-Teusaquillo'
-> Procesando métricas de: 'MARTIRES.xlsx' -> Destino: '